<a href="https://colab.research.google.com/github/tiagoeletro-bot/Ar-Condicionado-HVAC---notebookLM-/blob/main/CHILLERS_CAG_E2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==========================================================
# CAG AUDIT FRAMEWORK
# Módulo 01 - Importação dos Dados
# Bloco 01 - Configuração do Ambiente
# ==========================================================

# Instala bibliotecas complementares

!pip install openpyxl xlsxwriter python-docx tqdm scipy -q

# ==========================================================
# IMPORTAÇÃO DAS BIBLIOTECAS
# ==========================================================

import os
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

from scipy import stats

from tqdm.notebook import tqdm

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 20)
pd.set_option("display.float_format", "{:.4f}".format)

print("Bibliotecas carregadas com sucesso.")

# ==========================================================
# IDENTIFICAÇÃO DO PROJETO
# ==========================================================

PROJETO = {
    "cliente": "Torre Olavo Setúbal",
    "sistema": "Central de Água Gelada",
    "versao": "1.0",
    "autor": "Framework CAG Audit",
    "data_execucao": datetime.now().strftime("%d/%m/%Y %H:%M:%S")
}

print("Projeto iniciado")

for chave, valor in PROJETO.items():
    print(f"{chave:20}: {valor}")

    # ==========================================================
# UPLOAD DO ARQUIVO
# ==========================================================

from google.colab import files

uploaded = files.upload()

# ==========================================================
# IDENTIFICA O ARQUIVO EXCEL
# ==========================================================

arquivo_excel = list(uploaded.keys())[0]

print("Arquivo encontrado:")
print(arquivo_excel)

# ==========================================================
# TESTE DE LEITURA
# ==========================================================

try:

    excel = pd.ExcelFile(arquivo_excel)

    print("Arquivo aberto com sucesso.")

except Exception as erro:

    print("Erro na abertura do arquivo.")

    print(erro)

    # ==========================================================
# IDENTIFICAÇÃO DAS ABAS
# ==========================================================

abas = excel.sheet_names

print(f"Foram encontradas {len(abas)} abas.\n")

for i, aba in enumerate(abas, start=1):

    print(f"{i:02d} - {aba}")

    # ==========================================================
# IMPORTAÇÃO DAS ABAS
# ==========================================================

dados = {}

for aba in tqdm(abas):

    dados[aba] = pd.read_excel(
        arquivo_excel,
        sheet_name=aba
    )

print()

print("Todas as abas foram importadas.")

# ==========================================================
# RESUMO DA IMPORTAÇÃO
# ==========================================================

resumo = []

for aba, df in dados.items():

    resumo.append({
        "Aba": aba,
        "Linhas": len(df),
        "Colunas": len(df.columns),
        "Nulos": int(df.isna().sum().sum())
    })

resumo_importacao = pd.DataFrame(resumo)

display(resumo_importacao)

# ==========================================================
# EXPORTAÇÃO DO LOG
# ==========================================================

nome_log = "00_Log_Importacao.xlsx"

with pd.ExcelWriter(nome_log, engine="xlsxwriter") as writer:

    resumo_importacao.to_excel(
        writer,
        index=False,
        sheet_name="Importacao"
    )

print(f"Log salvo: {nome_log}")



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 11.8 MB/s eta 0:00:00
Bibliotecas carregadas com sucesso.
Projeto iniciado
cliente             : Torre Olavo Setúbal
sistema             : Central de Água Gelada
versao              : 1.0
autor               : Framework CAG Audit
data_execucao       : 23/07/2026 12:22:37


Saving 01 - Levantamentos de dados CAG E2.xlsx to 01 - Levantamentos de dados CAG E2.xlsx
Arquivo encontrado:
01 - Levantamentos de dados CAG E2.xlsx
Arquivo aberto com sucesso.
Foram encontradas 9 abas.

01 - RELAÇÃO DE CONSUMO
02 - COP
03 - APPROACH EVAPORADOR
04 - APPROACH CONDENSADOR
05 - CARGA TÉRMICA (TR)
06 - CONSUMO CAG
07 - POTÊNCIA ATIVA CAG
08 - VAZÃO AG
09 - FREQUÊNCIA VSD


  0%|          | 0/9 [00:00<?, ?it/s]


Todas as abas foram importadas.


,Aba,Linhas,Colunas,Nulos
0,RELAÇÃO DE CONSUMO,4295,4,0
1,COP,4295,4,0
2,APPROACH EVAPORADOR,4295,7,0
3,APPROACH CONDENSADOR,4295,7,0
4,CARGA TÉRMICA (TR),4295,4,0
5,CONSUMO CAG,31,2,0
6,POTÊNCIA ATIVA CAG,4295,2,0
7,VAZÃO AG,4295,4,0
8,FREQUÊNCIA VSD,4288,4,0


Log salvo: 00_Log_Importacao.xlsx


In [2]:
# ==========================================================
# BLOCO 02
# FUNÇÕES AUXILIARES
# ==========================================================

import re
import numpy as np
import pandas as pd

def normalizar_nome(texto):

    texto = str(texto)

    texto = texto.strip()

    texto = texto.upper()

    texto = texto.replace("\n", " ")

    texto = re.sub(r"\s+", " ", texto)

    return texto


# ==========================================================
# DETECÇÃO ROBUSTA DA COLUNA DE DATA
# ==========================================================

def detectar_coluna_data(df):
    """
    Detecta automaticamente a coluna de data/hora.
    Primeiro tenta pelo nome da coluna.
    Se não encontrar, tenta pelo conteúdo.
    """

    candidatos = [
        "DATAHORA",
        "DATA_HORA",
        "DATA HORA",
        "DATA/HORA",
        "DATAHORA"
        "DataHora"
        "TIMESTAMP",
        "TIME",
        "DATE",
        "DATA",
        "HORARIO",
        "HORA",
        "INSTANT"
    ]

    # Procura pelo nome
    for coluna in df.columns:

        nome = (
            str(coluna)
            .strip()
            .upper()
            .replace("/", "")
            .replace("_", "")
            .replace(" ", "")
        )

        if nome in [c.replace("/", "").replace("_", "").replace(" ", "") for c in candidatos]:
            return coluna

    # Procura pelo conteúdo
    for coluna in df.columns:

        teste = pd.to_datetime(df[coluna], errors="coerce")

        if teste.notna().mean() > 0.90:
            return coluna

    return None

    # ==========================================================
# PADRONIZAÇÃO DAS COLUNAS
# ==========================================================

dados_padronizados = {}

for aba, df in dados.items():

    copia = df.copy()

    copia.columns = [
        normalizar_nome(c)
        for c in copia.columns
    ]

    dados_padronizados[aba] = copia

print("Colunas padronizadas.")

# ==========================================================
# IDENTIFICAÇÃO DA DATA
# ==========================================================

datas = {}

for aba, df in dados_padronizados.items():

    coluna_data = detectar_coluna_data(df)

    datas[aba] = coluna_data

datas

# ==========================================================
# CONVERSÃO DA DATA
# ==========================================================

for aba, df in dados_padronizados.items():

    coluna = datas[aba]

    if coluna is not None:

        df[coluna] = pd.to_datetime(
            df[coluna],
            errors="coerce"
        )

print("Datas convertidas.")

# ==========================================================
# CONVERSÃO NUMÉRICA
# ==========================================================

for aba, df in dados_padronizados.items():

    for coluna in df.columns:

        if coluna != datas[aba]:

            try:

                df[coluna] = (
                    df[coluna]
                    .astype(str)
                    .str.replace(",", ".", regex=False)
                )

                df[coluna] = pd.to_numeric(
                    df[coluna],
                    errors="ignore"
                )

            except:

                pass

print("Conversão concluída.")

# ==========================================================
# QUALIDADE DOS DADOS
# ==========================================================

qualidade = []

for aba, df in dados_padronizados.items():

    qualidade.append({

        "ABA": aba,

        "LINHAS": len(df),

        "COLUNAS": len(df.columns),

        "NULOS": int(df.isna().sum().sum()),

        "DUPLICADOS": int(df.duplicated().sum()),

        "COLUNA_DATA": datas[aba]

    })

qualidade = pd.DataFrame(qualidade)

display(qualidade)

# ==========================================================
# PERFIL DAS VARIÁVEIS
# ==========================================================

perfil = []

for aba, df in dados_padronizados.items():

    for coluna in df.columns:

        perfil.append({

            "ABA": aba,

            "VARIAVEL": coluna,

            "TIPO": str(df[coluna].dtype),

            "NULOS": int(df[coluna].isna().sum()),

            "UNICOS": df[coluna].nunique()

        })

perfil = pd.DataFrame(perfil)

display(perfil.head())

# ==========================================================
# EXPORTAÇÃO
# ==========================================================

arquivo_saida = "01_Qualidade_Dados.xlsx"

with pd.ExcelWriter(
    arquivo_saida,
    engine="xlsxwriter"
) as writer:

    qualidade.to_excel(
        writer,
        sheet_name="Resumo",
        index=False
    )

    perfil.to_excel(
        writer,
        sheet_name="Variaveis",
        index=False
    )

print("Arquivo criado:", arquivo_saida)

# ==========================================================
# ESTATÍSTICAS GERAIS
# ==========================================================

estatisticas = []

for aba, df in dados_padronizados.items():

    for coluna in df.select_dtypes(include=np.number):

        serie = df[coluna]

        estatisticas.append({

            "ABA": aba,

            "VARIAVEL": coluna,

            "MEDIA": serie.mean(),

            "MINIMO": serie.min(),

            "MAXIMO": serie.max(),

            "DESVIO": serie.std()

        })

estatisticas = pd.DataFrame(estatisticas)

display(estatisticas.head())



Colunas padronizadas.
Datas convertidas.
Conversão concluída.


,ABA,LINHAS,COLUNAS,NULOS,DUPLICADOS,COLUNA_DATA
0,RELAÇÃO DE CONSUMO,4295,4,0,0,DATAHORA
1,COP,4295,4,0,0,DATAHORA
2,APPROACH EVAPORADOR,4295,7,0,0,DATAHORA
3,APPROACH CONDENSADOR,4295,7,0,0,DATAHORA
4,CARGA TÉRMICA (TR),4295,4,0,0,DATAHORA
5,CONSUMO CAG,31,2,0,0,DATAHORA
6,POTÊNCIA ATIVA CAG,4295,2,0,0,DATAHORA
7,VAZÃO AG,4295,4,0,0,DATAHORA
8,FREQUÊNCIA VSD,4288,4,0,0,DATAHORA


,ABA,VARIAVEL,TIPO,NULOS,UNICOS
0,RELAÇÃO DE CONSUMO,DATAHORA,datetime64[ns],0,4295
1,RELAÇÃO DE CONSUMO,CEITE2PGCAG1CHAG01_VI_RCO,float64,0,735
2,RELAÇÃO DE CONSUMO,CEITE2PGCAG1CHAG02_VI_RCO,float64,0,46
3,RELAÇÃO DE CONSUMO,CEITE2PGCAG1CHAG03_VI_RCO,float64,0,1992
4,COP,DATAHORA,datetime64[ns],0,4295


Arquivo criado: 01_Qualidade_Dados.xlsx


,ABA,VARIAVEL,MEDIA,MINIMO,MAXIMO,DESVIO
0,RELAÇÃO DE CONSUMO,CEITE2PGCAG1CHAG01_VI_RCO,-16.4911,-688.5404,50.4689,110.9916
1,RELAÇÃO DE CONSUMO,CEITE2PGCAG1CHAG02_VI_RCO,-4.9438,-22.3464,51.9863,7.1978
2,RELAÇÃO DE CONSUMO,CEITE2PGCAG1CHAG03_VI_RCO,1.5606,-25.0106,70.7780,1.3406
3,COP,CEITE2PGCAG1CHAG01_IE-COP,0.4397,0.0000,3.9118,1.0453
4,COP,CEITE2PGCAG1CHAG02_IE-COP,0.0257,0.0000,3.4108,0.2806


In [3]:
datas = {}

for aba, df in dados_padronizados.items():
    coluna_data = detectar_coluna_data(df)
    datas[aba] = coluna_data

print(datas)

{'RELAÇÃO DE CONSUMO': 'DATAHORA', 'COP': 'DATAHORA', 'APPROACH EVAPORADOR': 'DATAHORA', 'APPROACH CONDENSADOR': 'DATAHORA', 'CARGA TÉRMICA (TR)': 'DATAHORA', 'CONSUMO CAG': 'DATAHORA', 'POTÊNCIA ATIVA CAG': 'DATAHORA', 'VAZÃO AG': 'DATAHORA', 'FREQUÊNCIA VSD': 'DATAHORA'}


In [4]:
# ==========================================================
# BLOCO 2.5
# FUNÇÕES DE ENGENHARIA
# ==========================================================

import re

def identificar_ura(tag):

    tag = str(tag).upper()

    if "CHAG01" in tag:
        return "URA01"

    elif "CHAG02" in tag:
        return "URA02"

    elif "CHAG03" in tag:
        return "URA03"

    return "CAG"


def identificar_circuito(tag):

    tag = str(tag).upper()

    if tag.endswith("1"):
        return "Circuito 1"

    elif tag.endswith("2"):
        return "Circuito 2"

    return "-"

    # ==========================================================
# IDENTIFICA A VARIÁVEL
# ==========================================================

def identificar_variavel(tag):

    tag = str(tag).upper()

    if "IE-COP" in tag:
        return "COP"

    elif "VI_RCO" in tag:
        return "kW/TR"

    elif "VI-CTE" in tag:
        return "Carga Térmica"

    elif "CH-WFL" in tag:
        return "Vazão"

    elif "CH-VOT" in tag:
        return "Frequência VSD"

    elif "APPEV" in tag:
        return "Approach Evaporador"

    elif "APPCN" in tag:
        return "Approach Condensador"

    elif "VI-KWH" in tag:
        return "Energia CAG"

    elif "VI-POT" in tag:
        return "Potência CAG"

    return "Outra"

    # ==========================================================
# CATEGORIA
# ==========================================================

def categoria(tag):

    v = identificar_variavel(tag)

    mapa = {

        "COP":"Eficiência",

        "kW/TR":"Eficiência",

        "Carga Térmica":"Carga",

        "Vazão":"Hidráulica",

        "Approach Evaporador":"Térmica",

        "Approach Condensador":"Térmica",

        "Frequência VSD":"Controle",

        "Energia CAG":"Energia",

        "Potência CAG":"Energia"

    }

    return mapa.get(v,"Outros")

    # ==========================================================
# UNIDADE
# ==========================================================

def unidade(tag):

    v = identificar_variavel(tag)

    unidades = {

        "COP":"adimensional",

        "kW/TR":"kW/TR",

        "Carga Térmica":"TR",

        "Vazão":"m³/h",

        "Approach Evaporador":"°C",

        "Approach Condensador":"°C",

        "Frequência VSD":"Hz",

        "Energia CAG":"kWh",

        "Potência CAG":"kW"

    }

    return unidades.get(v,"")

    # ==========================================================
# CATÁLOGO DE TAGS
# ==========================================================

lista = []

for aba, df in dados_padronizados.items():

    coluna_data = datas[aba]

    for coluna in df.columns:

        if coluna == coluna_data:
            continue

        lista.append({

            "TAG": coluna,

            "ABA": aba,

            "URA": identificar_ura(coluna),

            "VARIAVEL": identificar_variavel(coluna),

            "CATEGORIA": categoria(coluna),

            "UNIDADE": unidade(coluna),

            "CIRCUITO": identificar_circuito(coluna),

            "SISTEMA": "Central Água Gelada"

        })

dim_tag = (
    pd.DataFrame(lista)
      .drop_duplicates()
      .sort_values(["URA","VARIAVEL"])
      .reset_index(drop=True)
)

display(dim_tag.head())

# ==========================================================
# INVENTÁRIO
# ==========================================================

print()

print("Número de TAGs:",len(dim_tag))

print()

print("URA")

display(
    dim_tag.groupby("URA").size()
)

print()

print("Categorias")

display(
    dim_tag.groupby("CATEGORIA").size()
)

print()

print("Variáveis")

display(
    dim_tag.groupby("VARIAVEL").size()
)

# ==========================================================
# QUALIDADE
# ==========================================================

display(

    dim_tag.isna().sum()

)

display(

    dim_tag.describe(include="all")

)

# ==========================================================
# EXPORTAÇÃO
# ==========================================================

arquivo="02_Catalogo_TAGs.xlsx"

with pd.ExcelWriter(
    arquivo,
    engine="xlsxwriter"
) as writer:

    dim_tag.to_excel(
        writer,
        sheet_name="DIM_TAG",
        index=False
    )

print()

print("Catálogo criado.")

print(arquivo)

,TAG,ABA,URA,VARIAVEL,CATEGORIA,UNIDADE,CIRCUITO,SISTEMA
0,CEITE2PGCM01QFCO00_VI-KWH,CONSUMO CAG,CAG,Energia CAG,Energia,kWh,-,Central Água Gelada
1,CEITE2PGCM01QFCO00_VI-POT,POTÊNCIA ATIVA CAG,CAG,Potência CAG,Energia,kW,-,Central Água Gelada
2,CEITE2PGCAG1CHAG01_APPCN1,APPROACH CONDENSADOR,URA01,Approach Condensador,Térmica,°C,Circuito 1,Central Água Gelada
3,CEITE2PGCAG1CHAG01_APPCN2,APPROACH CONDENSADOR,URA01,Approach Condensador,Térmica,°C,Circuito 2,Central Água Gelada
4,CEITE2PGCAG1CHAG01_APPEV1,APPROACH EVAPORADOR,URA01,Approach Evaporador,Térmica,°C,Circuito 1,Central Água Gelada



Número de TAGs: 29

URA


,0
URA,
CAG,2
URA01,9
URA02,9
URA03,9



Categorias


,0
CATEGORIA,
Carga,3
Controle,3
Eficiência,6
Energia,2
Hidráulica,3
Térmica,12



Variáveis


,0
VARIAVEL,
Approach Condensador,6
Approach Evaporador,6
COP,3
Carga Térmica,3
Energia CAG,1
Frequência VSD,3
Potência CAG,1
Vazão,3
kW/TR,3


,0
TAG,0
ABA,0
URA,0
VARIAVEL,0
CATEGORIA,0
UNIDADE,0
CIRCUITO,0
SISTEMA,0


,TAG,ABA,URA,VARIAVEL,CATEGORIA,UNIDADE,CIRCUITO,SISTEMA
count,29,29,29,29,29,29,29,29
unique,29,9,4,9,6,8,3,1
top,CEITE2PGCM01QFCO00_VI-KWH,APPROACH CONDENSADOR,URA01,Approach Condensador,Térmica,°C,-,Central Água Gelada
freq,1,6,9,6,12,12,17,29



Catálogo criado.
02_Catalogo_TAGs.xlsx


In [5]:
# ==========================================================
# BLOCO 03
# DIM_TEMPO
# ==========================================================

# Localiza automaticamente todas as datas existentes
datas_unicas = set()

for aba, df in dados_padronizados.items():

    coluna_data = datas.get(aba)

    if coluna_data is None:
        continue

    serie = pd.to_datetime(
        df[coluna_data],
        errors="coerce"
    )

    datas_unicas.update(serie.dropna().unique())

datas_unicas = sorted(list(datas_unicas))

dim_tempo = pd.DataFrame({

    "DATA_HORA": datas_unicas

})

dim_tempo["ID_TEMPO"] = range(1, len(dim_tempo)+1)

dim_tempo["ANO"] = dim_tempo["DATA_HORA"].dt.year

dim_tempo["MES"] = dim_tempo["DATA_HORA"].dt.month

dim_tempo["DIA"] = dim_tempo["DATA_HORA"].dt.day

dim_tempo["HORA"] = dim_tempo["DATA_HORA"].dt.hour

dim_tempo["MINUTO"] = dim_tempo["DATA_HORA"].dt.minute

dim_tempo["TRIMESTRE"] = dim_tempo["DATA_HORA"].dt.quarter

dim_tempo["SEMANA"] = dim_tempo["DATA_HORA"].dt.isocalendar().week

dim_tempo["DIA_SEMANA"] = dim_tempo["DATA_HORA"].dt.day_name()

dim_tempo["FIM_SEMANA"] = dim_tempo["DIA_SEMANA"].isin(
    ["Saturday","Sunday"]
)

display(dim_tempo.head())

# ==========================================================
# DIM_EQUIPAMENTO
# ==========================================================

equipamentos = sorted(dim_tag["URA"].unique())

dim_equipamento = pd.DataFrame({

    "URA": equipamentos

})

dim_equipamento["ID_EQUIPAMENTO"] = range(
    1,
    len(dim_equipamento)+1
)

dim_equipamento["EQUIPAMENTO"] = dim_equipamento["URA"].replace({

    "URA01":"Chiller 01",

    "URA02":"Chiller 02",

    "URA03":"Chiller 03",

    "CAG":"Central Água Gelada"

})

dim_equipamento["TIPO"] = "Chiller"

dim_equipamento.loc[
    dim_equipamento["URA"]=="CAG",
    "TIPO"
] = "Sistema"

dim_equipamento["FABRICANTE"] = "A definir"

dim_equipamento["CAPACIDADE_TR"] = np.nan

display(dim_equipamento)

# ==========================================================
# DIM_VARIAVEL
# ==========================================================

dim_variavel = (

    dim_tag[

        [

            "VARIAVEL",

            "UNIDADE",

            "CATEGORIA"

        ]

    ]

    .drop_duplicates()

    .sort_values("VARIAVEL")

    .reset_index(drop=True)

)

dim_variavel["ID_VARIAVEL"] = range(
    1,
    len(dim_variavel)+1
)

dim_variavel["TIPO"] = "Processo"

dim_variavel.loc[

    dim_variavel["VARIAVEL"].isin(

        [

            "COP",

            "kW/TR"

        ]

    ),

    "TIPO"

] = "KPI"

display(dim_variavel)

# ==========================================================
# DIM_SISTEMA
# ==========================================================

dim_sistema = pd.DataFrame({

    "ID_SISTEMA":[1],

    "SISTEMA":[

        "Central Água Gelada"

    ],

    "SIGLA":[

        "CAG"

    ]

})

display(dim_sistema)

# ==========================================================
# ENRIQUECIMENTO DIM_TAG
# ==========================================================

dim_tag = dim_tag.merge(

    dim_equipamento[

        [

            "URA",

            "ID_EQUIPAMENTO"

        ]

    ],

    on="URA",

    how="left"

)

dim_tag = dim_tag.merge(

    dim_variavel[

        [

            "VARIAVEL",

            "ID_VARIAVEL"

        ]

    ],

    on="VARIAVEL",

    how="left"

)

dim_tag["ID_TAG"] = range(

    1,

    len(dim_tag)+1

)

display(dim_tag.head())

# ==========================================================
# RESUMO
# ==========================================================

print("="*60)

print("DIMENSÕES ANALÍTICAS")

print("="*60)

print()

print("DIM_TEMPO")

print(len(dim_tempo))

print()

print("DIM_TAG")

print(len(dim_tag))

print()

print("DIM_VARIAVEL")

print(len(dim_variavel))

print()

print("DIM_EQUIPAMENTO")

print(len(dim_equipamento))

print()

print("DIM_SISTEMA")

print(len(dim_sistema))

# ==========================================================
# EXPORTAÇÃO
# ==========================================================

arquivo = "03_Modelo_Dimensional.xlsx"

with pd.ExcelWriter(

    arquivo,

    engine="xlsxwriter"

) as writer:

    dim_tempo.to_excel(

        writer,

        sheet_name="DIM_TEMPO",

        index=False

    )

    dim_tag.to_excel(

        writer,

        sheet_name="DIM_TAG",

        index=False

    )

    dim_variavel.to_excel(

        writer,

        sheet_name="DIM_VARIAVEL",

        index=False

    )

    dim_equipamento.to_excel(

        writer,

        sheet_name="DIM_EQUIPAMENTO",

        index=False

    )

    dim_sistema.to_excel(

        writer,

        sheet_name="DIM_SISTEMA",

        index=False

    )

print()

print("Modelo Dimensional criado com sucesso.")

print()

print(arquivo)



,DATA_HORA,ID_TEMPO,ANO,MES,DIA,HORA,MINUTO,TRIMESTRE,SEMANA,DIA_SEMANA,FIM_SEMANA
0,2026-06-01 00:00:00,1,2026,6,1,0,0,2,23,Monday,False
1,2026-06-01 00:10:00,2,2026,6,1,0,10,2,23,Monday,False
2,2026-06-01 00:20:00,3,2026,6,1,0,20,2,23,Monday,False
3,2026-06-01 00:30:00,4,2026,6,1,0,30,2,23,Monday,False
4,2026-06-01 00:40:00,5,2026,6,1,0,40,2,23,Monday,False


,URA,ID_EQUIPAMENTO,EQUIPAMENTO,TIPO,FABRICANTE,CAPACIDADE_TR
0,CAG,1,Central Água Gelada,Sistema,A definir,NaN
1,URA01,2,Chiller 01,Chiller,A definir,NaN
2,URA02,3,Chiller 02,Chiller,A definir,NaN
3,URA03,4,Chiller 03,Chiller,A definir,NaN


,VARIAVEL,UNIDADE,CATEGORIA,ID_VARIAVEL,TIPO
0,Approach Condensador,°C,Térmica,1,Processo
1,Approach Evaporador,°C,Térmica,2,Processo
2,COP,adimensional,Eficiência,3,KPI
3,Carga Térmica,TR,Carga,4,Processo
4,Energia CAG,kWh,Energia,5,Processo
5,Frequência VSD,Hz,Controle,6,Processo
6,Potência CAG,kW,Energia,7,Processo
7,Vazão,m³/h,Hidráulica,8,Processo
8,kW/TR,kW/TR,Eficiência,9,KPI


,ID_SISTEMA,SISTEMA,SIGLA
0,1,Central Água Gelada,CAG


,TAG,ABA,URA,VARIAVEL,CATEGORIA,UNIDADE,CIRCUITO,SISTEMA,ID_EQUIPAMENTO,ID_VARIAVEL,ID_TAG
0,CEITE2PGCM01QFCO00_VI-KWH,CONSUMO CAG,CAG,Energia CAG,Energia,kWh,-,Central Água Gelada,1,5,1
1,CEITE2PGCM01QFCO00_VI-POT,POTÊNCIA ATIVA CAG,CAG,Potência CAG,Energia,kW,-,Central Água Gelada,1,7,2
2,CEITE2PGCAG1CHAG01_APPCN1,APPROACH CONDENSADOR,URA01,Approach Condensador,Térmica,°C,Circuito 1,Central Água Gelada,2,1,3
3,CEITE2PGCAG1CHAG01_APPCN2,APPROACH CONDENSADOR,URA01,Approach Condensador,Térmica,°C,Circuito 2,Central Água Gelada,2,1,4
4,CEITE2PGCAG1CHAG01_APPEV1,APPROACH EVAPORADOR,URA01,Approach Evaporador,Térmica,°C,Circuito 1,Central Água Gelada,2,2,5


DIMENSÕES ANALÍTICAS

DIM_TEMPO
4295

DIM_TAG
29

DIM_VARIAVEL
9

DIM_EQUIPAMENTO
4

DIM_SISTEMA
1

Modelo Dimensional criado com sucesso.

03_Modelo_Dimensional.xlsx


In [6]:
# ==========================================================
# BLOCO 4.1
# CONSTRUÇÃO DA FACT_MEDICOES
# ETAPA 01
# ==========================================================

import pandas as pd
import numpy as np

print("="*70)
print("BLOCO 4.1 - CONSTRUÇÃO DA FACT_MEDICOES")
print("="*70)

fact_medicoes = []

# ==========================================================
# ETAPA 02
# LEITURA DAS ABAS
# ==========================================================

for aba, df in dados_padronizados.items():

    coluna_data = datas.get(aba)

    if coluna_data is None:

        print(f"Aba ignorada: {aba}")

        continue

    tags = [c for c in df.columns if c != coluna_data]

    print(f"{aba:30} -> {len(tags)} TAGs")

    # ==========================================================
# ETAPA 03
# LONG FORMAT
# ==========================================================

fact_medicoes = []

for aba, df in dados_padronizados.items():

    coluna_data = datas.get(aba)

    if coluna_data is None:
        continue

    tags = [c for c in df.columns if c != coluna_data]

    for tag in tags:

        temp = pd.DataFrame({

            "DATA_HORA": df[coluna_data],

            "TAG": tag,

            "VALOR": df[tag],

            "ABA_ORIGEM": aba

        })

        fact_medicoes.append(temp)

# ==========================================================
# ETAPA 04
# CONCATENAÇÃO
# ==========================================================

fact_medicoes = pd.concat(
    fact_medicoes,
    ignore_index=True
)

print()

print("Número de registros:")

print(f"{len(fact_medicoes):,}")

# ==========================================================
# ETAPA 05
# LIMPEZA
# ==========================================================

fact_medicoes["DATA_HORA"] = pd.to_datetime(

    fact_medicoes["DATA_HORA"],

    errors="coerce"

)

fact_medicoes["VALOR"] = pd.to_numeric(

    fact_medicoes["VALOR"],

    errors="coerce"

)

fact_medicoes = fact_medicoes.dropna(

    subset=["DATA_HORA","VALOR"]

)

fact_medicoes = fact_medicoes.reset_index(drop=True)

print()

print("Registros válidos")

print(len(fact_medicoes))

# ==========================================================
# ETAPA 06
# MERGE COM DIM_TAG
# ==========================================================

fact_medicoes = fact_medicoes.merge(

    dim_tag[

        [

            "TAG",

            "URA",

            "VARIAVEL",

            "UNIDADE",

            "CATEGORIA",

            "CIRCUITO",

            "ID_TAG",

            "ID_EQUIPAMENTO",

            "ID_VARIAVEL"

        ]

    ],

    on="TAG",

    how="left"

)

# ==========================================================
# ETAPA 07
# MERGE COM DIM_TEMPO
# ==========================================================

fact_medicoes = fact_medicoes.merge(

    dim_tempo[

        [

            "DATA_HORA",

            "ID_TEMPO"

        ]

    ],

    on="DATA_HORA",

    how="left"

)

# ==========================================================
# ETAPA 08
# CHAVE PRIMÁRIA
# ==========================================================

fact_medicoes.insert(

    0,

    "ID_MEDICAO",

    range(

        1,

        len(fact_medicoes)+1

    )

)

# ==========================================================
# ETAPA 09
# ORGANIZAÇÃO
# ==========================================================

ordem = [

    "ID_MEDICAO",

    "ID_TEMPO",

    "ID_TAG",

    "ID_VARIAVEL",

    "ID_EQUIPAMENTO",

    "DATA_HORA",

    "URA",

    "TAG",

    "VARIAVEL",

    "UNIDADE",

    "CATEGORIA",

    "CIRCUITO",

    "VALOR",

    "ABA_ORIGEM"

]

fact_medicoes = fact_medicoes[ordem]

# ==========================================================
# ETAPA 10
# RESULTADO
# ==========================================================

display(

    fact_medicoes.head()

)

print()

print("="*60)

print("FACT_MEDICOES")

print("="*60)

print()

print("Registros :",len(fact_medicoes))

print("TAGs      :",fact_medicoes.TAG.nunique())

print("URA       :",fact_medicoes.URA.unique())

print("Variáveis :",fact_medicoes.VARIAVEL.unique())

# ==========================================================
# EXPORTAÇÃO
# ==========================================================

arquivo = "04_FACT_MEDICOES_BASE.xlsx"

with pd.ExcelWriter(

    arquivo,

    engine="xlsxwriter"

) as writer:

    fact_medicoes.to_excel(

        writer,

        sheet_name="FACT_MEDICOES",

        index=False

    )

print()

print("Arquivo criado.")

print(arquivo)





BLOCO 4.1 - CONSTRUÇÃO DA FACT_MEDICOES
RELAÇÃO DE CONSUMO             -> 3 TAGs
COP                            -> 3 TAGs
APPROACH EVAPORADOR            -> 6 TAGs
APPROACH CONDENSADOR           -> 6 TAGs
CARGA TÉRMICA (TR)             -> 3 TAGs
CONSUMO CAG                    -> 1 TAGs
POTÊNCIA ATIVA CAG             -> 1 TAGs
VAZÃO AG                       -> 3 TAGs
FREQUÊNCIA VSD                 -> 3 TAGs

Número de registros:
120,270

Registros válidos
120270


,ID_MEDICAO,ID_TEMPO,ID_TAG,ID_VARIAVEL,ID_EQUIPAMENTO,DATA_HORA,URA,TAG,VARIAVEL,UNIDADE,CATEGORIA,CIRCUITO,VALOR,ABA_ORIGEM
0,1,1,11,9,2,2026-06-01 00:00:00,URA01,CEITE2PGCAG1CHAG01_VI_RCO,kW/TR,kW/TR,Eficiência,-,1.7932,RELAÇÃO DE CONSUMO
1,2,2,11,9,2,2026-06-01 00:10:00,URA01,CEITE2PGCAG1CHAG01_VI_RCO,kW/TR,kW/TR,Eficiência,-,1.7932,RELAÇÃO DE CONSUMO
2,3,3,11,9,2,2026-06-01 00:20:00,URA01,CEITE2PGCAG1CHAG01_VI_RCO,kW/TR,kW/TR,Eficiência,-,1.7932,RELAÇÃO DE CONSUMO
3,4,4,11,9,2,2026-06-01 00:30:00,URA01,CEITE2PGCAG1CHAG01_VI_RCO,kW/TR,kW/TR,Eficiência,-,1.7932,RELAÇÃO DE CONSUMO
4,5,5,11,9,2,2026-06-01 00:40:00,URA01,CEITE2PGCAG1CHAG01_VI_RCO,kW/TR,kW/TR,Eficiência,-,1.7932,RELAÇÃO DE CONSUMO



FACT_MEDICOES

Registros : 120270
TAGs      : 29
URA       : ['URA01' 'URA02' 'URA03' 'CAG']
Variáveis : ['kW/TR' 'COP' 'Approach Evaporador' 'Approach Condensador'
 'Carga Térmica' 'Energia CAG' 'Potência CAG' 'Vazão' 'Frequência VSD']

Arquivo criado.
04_FACT_MEDICOES_BASE.xlsx


In [7]:
# Backup em formatos eficientes
fact_medicoes.to_parquet("04_FACT_MEDICOES.parquet", index=False)
fact_medicoes.to_pickle("04_FACT_MEDICOES.pkl")

In [8]:
# ==========================================================
# BLOCO 4.2.1
# INTEGRIDADE REFERENCIAL
# ==========================================================

print("="*70)
print("BLOCO 4.2.1 - VALIDAÇÃO DA FACT_MEDICOES")
print("="*70)

fact_analitica = fact_medicoes.copy()

# ==========================================================
# VERIFICAÇÃO DAS CHAVES
# ==========================================================

print()

print("Valores nulos nas chaves:")

display(

fact_analitica[
[
"ID_TEMPO",
"ID_TAG",
"ID_VARIAVEL",
"ID_EQUIPAMENTO"
]
].isna().sum()

)

# ==========================================================
# STATUS DOS REGISTROS
# ==========================================================

fact_analitica["STATUS_REGISTRO"] = "OK"

fact_analitica.loc[
fact_analitica["ID_TAG"].isna(),
"STATUS_REGISTRO"
] = "TAG NÃO CADASTRADA"

fact_analitica.loc[
fact_analitica["ID_TEMPO"].isna(),
"STATUS_REGISTRO"
] = "DATA INVÁLIDA"

fact_analitica.loc[
fact_analitica["VALOR"].isna(),
"STATUS_REGISTRO"
] = "VALOR NULO"

# ==========================================================
# QUALIDADE
# ==========================================================

fact_analitica["REGISTRO_VALIDO"] = (

fact_analitica["STATUS_REGISTRO"]=="OK"

)

# ==========================================================
# KPIs
# ==========================================================

total = len(fact_analitica)

validos = fact_analitica["REGISTRO_VALIDO"].sum()

integridade = validos/total*100

print()

print(f"Registros : {total:,}")

print(f"Válidos   : {validos:,}")

print(f"Integridade : {integridade:.2f}%")

# ==========================================================
# QUALIDADE POR URA
# ==========================================================

relatorio = (

fact_analitica

.groupby("URA")

.agg(

Registros=("ID_MEDICAO","count"),

Validos=("REGISTRO_VALIDO","sum")

)

)

relatorio["Integridade (%)"] = (

100*

relatorio["Validos"]

/

relatorio["Registros"]

)

display(relatorio)

# ==========================================================
# VARIÁVEIS
# ==========================================================

display(

fact_analitica

.groupby(

["VARIAVEL","STATUS_REGISTRO"]

)

.size()

.unstack(fill_value=0)

)

# ==========================================================
# EXPORTAÇÃO
# ==========================================================

arquivo="04_FACT_ANALITICA_V1.xlsx"

with pd.ExcelWriter(
arquivo,
engine="xlsxwriter"
) as writer:

    fact_analitica.to_excel(
        writer,
        sheet_name="FACT_ANALITICA",
        index=False
    )

    relatorio.to_excel(
        writer,
        sheet_name="QUALIDADE"
    )

print()

print("Arquivo criado.")

print(arquivo)



BLOCO 4.2.1 - VALIDAÇÃO DA FACT_MEDICOES

Valores nulos nas chaves:


,0
ID_TEMPO,0
ID_TAG,0
ID_VARIAVEL,0
ID_EQUIPAMENTO,0



Registros : 120,270
Válidos   : 120,270
Integridade : 100.00%


,Registros,Validos,Integridade (%)
URA,,,
CAG,4326,4326,100.0000
URA01,38648,38648,100.0000
URA02,38648,38648,100.0000
URA03,38648,38648,100.0000


STATUS_REGISTRO,OK
VARIAVEL,
Approach Condensador,25770
Approach Evaporador,25770
COP,12885
Carga Térmica,12885
Energia CAG,31
Frequência VSD,12864
Potência CAG,4295
Vazão,12885
kW/TR,12885



Arquivo criado.
04_FACT_ANALITICA_V1.xlsx


In [9]:
# ==========================================================
# BLOCO 4.2.2
# ENGENHARIA DA QUALIDADE DOS DADOS
# ==========================================================

print("="*70)
print("BLOCO 4.2.2 - ENGENHARIA DA QUALIDADE")
print("="*70)

fact_analitica_v2 = fact_analitica.copy()

# ==========================================================
# COMPONENTES TEMPORAIS
# ==========================================================

fact_analitica_v2["ANO"] = fact_analitica_v2["DATA_HORA"].dt.year

fact_analitica_v2["MES"] = fact_analitica_v2["DATA_HORA"].dt.month

fact_analitica_v2["DIA"] = fact_analitica_v2["DATA_HORA"].dt.day

fact_analitica_v2["HORA"] = fact_analitica_v2["DATA_HORA"].dt.hour

fact_analitica_v2["MINUTO"] = fact_analitica_v2["DATA_HORA"].dt.minute

fact_analitica_v2["TRIMESTRE"] = fact_analitica_v2["DATA_HORA"].dt.quarter

fact_analitica_v2["DIA_SEMANA"] = fact_analitica_v2["DATA_HORA"].dt.day_name()

fact_analitica_v2["MES_NOME"] = fact_analitica_v2["DATA_HORA"].dt.month_name()

# ==========================================================
# TURNO
# ==========================================================

def turno(hora):

    if 6 <= hora < 12:
        return "Manhã"

    elif 12 <= hora < 18:
        return "Tarde"

    elif 18 <= hora < 24:
        return "Noite"

    return "Madrugada"


fact_analitica_v2["TURNO"] = (
    fact_analitica_v2["HORA"]
    .apply(turno)
)

# ==========================================================
# FAIXA DE CARGA
# ==========================================================

fact_analitica_v2["FAIXA_CARGA"] = np.nan

mask = fact_analitica_v2["VARIAVEL"] == "Carga Térmica"

fact_analitica_v2.loc[
    mask & (fact_analitica_v2["VALOR"] < 200),
    "FAIXA_CARGA"
] = "Baixa"

fact_analitica_v2.loc[
    mask & (fact_analitica_v2["VALOR"] >= 200) &
    (fact_analitica_v2["VALOR"] < 400),
    "FAIXA_CARGA"
] = "Média"

fact_analitica_v2.loc[
    mask & (fact_analitica_v2["VALOR"] >= 400),
    "FAIXA_CARGA"
] = "Alta"

# ==========================================================
# COP
# ==========================================================

fact_analitica_v2["STATUS_COP"] = np.nan

mask = fact_analitica_v2["VARIAVEL"] == "COP"

fact_analitica_v2.loc[
    mask & (fact_analitica_v2["VALOR"] >= 6.0),
    "STATUS_COP"
] = "Excelente"

fact_analitica_v2.loc[
    mask &
    (fact_analitica_v2["VALOR"] >= 5.5) &
    (fact_analitica_v2["VALOR"] < 6.0),
    "STATUS_COP"
] = "Bom"

fact_analitica_v2.loc[
    mask &
    (fact_analitica_v2["VALOR"] >= 5.0) &
    (fact_analitica_v2["VALOR"] < 5.5),
    "STATUS_COP"
] = "Regular"

fact_analitica_v2.loc[
    mask &
    (fact_analitica_v2["VALOR"] < 5.0),
    "STATUS_COP"
] = "Crítico"

# ==========================================================
# KW/TR
# ==========================================================

fact_analitica_v2["STATUS_KWTR"] = np.nan

mask = fact_analitica_v2["VARIAVEL"] == "kW/TR"

fact_analitica_v2.loc[
    mask & (fact_analitica_v2["VALOR"] <= 0.55),
    "STATUS_KWTR"
] = "Excelente"

fact_analitica_v2.loc[
    mask &
    (fact_analitica_v2["VALOR"] > 0.55) &
    (fact_analitica_v2["VALOR"] <= 0.65),
    "STATUS_KWTR"
] = "Bom"

fact_analitica_v2.loc[
    mask &
    (fact_analitica_v2["VALOR"] > 0.65) &
    (fact_analitica_v2["VALOR"] <= 0.75),
    "STATUS_KWTR"
] = "Regular"

fact_analitica_v2.loc[
    mask &
    (fact_analitica_v2["VALOR"] > 0.75),
    "STATUS_KWTR"
] = "Crítico"

# ==========================================================
# APPROACH
# ==========================================================

fact_analitica_v2["STATUS_APPROACH"] = np.nan

mask = fact_analitica_v2["VARIAVEL"].str.contains("Approach", na=False)

fact_analitica_v2.loc[
    mask &
    (fact_analitica_v2["VALOR"] <= 2),
    "STATUS_APPROACH"
] = "Excelente"

fact_analitica_v2.loc[
    mask &
    (fact_analitica_v2["VALOR"] > 2) &
    (fact_analitica_v2["VALOR"] <= 4),
    "STATUS_APPROACH"
] = "Bom"

fact_analitica_v2.loc[
    mask &
    (fact_analitica_v2["VALOR"] > 4),
    "STATUS_APPROACH"
] = "Crítico"

# ==========================================================
# RASTREABILIDADE
# ==========================================================

from datetime import datetime

fact_analitica_v2["DATA_PROCESSAMENTO"] = datetime.now()

fact_analitica_v2["VERSAO_ETL"] = "2.2"

fact_analitica_v2["LOTE"] = 1

# ==========================================================
# RESUMO
# ==========================================================

print()

print("Total de Registros")

print(len(fact_analitica_v2))

print()

print("Status COP")

display(

fact_analitica_v2["STATUS_COP"]

.value_counts(dropna=False)

)

print()

print("Status kW/TR")

display(

fact_analitica_v2["STATUS_KWTR"]

.value_counts(dropna=False)

)

print()

print("Status Approach")

display(

fact_analitica_v2["STATUS_APPROACH"]

.value_counts(dropna=False)

)

# ==========================================================
# EXPORTAÇÃO
# ==========================================================

arquivo = "04_FACT_ANALITICA_V2.xlsx"

with pd.ExcelWriter(
    arquivo,
    engine="xlsxwriter"
) as writer:

    fact_analitica_v2.to_excel(
        writer,
        sheet_name="FACT_ANALITICA",
        index=False
    )

print()

print("Arquivo exportado.")

print(arquivo)



BLOCO 4.2.2 - ENGENHARIA DA QUALIDADE

Total de Registros
120270

Status COP


,count
STATUS_COP,
NaN,107385
Crítico,12885



Status kW/TR


,count
STATUS_KWTR,
NaN,107385
Crítico,7995
Excelente,3207
Bom,1440
Regular,243



Status Approach


,count
STATUS_APPROACH,
NaN,68730
Excelente,45179
Bom,4489
Crítico,1872



Arquivo exportado.
04_FACT_ANALITICA_V2.xlsx


In [10]:
# ==========================================================
# BLOCO 4.2.3 V2
# MOTOR ESTATÍSTICO DE ENGENHARIA
# ==========================================================

import numpy as np
import pandas as pd

print("="*80)
print("BLOCO 4.2.3 V2 - MOTOR ESTATÍSTICO DE ENGENHARIA")
print("="*80)

fact_estatistica = fact_analitica_v2.copy()

# ==========================================================
# ESTADO OPERACIONAL
# ==========================================================

fact_estatistica["EM_OPERACAO"] = True

variaveis_operacionais = [

    "COP",

    "kW/TR",

    "Approach Evaporador",

    "Approach Condensador",

    "Carga Térmica",

    "Vazão"

]

mask = (

    fact_estatistica["VARIAVEL"].isin(variaveis_operacionais)

) & (

    fact_estatistica["VALOR"] <= 0

)

fact_estatistica.loc[

    mask,

    "EM_OPERACAO"

] = False

print()

print("Equipamentos em operação:")

display(

fact_estatistica["EM_OPERACAO"]

.value_counts()

)

# ==========================================================
# BASE ESTATÍSTICA
# ==========================================================

base = (

fact_estatistica

.loc[

fact_estatistica["EM_OPERACAO"]

]

.copy()

)

print()

print("Registros válidos:")

print(len(base))

# ==========================================================
# ESTATÍSTICA DESCRITIVA
# ==========================================================

estatistica = (

base

.groupby(

["URA","VARIAVEL"]

)

["VALOR"]

.agg(

Quantidade="count",

Media="mean",

Mediana="median",

Desvio="std",

Variancia="var",

Minimo="min",

Maximo="max"

)

.reset_index()

)

estatistica["CV (%)"] = (

100*

estatistica["Desvio"]

/

estatistica["Media"]

)

display(estatistica.head())

# ==========================================================
# QUARTIS
# ==========================================================

q = (

base

.groupby(

["URA","VARIAVEL"]

)

["VALOR"]

.quantile(

[0.05,0.25,0.50,0.75,0.95]

)

.unstack()

)

q.columns=[

"P5",

"Q1",

"Q2",

"Q3",

"P95"

]

q=q.reset_index()

estatistica=estatistica.merge(

q,

on=[

"URA",

"VARIAVEL"

]

)

estatistica["IQR"]=estatistica["Q3"]-estatistica["Q1"]

# ==========================================================
# LIMITES
# ==========================================================

estatistica["LIM_INF"] = (

estatistica["Q1"]

-

1.5*

estatistica["IQR"]

)

estatistica["LIM_SUP"] = (

estatistica["Q3"]

+

1.5*

estatistica["IQR"]

)

# ==========================================================
# MERGE
# ==========================================================

base = base.merge(

estatistica,

on=[

"URA",

"VARIAVEL"

],

how="left"

)

# ==========================================================
# Z SCORE
# ==========================================================

base["Z_SCORE"] = np.where(

base["Desvio"] > 0,

(base["VALOR"] - base["Media"]) / base["Desvio"],

np.nan

)

# ==========================================================
# OUTLIERS
# ==========================================================

base["OUTLIER"] = (

(base["VALOR"] < base["LIM_INF"])

|

(base["VALOR"] > base["LIM_SUP"])

)

# ==========================================================
# BENCHMARK
# ==========================================================

benchmark = (

estatistica

.pivot_table(

index="VARIAVEL",

columns="URA",

values="Media"

)

)

display(benchmark)

# ==========================================================
# EXPORTAÇÃO
# ==========================================================

arquivo = "05_Motor_Estatistico_Engenharia.xlsx"

with pd.ExcelWriter(
    arquivo,
    engine="xlsxwriter"
) as writer:

    estatistica.to_excel(
        writer,
        sheet_name="ESTATISTICA",
        index=False
    )

    benchmark.to_excel(
        writer,
        sheet_name="BENCHMARK"
    )

    base.to_excel(
        writer,
        sheet_name="BASE_ESTATISTICA",
        index=False
    )

print()

print("Motor Estatístico criado.")

print()

print(arquivo)



BLOCO 4.2.3 V2 - MOTOR ESTATÍSTICO DE ENGENHARIA

Equipamentos em operação:


,count
EM_OPERACAO,
False,65349
True,54921



Registros válidos:
54921


,URA,VARIAVEL,Quantidade,Media,Mediana,Desvio,Variancia,Minimo,Maximo,CV (%)
0,CAG,Energia CAG,31,1804.1020,2043.3500,642.5385,412855.6981,590.8742,2537.1310,35.6154
1,CAG,Potência CAG,4295,76.6065,86.0000,62.2876,3879.7394,2.0000,264.0000,81.3084
2,URA01,Approach Condensador,1276,4.4296,4.3887,1.7415,3.0329,0.1907,12.1421,39.3152
3,URA01,Approach Evaporador,1376,3.3817,2.7799,2.0100,4.0402,0.2748,11.8623,59.4374
4,URA01,COP,700,2.6980,3.0065,0.7812,0.6103,0.0683,3.9118,28.9567


URA,CAG,URA01,URA02,URA03
VARIAVEL,,,,
Approach Condensador,NaN,4.4296,3.8440,2.9455
Approach Evaporador,NaN,3.3817,6.4196,2.0584
COP,NaN,2.6980,2.8307,2.6812
Carga Térmica,NaN,107.1355,116.2400,95.4530
Energia CAG,1804.1020,NaN,NaN,NaN
Frequência VSD,NaN,15.0250,1.6381,36.0728
Potência CAG,76.6065,NaN,NaN,NaN
Vazão,NaN,19.3521,10.1935,60.3767
kW/TR,NaN,1.8549,0.7260,1.5789



Motor Estatístico criado.

05_Motor_Estatistico_Engenharia.xlsx


In [11]:
# ==========================================================
# BLOCO 4.2.4
# ENGINEERING RULES ENGINE
# ==========================================================

print("="*80)
print("BLOCO 4.2.4 - ENGINEERING RULES ENGINE")
print("="*80)

fact_engineering = base.copy()

# ==========================================================
# TABELA DE REGRAS
# ==========================================================

regras = pd.DataFrame({

    "VARIAVEL":[

        "COP",

        "kW/TR",

        "Approach Evaporador",

        "Approach Condensador",

        "Carga Térmica",

        "Vazão"

    ],

    "TIPO":[

        "MAIOR_MELHOR",

        "MENOR_MELHOR",

        "MENOR_MELHOR",

        "MENOR_MELHOR",

        "MAIOR_MELHOR",

        "MAIOR_MELHOR"

    ],

    "LIMITE_BOM":[

        6.0,

        0.55,

        2.0,

        2.0,

        np.nan,

        np.nan

    ],

    "LIMITE_REGULAR":[

        5.5,

        0.65,

        4.0,

        4.0,

        np.nan,

        np.nan

    ],

    "PESO":[

        10,

        10,

        8,

        8,

        7,

        7

    ],

    "CRITICIDADE":[

        "Alta",

        "Alta",

        "Média",

        "Média",

        "Média",

        "Baixa"

    ]

})

display(regras)

# ==========================================================
# MERGE
# ==========================================================

fact_engineering = fact_engineering.merge(

    regras,

    on="VARIAVEL",

    how="left"

)

# ==========================================================
# SCORE DE ENGENHARIA
# ==========================================================

fact_engineering["CLASSE_ENGENHARIA"] = "Sem Regra"

# COP
mask = fact_engineering["VARIAVEL"] == "COP"

fact_engineering.loc[
    mask & (fact_engineering["VALOR"] >= 6),
    "CLASSE_ENGENHARIA"
] = "Excelente"

fact_engineering.loc[
    mask &
    (fact_engineering["VALOR"] >= 5.5) &
    (fact_engineering["VALOR"] < 6),
    "CLASSE_ENGENHARIA"
] = "Bom"

fact_engineering.loc[
    mask &
    (fact_engineering["VALOR"] < 5.5),
    "CLASSE_ENGENHARIA"
] = "Crítico"

# kW/TR
mask = fact_engineering["VARIAVEL"] == "kW/TR"

fact_engineering.loc[
    mask & (fact_engineering["VALOR"] <= 0.55),
    "CLASSE_ENGENHARIA"
] = "Excelente"

fact_engineering.loc[
    mask &
    (fact_engineering["VALOR"] > 0.55) &
    (fact_engineering["VALOR"] <= 0.65),
    "CLASSE_ENGENHARIA"
] = "Bom"

fact_engineering.loc[
    mask &
    (fact_engineering["VALOR"] > 0.65),
    "CLASSE_ENGENHARIA"
] = "Crítico"

# ==========================================================
# SCORE
# ==========================================================

pontuacao = {

    "Excelente":100,

    "Bom":85,

    "Regular":70,

    "Crítico":40,

    "Sem Regra":np.nan

}

fact_engineering["NOTA_ENGENHARIA"] = (

fact_engineering["CLASSE_ENGENHARIA"]

.map(pontuacao)

)

# ==========================================================
# PRIORIDADE
# ==========================================================

fact_engineering["PRIORIDADE"] = "Normal"

fact_engineering.loc[

(fact_engineering["CRITICIDADE"]=="Alta")

&

(fact_engineering["CLASSE_ENGENHARIA"]=="Crítico"),

"PRIORIDADE"

]="Intervenção Imediata"

fact_engineering.loc[

(fact_engineering["CRITICIDADE"]=="Média")

&

(fact_engineering["CLASSE_ENGENHARIA"]=="Crítico"),

"PRIORIDADE"

]="Programar Manutenção"

# ==========================================================
# RESUMO
# ==========================================================

resumo = (

fact_engineering

.groupby(

["URA","CLASSE_ENGENHARIA"]

)

.size()

.unstack(fill_value=0)

)

display(resumo)

# ==========================================================
# EXPORTAÇÃO
# ==========================================================

arquivo = "06_ENGINEERING_RULES_ENGINE.xlsx"

with pd.ExcelWriter(
    arquivo,
    engine="xlsxwriter"
) as writer:

    regras.to_excel(
        writer,
        sheet_name="REGRAS",
        index=False
    )

    fact_engineering.to_excel(
        writer,
        sheet_name="FACT_ENGINEERING",
        index=False
    )

    resumo.to_excel(
        writer,
        sheet_name="RESUMO"
    )

print()
print("Engineering Rules Engine criado com sucesso.")
print(arquivo)



BLOCO 4.2.4 - ENGINEERING RULES ENGINE


,VARIAVEL,TIPO,LIMITE_BOM,LIMITE_REGULAR,PESO,CRITICIDADE
0,COP,MAIOR_MELHOR,6.0000,5.5000,10,Alta
1,kW/TR,MENOR_MELHOR,0.5500,0.6500,10,Alta
2,Approach Evaporador,MENOR_MELHOR,2.0000,4.0000,8,Média
3,Approach Condensador,MENOR_MELHOR,2.0000,4.0000,8,Média
4,Carga Térmica,MAIOR_MELHOR,NaN,NaN,7,Média
5,Vazão,MAIOR_MELHOR,NaN,NaN,7,Baixa


CLASSE_ENGENHARIA,Bom,Crítico,Sem Regra
URA,,,
CAG,0,0,4326
URA01,168,4701,11934
URA02,1207,80,8710
URA03,65,6108,17622



Engineering Rules Engine criado com sucesso.
06_ENGINEERING_RULES_ENGINE.xlsx


In [12]:
print("="*80)
print("COLUNAS DA FACT_ENGINEERING")
print("="*80)

print(fact_engineering.columns.tolist())

COLUNAS DA FACT_ENGINEERING
['ID_MEDICAO', 'ID_TEMPO', 'ID_TAG', 'ID_VARIAVEL', 'ID_EQUIPAMENTO', 'DATA_HORA', 'URA', 'TAG', 'VARIAVEL', 'UNIDADE', 'CATEGORIA', 'CIRCUITO', 'VALOR', 'ABA_ORIGEM', 'STATUS_REGISTRO', 'REGISTRO_VALIDO', 'ANO', 'MES', 'DIA', 'HORA', 'MINUTO', 'TRIMESTRE', 'DIA_SEMANA', 'MES_NOME', 'TURNO', 'FAIXA_CARGA', 'STATUS_COP', 'STATUS_KWTR', 'STATUS_APPROACH', 'DATA_PROCESSAMENTO', 'VERSAO_ETL', 'LOTE', 'EM_OPERACAO', 'Quantidade', 'Media', 'Mediana', 'Desvio', 'Variancia', 'Minimo', 'Maximo', 'CV (%)', 'P5', 'Q1', 'Q2', 'Q3', 'P95', 'IQR', 'LIM_INF', 'LIM_SUP', 'Z_SCORE', 'OUTLIER', 'TIPO', 'LIMITE_BOM', 'LIMITE_REGULAR', 'PESO', 'CRITICIDADE', 'CLASSE_ENGENHARIA', 'NOTA_ENGENHARIA', 'PRIORIDADE']


In [13]:
fact_engineering.head()

,ID_MEDICAO,ID_TEMPO,ID_TAG,ID_VARIAVEL,ID_EQUIPAMENTO,DATA_HORA,URA,TAG,VARIAVEL,UNIDADE,CATEGORIA,CIRCUITO,VALOR,ABA_ORIGEM,STATUS_REGISTRO,REGISTRO_VALIDO,ANO,MES,DIA,HORA,MINUTO,TRIMESTRE,DIA_SEMANA,MES_NOME,TURNO,FAIXA_CARGA,STATUS_COP,STATUS_KWTR,STATUS_APPROACH,DATA_PROCESSAMENTO,VERSAO_ETL,LOTE,EM_OPERACAO,Quantidade,Media,Mediana,Desvio,Variancia,Minimo,Maximo,CV (%),P5,Q1,Q2,Q3,P95,IQR,LIM_INF,LIM_SUP,Z_SCORE,OUTLIER,TIPO,LIMITE_BOM,LIMITE_REGULAR,PESO,CRITICIDADE,CLASSE_ENGENHARIA,NOTA_ENGENHARIA,PRIORIDADE
0,1,1,11,9,2,2026-06-01 00:00:00,URA01,CEITE2PGCAG1CHAG01_VI_RCO,kW/TR,kW/TR,Eficiência,-,1.7932,RELAÇÃO DE CONSUMO,OK,True,2026,6,1,0,0,2,Monday,June,Madrugada,NaN,NaN,Crítico,NaN,2026-07-23 12:23:38.183885,2.2,1,True,4169,1.8549,1.7932,1.1642,1.3553,0.5917,50.4689,62.7600,0.7407,1.0752,1.7932,2.3288,3.1343,1.2536,-0.8052,4.2091,-0.0531,False,MENOR_MELHOR,0.5500,0.6500,10.0000,Alta,Crítico,40.0000,Intervenção Imediata
1,2,2,11,9,2,2026-06-01 00:10:00,URA01,CEITE2PGCAG1CHAG01_VI_RCO,kW/TR,kW/TR,Eficiência,-,1.7932,RELAÇÃO DE CONSUMO,OK,True,2026,6,1,0,10,2,Monday,June,Madrugada,NaN,NaN,Crítico,NaN,2026-07-23 12:23:38.183885,2.2,1,True,4169,1.8549,1.7932,1.1642,1.3553,0.5917,50.4689,62.7600,0.7407,1.0752,1.7932,2.3288,3.1343,1.2536,-0.8052,4.2091,-0.0531,False,MENOR_MELHOR,0.5500,0.6500,10.0000,Alta,Crítico,40.0000,Intervenção Imediata
2,3,3,11,9,2,2026-06-01 00:20:00,URA01,CEITE2PGCAG1CHAG01_VI_RCO,kW/TR,kW/TR,Eficiência,-,1.7932,RELAÇÃO DE CONSUMO,OK,True,2026,6,1,0,20,2,Monday,June,Madrugada,NaN,NaN,Crítico,NaN,2026-07-23 12:23:38.183885,2.2,1,True,4169,1.8549,1.7932,1.1642,1.3553,0.5917,50.4689,62.7600,0.7407,1.0752,1.7932,2.3288,3.1343,1.2536,-0.8052,4.2091,-0.0531,False,MENOR_MELHOR,0.5500,0.6500,10.0000,Alta,Crítico,40.0000,Intervenção Imediata
3,4,4,11,9,2,2026-06-01 00:30:00,URA01,CEITE2PGCAG1CHAG01_VI_RCO,kW/TR,kW/TR,Eficiência,-,1.7932,RELAÇÃO DE CONSUMO,OK,True,2026,6,1,0,30,2,Monday,June,Madrugada,NaN,NaN,Crítico,NaN,2026-07-23 12:23:38.183885,2.2,1,True,4169,1.8549,1.7932,1.1642,1.3553,0.5917,50.4689,62.7600,0.7407,1.0752,1.7932,2.3288,3.1343,1.2536,-0.8052,4.2091,-0.0531,False,MENOR_MELHOR,0.5500,0.6500,10.0000,Alta,Crítico,40.0000,Intervenção Imediata
4,5,5,11,9,2,2026-06-01 00:40:00,URA01,CEITE2PGCAG1CHAG01_VI_RCO,kW/TR,kW/TR,Eficiência,-,1.7932,RELAÇÃO DE CONSUMO,OK,True,2026,6,1,0,40,2,Monday,June,Madrugada,NaN,NaN,Crítico,NaN,2026-07-23 12:23:38.183885,2.2,1,True,4169,1.8549,1.7932,1.1642,1.3553,0.5917,50.4689,62.7600,0.7407,1.0752,1.7932,2.3288,3.1343,1.2536,-0.8052,4.2091,-0.0531,False,MENOR_MELHOR,0.5500,0.6500,10.0000,Alta,Crítico,40.0000,Intervenção Imediata


In [14]:
fact_engineering.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 54921 entries, 0 to 54920
Data columns (total 59 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   ID_MEDICAO          54921 non-null  int64         
 1   ID_TEMPO            54921 non-null  int64         
 2   ID_TAG              54921 non-null  int64         
 3   ID_VARIAVEL         54921 non-null  int64         
 4   ID_EQUIPAMENTO      54921 non-null  int64         
 5   DATA_HORA           54921 non-null  datetime64[ns]
 6   URA                 54921 non-null  object        
 7   TAG                 54921 non-null  object        
 8   VARIAVEL            54921 non-null  object        
 9   UNIDADE             54921 non-null  object        
 10  CATEGORIA           54921 non-null  object        
 11  CIRCUITO            54921 non-null  object        
 12  VALOR               54921 non-null  float64       
 13  ABA_ORIGEM          54921 non-null  object    

In [15]:
# ==========================================================
# BLOCO 4.9
# AUDITORIA ESTRUTURAL
# ==========================================================

import pandas as pd
import numpy as np
import json
from pathlib import Path

print("="*90)
print("BLOCO 4.9 - AUDITORIA ESTRUTURAL")
print("="*90)

base = fact_engineering.copy()

PASTA = Path("Resultados")
PASTA.mkdir(exist_ok=True)

# ==========================================================
# ESTRUTURA
# ==========================================================

estrutura = pd.DataFrame({

    "Campo":base.columns,

    "Tipo":base.dtypes.astype(str),

    "Nao_Nulos":base.notna().sum(),

    "Nulos":base.isna().sum(),

    "Percentual_Nulos":

        round(

            base.isna().mean()*100,

            2

        )

})

estrutura["Percentual_Preenchimento"]=(
100-
estrutura["Percentual_Nulos"]
)

display(estrutura.head())

# ==========================================================
# RESUMO
# ==========================================================

resumo = pd.DataFrame({

"Indicador":[

"Registros",

"Colunas",

"Memória (MB)",

"Duplicados",

"Outliers",

"Registros Válidos"

],

"Valor":[

len(base),

len(base.columns),

round(base.memory_usage().sum()/1024/1024,2),

base.duplicated().sum(),

base["OUTLIER"].sum(),

base["REGISTRO_VALIDO"].sum()

]

})

display(resumo)

# ==========================================================
# CHAVES
# ==========================================================

ids=[

"ID_MEDICAO",

"ID_TEMPO",

"ID_TAG",

"ID_VARIAVEL",

"ID_EQUIPAMENTO"

]

integridade=[]

for coluna in ids:

    integridade.append({

        "Campo":coluna,

        "Nulos":base[coluna].isna().sum(),

        "Duplicados":base[coluna].duplicated().sum(),

        "Únicos":base[coluna].nunique()

    })

integridade=pd.DataFrame(integridade)

display(integridade)

# ==========================================================
# DISTRIBUIÇÕES
# ==========================================================

ura = (

base

.groupby("URA")

.size()

.reset_index(name="Quantidade")

)

variavel=(

base

.groupby("VARIAVEL")

.size()

.reset_index(name="Quantidade")

)

categoria=(

base

.groupby("CATEGORIA")

.size()

.reset_index(name="Quantidade")

)

# ==========================================================
# ENGENHARIA
# ==========================================================

engenharia=pd.DataFrame({

"Indicador":[

"Peso",

"Nota",

"Criticidade",

"Classe"

],

"Preenchimento (%)":[

100-base["PESO"].isna().mean()*100,

100-base["NOTA_ENGENHARIA"].isna().mean()*100,

100-base["CRITICIDADE"].isna().mean()*100,

100-base["CLASSE_ENGENHARIA"].isna().mean()*100

]

})

display(engenharia)

# ==========================================================
# TEMPORAL
# ==========================================================

datas=pd.DataFrame({

"Indicador":[

"Início",

"Fim",

"Dias Monitorados",

"Registros"

],

"Valor":[

base["DATA_HORA"].min(),

base["DATA_HORA"].max(),

(base["DATA_HORA"].max()-base["DATA_HORA"].min()).days,

len(base)

]

})

display(datas)

# ==========================================================
# SCORE
# ==========================================================

score={}

score["Integridade"]=100-base.isna().mean().mean()*100

score["Outliers"]=100-base["OUTLIER"].mean()*100

score["Registros Válidos"]=base["REGISTRO_VALIDO"].mean()*100

score["Duplicidade"]=100-(base.duplicated().mean()*100)

score["Cobertura Engenharia"]=engenharia["Preenchimento (%)"].mean()

score_df=pd.DataFrame({

"Indicador":score.keys(),

"Score":score.values()

})

score_final=round(score_df["Score"].mean(),2)

print()

print("="*90)

print("SCORE GERAL DA BASE")

print("="*90)

print(score_final)

# ==========================================================
# DIAGNÓSTICO
# ==========================================================

if score_final >= 95:
    parecer = "EXCELENTE"

elif score_final >= 90:
    parecer = "MUITO BOA"

elif score_final >= 80:
    parecer = "BOA"

elif score_final >= 70:
    parecer = "REGULAR"

else:
    parecer = "CRÍTICA"

print()

print("="*90)

print("PARECER")

print("="*90)

print(parecer)

# ==========================================================
# EXPORTAÇÃO
# ==========================================================

with pd.ExcelWriter(
    PASTA/"12_AUDITORIA_ESTRUTURAL.xlsx",
    engine="xlsxwriter"
) as writer:

    estrutura.to_excel(
        writer,
        sheet_name="Estrutura",
        index=False
    )

    resumo.to_excel(
        writer,
        sheet_name="Resumo",
        index=False
    )

    integridade.to_excel(
        writer,
        sheet_name="Integridade",
        index=False
    )

    engenharia.to_excel(
        writer,
        sheet_name="Engenharia",
        index=False
    )

    ura.to_excel(
        writer,
        sheet_name="URA",
        index=False
    )

    variavel.to_excel(
        writer,
        sheet_name="Variaveis",
        index=False
    )

    categoria.to_excel(
        writer,
        sheet_name="Categorias",
        index=False
    )

    datas.to_excel(
        writer,
        sheet_name="Temporal",
        index=False
    )

    score_df.to_excel(
        writer,
        sheet_name="Score",
        index=False
    )

# JSON
with open(PASTA/"auditoria_estrutural.json", "w", encoding="utf-8") as f:
    json.dump(
        {
            "score_final": score_final,
            "parecer": parecer,
            "estrutura": estrutura.to_dict(orient="records")
        },
        f,
        indent=4,
        ensure_ascii=False,
        default=str
    )

# TXT
with open(PASTA/"auditoria_estrutural.txt", "w", encoding="utf-8") as f:
    f.write("AUDITORIA ESTRUTURAL DA BASE\n")
    f.write("="*60 + "\n")
    f.write(f"Score Geral: {score_final}\n")
    f.write(f"Parecer: {parecer}\n")

print("\n" + "="*90)
print("AUDITORIA ESTRUTURAL CONCLUÍDA")
print("="*90)



BLOCO 4.9 - AUDITORIA ESTRUTURAL


,Campo,Tipo,Nao_Nulos,Nulos,Percentual_Nulos,Percentual_Preenchimento
ID_MEDICAO,ID_MEDICAO,int64,54921,0,0.0000,100.0000
ID_TEMPO,ID_TEMPO,int64,54921,0,0.0000,100.0000
ID_TAG,ID_TAG,int64,54921,0,0.0000,100.0000
ID_VARIAVEL,ID_VARIAVEL,int64,54921,0,0.0000,100.0000
ID_EQUIPAMENTO,ID_EQUIPAMENTO,int64,54921,0,0.0000,100.0000


,Indicador,Valor
0,Registros,54921.0000
1,Colunas,59.0000
2,Memória (MB),22.3600
3,Duplicados,0.0000
4,Outliers,2066.0000
5,Registros Válidos,54921.0000


,Campo,Nulos,Duplicados,Únicos
0,ID_MEDICAO,0,0,54921
1,ID_TEMPO,0,50626,4295
2,ID_TAG,0,54893,28
3,ID_VARIAVEL,0,54912,9
4,ID_EQUIPAMENTO,0,54917,4


,Indicador,Preenchimento (%)
0,Peso,68.7005
1,Nota,22.4486
2,Criticidade,68.7005
3,Classe,100.0000


,Indicador,Valor
0,Início,2026-06-01 00:00:00
1,Fim,2026-07-01 00:00:00
2,Dias Monitorados,30
3,Registros,54921



SCORE GERAL DA BASE
90.05

PARECER
MUITO BOA

AUDITORIA ESTRUTURAL CONCLUÍDA


In [16]:
# ==========================================================
# BLOCO 4.10
# FACT_ANALYTICS
# ==========================================================

import pandas as pd
import numpy as np
from pathlib import Path

print("="*90)
print("BLOCO 4.10 - FACT_ANALYTICS")
print("="*90)

PASTA = Path("Resultados")
PASTA.mkdir(exist_ok=True)

fact_analytics = fact_engineering.copy()

print(f"Registros : {len(fact_analytics):,}")
print(f"Colunas   : {len(fact_analytics.columns)}")

# ==========================================================
# PADRONIZAÇÃO
# ==========================================================

fact_analytics.rename(
    columns={
        "DATA_HORA":"DATAHORA",
        "CV (%)":"CV",
        "Media":"MEDIA",
        "Mediana":"MEDIANA",
        "Variancia":"VARIANCIA",
        "Desvio":"DESVIO",
        "Minimo":"MINIMO",
        "Maximo":"MAXIMO"
    },
    inplace=True
)

# ==========================================================
# CALENDÁRIO ANALÍTICO
# ==========================================================

fact_analytics["DATA"] = fact_analytics["DATAHORA"].dt.date

fact_analytics["MES_ANO"] = (
    fact_analytics["DATAHORA"]
    .dt.strftime("%Y-%m")
)

fact_analytics["SEMANA_ANO"] = (
    fact_analytics["DATAHORA"]
    .dt.isocalendar()
    .week
)

fact_analytics["FIM_DE_SEMANA"] = (
    fact_analytics["DATAHORA"]
    .dt.weekday >= 5
)

fact_analytics["HORA_DECIMAL"] = (

    fact_analytics["HORA"]

    +

    fact_analytics["MINUTO"]/60

)

# ==========================================================
# STATUS OPERACIONAL
# ==========================================================

fact_analytics["STATUS_OPERACAO"] = np.where(

    fact_analytics["EM_OPERACAO"],

    "Operando",

    "Parado"

)

fact_analytics["STATUS_OUTLIER"] = np.where(

    fact_analytics["OUTLIER"],

    "Outlier",

    "Normal"

)

# ==========================================================
# CLASSIFICAÇÃO
# ==========================================================

fact_analytics["FAIXA_NOTA"] = pd.cut(

    fact_analytics["NOTA_ENGENHARIA"],

    bins=[0,40,60,80,100],

    labels=[

        "Crítica",

        "Regular",

        "Boa",

        "Excelente"

    ]

)

fact_analytics["PRIORIDADE_NUMERICA"] = (

    fact_analytics["PRIORIDADE"]

    .map({

        "Baixa":1,

        "Média":2,

        "Alta":3,

        "Crítica":4

    })

)

# ==========================================================
# FLAGS ANALÍTICAS
# ==========================================================

fact_analytics["FLAG_COP"] = (

    fact_analytics["VARIAVEL"]

    ==

    "COP"

)

fact_analytics["FLAG_KWTR"] = (

    fact_analytics["VARIAVEL"]

    ==

    "kW/TR"

)

fact_analytics["FLAG_APPROACH"] = (

    fact_analytics["VARIAVEL"]

    .str.contains(

        "Approach",

        case=False,

        na=False

    )

)

fact_analytics["FLAG_CARGA"] = (

    fact_analytics["VARIAVEL"]

    .str.contains(

        "Carga",

        case=False,

        na=False

    )

)

# ==========================================================
# SCORE DA MEDIÇÃO
# ==========================================================

fact_analytics["SCORE_MEDICAO"] = 100

fact_analytics.loc[

    fact_analytics["OUTLIER"],

    "SCORE_MEDICAO"

] -= 30

fact_analytics.loc[

    ~fact_analytics["REGISTRO_VALIDO"],

    "SCORE_MEDICAO"

] -= 40

fact_analytics.loc[

    fact_analytics["NOTA_ENGENHARIA"] < 60,

    "SCORE_MEDICAO"

] -= 20

fact_analytics["SCORE_MEDICAO"] = (

    fact_analytics["SCORE_MEDICAO"]

    .clip(lower=0)

)

# ==========================================================
# COLUNAS ANALÍTICAS
# ==========================================================

colunas = [

'ID_MEDICAO',

'DATAHORA',

'URA',

'TAG',

'VARIAVEL',

'VALOR',

'UNIDADE',

'CATEGORIA',

'CIRCUITO',

'EM_OPERACAO',

'STATUS_OPERACAO',

'OUTLIER',

'STATUS_OUTLIER',

'REGISTRO_VALIDO',

'ANO',

'MES',

'DIA',

'HORA',

'MINUTO',

'MES_ANO',

'SEMANA_ANO',

'FIM_DE_SEMANA',

'HORA_DECIMAL',

'FAIXA_CARGA',

'STATUS_COP',

'STATUS_KWTR',

'STATUS_APPROACH',

'MEDIA',

'DESVIO',

'CV',

'P5',

'Q1',

'Q2',

'Q3',

'P95',

'LIM_INF',

'LIM_SUP',

'Z_SCORE',

'CLASSE_ENGENHARIA',

'NOTA_ENGENHARIA',

'FAIXA_NOTA',

'PESO',

'CRITICIDADE',

'PRIORIDADE',

'PRIORIDADE_NUMERICA',

'SCORE_MEDICAO'

]

fact_analytics = fact_analytics[colunas]

# ==========================================================
# AUDITORIA
# ==========================================================

print()

print("="*90)

print("FACT_ANALYTICS")

print("="*90)

print()

print(fact_analytics.info())

print()

print(fact_analytics.head())

print()

print(f"Registros : {len(fact_analytics):,}")

print(f"Colunas   : {len(fact_analytics.columns)}")

# ==========================================================
# EXPORTAÇÃO
# ==========================================================

fact_analytics.to_excel(

    PASTA/"13_FACT_ANALYTICS.xlsx",

    index=False

)

fact_analytics.to_csv(

    PASTA/"13_FACT_ANALYTICS.csv",

    sep=";",

    index=False

)

fact_analytics.to_parquet(

    PASTA/"13_FACT_ANALYTICS.parquet",

    index=False

)

print()

print("="*90)

print("FACT_ANALYTICS GERADA COM SUCESSO")

print("="*90)



BLOCO 4.10 - FACT_ANALYTICS
Registros : 54,921
Colunas   : 59

FACT_ANALYTICS

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 54921 entries, 0 to 54920
Data columns (total 46 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   ID_MEDICAO           54921 non-null  int64         
 1   DATAHORA             54921 non-null  datetime64[ns]
 2   URA                  54921 non-null  object        
 3   TAG                  54921 non-null  object        
 4   VARIAVEL             54921 non-null  object        
 5   VALOR                54921 non-null  float64       
 6   UNIDADE              54921 non-null  object        
 7   CATEGORIA            54921 non-null  object        
 8   CIRCUITO             54921 non-null  object        
 9   EM_OPERACAO          54921 non-null  bool          
 10  STATUS_OPERACAO      54921 non-null  object        
 11  OUTLIER              54921 non-null  bool          
 12  STATUS_OU

In [17]:
# ==========================================================
# BLOCO 5.1.1
# ANALYTICAL CORE
# PREPARAÇÃO DA BASE ANALÍTICA
# ==========================================================

import pandas as pd
import numpy as np
from pathlib import Path

print("="*100)
print("BLOCO 5.1.1 - ANALYTICAL CORE")
print("="*100)

PASTA = Path("Resultados")
PASTA.mkdir(exist_ok=True)

core = fact_analytics.copy()

print(f"Registros : {len(core):,}")
print(f"Colunas   : {len(core.columns)}")

# ==========================================================
# ORGANIZAÇÃO
# ==========================================================

core = (
    core
    .sort_values(
        [
            "URA",
            "VARIAVEL",
            "DATAHORA"
        ]
    )
    .reset_index(drop=True)
)

print("Base organizada.")

# ==========================================================
# IDENTIFICADOR ANALÍTICO
# ==========================================================

core["ID_ANALYTICS"] = np.arange(
    1,
    len(core)+1
)

core["ID_SERIE"] = (

    core["URA"]

    + "_"

    +

    core["VARIAVEL"]

)

# ==========================================================
# SEQUÊNCIA
# ==========================================================

core["ORDEM_SERIE"] = (

    core

    .groupby(

        "ID_SERIE"

    )

    .cumcount()

    +1

)

# ==========================================================
# DELTA T
# ==========================================================

core["DELTA_MIN"] = (

    core

    .groupby(

        "ID_SERIE"

    )["DATAHORA"]

    .diff()

    .dt.total_seconds()

    /60

)

core["DELTA_MIN"] = (

    core["DELTA_MIN"]

    .fillna(0)

)

# ==========================================================
# INTERVALO PADRÃO
# ==========================================================

intervalo = (

    core

    .groupby("ID_SERIE")["DELTA_MIN"]

    .median()

)

core = core.merge(

    intervalo.rename(

        "INTERVALO_PADRAO"

    ),

    on="ID_SERIE"

)

# ==========================================================
# DESVIO
# ==========================================================

core["ERRO_INTERVALO"] = (

    core["DELTA_MIN"]

    -

    core["INTERVALO_PADRAO"]

)

core["FALHA_AQUISICAO"] = (

    abs(

        core["ERRO_INTERVALO"]

    )>

    2

)

# ==========================================================
# QUALIDADE TEMPORAL
# ==========================================================

core["QUALIDADE_TEMPORAL"] = np.where(

    core["FALHA_AQUISICAO"],

    "Falha",

    "OK"

)

# ==========================================================
# TEMPO
# ==========================================================

core["TEMPO_ACUMULADO_MIN"] = (

    core

    .groupby(

        "ID_SERIE"

    )["DELTA_MIN"]

    .cumsum()

)

core["HORAS_MONITORADAS"] = (

    core["TEMPO_ACUMULADO_MIN"]

    /60

)

# ==========================================================
# PERÍODO
# ==========================================================

periodo = (

core

.groupby(

"ID_SERIE"

)

.agg(

INICIO=("DATAHORA","min"),

FIM=("DATAHORA","max")

)

.reset_index()

)

core = core.merge(

periodo,

on="ID_SERIE"

)

# ==========================================================
# DIAS
# ==========================================================

core["DIAS_MONITORADOS"] = (

    core["FIM"]

    -

    core["INICIO"]

).dt.days+1

# ==========================================================
# OPERAÇÃO
# ==========================================================

operacao = (

core

.groupby(

"ID_SERIE"

)["EM_OPERACAO"]

.mean()

*100

)

core = core.merge(

operacao.rename(

"PERCENTUAL_OPERACAO"

),

on="ID_SERIE"

)

# ==========================================================
# RESUMO
# ==========================================================

print()

print("="*100)

print("ANALYTICAL CORE")

print("="*100)

print()

print(core.info())

print()

print(core.head())

print()

print(f"Registros : {len(core):,}")

print(f"Colunas   : {len(core.columns)}")

# ==========================================================
# EXPORTAÇÃO
# ==========================================================

core.to_excel(

PASTA/

"14_ANALYTICAL_CORE.xlsx",

index=False

)

core.to_csv(

PASTA/

"14_ANALYTICAL_CORE.csv",

sep=";",

index=False

)

core.to_parquet(

PASTA/

"14_ANALYTICAL_CORE.parquet",

index=False

)

analytical_core = core.copy()

print()

print("="*100)

print("ANALYTICAL CORE CRIADO")

print("="*100)

BLOCO 5.1.1 - ANALYTICAL CORE
Registros : 54,921
Colunas   : 46
Base organizada.

ANALYTICAL CORE

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 54921 entries, 0 to 54920
Data columns (total 60 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   ID_MEDICAO           54921 non-null  int64         
 1   DATAHORA             54921 non-null  datetime64[ns]
 2   URA                  54921 non-null  object        
 3   TAG                  54921 non-null  object        
 4   VARIAVEL             54921 non-null  object        
 5   VALOR                54921 non-null  float64       
 6   UNIDADE              54921 non-null  object        
 7   CATEGORIA            54921 non-null  object        
 8   CIRCUITO             54921 non-null  object        
 9   EM_OPERACAO          54921 non-null  bool          
 10  STATUS_OPERACAO      54921 non-null  object        
 11  OUTLIER              54921 non-null  bool     

In [18]:
# ==========================================================
# BLOCO 5.1.2
# ROLLING STATISTICS ENGINE
# ==========================================================

import pandas as pd
import numpy as np

print("="*100)
print("BLOCO 5.1.2 - ROLLING STATISTICS ENGINE")
print("="*100)

rolling_core = analytical_core.copy()

WINDOW = 12

rolling_core = (
    rolling_core
    .sort_values(
        ["ID_SERIE", "DATAHORA"]
    )
    .reset_index(drop=True)
)

rolling_core["ROLLING_MEDIA"] = (

    rolling_core

    .groupby("ID_SERIE")["VALOR"]

    .transform(

        lambda s:

        s.rolling(

            WINDOW,

            min_periods=1

        ).mean()

    )

)

rolling_core["ROLLING_MEDIANA"] = (

    rolling_core

    .groupby("ID_SERIE")["VALOR"]

    .transform(

        lambda s:

        s.rolling(

            WINDOW,

            min_periods=1

        ).median()

    )

)

rolling_core["ROLLING_STD"] = (

    rolling_core

    .groupby("ID_SERIE")["VALOR"]

    .transform(

        lambda s:

        s.rolling(

            WINDOW,

            min_periods=2

        ).std()

    )

)

rolling_core["ROLLING_VARIANCIA"] = (

    rolling_core

    .groupby("ID_SERIE")["VALOR"]

    .transform(

        lambda s:

        s.rolling(

            WINDOW,

            min_periods=2

        ).var()

    )

)

rolling_core["ROLLING_CV"] = (

    rolling_core["ROLLING_STD"]

    /

    rolling_core["ROLLING_MEDIA"]

) * 100

rolling_core["ROLLING_MIN"] = (

    rolling_core

    .groupby("ID_SERIE")["VALOR"]

    .transform(

        lambda s:

        s.rolling(

            WINDOW,

            min_periods=1

        ).min()

    )

)

rolling_core["ROLLING_MAX"] = (

    rolling_core

    .groupby("ID_SERIE")["VALOR"]

    .transform(

        lambda s:

        s.rolling(

            WINDOW,

            min_periods=1

        ).max()

    )

)

rolling_core["ROLLING_RANGE"] = (

    rolling_core["ROLLING_MAX"]

    -

    rolling_core["ROLLING_MIN"]

)

rolling_core["DELTA_VALOR"] = (

    rolling_core

    .groupby("ID_SERIE")["VALOR"]

    .diff()

)

rolling_core["TENDENCIA"] = np.where(

    rolling_core["DELTA_VALOR"] > 0,

    "Crescente",

    np.where(

        rolling_core["DELTA_VALOR"] < 0,

        "Decrescente",

        "Estável"

    )

)

rolling_core["ROLLING_Z"] = (

    rolling_core["VALOR"]

    -

    rolling_core["ROLLING_MEDIA"]

)

/

rolling_core["ROLLING_STD"]

rolling_core["ROLLING_Z"] = (

    rolling_core["ROLLING_Z"]

    .replace(

        [np.inf, -np.inf],

        np.nan

    )

)

rolling_core["LIMITE_SUP"] = (

    rolling_core["ROLLING_MEDIA"]

    +

    3 *

    rolling_core["ROLLING_STD"]

)

rolling_core["LIMITE_INF"] = (

    rolling_core["ROLLING_MEDIA"]

    -

    3 *

    rolling_core["ROLLING_STD"]

)

rolling_core["FLAG_ESTATISTICA"] = (

    (

        rolling_core["VALOR"]

        >

        rolling_core["LIMITE_SUP"]

    )

    |

    (

        rolling_core["VALOR"]

        <

        rolling_core["LIMITE_INF"]

    )

)

print()

print("="*100)

print("ROLLING STATISTICS")

print("="*100)

print()

print(rolling_core.info())

print()

print(rolling_core[

    [

        "VALOR",

        "ROLLING_MEDIA",

        "ROLLING_STD",

        "ROLLING_CV",

        "ROLLING_Z"

    ]

].head())

print()

print(

"Flags:",

rolling_core["FLAG_ESTATISTICA"].sum()

)

rolling_core.to_excel(

    "Resultados/15_ANALYTICAL_CORE_ROLLING.xlsx",

    index=False

)

rolling_core.to_csv(

    "Resultados/15_ANALYTICAL_CORE_ROLLING.csv",

    sep=";",

    index=False

)

rolling_core.to_parquet(

    "Resultados/15_ANALYTICAL_CORE_ROLLING.parquet",

    index=False

)

analytical_core = rolling_core.copy()

print()

print("="*100)

print("ROLLING ENGINE FINALIZADO")

print("="*100)

BLOCO 5.1.2 - ROLLING STATISTICS ENGINE

ROLLING STATISTICS

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 54921 entries, 0 to 54920
Data columns (total 74 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   ID_MEDICAO           54921 non-null  int64         
 1   DATAHORA             54921 non-null  datetime64[ns]
 2   URA                  54921 non-null  object        
 3   TAG                  54921 non-null  object        
 4   VARIAVEL             54921 non-null  object        
 5   VALOR                54921 non-null  float64       
 6   UNIDADE              54921 non-null  object        
 7   CATEGORIA            54921 non-null  object        
 8   CIRCUITO             54921 non-null  object        
 9   EM_OPERACAO          54921 non-null  bool          
 10  STATUS_OPERACAO      54921 non-null  object        
 11  OUTLIER              54921 non-null  bool          
 12  STATUS_OUTLIER       54921 

In [19]:
# ==========================================================
# BLOCO 5.1.3
# TREND ENGINE
# ==========================================================

import pandas as pd
import numpy as np
from pathlib import Path

print("="*100)
print("BLOCO 5.1.3 - TREND ENGINE")
print("="*100)

trend_core = analytical_core.copy()

WINDOW = 12

trend_core = (
    trend_core
    .sort_values(["ID_SERIE", "DATAHORA"])
    .reset_index(drop=True)
)

# ==========================================================
# REGRESSÃO LINEAR
# ==========================================================

def calcular_regressao(janela):

    y = janela.values

    if len(y) < 2:
        return np.nan, np.nan, np.nan

    x = np.arange(len(y))

    slope, intercept = np.polyfit(x, y, 1)

    y_hat = slope * x + intercept

    ss_res = np.sum((y - y_hat) ** 2)
    ss_tot = np.sum((y - np.mean(y)) ** 2)

    if ss_tot == 0:
        r2 = 1

    else:
        r2 = 1 - (ss_res / ss_tot)

    return slope, intercept, r2

    # ==========================================================
# TENDÊNCIA
# ==========================================================

trend_core["TREND_SLOPE"] = np.nan
trend_core["TREND_INTERCEPT"] = np.nan
trend_core["TREND_R2"] = np.nan

for serie, indice in trend_core.groupby("ID_SERIE").groups.items():

    valores = trend_core.loc[indice, "VALOR"]

    slopes = []
    interceptos = []
    r2s = []

    for i in range(len(valores)):

        inicio = max(0, i - WINDOW + 1)

        janela = valores.iloc[inicio:i+1]

        slope, intercept, r2 = calcular_regressao(janela)

        slopes.append(slope)
        interceptos.append(intercept)
        r2s.append(r2)

    trend_core.loc[indice, "TREND_SLOPE"] = slopes
    trend_core.loc[indice, "TREND_INTERCEPT"] = interceptos
    trend_core.loc[indice, "TREND_R2"] = r2s

    trend_core["TREND_PREVISTO"] = (

    trend_core["TREND_INTERCEPT"]

    +

    trend_core["TREND_SLOPE"]

    *

    (trend_core["ORDEM_SERIE"] - 1)

)

    trend_core["TREND_RESIDUO"] = (

    trend_core["VALOR"]

    -

    trend_core["TREND_PREVISTO"]

)

    trend_core["VELOCIDADE"] = (

    trend_core["TREND_SLOPE"]

    /

    trend_core["INTERVALO_PADRAO"]

)

    LIMIAR = 0.001

trend_core["CLASSIFICACAO_TENDENCIA"] = np.select(

    [

        trend_core["TREND_SLOPE"] > LIMIAR,

        trend_core["TREND_SLOPE"] < -LIMIAR

    ],

    [

        "Crescente",

        "Decrescente"

    ],

    default="Estável"

)

trend_core["INTENSIDADE_TENDENCIA"] = pd.cut(

    trend_core["TREND_SLOPE"].abs(),

    bins=[0, 0.01, 0.05, np.inf],

    labels=[

        "Fraca",

        "Moderada",

        "Forte"

    ],

    include_lowest=True

)

trend_core["SCORE_ESTABILIDADE"] = (

    100

    -

    (

        trend_core["ROLLING_CV"]

        .fillna(0)

        * 2

    )

)

trend_core["SCORE_ESTABILIDADE"] = (

    trend_core["SCORE_ESTABILIDADE"]

    .clip(

        lower=0,

        upper=100

    )

)

print()

print("="*100)
print("TREND ENGINE")
print("="*100)

print()

print(

trend_core[

[
"VALOR",
"TREND_SLOPE",
"TREND_R2",
"CLASSIFICACAO_TENDENCIA",
"INTENSIDADE_TENDENCIA",
"SCORE_ESTABILIDADE"

]

].head(15)

)

print()

print("Distribuição das tendências")

print(

trend_core["CLASSIFICACAO_TENDENCIA"]

.value_counts()

)

PASTA = Path("Resultados")

trend_core.to_excel(
    PASTA / "16_ANALYTICAL_CORE_TREND.xlsx",
    index=False
)

trend_core.to_csv(
    PASTA / "16_ANALYTICAL_CORE_TREND.csv",
    sep=";",
    index=False
)

trend_core.to_parquet(
    PASTA / "16_ANALYTICAL_CORE_TREND.parquet",
    index=False
)

analytical_core = trend_core.copy()

print()

print("="*100)
print("TREND ENGINE FINALIZADO")
print("="*100)



BLOCO 5.1.3 - TREND ENGINE

TREND ENGINE

       VALOR  TREND_SLOPE  TREND_R2 CLASSIFICACAO_TENDENCIA  \
0  1286.0760          NaN       NaN                 Estável   
1  2260.2370     974.1610    1.0000               Crescente   
2  2241.6330     477.7785    0.7354               Crescente   
3  2285.4020     297.9374    0.6200               Crescente   
4  1099.7830     -34.7421    0.0087             Decrescente   
5  2252.0570      39.7803    0.0180               Crescente   
6  1173.0180     -53.4780    0.0402             Decrescente   
7  1273.9520     -79.4680    0.1186             Decrescente   
8  2537.1310      -2.0868    0.0001             Decrescente   
9  2445.6190      32.4295    0.0275               Crescente   
10 2533.5840      53.7809    0.0899               Crescente   
11 2383.0070      58.2391    0.1305               Crescente   
12 2114.9000      31.8051    0.0453               Crescente   
13  886.7126      -3.3827    0.0004             Decrescente   
14  853.1733 

In [20]:
# ==========================================================
# BLOCO 5.1.4
# OPERATIONAL STATE ENGINE
# ==========================================================

import pandas as pd
import numpy as np
from pathlib import Path

print("="*100)
print("BLOCO 5.1.4 - OPERATIONAL STATE ENGINE")
print("="*100)

PASTA = Path("Resultados")
PASTA.mkdir(exist_ok=True)

operational_core = analytical_core.copy()

# ==========================================================
# CONFIGURAÇÃO DE ENGENHARIA
# ==========================================================

ENG_CONFIG = {

    "CAPACIDADE_CHILLER_TR": 300.0,
    "CAPACIDADE_TOTAL_TR": 900.0,

    "CARGA_MINIMA": 0.20,
    "CARGA_BAIXA": 0.40,
    "CARGA_MEDIA": 0.60,
    "CARGA_ALTA": 0.80

}

print(f"Registros: {len(operational_core):,}")

# ==========================================================
# CAPACIDADE
# ==========================================================

operational_core["CAPACIDADE_TR"] = ENG_CONFIG["CAPACIDADE_CHILLER_TR"]

# ==========================================================
# PERCENTUAL DE CARGA
# ==========================================================

operational_core["PERCENTUAL_CARGA"] = np.nan

mask = (
    operational_core["VARIAVEL"]
    .str.contains("Carga", case=False, na=False)
)

operational_core.loc[mask, "PERCENTUAL_CARGA"] = (

    operational_core.loc[mask, "VALOR"]

    /

    ENG_CONFIG["CAPACIDADE_CHILLER_TR"]

)

# ==========================================================
# FAIXA OPERACIONAL
# ==========================================================

def faixa_operacao(valor):

    if pd.isna(valor):
        return np.nan

    if valor < 0.20:
        return "Muito Baixa"

    elif valor < 0.40:
        return "Baixa"

    elif valor < 0.60:
        return "Média"

    elif valor < 0.80:
        return "Alta"

    return "Plena"


operational_core["FAIXA_OPERACAO"] = (

    operational_core["PERCENTUAL_CARGA"]

    .apply(faixa_operacao)

)

# ==========================================================
# FLAGS
# ==========================================================

operational_core["ABAIXO_CARGA_MINIMA"] = (

    operational_core["PERCENTUAL_CARGA"]

    <

    ENG_CONFIG["CARGA_MINIMA"]

)

operational_core["ACIMA_CAPACIDADE"] = (

    operational_core["PERCENTUAL_CARGA"]

    >

    1.0

)

# ==========================================================
# ESTADO
# ==========================================================

def estado(row):

    if not row["EM_OPERACAO"]:
        return "Parado"

    if pd.isna(row["PERCENTUAL_CARGA"]):
        return "Operando"

    if row["PERCENTUAL_CARGA"] > 1:
        return "Sobrecarga"

    if row["PERCENTUAL_CARGA"] < 0.20:
        return "Carga Muito Baixa"

    return "Operação Normal"


operational_core["ESTADO_OPERACIONAL"] = (

    operational_core.apply(

        estado,

        axis=1

    )

)

# ==========================================================
# HORAS EQUIVALENTES
# ==========================================================

operational_core["HORAS_EQUIVALENTES"] = np.where(

    operational_core["EM_OPERACAO"],

    operational_core["HORAS_MONITORADAS"],

    0

)

# ==========================================================
# EVENTOS
# ==========================================================

operational_core["EVENTO_OPERACIONAL"] = "Continuação"

mudou = (

    operational_core

    .groupby("ID_SERIE")["EM_OPERACAO"]

    .diff()

)

operational_core.loc[

    mudou == 1,

    "EVENTO_OPERACIONAL"

] = "Partida"

operational_core.loc[

    mudou == -1,

    "EVENTO_OPERACIONAL"

] = "Parada"

# ==========================================================
# SCORE DE CARGA
# ==========================================================

operational_core["SCORE_CARGA"] = np.nan

mask = operational_core["PERCENTUAL_CARGA"].notna()

pc = operational_core.loc[mask, "PERCENTUAL_CARGA"]

score = np.select(

    [

        (pc >= 0.40) & (pc <= 0.80),

        (pc >= 0.20) & (pc < 0.40),

        (pc > 0.80) & (pc <= 1.00),

        pc < 0.20,

        pc > 1.00

    ],

    [

        100,

        85,

        75,

        50,

        0

    ],

    default=70

)

operational_core.loc[mask, "SCORE_CARGA"] = score

# ==========================================================
# SCORE OPERACIONAL
# ==========================================================

operational_core["SCORE_OPERACIONAL"] = 100

operational_core.loc[
    operational_core["OUTLIER"],
    "SCORE_OPERACIONAL"
] -= 20

operational_core.loc[
    operational_core["FLAG_ESTATISTICA"],
    "SCORE_OPERACIONAL"
] -= 10

operational_core.loc[
    operational_core["ABAIXO_CARGA_MINIMA"],
    "SCORE_OPERACIONAL"
] -= 15

operational_core.loc[
    operational_core["ACIMA_CAPACIDADE"],
    "SCORE_OPERACIONAL"
] -= 40

operational_core["SCORE_OPERACIONAL"] = (
    operational_core["SCORE_OPERACIONAL"]
    .clip(0, 100)
)

# ==========================================================
# AUDITORIA
# ==========================================================

print("\n" + "="*100)
print("RESUMO OPERACIONAL")
print("="*100)

print("\nEstados Operacionais:")

print(
    operational_core["ESTADO_OPERACIONAL"]
    .value_counts(dropna=False)
)

print("\nFaixas de Operação:")

print(
    operational_core["FAIXA_OPERACAO"]
    .value_counts(dropna=False)
)

print("\nScore Médio Operacional:")

print(
    round(
        operational_core["SCORE_OPERACIONAL"].mean(),
        2
    )
)

# ==========================================================
# EXPORTAÇÃO
# ==========================================================

operational_core.to_excel(
    PASTA / "17_ANALYTICAL_CORE_OPERATIONAL.xlsx",
    index=False
)

operational_core.to_csv(
    PASTA / "17_ANALYTICAL_CORE_OPERATIONAL.csv",
    sep=";",
    index=False
)

operational_core.to_parquet(
    PASTA / "17_ANALYTICAL_CORE_OPERATIONAL.parquet",
    index=False
)

analytical_core = operational_core.copy()

print("\n" + "="*100)
print("OPERATIONAL STATE ENGINE FINALIZADO")
print("="*100)



BLOCO 5.1.4 - OPERATIONAL STATE ENGINE
Registros: 54,921

RESUMO OPERACIONAL

Estados Operacionais:
ESTADO_OPERACIONAL
Operando             52273
Operação Normal       1937
Carga Muito Baixa      711
Name: count, dtype: int64

Faixas de Operação:
FAIXA_OPERACAO
NaN            52273
Baixa            968
Média            900
Muito Baixa      711
Alta              58
Plena             11
Name: count, dtype: int64

Score Médio Operacional:
98.91

OPERATIONAL STATE ENGINE FINALIZADO


In [21]:
# ==========================================================
# BLOCO 5.1.5
# STABILITY ENGINE
# ==========================================================

import pandas as pd
import numpy as np
from pathlib import Path

print("="*100)
print("BLOCO 5.1.5 - STABILITY ENGINE")
print("="*100)

PASTA = Path("Resultados")
PASTA.mkdir(exist_ok=True)

stability_core = analytical_core.copy()

print(f"Registros : {len(stability_core):,}")

# ==========================================================
# CICLOS OPERACIONAIS
# ==========================================================

mudanca_estado = (

    stability_core
    .groupby("ID_SERIE")["EM_OPERACAO"]
    .transform(lambda s: s.ne(s.shift()))

)

stability_core["ID_CICLO"] = (

    mudanca_estado
    .groupby(stability_core["ID_SERIE"])
    .cumsum()

)

# ==========================================================
# DURAÇÃO
# ==========================================================

ciclos = (

    stability_core

    .groupby(

        ["ID_SERIE","ID_CICLO"]

    )

    .agg(

        INICIO=("DATAHORA","min"),

        FIM=("DATAHORA","max"),

        EM_OPERACAO=("EM_OPERACAO","first"),

        REGISTROS=("ID_ANALYTICS","count")

    )

    .reset_index()

)

ciclos["DURACAO_MIN"] = (

    ciclos["FIM"]

    -

    ciclos["INICIO"]

).dt.total_seconds()/60

ciclos["DURACAO_H"] = (

    ciclos["DURACAO_MIN"]

    /60

)

# ==========================================================
# RESUMO
# ==========================================================

resumo = (

    ciclos

    .groupby("ID_SERIE")

    .agg(

        NUM_CICLOS=("ID_CICLO","count"),

        TEMPO_MEDIO_H=("DURACAO_H","mean"),

        TEMPO_MAX_H=("DURACAO_H","max"),

        TEMPO_MIN_H=("DURACAO_H","min")

    )

    .reset_index()

)

# ==========================================================
# PARTIDAS
# ==========================================================

partidas = (

    stability_core

    .groupby("ID_SERIE")["EVENTO_OPERACIONAL"]

    .apply(

        lambda x:

        (x=="Partida").sum()

    )

)

paradas = (

    stability_core

    .groupby("ID_SERIE")["EVENTO_OPERACIONAL"]

    .apply(

        lambda x:

        (x=="Parada").sum()

    )

)

resumo["PARTIDAS"] = (

    resumo["ID_SERIE"]

    .map(partidas)

)

resumo["PARADAS"] = (

    resumo["ID_SERIE"]

    .map(paradas)

)

# ==========================================================
# OSCILAÇÃO
# ==========================================================

variacao = (

    stability_core

    .groupby("ID_SERIE")["VALOR"]

    .std()

)

resumo["OSCILACAO"] = (

    resumo["ID_SERIE"]

    .map(variacao)

)

# ==========================================================
# ESTABILIDADE
# ==========================================================

resumo["INDICE_ESTABILIDADE"] = (

    100

    -

    (

        resumo["OSCILACAO"]

        .fillna(0)

    )

)

resumo["INDICE_ESTABILIDADE"] = (

    resumo["INDICE_ESTABILIDADE"]

    .clip(0,100)

)

# ==========================================================
# CONFIABILIDADE
# ==========================================================

resumo["INDICE_CONFIABILIDADE"] = (

    resumo["INDICE_ESTABILIDADE"]

    -

    resumo["PARTIDAS"]*2

)

resumo["INDICE_CONFIABILIDADE"] = (

    resumo["INDICE_CONFIABILIDADE"]

    .clip(0,100)

)

# ==========================================================
# MERGE
# ==========================================================

stability_core = stability_core.merge(

    resumo,

    on="ID_SERIE",

    how="left"

)

# ==========================================================
# SCORE
# ==========================================================

stability_core["SCORE_ESTABILIDADE"] = (

    0.40*stability_core["INDICE_ESTABILIDADE"]

    +

    0.60*stability_core["INDICE_CONFIABILIDADE"]

)

stability_core["SCORE_ESTABILIDADE"] = (

    stability_core["SCORE_ESTABILIDADE"]

    .round(2)

)

# ==========================================================
# CLASSIFICAÇÃO
# ==========================================================

stability_core["CLASSE_ESTABILIDADE"] = pd.cut(

    stability_core["SCORE_ESTABILIDADE"],

    bins=[0,40,60,80,100],

    labels=[

        "Crítica",

        "Regular",

        "Boa",

        "Excelente"

    ],

    include_lowest=True

)

# ==========================================================
# AUDITORIA
# ==========================================================

print("\n"+"="*100)
print("STABILITY ENGINE")
print("="*100)

print()

print(resumo.head())

print()

print(

stability_core[

[
"ID_SERIE",
"SCORE_ESTABILIDADE",
"CLASSE_ESTABILIDADE"

]

]

.drop_duplicates()

.head(20)

)

print()

print("Distribuição:")

print(

stability_core["CLASSE_ESTABILIDADE"]

.value_counts()

)

# ==========================================================
# EXPORTAÇÃO
# ==========================================================

stability_core.to_excel(

    PASTA/"18_ANALYTICAL_CORE_STABILITY.xlsx",

    index=False

)

stability_core.to_csv(

    PASTA/"18_ANALYTICAL_CORE_STABILITY.csv",

    sep=";",

    index=False

)

stability_core.to_parquet(

    PASTA/"18_ANALYTICAL_CORE_STABILITY.parquet",

    index=False

)

analytical_core = stability_core.copy()

print()

print("="*100)
print("STABILITY ENGINE FINALIZADO")
print("="*100)



BLOCO 5.1.5 - STABILITY ENGINE
Registros : 54,921

STABILITY ENGINE

                     ID_SERIE  NUM_CICLOS  TEMPO_MEDIO_H  TEMPO_MAX_H  \
0             CAG_Energia CAG           1       720.0000     720.0000   
1            CAG_Potência CAG           1       720.0000     720.0000   
2  URA01_Approach Condensador           1       692.6667     692.6667   
3   URA01_Approach Evaporador           1       692.6667     692.6667   
4                   URA01_COP           1       692.6667     692.6667   

   TEMPO_MIN_H  PARTIDAS  PARADAS  OSCILACAO  INDICE_ESTABILIDADE  \
0     720.0000         0        0   642.5385               0.0000   
1     720.0000         0        0    62.2876              37.7124   
2     692.6667         0        0     1.7415              98.2585   
3     692.6667         0        0     2.0100              97.9900   
4     692.6667         0        0     0.7812              99.2188   

   INDICE_CONFIABILIDADE  
0                 0.0000  
1                37.712

In [22]:
# ==========================================================
# BLOCO 5.1.6
# ANALYTICAL CORE FINAL
# ==========================================================

import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime

print("="*100)
print("BLOCO 5.1.6 - ANALYTICAL CORE FINAL")
print("="*100)

PASTA = Path("Resultados")
PASTA.mkdir(exist_ok=True)

core = analytical_core.copy()

print(f"Registros : {len(core):,}")

# ==========================================================
# IDENTIFICADOR
# ==========================================================

core = core.reset_index(drop=True)

core["ID_ANALYTICAL_CORE"] = np.arange(
    1,
    len(core) + 1
)

# ==========================================================
# VALIDAÇÃO
# ==========================================================

campos_obrigatorios = [

    "ID_SERIE",
    "DATAHORA",
    "VALOR",
    "URA",
    "VARIAVEL"

]

core["STATUS_ANALYTICAL"] = "OK"

for coluna in campos_obrigatorios:

    core.loc[
        core[coluna].isna(),
        "STATUS_ANALYTICAL"
    ] = "INCONSISTENTE"

    # ==========================================================
# DUPLICIDADE
# ==========================================================

duplicados = core.duplicated(

    subset=[
        "ID_SERIE",
        "DATAHORA"
    ]

)

core["REGISTRO_DUPLICADO"] = duplicados

# ==========================================================
# SCORE QUALIDADE
# ==========================================================

core["SCORE_QUALIDADE"] = 100

core.loc[
    core["STATUS_ANALYTICAL"] != "OK",
    "SCORE_QUALIDADE"
] -= 40

core.loc[
    core["REGISTRO_DUPLICADO"],
    "SCORE_QUALIDADE"
] -= 30

core.loc[
    core["OUTLIER"],
    "SCORE_QUALIDADE"
] -= 10

core.loc[
    core["FLAG_ESTATISTICA"],
    "SCORE_QUALIDADE"
] -= 10

core["SCORE_QUALIDADE"] = (
    core["SCORE_QUALIDADE"]
    .clip(0, 100)
)

# ==========================================================
# SCORE GLOBAL
# ==========================================================

componentes = [
    "SCORE_QUALIDADE",
    "SCORE_OPERACIONAL",
    "SCORE_ESTABILIDADE"
]

existentes = [c for c in componentes if c in core.columns]

core["SCORE_GLOBAL"] = (
    core[existentes]
    .mean(axis=1)
    .round(2)
)

# ==========================================================
# CLASSIFICAÇÃO
# ==========================================================

core["CLASSE_GLOBAL"] = pd.cut(

    core["SCORE_GLOBAL"],

    bins=[0,40,60,80,100],

    labels=[
        "Crítica",
        "Regular",
        "Boa",
        "Excelente"
    ],

    include_lowest=True

)

# ==========================================================
# METADADOS
# ==========================================================

core["DATA_PROCESSAMENTO_FINAL"] = datetime.now()

core["VERSAO_ANALYTICAL"] = "3.0"

core["BASE_APROVADA"] = (

    (core["STATUS_ANALYTICAL"] == "OK")

    &

    (~core["REGISTRO_DUPLICADO"])

)

# ==========================================================
# AUDITORIA
# ==========================================================

print("\n" + "="*100)
print("AUDITORIA DO ANALYTICAL CORE")
print("="*100)

print(f"Registros.....................: {len(core):,}")
print(f"Registros aprovados...........: {core['BASE_APROVADA'].sum():,}")
print(f"Duplicados....................: {core['REGISTRO_DUPLICADO'].sum():,}")
print(f"Score médio...................: {core['SCORE_GLOBAL'].mean():.2f}")

print("\nDistribuição da qualidade:")

print(
    core["CLASSE_GLOBAL"]
    .value_counts(dropna=False)
)

print("\nStatus Analytical:")

print(
    core["STATUS_ANALYTICAL"]
    .value_counts(dropna=False)
)

# ==========================================================
# EXPORTAÇÃO
# ==========================================================

core.to_excel(
    PASTA / "19_ANALYTICAL_CORE_FINAL.xlsx",
    index=False
)

core.to_csv(
    PASTA / "19_ANALYTICAL_CORE_FINAL.csv",
    sep=";",
    index=False
)

core.to_parquet(
    PASTA / "19_ANALYTICAL_CORE_FINAL.parquet",
    index=False
)

analytical_core = core.copy()

print("\n" + "="*100)
print("ANALYTICAL CORE CONSOLIDADO")
print("="*100)

# ==========================================================
# EXPORTAÇÃO
# ==========================================================

core.to_excel(
    PASTA / "19_ANALYTICAL_CORE_FINAL.xlsx",
    index=False
)

core.to_csv(
    PASTA / "19_ANALYTICAL_CORE_FINAL.csv",
    sep=";",
    index=False
)

core.to_parquet(
    PASTA / "19_ANALYTICAL_CORE_FINAL.parquet",
    index=False
)

analytical_core = core.copy()

print("\n" + "="*100)
print("ANALYTICAL CORE CONSOLIDADO")
print("="*100)



BLOCO 5.1.6 - ANALYTICAL CORE FINAL
Registros : 54,921

AUDITORIA DO ANALYTICAL CORE
Registros.....................: 54,921
Registros aprovados...........: 50,312
Duplicados....................: 4,609
Score médio...................: 90.81

Distribuição da qualidade:
CLASSE_GLOBAL
Excelente    48028
Boa           6893
Regular          0
Crítica          0
Name: count, dtype: int64

Status Analytical:
STATUS_ANALYTICAL
OK    54921
Name: count, dtype: int64

ANALYTICAL CORE CONSOLIDADO

ANALYTICAL CORE CONSOLIDADO


In [23]:
# ==========================================================
# BLOCO 5.2.1
# PERFORMANCE ENGINE - INICIALIZAÇÃO
# ==========================================================

import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime

print("=" * 100)
print("BLOCO 5.2.1 - PERFORMANCE ENGINE")
print("=" * 100)

PASTA = Path("Resultados")
PASTA.mkdir(exist_ok=True)

performance_core = analytical_core.copy()

print(f"Registros carregados: {len(performance_core):,}")

# ==========================================================
# CONFIGURAÇÃO DE ENGENHARIA
# ==========================================================

ENG_PERFORMANCE = {

    "CAPACIDADE_CHILLER_TR": 300.0,

    "COP_META": 6.00,
    "COP_MINIMO": 5.50,

    "KWTR_META": 0.58,
    "KWTR_MAXIMO": 0.70,

    "APP_EVAP_MAX": 1.50,
    "APP_COND_MAX": 2.00

}

print("\nParâmetros carregados:")
for chave, valor in ENG_PERFORMANCE.items():
    print(f"{chave:<25}: {valor}")

    # ==========================================================
# VALIDAÇÃO DA ESTRUTURA
# ==========================================================

colunas_obrigatorias = [

    "URA",
    "VARIAVEL",
    "VALOR",
    "DATAHORA"

]

faltantes = [

    c for c in colunas_obrigatorias

    if c not in performance_core.columns

]

if faltantes:

    raise Exception(f"Colunas ausentes: {faltantes}")

print("\nEstrutura validada com sucesso.")

# ==========================================================
# COBERTURA DOS DADOS
# ==========================================================

cobertura = (

    performance_core

    .groupby(

        ["URA", "VARIAVEL"]

    )

    .agg(

        Registros=("VALOR", "count"),

        Primeiro=("DATAHORA", "min"),

        Ultimo=("DATAHORA", "max")

    )

    .reset_index()

)

print("\nCobertura das séries:")
print(cobertura.head(15))

# ==========================================================
# EXPORTAÇÃO
# ==========================================================

cobertura.to_excel(
    PASTA / "20_PERFORMANCE_INVENTARIO.xlsx",
    index=False
)

performance_core.to_parquet(
    PASTA / "20_PERFORMANCE_CORE.parquet",
    index=False
)

print("\nArquivos iniciais do Performance Engine gerados com sucesso.")



BLOCO 5.2.1 - PERFORMANCE ENGINE
Registros carregados: 54,921

Parâmetros carregados:
CAPACIDADE_CHILLER_TR    : 300.0
COP_META                 : 6.0
COP_MINIMO               : 5.5
KWTR_META                : 0.58
KWTR_MAXIMO              : 0.7
APP_EVAP_MAX             : 1.5
APP_COND_MAX             : 2.0

Estrutura validada com sucesso.

Cobertura das séries:
      URA              VARIAVEL  Registros            Primeiro  \
0     CAG           Energia CAG         31 2026-06-01 00:00:00   
1     CAG          Potência CAG       4295 2026-06-01 00:00:00   
2   URA01  Approach Condensador       1276 2026-06-01 23:20:00   
3   URA01   Approach Evaporador       1376 2026-06-01 23:20:00   
4   URA01                   COP        700 2026-06-01 23:20:00   
5   URA01         Carga Térmica        699 2026-06-01 23:20:00   
6   URA01        Frequência VSD       4288 2026-06-01 00:00:00   
7   URA01                 Vazão       4295 2026-06-01 00:00:00   
8   URA01                 kW/TR       4169 2

In [24]:
# ==========================================================
# BLOCO 5.2.2
# PERFORMANCE METRICS ENGINE
# ==========================================================

import pandas as pd
import numpy as np
from pathlib import Path

print("=" * 100)
print("BLOCO 5.2.2 - PERFORMANCE METRICS ENGINE")
print("=" * 100)

PASTA = Path("Resultados")
PASTA.mkdir(exist_ok=True)

base = analytical_core.copy()

# ==========================================================
# VARIÁVEIS DE INTERESSE
# ==========================================================

variaveis = [

    "COP",
    "Carga Térmica (TR)",
    "Potência CAG",
    "Potência",
    "Approach Evaporador",
    "Approach Condensador"

]

performance = base[
    base["VARIAVEL"].isin(variaveis)
].copy()

print("Registros selecionados:", len(performance))

# ==========================================================
# TABELA DE PERFORMANCE
# ==========================================================

performance_metrics = (

    performance

    .pivot_table(

        index=[

            "URA",
            "DATAHORA"

        ],

        columns="VARIAVEL",

        values="VALOR",

        aggfunc="mean"

    )

    .reset_index()

)

performance_metrics.columns.name = None

# ==========================================================
# PADRONIZAÇÃO
# ==========================================================

performance_metrics.columns = [

    c.upper()

    .replace(" ", "_")

    .replace("(", "")

    .replace(")", "")

    .replace("/", "_")

    .replace("-", "_")

    for c in performance_metrics.columns

]

# ==========================================================
# KW/TR
# ==========================================================

if (

    "POTÊNCIA" in performance_metrics.columns

    and

    "CARGA_TÉRMICA_TR" in performance_metrics.columns

):

    performance_metrics["KW_TR_CALCULADO"] = (

        performance_metrics["POTÊNCIA"]

        /

        performance_metrics["CARGA_TÉRMICA_TR"]

    )

    # ==========================================================
# % CARGA
# ==========================================================

if "CARGA_TÉRMICA_TR" in performance_metrics.columns:

    performance_metrics["PERCENTUAL_CARGA"] = (

        performance_metrics["CARGA_TÉRMICA_TR"]

        /300

    )*100

    # ==========================================================
# VALIDAÇÃO
# ==========================================================

performance_metrics["STATUS_PERFORMANCE"] = "OK"

performance_metrics.loc[

    performance_metrics.isna().any(axis=1),

    "STATUS_PERFORMANCE"

] = "INCOMPLETO"

# ==========================================================
# RESUMO
# ==========================================================

print()

print("="*100)

print("RESUMO PERFORMANCE")

print("="*100)

print()

print(

performance_metrics.describe(

    include="all"

)

)

# ==========================================================
# EXPORTAÇÃO
# ==========================================================

performance_metrics.to_excel(

    PASTA/"21_PERFORMANCE_METRICS.xlsx",

    index=False

)

performance_metrics.to_csv(

    PASTA/"21_PERFORMANCE_METRICS.csv",

    sep=";",

    index=False

)

performance_metrics.to_parquet(

    PASTA/"21_PERFORMANCE_METRICS.parquet",

    index=False

)

print()

print("="*100)
print("PERFORMANCE METRICS FINALIZADO")
print("="*100)



BLOCO 5.2.2 - PERFORMANCE METRICS ENGINE
Registros selecionados: 16815

RESUMO PERFORMANCE

         URA                       DATAHORA  APPROACH_CONDENSADOR  \
count   6946                           6946             2610.0000   
unique     4                            NaN                   NaN   
top      CAG                            NaN                   NaN   
freq    4295                            NaN                   NaN   
mean     NaN  2026-06-15 16:52:46.801036544                3.1171   
min      NaN            2026-06-01 00:00:00                0.0097   
25%      NaN            2026-06-08 07:00:00                1.8941   
50%      NaN            2026-06-15 13:55:00                3.1284   
75%      NaN            2026-06-23 07:40:00                4.0860   
max      NaN            2026-07-01 00:00:00               11.1399   
std      NaN                            NaN                1.7260   

        APPROACH_EVAPORADOR       COP  POTÊNCIA_CAG STATUS_PERFORMANCE  
count 

In [25]:
# ==========================================================
# BLOCO 5.2.3
# COP PERFORMANCE ENGINE
# ==========================================================

import pandas as pd
import numpy as np
from pathlib import Path

print("=" * 100)
print("BLOCO 5.2.3 - COP PERFORMANCE ENGINE")
print("=" * 100)

PASTA = Path("Resultados")
PASTA.mkdir(exist_ok=True)

cop_core = performance_metrics.copy()

print(f"Registros: {len(cop_core):,}")

# ==========================================================
# PARÂMETROS
# ==========================================================

COP_META = 6.00
COP_BOM = 5.50
COP_REGULAR = 5.00
COP_MINIMO = 4.50

# ==========================================================
# LOCALIZAÇÃO DA COLUNA COP
# ==========================================================

colunas_cop = [

    c

    for c in cop_core.columns

    if "COP" in c.upper()

]

if len(colunas_cop) == 0:

    raise Exception("Coluna COP não encontrada.")

COL_COP = colunas_cop[0]

print("Coluna utilizada:", COL_COP)

# ==========================================================
# EFICIÊNCIA
# ==========================================================

cop_core["EFICIENCIA_RELATIVA"] = (

    cop_core[COL_COP]

    /

    COP_META

) * 100

# ==========================================================
# DESVIO
# ==========================================================

cop_core["DESVIO_COP"] = (

    cop_core[COL_COP]

    -

    COP_META

)

# ==========================================================
# CLASSIFICAÇÃO
# ==========================================================

condicoes = [

    cop_core[COL_COP] >= COP_META,

    (cop_core[COL_COP] >= COP_BOM) &
    (cop_core[COL_COP] < COP_META),

    (cop_core[COL_COP] >= COP_REGULAR) &
    (cop_core[COL_COP] < COP_BOM),

    cop_core[COL_COP] < COP_REGULAR

]

classes = [

    "Excelente",

    "Bom",

    "Regular",

    "Crítico"

]

cop_core["CLASSE_COP"] = np.select(

    condicoes,

    classes,

    default="Sem Dados"

)

# ==========================================================
# SCORE
# ==========================================================

score = np.select(

    [

        cop_core["CLASSE_COP"]=="Excelente",

        cop_core["CLASSE_COP"]=="Bom",

        cop_core["CLASSE_COP"]=="Regular",

        cop_core["CLASSE_COP"]=="Crítico"

    ],

    [

        100,

        85,

        65,

        30

    ],

    default=np.nan

)

cop_core["SCORE_COP"] = score

# ==========================================================
# FAIXA DE CARGA
# ==========================================================

if "PERCENTUAL_CARGA" in cop_core.columns:

    cop_core["FAIXA_CARGA"] = pd.cut(

        cop_core["PERCENTUAL_CARGA"],

        bins=[

            0,

            20,

            40,

            60,

            80,

            100,

            np.inf

        ],

        labels=[

            "<20%",

            "20-40%",

            "40-60%",

            "60-80%",

            "80-100%",

            ">100%"

        ],

        include_lowest=True

    )

    # ==========================================================
# RANKING
# ==========================================================

cop_core["RANK_COP"] = (

    cop_core

    .groupby(

        "DATAHORA"

    )[COL_COP]

    .rank(

        ascending=False,

        method="dense"

    )

)

# ==========================================================
# RESUMO
# ==========================================================

print()

print("="*100)

print("COP PERFORMANCE")

print("="*100)

print()

print(

cop_core[

[
"URA",
COL_COP,
"EFICIENCIA_RELATIVA",
"CLASSE_COP",
"SCORE_COP",
"RANK_COP"

]

].head(20)

)

print()

print(

cop_core["CLASSE_COP"]

.value_counts()

)

# ==========================================================
# EXPORTAÇÃO
# ==========================================================

cop_core.to_excel(

    PASTA / "22_COP_PERFORMANCE.xlsx",

    index=False

)

cop_core.to_csv(

    PASTA / "22_COP_PERFORMANCE.csv",

    sep=";",

    index=False

)

cop_core.to_parquet(

    PASTA / "22_COP_PERFORMANCE.parquet",

    index=False

)

performance_metrics = cop_core.copy()

print()

print("="*100)
print("COP PERFORMANCE ENGINE FINALIZADO")
print("="*100)



BLOCO 5.2.3 - COP PERFORMANCE ENGINE
Registros: 6,946
Coluna utilizada: COP

COP PERFORMANCE

    URA  COP  EFICIENCIA_RELATIVA CLASSE_COP  SCORE_COP  RANK_COP
0   CAG  NaN                  NaN  Sem Dados        NaN       NaN
1   CAG  NaN                  NaN  Sem Dados        NaN       NaN
2   CAG  NaN                  NaN  Sem Dados        NaN       NaN
3   CAG  NaN                  NaN  Sem Dados        NaN       NaN
4   CAG  NaN                  NaN  Sem Dados        NaN       NaN
5   CAG  NaN                  NaN  Sem Dados        NaN       NaN
6   CAG  NaN                  NaN  Sem Dados        NaN       NaN
7   CAG  NaN                  NaN  Sem Dados        NaN       NaN
8   CAG  NaN                  NaN  Sem Dados        NaN       NaN
9   CAG  NaN                  NaN  Sem Dados        NaN       NaN
10  CAG  NaN                  NaN  Sem Dados        NaN       NaN
11  CAG  NaN                  NaN  Sem Dados        NaN       NaN
12  CAG  NaN                  NaN  Sem Dados    

In [26]:
# ==========================================================
# BLOCO 5.2.4
# COP REFERENCE CURVE ENGINE
# ==========================================================

import pandas as pd
import numpy as np
from pathlib import Path

print("=" * 100)
print("BLOCO 5.2.4 - COP REFERENCE CURVE ENGINE")
print("=" * 100)

PASTA = Path("Resultados")
PASTA.mkdir(exist_ok=True)

cop_curve = performance_metrics.copy()

print(f"Registros: {len(cop_curve):,}")

# ==========================================================
# IDENTIFICAÇÃO DA COLUNA COP
# ==========================================================

colunas_cop = [c for c in cop_curve.columns if "COP" in c.upper()]

if not colunas_cop:
    raise Exception("Nenhuma coluna de COP encontrada.")

COL_COP = colunas_cop[0]

print("Coluna COP:", COL_COP)

# ==========================================================
# CURVA DE REFERÊNCIA
# ==========================================================

def cop_referencia(percentual):

    if pd.isna(percentual):
        return np.nan

    if percentual < 20:
        return 4.80

    elif percentual < 40:
        return 5.50

    elif percentual < 60:
        return 6.00

    elif percentual < 80:
        return 6.30

    elif percentual <= 100:
        return 6.10

    return 5.80

    # ==========================================================
# LOCALIZAÇÃO DA CARGA TÉRMICA
# ==========================================================

colunas_carga = [

    c

    for c in cop_curve.columns

    if ("CARGA" in c.upper()) and ("TR" in c.upper())

]

if len(colunas_carga) > 0:

    COL_CARGA = colunas_carga[0]

    print(f"Coluna de carga encontrada: {COL_CARGA}")

    cop_curve["PERCENTUAL_CARGA"] = (

        cop_curve[COL_CARGA]

        /300

    )*100

else:

    print("Nenhuma coluna de carga térmica encontrada.")

    cop_curve["PERCENTUAL_CARGA"] = np.nan

    # ==========================================================
# COP ESPERADO
# ==========================================================

cop_curve["COP_REFERENCIA"] = (

    cop_curve["PERCENTUAL_CARGA"]

    .apply(cop_referencia)

)

# ==========================================================
# EFICIÊNCIA
# ==========================================================

cop_curve["DESVIO_COP"] = (

    cop_curve[COL_COP]

    -

    cop_curve["COP_REFERENCIA"]

)

cop_curve["DESVIO_PERCENTUAL"] = (

    100

    *

    cop_curve["DESVIO_COP"]

    /

    cop_curve["COP_REFERENCIA"]

)

cop_curve["EFICIENCIA_RELATIVA"] = (

    100

    *

    cop_curve[COL_COP]

    /

    cop_curve["COP_REFERENCIA"]

)

# ==========================================================
# CLASSIFICAÇÃO
# ==========================================================

condicoes = [

    cop_curve["EFICIENCIA_RELATIVA"] >= 100,

    cop_curve["EFICIENCIA_RELATIVA"].between(95, 100, inclusive="left"),

    cop_curve["EFICIENCIA_RELATIVA"].between(90, 95, inclusive="left"),

    cop_curve["EFICIENCIA_RELATIVA"] < 90

]

classes = [

    "Excelente",

    "Bom",

    "Regular",

    "Crítico"

]

cop_curve["CLASSE_PERFORMANCE"] = np.select(

    condicoes,

    classes,

    default="Sem Dados"

)

# ==========================================================
# SCORE
# ==========================================================

score = {

    "Excelente": 100,

    "Bom": 85,

    "Regular": 70,

    "Crítico": 40,

    "Sem Dados": np.nan

}

cop_curve["SCORE_PERFORMANCE"] = (

    cop_curve["CLASSE_PERFORMANCE"]

    .map(score)

)

# ==========================================================
# RANKING POR DATA
# ==========================================================

cop_curve["RANK_PERFORMANCE"] = (

    cop_curve

    .groupby("DATAHORA")["EFICIENCIA_RELATIVA"]

    .rank(

        ascending=False,

        method="dense"

    )

)

# ==========================================================
# AUDITORIA
# ==========================================================

print("\n" + "=" * 100)
print("COP PERFORMANCE POR CURVA")
print("=" * 100)

print()

print(

cop_curve[

[
"URA",
COL_COP,
"COP_REFERENCIA",
"EFICIENCIA_RELATIVA",
"CLASSE_PERFORMANCE",
"SCORE_PERFORMANCE"

]

].head(20)

)

print("\nDistribuição:")

print(

cop_curve["CLASSE_PERFORMANCE"]

.value_counts(dropna=False)

)

# ==========================================================
# EXPORTAÇÃO
# ==========================================================

cop_curve.to_excel(

    PASTA / "23_COP_REFERENCE_ENGINE.xlsx",

    index=False

)

cop_curve.to_csv(

    PASTA / "23_COP_REFERENCE_ENGINE.csv",

    sep=";",

    index=False

)

cop_curve.to_parquet(

    PASTA / "23_COP_REFERENCE_ENGINE.parquet",

    index=False

)

performance_metrics = cop_curve.copy()

print("\n" + "=" * 100)
print("COP REFERENCE CURVE ENGINE FINALIZADO")
print("=" * 100)

BLOCO 5.2.4 - COP REFERENCE CURVE ENGINE
Registros: 6,946
Coluna COP: COP
Nenhuma coluna de carga térmica encontrada.

COP PERFORMANCE POR CURVA

    URA  COP  COP_REFERENCIA  EFICIENCIA_RELATIVA CLASSE_PERFORMANCE  \
0   CAG  NaN             NaN                  NaN          Sem Dados   
1   CAG  NaN             NaN                  NaN          Sem Dados   
2   CAG  NaN             NaN                  NaN          Sem Dados   
3   CAG  NaN             NaN                  NaN          Sem Dados   
4   CAG  NaN             NaN                  NaN          Sem Dados   
5   CAG  NaN             NaN                  NaN          Sem Dados   
6   CAG  NaN             NaN                  NaN          Sem Dados   
7   CAG  NaN             NaN                  NaN          Sem Dados   
8   CAG  NaN             NaN                  NaN          Sem Dados   
9   CAG  NaN             NaN                  NaN          Sem Dados   
10  CAG  NaN             NaN                  NaN          Sem

In [27]:
# Todas as variáveis existentes

sorted(

    analytical_core["VARIAVEL"]

    .dropna()

    .unique()

)

['Approach Condensador',
 'Approach Evaporador',
 'COP',
 'Carga Térmica',
 'Energia CAG',
 'Frequência VSD',
 'Potência CAG',
 'Vazão',
 'kW/TR']

In [28]:
pd.DataFrame(

    sorted(

        analytical_core["VARIAVEL"]

        .dropna()

        .unique()

    ),

    columns=["VARIAVEL"]

)

,VARIAVEL
0,Approach Condensador
1,Approach Evaporador
2,COP
3,Carga Térmica
4,Energia CAG
5,Frequência VSD
6,Potência CAG
7,Vazão
8,kW/TR


In [29]:
# ==========================================================
# BLOCO 5.2.2A
# VARIABLE REGISTRY ENGINE
# ==========================================================

import pandas as pd
import numpy as np
from pathlib import Path

print("=" * 100)
print("BLOCO 5.2.2A - VARIABLE REGISTRY ENGINE")
print("=" * 100)

PASTA = Path("Resultados")
PASTA.mkdir(exist_ok=True)

registry_core = analytical_core.copy()

print(f"Registros: {len(registry_core):,}")

# ==========================================================
# DICIONÁRIO CORPORATIVO
# ==========================================================

VARIABLE_REGISTRY = {

    "COP": {
        "CANONICA": "COP",
        "UNIDADE": "-",
        "TIPO": "Performance",
        "MIN": 0.0,
        "MAX": 12.0,
        "CRITICIDADE": "Alta"
    },

    "Carga Térmica": {
        "CANONICA": "CARGA_TR",
        "UNIDADE": "TR",
        "TIPO": "Carga",
        "MIN": 0.0,
        "MAX": 300.0,
        "CRITICIDADE": "Alta"
    },

    "Potência CAG": {
        "CANONICA": "POTENCIA",
        "UNIDADE": "kW",
        "TIPO": "Energia",
        "MIN": 0.0,
        "MAX": 500.0,
        "CRITICIDADE": "Alta"
    },

    "Energia CAG": {
        "CANONICA": "ENERGIA",
        "UNIDADE": "kWh",
        "TIPO": "Energia",
        "MIN": 0.0,
        "MAX": None,
        "CRITICIDADE": "Média"
    },

    "kW/TR": {
        "CANONICA": "KW_TR",
        "UNIDADE": "kW/TR",
        "TIPO": "Performance",
        "MIN": 0.20,
        "MAX": 2.00,
        "CRITICIDADE": "Alta"
    },

    "Approach Evaporador": {
        "CANONICA": "APPROACH_EVAP",
        "UNIDADE": "°C",
        "TIPO": "Troca Térmica",
        "MIN": -5.0,
        "MAX": 15.0,
        "CRITICIDADE": "Alta"
    },

    "Approach Condensador": {
        "CANONICA": "APPROACH_COND",
        "UNIDADE": "°C",
        "TIPO": "Troca Térmica",
        "MIN": -5.0,
        "MAX": 20.0,
        "CRITICIDADE": "Alta"
    },

    "Vazão": {
        "CANONICA": "VAZAO",
        "UNIDADE": "m³/h",
        "TIPO": "Hidráulica",
        "MIN": 0.0,
        "MAX": None,
        "CRITICIDADE": "Média"
    },

    "Frequência VSD": {
        "CANONICA": "FREQUENCIA",
        "UNIDADE": "Hz",
        "TIPO": "Acionamento",
        "MIN": 0.0,
        "MAX": 60.0,
        "CRITICIDADE": "Baixa"
    }

}

# ==========================================================
# CATÁLOGO CORPORATIVO
# ==========================================================

catalogo = []

for nome, info in VARIABLE_REGISTRY.items():

    catalogo.append({

        "VARIAVEL_CANONICA": info["CANONICA"],

        "REG_VARIAVEL_ORIGINAL": nome,

        "REG_UNIDADE": info["UNIDADE"],

        "REG_TIPO": info["TIPO"],

        "REG_LIMITE_MIN": info["MIN"],

        "REG_LIMITE_MAX": info["MAX"],

        "REG_CRITICIDADE": info["CRITICIDADE"]

    })

dim_variavel_registry = pd.DataFrame(catalogo)

# ==========================================================
# PADRONIZAÇÃO
# ==========================================================

mapa = {

    k: v["CANONICA"]

    for k, v in VARIABLE_REGISTRY.items()

}

registry_core["VARIAVEL_CANONICA"] = (

    registry_core["VARIAVEL"]

    .map(mapa)

)

registry_core["VARIAVEL_CANONICA"] = (

    registry_core["VARIAVEL_CANONICA"]

    .fillna(registry_core["VARIAVEL"])

)

# ==========================================================
# METADADOS
# ==========================================================

registry_core = registry_core.merge(

    dim_variavel_registry,

    on="VARIAVEL_CANONICA",

    how="left"

)

# ==========================================================
# AUDITORIA
# ==========================================================

print("\nResumo do Registry\n")

print(

    registry_core[

        [

            "VARIAVEL",

            "VARIAVEL_CANONICA",

            "REG_TIPO",

            "REG_UNIDADE",

            "REG_CRITICIDADE"

        ]

    ]



    .drop_duplicates()

    .sort_values("VARIAVEL_CANONICA")

)

# ==========================================================
# EXPORTAÇÃO
# ==========================================================

dim_variavel_registry.to_excel(
    PASTA / "19_DIM_VARIAVEL_REGISTRY.xlsx",
    index=False
)

registry_core.to_parquet(
    PASTA / "19_ANALYTICAL_CORE_REGISTRY.parquet",
    index=False
)

registry_core.to_csv(
    PASTA / "19_ANALYTICAL_CORE_REGISTRY.csv",
    sep=";",
    index=False
)

# Atualiza a base principal
analytical_core = registry_core.copy()

print("\n" + "=" * 100)
print("VARIABLE REGISTRY ENGINE FINALIZADO")
print("=" * 100)

BLOCO 5.2.2A - VARIABLE REGISTRY ENGINE
Registros: 54,921

Resumo do Registry

                   VARIAVEL VARIAVEL_CANONICA       REG_TIPO REG_UNIDADE  \
4326   Approach Condensador     APPROACH_COND  Troca Térmica          °C   
5602    Approach Evaporador     APPROACH_EVAP  Troca Térmica          °C   
7678          Carga Térmica          CARGA_TR          Carga          TR   
6978                    COP               COP    Performance           -   
0               Energia CAG           ENERGIA        Energia         kWh   
8377         Frequência VSD        FREQUENCIA    Acionamento          Hz   
16960                 kW/TR             KW_TR    Performance       kW/TR   
31             Potência CAG          POTENCIA        Energia          kW   
12665                 Vazão             VAZAO     Hidráulica        m³/h   

      REG_CRITICIDADE  
4326             Alta  
5602             Alta  
7678             Alta  
6978             Alta  
0               Média  
8377            

In [30]:
print(analytical_core.columns.tolist())

['ID_MEDICAO', 'DATAHORA', 'URA', 'TAG', 'VARIAVEL', 'VALOR', 'UNIDADE', 'CATEGORIA', 'CIRCUITO', 'EM_OPERACAO', 'STATUS_OPERACAO', 'OUTLIER', 'STATUS_OUTLIER', 'REGISTRO_VALIDO', 'ANO', 'MES', 'DIA', 'HORA', 'MINUTO', 'MES_ANO', 'SEMANA_ANO', 'FIM_DE_SEMANA', 'HORA_DECIMAL', 'FAIXA_CARGA', 'STATUS_COP', 'STATUS_KWTR', 'STATUS_APPROACH', 'MEDIA', 'DESVIO', 'CV', 'P5', 'Q1', 'Q2', 'Q3', 'P95', 'LIM_INF', 'LIM_SUP', 'Z_SCORE', 'CLASSE_ENGENHARIA', 'NOTA_ENGENHARIA', 'FAIXA_NOTA', 'PESO', 'CRITICIDADE', 'PRIORIDADE', 'PRIORIDADE_NUMERICA', 'SCORE_MEDICAO', 'ID_ANALYTICS', 'ID_SERIE', 'ORDEM_SERIE', 'DELTA_MIN', 'INTERVALO_PADRAO', 'ERRO_INTERVALO', 'FALHA_AQUISICAO', 'QUALIDADE_TEMPORAL', 'TEMPO_ACUMULADO_MIN', 'HORAS_MONITORADAS', 'INICIO', 'FIM', 'DIAS_MONITORADOS', 'PERCENTUAL_OPERACAO', 'ROLLING_MEDIA', 'ROLLING_MEDIANA', 'ROLLING_STD', 'ROLLING_VARIANCIA', 'ROLLING_CV', 'ROLLING_MIN', 'ROLLING_MAX', 'ROLLING_RANGE', 'DELTA_VALOR', 'TENDENCIA', 'ROLLING_Z', 'LIMITE_SUP', 'LIMITE_INF',

In [31]:
# ==============================================================================
# BLOCO 5.2.3 - ANALYTICAL MART ENGINE
# ==============================================================================

import pandas as pd
import numpy as np
from pathlib import Path

print("="*100)
print("BLOCO 5.2.3 - ANALYTICAL MART ENGINE")
print("="*100)

PASTA = Path("Resultados")
PASTA.mkdir(exist_ok=True)

mart = analytical_core.copy()

print(f"Registros de entrada : {len(mart):,}")

# ==============================================================================
# VALIDAÇÃO
# ==============================================================================

colunas_obrigatorias = [

    "DATAHORA",
    "URA",
    "VARIAVEL_CANONICA",
    "VALOR"

]

faltantes = [

    c for c in colunas_obrigatorias

    if c not in mart.columns

]

if faltantes:

    raise Exception(

        f"Colunas obrigatórias ausentes: {faltantes}"

    )

print("Estrutura validada.")

# ==============================================================================
# VARIÁVEIS ANALÍTICAS
# ==============================================================================

variaveis = [

    "COP",
    "CARGA_TR",
    "POTENCIA",
    "ENERGIA",
    "KW_TR",
    "APPROACH_EVAP",
    "APPROACH_COND",
    "VAZAO",
    "FREQUENCIA"

]

mart = mart.loc[

    mart["VARIAVEL_CANONICA"]

    .isin(variaveis)

].copy()

print(f"Registros analíticos : {len(mart):,}")

# ==============================================================================
# PIVOT
# ==============================================================================

analytical_mart = (

    mart

    .pivot_table(

        index=[

            "DATAHORA",

            "URA"

        ],

        columns="VARIAVEL_CANONICA",

        values="VALOR",

        aggfunc="mean"

    )

    .reset_index()

)

analytical_mart.columns.name = None

print(f"Linhas do Mart : {len(analytical_mart):,}")
print(f"Colunas        : {len(analytical_mart.columns)}")

# ==============================================================================
# CARGA NOMINAL
# ==============================================================================

CAPACIDADE_NOMINAL = 300

if "CARGA_TR" in analytical_mart.columns:

    analytical_mart["CAPACIDADE_TR"] = CAPACIDADE_NOMINAL

    analytical_mart["PERCENTUAL_CARGA"] = (

        analytical_mart["CARGA_TR"]

        /

        CAPACIDADE_NOMINAL

    ) * 100

    # ==============================================================================
# KW/TR
# ==============================================================================

if "KW_TR" not in analytical_mart.columns:

    if (

        "POTENCIA" in analytical_mart.columns

        and

        "CARGA_TR" in analytical_mart.columns

    ):

        analytical_mart["KW_TR"] = (

            analytical_mart["POTENCIA"]

            /

            analytical_mart["CARGA_TR"]

        )

        # ==============================================================================
# FLAGS
# ==============================================================================

for coluna in [

    "COP",

    "KW_TR",

    "CARGA_TR",

    "POTENCIA",

    "APPROACH_EVAP",

    "APPROACH_COND"

]:

    analytical_mart[f"FLAG_{coluna}"] = (

        analytical_mart[coluna].notna()

        if coluna in analytical_mart.columns

        else False

    )

analytical_mart["REGISTRO_COMPLETO"] = (

    analytical_mart.filter(like="FLAG_")

    .all(axis=1)

)

# ==============================================================================
# SCORE
# ==============================================================================

flags = analytical_mart.filter(like="FLAG_")

analytical_mart["SCORE_DISPONIBILIDADE"] = (

    flags.sum(axis=1)

    /

    len(flags.columns)

) * 100

# ==============================================================================
# AUDITORIA
# ==============================================================================

print("\nResumo do Analytical Mart\n")

print(analytical_mart.info())

print()

print(analytical_mart.head())

print()

print(

analytical_mart

.describe(

include="all"

)

)

# ==============================================================================
# EXPORTAÇÃO
# ==============================================================================

analytical_mart.to_parquet(

    PASTA / "20_ANALYTICAL_MART.parquet",

    index=False

)

analytical_mart.to_excel(

    PASTA / "20_ANALYTICAL_MART.xlsx",

    index=False

)

analytical_mart.to_csv(

    PASTA / "20_ANALYTICAL_MART.csv",

    sep=";",

    index=False

)

print()

print("="*100)
print("ANALYTICAL MART ENGINE FINALIZADO")
print("="*100)



BLOCO 5.2.3 - ANALYTICAL MART ENGINE
Registros de entrada : 54,921
Estrutura validada.
Registros analíticos : 54,921
Linhas do Mart : 17,180
Colunas        : 11

Resumo do Analytical Mart

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17180 entries, 0 to 17179
Data columns (total 21 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   DATAHORA               17180 non-null  datetime64[ns]
 1   URA                    17180 non-null  object        
 2   APPROACH_COND          2610 non-null   float64       
 3   APPROACH_EVAP          2650 non-null   float64       
 4   CARGA_TR               2648 non-null   float64       
 5   COP                    2651 non-null   float64       
 6   ENERGIA                31 non-null     float64       
 7   FREQUENCIA             12864 non-null  float64       
 8   KW_TR                  9678 non-null   float64       
 9   POTENCIA               4295 non-null   float64  

In [32]:
# ==============================================================================
# BLOCO 5.2.4
# COP PERFORMANCE ENGINE
# ==============================================================================

import pandas as pd
import numpy as np
from pathlib import Path

print("="*100)
print("BLOCO 5.2.4 - COP PERFORMANCE ENGINE")
print("="*100)

PASTA = Path("Resultados")

cop_engine = analytical_mart.copy()

print(f"Registros de entrada: {len(cop_engine):,}")

# ==============================================================================
# REGISTROS VÁLIDOS
# ==============================================================================

cop_engine = cop_engine.loc[
    cop_engine["COP"].notna()
].copy()

print(f"Registros com COP: {len(cop_engine):,}")

# ==============================================================================
# CURVA DE REFERÊNCIA
# ==============================================================================

def cop_referencia(carga):

    if pd.isna(carga):
        return np.nan

    if carga < 20:
        return 2.60

    elif carga < 40:
        return 3.00

    elif carga < 60:
        return 3.50

    elif carga < 80:
        return 4.00

    else:
        return 4.50


cop_engine["COP_REFERENCIA"] = (
    cop_engine["PERCENTUAL_CARGA"]
    .apply(cop_referencia)
)

# ==============================================================================
# EFICIÊNCIA
# ==============================================================================

cop_engine["EFICIENCIA_RELATIVA"] = (
    cop_engine["COP"]
    /
    cop_engine["COP_REFERENCIA"]
) * 100

# ==============================================================================
# PERDA
# ==============================================================================

cop_engine["PERDA_COP"] = (
    cop_engine["COP_REFERENCIA"]
    -
    cop_engine["COP"]
)

cop_engine["PERDA_COP"] = (
    cop_engine["PERDA_COP"]
    .clip(lower=0)
)

# ==============================================================================
# CLASSIFICAÇÃO
# ==============================================================================

def classificar(valor):

    if pd.isna(valor):
        return "Sem dados"

    elif valor >= 95:
        return "Excelente"

    elif valor >= 85:
        return "Bom"

    elif valor >= 70:
        return "Regular"

    else:
        return "Crítico"


cop_engine["STATUS_COP_ENGINE"] = (
    cop_engine["EFICIENCIA_RELATIVA"]
    .apply(classificar)
)

# ==============================================================================
# SCORE
# ==============================================================================

cop_engine["SCORE_COP"] = (
    cop_engine["EFICIENCIA_RELATIVA"]
    .clip(0,100)
)

# ==============================================================================
# RESUMO
# ==============================================================================

print()

print(cop_engine[
[
"COP",
"COP_REFERENCIA",
"EFICIENCIA_RELATIVA",
"PERDA_COP",
"SCORE_COP"
]
].describe().round(3))

print()

print(
cop_engine["STATUS_COP_ENGINE"]
.value_counts()
)

# ==============================================================================
# EXPORTAÇÃO
# ==============================================================================

cop_engine.to_excel(

    PASTA / "21_COP_ENGINE.xlsx",

    index=False

)

cop_engine.to_csv(

    PASTA / "21_COP_ENGINE.csv",

    sep=";",

    index=False

)

cop_engine.to_parquet(

    PASTA / "21_COP_ENGINE.parquet",

    index=False

)

print()

print("="*100)
print("COP PERFORMANCE ENGINE FINALIZADO")
print("="*100)



BLOCO 5.2.4 - COP PERFORMANCE ENGINE
Registros de entrada: 17,180
Registros com COP: 2,651

            COP  COP_REFERENCIA  EFICIENCIA_RELATIVA  PERDA_COP  SCORE_COP
count 2651.0000       2648.0000            2648.0000  2648.0000  2648.0000
mean     2.6880          3.0910              85.3220     0.4390    84.1170
std      0.8480          0.3890              21.0460     0.5200    19.9230
min      0.0680          2.6000               2.6080     0.0000     2.6080
25%      1.8970          2.6000              72.2590     0.0000    72.2590
50%      3.0350          3.0000              93.9810     0.1990    93.9810
75%      3.3150          3.5000             100.4640     0.7980   100.0000
max      4.6810          4.5000             128.7870     2.5320   100.0000

STATUS_COP_ENGINE
Excelente    1229
Crítico       641
Bom           549
Regular       229
Sem dados       3
Name: count, dtype: int64

COP PERFORMANCE ENGINE FINALIZADO


In [33]:
# ==============================================================================
# BLOCO 5.2.5
# KW/TR PERFORMANCE ENGINE
# ==============================================================================

import pandas as pd
import numpy as np
from pathlib import Path

print("=" * 100)
print("BLOCO 5.2.5 - KW/TR PERFORMANCE ENGINE")
print("=" * 100)

PASTA = Path("Resultados")

kwtr_engine = analytical_mart.copy()

print(f"Registros de entrada: {len(kwtr_engine):,}")

# ==============================================================================
# REGISTROS VÁLIDOS
# ==============================================================================

kwtr_engine = kwtr_engine.loc[
    kwtr_engine["KW_TR"].notna()
].copy()

print(f"Registros válidos: {len(kwtr_engine):,}")

# ==============================================================================
# REFERÊNCIA KW/TR
# ==============================================================================

def kwtr_referencia(carga):

    if pd.isna(carga):
        return np.nan

    if carga < 20:
        return 1.20

    elif carga < 40:
        return 0.95

    elif carga < 60:
        return 0.80

    elif carga < 80:
        return 0.70

    else:
        return 0.65


kwtr_engine["KWTR_REFERENCIA"] = (
    kwtr_engine["PERCENTUAL_CARGA"]
    .apply(kwtr_referencia)
)

# ==============================================================================
# EFICIÊNCIA
# ==============================================================================

kwtr_engine["EFICIENCIA_KWTR"] = (
    kwtr_engine["KWTR_REFERENCIA"]
    /
    kwtr_engine["KW_TR"]
) * 100

# ==============================================================================
# EXCESSO DE CONSUMO
# ==============================================================================

kwtr_engine["EXCESSO_CONSUMO"] = (
    kwtr_engine["KW_TR"]
    -
    kwtr_engine["KWTR_REFERENCIA"]
)

kwtr_engine["EXCESSO_CONSUMO"] = (
    kwtr_engine["EXCESSO_CONSUMO"]
    .clip(lower=0)
)

# ==============================================================================
# CLASSIFICAÇÃO
# ==============================================================================

def classificar_kwtr(valor):

    if pd.isna(valor):
        return "Sem dados"

    elif valor >= 100:
        return "Excelente"

    elif valor >= 90:
        return "Bom"

    elif valor >= 75:
        return "Regular"

    else:
        return "Crítico"


kwtr_engine["STATUS_KWTR_ENGINE"] = (
    kwtr_engine["EFICIENCIA_KWTR"]
    .apply(classificar_kwtr)
)

# ==============================================================================
# SCORE
# ==============================================================================

kwtr_engine["SCORE_KWTR"] = (
    kwtr_engine["EFICIENCIA_KWTR"]
    .clip(0, 100)
)

# ==============================================================================
# POTENCIAL DE ECONOMIA
# ==============================================================================

if "CARGA_TR" in kwtr_engine.columns:

    kwtr_engine["POTENCIAL_ECONOMIA_KW"] = (
        kwtr_engine["EXCESSO_CONSUMO"]
        *
        kwtr_engine["CARGA_TR"]
    )

    # ==============================================================================
# RESUMO
# ==============================================================================

print()

print(
    kwtr_engine[
        [
            "KW_TR",
            "KWTR_REFERENCIA",
            "EFICIENCIA_KWTR",
            "EXCESSO_CONSUMO",
            "SCORE_KWTR",
        ]
    ].describe().round(3)
)

print()

print(
    kwtr_engine["STATUS_KWTR_ENGINE"]
    .value_counts()
)

# ==============================================================================
# EXPORTAÇÃO
# ==============================================================================

kwtr_engine.to_excel(
    PASTA / "22_KWTR_ENGINE.xlsx",
    index=False
)

kwtr_engine.to_csv(
    PASTA / "22_KWTR_ENGINE.csv",
    sep=";",
    index=False
)

kwtr_engine.to_parquet(
    PASTA / "22_KWTR_ENGINE.parquet",
    index=False
)

print()

print("=" * 100)
print("KW/TR PERFORMANCE ENGINE FINALIZADO")
print("=" * 100)



BLOCO 5.2.5 - KW/TR PERFORMANCE ENGINE
Registros de entrada: 17,180
Registros válidos: 9,678

          KW_TR  KWTR_REFERENCIA  EFICIENCIA_KWTR  EXCESSO_CONSUMO  SCORE_KWTR
count 9678.0000        2648.0000        2648.0000        2648.0000   2648.0000
mean     1.5880           0.9590          69.7920           0.6340     69.7870
std      1.3380           0.1620          15.3800           2.0740     15.3670
min      0.5920           0.6500           1.6950           0.0000      1.6950
25%      0.9790           0.8000          61.4050           0.2070     61.4050
50%      1.2550           0.9500          75.2940           0.2780     75.2940
75%      2.2990           1.2000          80.5680           0.6670     80.5680
max     70.7780           1.2000         112.3760          69.5780    100.0000

STATUS_KWTR_ENGINE
Sem dados    7030
Regular      1329
Crítico      1287
Bom            30
Excelente       2
Name: count, dtype: int64

KW/TR PERFORMANCE ENGINE FINALIZADO


In [34]:
# ==============================================================================
# BLOCO 5.2.6
# APPROACH ENGINE
# ==============================================================================

import pandas as pd
import numpy as np
from pathlib import Path

print("="*100)
print("BLOCO 5.2.6 - APPROACH ENGINE")
print("="*100)

PASTA = Path("Resultados")

approach_engine = analytical_mart.copy()

print(f"Registros de entrada: {len(approach_engine):,}")

# ==============================================================================
# REGISTROS VÁLIDOS
# ==============================================================================

approach_engine = approach_engine.loc[
    (
        approach_engine["APPROACH_EVAP"].notna()
    )
    |
    (
        approach_engine["APPROACH_COND"].notna()
    )
].copy()

print(f"Registros válidos: {len(approach_engine):,}")

# ==============================================================================
# EVAPORADOR
# ==============================================================================

def classe_evap(valor):

    if pd.isna(valor):
        return "Sem dados"

    elif valor <= 2:
        return "Excelente"

    elif valor <= 4:
        return "Bom"

    elif valor <= 6:
        return "Regular"

    else:
        return "Crítico"


approach_engine["STATUS_EVAP"] = (
    approach_engine["APPROACH_EVAP"]
    .apply(classe_evap)
)

# ==============================================================================
# CONDENSADOR
# ==============================================================================

def classe_cond(valor):

    if pd.isna(valor):
        return "Sem dados"

    elif valor <= 3:
        return "Excelente"

    elif valor <= 5:
        return "Bom"

    elif valor <= 7:
        return "Regular"

    else:
        return "Crítico"


approach_engine["STATUS_COND"] = (
    approach_engine["APPROACH_COND"]
    .apply(classe_cond)
)

# ==============================================================================
# SCORE EVAP
# ==============================================================================

def score_evap(valor):

    if pd.isna(valor):
        return np.nan

    score = 100 - (valor * 15)

    return max(min(score,100),0)


approach_engine["SCORE_EVAP"] = (
    approach_engine["APPROACH_EVAP"]
    .apply(score_evap)
)

# ==============================================================================
# SCORE COND
# ==============================================================================

def score_cond(valor):

    if pd.isna(valor):
        return np.nan

    score = 100 - (valor * 12)

    return max(min(score,100),0)


approach_engine["SCORE_COND"] = (
    approach_engine["APPROACH_COND"]
    .apply(score_cond)
)

# ==============================================================================
# THERMAL HEALTH INDEX
# ==============================================================================

approach_engine["THERMAL_HEALTH_INDEX"] = (

    approach_engine[

        [

            "SCORE_EVAP",

            "SCORE_COND"

        ]

    ]

    .mean(axis=1)

)

# ==============================================================================
# DIAGNÓSTICO
# ==============================================================================

def diagnostico(row):

    evap = row["APPROACH_EVAP"]
    cond = row["APPROACH_COND"]

    if pd.isna(evap) and pd.isna(cond):
        return "Sem dados"

    if (not pd.isna(evap)) and evap > 6:
        return "Possível incrustação no evaporador"

    if (not pd.isna(cond)) and cond > 7:
        return "Possível incrustação no condensador"

    if (
        (not pd.isna(evap))
        and
        (not pd.isna(cond))
        and
        evap <= 2
        and
        cond <= 3
    ):
        return "Troca térmica normal"

    return "Monitorar desempenho"


approach_engine["DIAGNOSTICO_TERMICO"] = (
    approach_engine.apply(diagnostico, axis=1)
)

# ==============================================================================
# RESUMO
# ==============================================================================

print()

print(

approach_engine[

[
"APPROACH_EVAP",
"APPROACH_COND",
"SCORE_EVAP",
"SCORE_COND",
"THERMAL_HEALTH_INDEX"
]

].describe().round(3)

)

print()

print(

approach_engine["DIAGNOSTICO_TERMICO"]

.value_counts()

)

# ==============================================================================
# EXPORTAÇÃO
# ==============================================================================

approach_engine.to_excel(

    PASTA / "23_APPROACH_ENGINE.xlsx",

    index=False

)

approach_engine.to_csv(

    PASTA / "23_APPROACH_ENGINE.csv",

    sep=";",

    index=False

)

approach_engine.to_parquet(

    PASTA / "23_APPROACH_ENGINE.parquet",

    index=False

)

print()

print("="*100)
print("APPROACH ENGINE FINALIZADO")
print("="*100)



BLOCO 5.2.6 - APPROACH ENGINE
Registros de entrada: 17,180
Registros válidos: 2,650

       APPROACH_EVAP  APPROACH_COND  SCORE_EVAP  SCORE_COND  \
count      2650.0000      2610.0000   2650.0000   2610.0000   
mean          2.4860         3.1170     63.4890     62.7050   
std           1.3300         1.7260     16.5490     20.3250   
min           0.4140         0.0100      0.0000      0.0000   
25%           1.8000         1.8940     61.3590     50.9680   
50%           2.0000         3.1280     70.0000     62.4600   
75%           2.5760         4.0860     73.0000     77.2710   
max          11.3770        11.1400     93.7970     99.8830   

       THERMAL_HEALTH_INDEX  
count             2650.0000  
mean                62.7790  
std                 15.8790  
min                  0.0000  
25%                 58.4030  
50%                 66.4550  
75%                 73.2710  
max                 93.7970  

DIAGNOSTICO_TERMICO
Monitorar desempenho                   1877
Troca térmic

In [35]:
# ==============================================================================
# BLOCO 5.2.7
# FLOW ENGINE
# ==============================================================================

import pandas as pd
import numpy as np
from pathlib import Path

print("="*100)
print("BLOCO 5.2.7 - FLOW ENGINE")
print("="*100)

PASTA = Path("Resultados")

flow_engine = analytical_mart.copy()

print(f"Registros de entrada: {len(flow_engine):,}")

# ==============================================================================
# REGISTROS VÁLIDOS
# ==============================================================================

flow_engine = flow_engine.loc[
    (
        flow_engine["VAZAO"].notna()
    )
    |
    (
        flow_engine["FREQUENCIA"].notna()
    )
].copy()

print(f"Registros válidos: {len(flow_engine):,}")

# ==============================================================================
# PERCENTUAL DO VSD
# ==============================================================================

FREQ_MAX = 60

flow_engine["PERCENTUAL_VSD"] = (

    flow_engine["FREQUENCIA"]

    /

    FREQ_MAX

) * 100

# ==============================================================================
# ÍNDICE HIDRÁULICO
# ==============================================================================

flow_engine["INDICE_HIDRAULICO"] = (

    flow_engine["VAZAO"]

    /

    flow_engine["CARGA_TR"]

)

# ==============================================================================
# CLASSIFICAÇÃO
# ==============================================================================

def classificar_fluxo(row):

    vazao = row["VAZAO"]
    carga = row["PERCENTUAL_CARGA"]

    if pd.isna(vazao):

        return "Sem dados"

    if pd.isna(carga):

        return "Sem dados"

    if carga < 20:

        return "Baixa carga"

    if vazao < 5:

        return "Baixa vazão"

    if vazao > 120:

        return "Excesso de vazão"

    return "Operação normal"


flow_engine["STATUS_HIDRAULICO"] = (

    flow_engine.apply(

        classificar_fluxo,

        axis=1

    )

)

# ==============================================================================
# SCORE
# ==============================================================================

def score(row):

    status = row["STATUS_HIDRAULICO"]

    tabela = {

        "Operação normal":100,

        "Baixa carga":90,

        "Baixa vazão":50,

        "Excesso de vazão":60,

        "Sem dados":np.nan

    }

    return tabela.get(status,np.nan)

flow_engine["FLOW_SCORE"] = (

    flow_engine.apply(

        score,

        axis=1

    )

)

# ==============================================================================
# DIAGNÓSTICO
# ==============================================================================

def diagnostico(row):

    status = row["STATUS_HIDRAULICO"]

    if status == "Operação normal":

        return "Sistema hidráulico operando normalmente"

    elif status == "Baixa vazão":

        return "Verificar bomba, filtros ou válvulas"

    elif status == "Excesso de vazão":

        return "Possível desperdício energético"

    elif status == "Baixa carga":

        return "Equipamento operando abaixo da carga ideal"

    return "Sem diagnóstico"


flow_engine["DIAGNOSTICO_HIDRAULICO"] = (

    flow_engine.apply(

        diagnostico,

        axis=1

    )

)

# ==============================================================================
# OPORTUNIDADE DE OTIMIZAÇÃO
# ==============================================================================

flow_engine["POTENCIAL_OTIMIZACAO"] = np.where(

    (flow_engine["PERCENTUAL_VSD"] > 80)

    &

    (flow_engine["PERCENTUAL_CARGA"] < 50),

    "Reduzir velocidade da bomba",

    "Sem oportunidade"

)

# ==============================================================================
# RESUMO
# ==============================================================================

print()

print(

flow_engine[

[
"VAZAO",
"FREQUENCIA",
"PERCENTUAL_VSD",
"FLOW_SCORE"

]

].describe().round(2)

)

print()

print(

flow_engine["STATUS_HIDRAULICO"]

.value_counts()

)

print()

print(

flow_engine["POTENCIAL_OTIMIZACAO"]

.value_counts()

)

# ==============================================================================
# EXPORTAÇÃO
# ==============================================================================

flow_engine.to_excel(

    PASTA/"24_FLOW_ENGINE.xlsx",

    index=False

)

flow_engine.to_csv(

    PASTA/"24_FLOW_ENGINE.csv",

    sep=";",

    index=False

)

flow_engine.to_parquet(

    PASTA/"24_FLOW_ENGINE.parquet",

    index=False

)

print()

print("="*100)
print("FLOW ENGINE FINALIZADO")
print("="*100)



BLOCO 5.2.7 - FLOW ENGINE
Registros de entrada: 17,180
Registros válidos: 12,885

           VAZAO  FREQUENCIA  PERCENTUAL_VSD  FLOW_SCORE
count 12885.0000  12864.0000      12864.0000   2648.0000
mean     29.9700     17.5800         29.3000     82.7800
std      44.5700     37.0900         61.8200     17.7900
min       1.0500      0.0000          0.0000     50.0000
25%       1.3200      0.0000          0.0000     60.0000
50%       9.1200      0.0000          0.0000     90.0000
75%      14.0200      0.0000          0.0000    100.0000
max     229.4800    200.0000        333.3300    100.0000

STATUS_HIDRAULICO
Sem dados           10237
Operação normal       981
Excesso de vazão      931
Baixa carga           711
Baixa vazão            25
Name: count, dtype: int64

POTENCIAL_OTIMIZACAO
Sem oportunidade               11155
Reduzir velocidade da bomba     1730
Name: count, dtype: int64

FLOW ENGINE FINALIZADO


In [36]:
# ==============================================================================
# BLOCO 5.2.8
# PERFORMANCE INDEX ENGINE
# ==============================================================================

import pandas as pd
import numpy as np
from pathlib import Path

print("="*100)
print("BLOCO 5.2.8 - PERFORMANCE INDEX ENGINE")
print("="*100)

PASTA = Path("Resultados")

performance_engine = analytical_mart.copy()

print(f"Registros: {len(performance_engine):,}")

# ==============================================================================
# COP
# ==============================================================================

performance_engine = performance_engine.merge(

    cop_engine[

        [

            "DATAHORA",
            "URA",
            "SCORE_COP"

        ]

    ],

    on=[

        "DATAHORA",
        "URA"

    ],

    how="left"

)

# ==============================================================================
# KW/TR
# ==============================================================================

performance_engine = performance_engine.merge(

    kwtr_engine[

        [

            "DATAHORA",
            "URA",
            "SCORE_KWTR"

        ]

    ],

    on=[

        "DATAHORA",
        "URA"

    ],

    how="left"

)

# ==============================================================================
# APPROACH
# ==============================================================================

performance_engine = performance_engine.merge(

    approach_engine[

        [

            "DATAHORA",
            "URA",
            "THERMAL_HEALTH_INDEX"

        ]

    ],

    on=[

        "DATAHORA",
        "URA"

    ],

    how="left"

)

# ==============================================================================
# FLOW
# ==============================================================================

performance_engine = performance_engine.merge(

    flow_engine[

        [

            "DATAHORA",
            "URA",
            "FLOW_SCORE"

        ]

    ],

    on=[

        "DATAHORA",
        "URA"

    ],

    how="left"
)

print("Scores integrados.")

# ==============================================================================
# PERFORMANCE INDEX
# ==============================================================================

pesos = {

    "SCORE_COP":0.35,

    "SCORE_KWTR":0.35,

    "THERMAL_HEALTH_INDEX":0.20,

    "FLOW_SCORE":0.10

}

performance_engine["PERFORMANCE_INDEX"] = (

      performance_engine["SCORE_COP"]*pesos["SCORE_COP"]

    + performance_engine["SCORE_KWTR"]*pesos["SCORE_KWTR"]

    + performance_engine["THERMAL_HEALTH_INDEX"]*pesos["THERMAL_HEALTH_INDEX"]

    + performance_engine["FLOW_SCORE"]*pesos["FLOW_SCORE"]

)

# ==============================================================================
# CONFIABILIDADE
# ==============================================================================

componentes = [

    "SCORE_COP",

    "SCORE_KWTR",

    "THERMAL_HEALTH_INDEX",

    "FLOW_SCORE"

]

performance_engine["CONFIDENCE_SCORE"] = (

    performance_engine[componentes]

    .notna()

    .sum(axis=1)

    /

    len(componentes)

) * 100

# ==============================================================================
# CLASSIFICAÇÃO
# ==============================================================================

def classe(score):

    if pd.isna(score):
        return "Sem dados"

    elif score >= 95:
        return "Classe A"

    elif score >= 90:
        return "Classe B"

    elif score >= 80:
        return "Classe C"

    elif score >= 70:
        return "Classe D"

    return "Classe E"

performance_engine["CLASSE_PERFORMANCE"] = (

    performance_engine["PERFORMANCE_INDEX"]

    .apply(classe)

)

# ==============================================================================
# PRIORIDADE
# ==============================================================================

def prioridade(classe):

    tabela = {

        "Classe A":"Muito Baixa",

        "Classe B":"Baixa",

        "Classe C":"Moderada",

        "Classe D":"Alta",

        "Classe E":"Crítica",

        "Sem dados":"Indefinida"

    }

    return tabela.get(classe,"Indefinida")

performance_engine["PRIORIDADE_INTERVENCAO"] = (

    performance_engine["CLASSE_PERFORMANCE"]

    .apply(prioridade)

)

# ==============================================================================
# DIAGNÓSTICO
# ==============================================================================

def diagnostico(row):

    problemas = []

    if row.get("SCORE_COP",100) < 80:
        problemas.append("COP")

    if row.get("SCORE_KWTR",100) < 80:
        problemas.append("kW/TR")

    if row.get("THERMAL_HEALTH_INDEX",100) < 80:
        problemas.append("Approach")

    if row.get("FLOW_SCORE",100) < 80:
        problemas.append("Fluxo")

    if len(problemas)==0:
        return "Operação Normal"

    return "Verificar: " + ", ".join(problemas)

performance_engine["DIAGNOSTICO_GERAL"] = (

    performance_engine.apply(

        diagnostico,

        axis=1

    )

)

# ==============================================================================
# RESUMO
# ==============================================================================

print()

print(

performance_engine[

[
"PERFORMANCE_INDEX",
"CONFIDENCE_SCORE"

]

].describe().round(2)

)

print()

print(

performance_engine["CLASSE_PERFORMANCE"]

.value_counts()

)

print()

print(

performance_engine["PRIORIDADE_INTERVENCAO"]

.value_counts()

)

# ==============================================================================
# EXPORTAÇÃO
# ==============================================================================

performance_engine.to_excel(

    PASTA/"25_PERFORMANCE_INDEX_ENGINE.xlsx",

    index=False

)

performance_engine.to_csv(

    PASTA/"25_PERFORMANCE_INDEX_ENGINE.csv",

    sep=";",

    index=False

)

performance_engine.to_parquet(

    PASTA/"25_PERFORMANCE_INDEX_ENGINE.parquet",

    index=False

)

print()

print("="*100)
print("PERFORMANCE INDEX ENGINE FINALIZADO")
print("="*100)



BLOCO 5.2.8 - PERFORMANCE INDEX ENGINE
Registros: 17,180
Scores integrados.

       PERFORMANCE_INDEX  CONFIDENCE_SCORE
count          2648.0000        17180.0000
mean             74.7000           15.4200
std              11.6900           36.1100
min              11.9500            0.0000
25%              65.2500            0.0000
50%              79.0300            0.0000
75%              83.4100            0.0000
max              93.9300          100.0000

CLASSE_PERFORMANCE
Sem dados    14532
Classe C      1148
Classe E       818
Classe D       649
Classe B        33
Name: count, dtype: int64

PRIORIDADE_INTERVENCAO
Indefinida    14532
Moderada       1148
Crítica         818
Alta            649
Baixa            33
Name: count, dtype: int64

PERFORMANCE INDEX ENGINE FINALIZADO


In [37]:
# ==============================================================================
# BLOCO 5.3.1
# HEALTH INDEX ENGINE
# ==============================================================================

import pandas as pd
import numpy as np
from pathlib import Path

print("="*100)
print("BLOCO 5.3.1 - HEALTH INDEX ENGINE")
print("="*100)

PASTA = Path("Resultados")

health_engine = performance_engine.copy()

print(f"Registros: {len(health_engine):,}")

# ==============================================================================
# SCORE DE ESTABILIDADE
# ==============================================================================

if "SCORE_ESTABILIDADE" not in health_engine.columns:

    health_engine["SCORE_ESTABILIDADE"] = 100

health_engine["SCORE_ESTABILIDADE"] = (

    health_engine["SCORE_ESTABILIDADE"]

    .fillna(100)

    .clip(0,100)

)

# ==============================================================================
# SCORE QUALIDADE
# ==============================================================================

if "SCORE_QUALIDADE" not in health_engine.columns:

    health_engine["SCORE_QUALIDADE"] = 100

health_engine["SCORE_QUALIDADE"] = (

    health_engine["SCORE_QUALIDADE"]

    .fillna(100)

    .clip(0,100)

)

# ==============================================================================
# HEALTH SCORE
# ==============================================================================

health_engine["HEALTH_SCORE"] = (

      health_engine["PERFORMANCE_INDEX"]*0.50

    + health_engine["SCORE_ESTABILIDADE"]*0.25

    + health_engine["SCORE_QUALIDADE"]*0.15

    + health_engine["CONFIDENCE_SCORE"]*0.10

)

# ==============================================================================
# CLASSE
# ==============================================================================

def classe(score):

    if pd.isna(score):

        return "Sem dados"

    if score >= 95:

        return "Excelente"

    if score >= 90:

        return "Muito Bom"

    if score >= 80:

        return "Bom"

    if score >= 70:

        return "Regular"

    if score >= 60:

        return "Ruim"

    return "Crítico"

health_engine["HEALTH_CLASS"] = (

    health_engine["HEALTH_SCORE"]

    .apply(classe)

)

# ==============================================================================
# RISCO
# ==============================================================================

risco = {

    "Excelente":"Muito Baixo",

    "Muito Bom":"Baixo",

    "Bom":"Moderado",

    "Regular":"Alto",

    "Ruim":"Muito Alto",

    "Crítico":"Crítico",

    "Sem dados":"Indefinido"

}

health_engine["RISK_LEVEL"] = (

    health_engine["HEALTH_CLASS"]

    .map(risco)

)

# ==============================================================================
# DIAGNÓSTICO
# ==============================================================================

def diagnostico(row):

    score = row["HEALTH_SCORE"]

    if pd.isna(score):
        return "Sem informações"

    if score >= 95:
        return "Equipamento em excelente condição"

    if score >= 90:
        return "Equipamento saudável"

    if score >= 80:
        return "Monitoramento recomendado"

    if score >= 70:
        return "Planejar inspeção"

    if score >= 60:
        return "Programar manutenção"

    return "Intervenção imediata"

health_engine["HEALTH_DIAGNOSIS"] = (

    health_engine.apply(

        diagnostico,

        axis=1

    )

)

# ==============================================================================
# ESTATÍSTICAS
# ==============================================================================

print()

print(

health_engine[

[
"HEALTH_SCORE",
"PERFORMANCE_INDEX",
"CONFIDENCE_SCORE"

]

].describe().round(2)

)

print()

print(

health_engine["HEALTH_CLASS"]

.value_counts()

)

print()

print(

health_engine["RISK_LEVEL"]

.value_counts()

)

# ==============================================================================
# EXPORTAÇÃO
# ==============================================================================

health_engine.to_excel(

    PASTA/"26_HEALTH_INDEX_ENGINE.xlsx",

    index=False

)

health_engine.to_csv(

    PASTA/"26_HEALTH_INDEX_ENGINE.csv",

    sep=";",

    index=False

)

health_engine.to_parquet(

    PASTA/"26_HEALTH_INDEX_ENGINE.parquet",

    index=False

)

print()

print("="*100)
print("HEALTH INDEX ENGINE FINALIZADO")
print("="*100)



BLOCO 5.3.1 - HEALTH INDEX ENGINE
Registros: 17,180

       HEALTH_SCORE  PERFORMANCE_INDEX  CONFIDENCE_SCORE
count     2648.0000          2648.0000        17180.0000
mean        87.3500            74.7000           15.4200
std          5.8400            11.6900           36.1100
min         55.9700            11.9500            0.0000
25%         82.6300            65.2500            0.0000
50%         89.5200            79.0300            0.0000
75%         91.7000            83.4100            0.0000
max         96.9700            93.9300          100.0000

HEALTH_CLASS
Sem dados    14532
Muito Bom     1148
Bom           1073
Regular        385
Excelente       33
Ruim             8
Crítico          1
Name: count, dtype: int64

RISK_LEVEL
Indefinido     14532
Baixo           1148
Moderado        1073
Alto             385
Muito Baixo       33
Muito Alto         8
Crítico            1
Name: count, dtype: int64

HEALTH INDEX ENGINE FINALIZADO


In [38]:
# ==============================================================================
# BLOCO 5.3.2
# DEGRADATION ENGINE
# ==============================================================================

import pandas as pd
import numpy as np
from pathlib import Path

print("="*100)
print("BLOCO 5.3.2 - DEGRADATION ENGINE")
print("="*100)

PASTA = Path("Resultados")

degradation_engine = health_engine.copy()

print(f"Registros: {len(degradation_engine):,}")

# ==============================================================================
# ORDENAÇÃO
# ==============================================================================

degradation_engine = (

    degradation_engine

    .sort_values(

        ["URA","DATAHORA"]

    )

    .reset_index(drop=True)

)

# ==============================================================================
# DELTA HEALTH
# ==============================================================================

degradation_engine["DELTA_HEALTH"] = (

    degradation_engine

    .groupby("URA")["HEALTH_SCORE"]

    .diff()

)

# ==============================================================================
# VELOCIDADE
# ==============================================================================

degradation_engine["VELOCIDADE_DEGRADACAO"] = (

    -degradation_engine["DELTA_HEALTH"]

)

degradation_engine["VELOCIDADE_DEGRADACAO"] = (

    degradation_engine["VELOCIDADE_DEGRADACAO"]

    .clip(lower=0)

)

# ==============================================================================
# DEGRADAÇÃO ACUMULADA
# ==============================================================================

degradation_engine["DEGRADACAO_ACUMULADA"] = (

    degradation_engine

    .groupby("URA")["VELOCIDADE_DEGRADACAO"]

    .cumsum()

)

# ==============================================================================
# MÉDIA MÓVEL
# ==============================================================================

degradation_engine["TREND_HEALTH"] = (

    degradation_engine

    .groupby("URA")["HEALTH_SCORE"]

    .transform(

        lambda x:

        x.rolling(

            24,

            min_periods=1

        ).mean()

    )

)

# ==============================================================================
# DETERIORATION INDEX
# ==============================================================================

degradation_engine["DETERIORATION_INDEX"] = (

    degradation_engine["DEGRADACAO_ACUMULADA"]

    /

    degradation_engine["HEALTH_SCORE"]

) * 100

degradation_engine["DETERIORATION_INDEX"] = (

    degradation_engine["DETERIORATION_INDEX"]

    .replace(

        [np.inf,-np.inf],

        np.nan

    )

)

# ==============================================================================
# CLASSIFICAÇÃO
# ==============================================================================

def classe(valor):

    if pd.isna(valor):

        return "Sem dados"

    if valor < 5:

        return "Estável"

    if valor < 10:

        return "Leve"

    if valor < 20:

        return "Moderada"

    if valor < 35:

        return "Alta"

    return "Crítica"

degradation_engine["DEGRADATION_CLASS"] = (

    degradation_engine["DETERIORATION_INDEX"]

    .apply(classe)

)

# ==============================================================================
# ALERTA
# ==============================================================================

def alerta(classe):

    tabela = {

        "Estável":"Sem alerta",

        "Leve":"Monitorar",

        "Moderada":"Inspecionar",

        "Alta":"Planejar manutenção",

        "Crítica":"Intervenção urgente",

        "Sem dados":"Indefinido"

    }

    return tabela.get(classe)

degradation_engine["DEGRADATION_ALERT"] = (

    degradation_engine["DEGRADATION_CLASS"]

    .map(alerta)

)

# ==============================================================================
# RESUMO
# ==============================================================================

print()

print(

degradation_engine[

[
"HEALTH_SCORE",
"VELOCIDADE_DEGRADACAO",
"DETERIORATION_INDEX"

]

].describe().round(2)

)

print()

print(

degradation_engine["DEGRADATION_CLASS"]

.value_counts()

)

print()

print(

degradation_engine["DEGRADATION_ALERT"]

.value_counts()

)

# ==============================================================================
# EXPORTAÇÃO
# ==============================================================================

degradation_engine.to_excel(

    PASTA/"27_DEGRADATION_ENGINE.xlsx",

    index=False

)

degradation_engine.to_csv(

    PASTA/"27_DEGRADATION_ENGINE.csv",

    sep=";",

    index=False

)

degradation_engine.to_parquet(

    PASTA/"27_DEGRADATION_ENGINE.parquet",

    index=False

)

print()

print("="*100)
print("DEGRADATION ENGINE FINALIZADO")
print("="*100)

BLOCO 5.3.2 - DEGRADATION ENGINE
Registros: 17,180

       HEALTH_SCORE  VELOCIDADE_DEGRADACAO  DETERIORATION_INDEX
count     2648.0000              2532.0000            2532.0000
mean        87.3500                 0.9500             887.8900
std          5.8400                 1.6500             608.3700
min         55.9700                 0.0000               0.0000
25%         82.6300                 0.0000             350.5000
50%         89.5200                 0.0400             764.9400
75%         91.7000                 1.2800            1356.9000
max         96.9700                12.8400            2369.3800

DEGRADATION_CLASS
Sem dados    14648
Crítica       2434
Alta            61
Moderada        18
Leve            11
Estável          8
Name: count, dtype: int64

DEGRADATION_ALERT
Indefinido             14648
Intervenção urgente     2434
Planejar manutenção       61
Inspecionar               18
Monitorar                 11
Sem alerta                 8
Name: count, dtype: 

In [39]:
# ==============================================================================
# BLOCO 5.3.3
# ANOMALY DETECTION ENGINE
# ==============================================================================

import pandas as pd
import numpy as np
from pathlib import Path

print("="*100)
print("BLOCO 5.3.3 - ANOMALY DETECTION ENGINE")
print("="*100)

PASTA = Path("Resultados")

anomaly_engine = degradation_engine.copy()

print(f"Registros: {len(anomaly_engine):,}")

# ==============================================================================
# SCORE ESTATÍSTICO
# ==============================================================================

if "ROLLING_Z" not in anomaly_engine.columns:

    anomaly_engine["ROLLING_Z"] = 0

anomaly_engine["ANOMALIA_Z"] = (

    anomaly_engine["ROLLING_Z"]

    .abs()

    > 3

)

# ==============================================================================
# PERFORMANCE
# ==============================================================================

anomaly_engine["ANOMALIA_PERFORMANCE"] = (

    anomaly_engine["PERFORMANCE_INDEX"]

    <

    70

)

# ==============================================================================
# HEALTH
# ==============================================================================

anomaly_engine["ANOMALIA_HEALTH"] = (

    anomaly_engine["HEALTH_SCORE"]

    <

    70

)

# ==============================================================================
# DEGRADAÇÃO
# ==============================================================================

anomaly_engine["ANOMALIA_DEGRADACAO"] = (

    anomaly_engine["DETERIORATION_INDEX"]

    >

    20

)

# ==============================================================================
# ANOMALIA OPERACIONAL
# ==============================================================================

# A operação é considerada anômala quando existe
# baixa confiança na medição ou baixa disponibilidade.

cond1 = anomaly_engine["CONFIDENCE_SCORE"] < 75

if "SCORE_DISPONIBILIDADE" in anomaly_engine.columns:

    cond2 = anomaly_engine["SCORE_DISPONIBILIDADE"] < 60

else:

    cond2 = False

anomaly_engine["ANOMALIA_OPERACIONAL"] = (

    cond1 | cond2

)

# ==============================================================================
# SCORE
# ==============================================================================

anomalias = [

    "ANOMALIA_Z",

    "ANOMALIA_PERFORMANCE",

    "ANOMALIA_HEALTH",

    "ANOMALIA_DEGRADACAO",

    "ANOMALIA_OPERACIONAL"

]

anomaly_engine["ANOMALY_SCORE"] = (

    anomaly_engine[anomalias]

    .sum(axis=1)

)

# ==============================================================================
# SEVERIDADE
# ==============================================================================

def severidade(score):

    if score == 0:

        return "Normal"

    if score == 1:

        return "Baixa"

    if score == 2:

        return "Moderada"

    if score == 3:

        return "Alta"

    return "Crítica"

anomaly_engine["ANOMALY_SEVERITY"] = (

    anomaly_engine["ANOMALY_SCORE"]

    .apply(severidade)

)

# ==============================================================================
# DIAGNÓSTICO
# ==============================================================================

def diagnostico(row):

    eventos = []

    if row["ANOMALIA_Z"]:

        eventos.append("Desvio Estatístico")

    if row["ANOMALIA_PERFORMANCE"]:

        eventos.append("Performance")

    if row["ANOMALIA_HEALTH"]:

        eventos.append("Health")

    if row["ANOMALIA_DEGRADACAO"]:

        eventos.append("Degradação")

    if row["ANOMALIA_OPERACIONAL"]:

        eventos.append("Disponibilidade/Confiabilidade")

    if len(eventos)==0:

        return "Operação Normal"

    return ", ".join(eventos)

anomaly_engine["ANOMALY_DESCRIPTION"] = (

    anomaly_engine.apply(

        diagnostico,

        axis=1

    )

)

BLOCO 5.3.3 - ANOMALY DETECTION ENGINE
Registros: 17,180


In [40]:
# ==============================================================================
# BLOCO 5.3.4.1
# FAILURE PROBABILITY ENGINE
# ==============================================================================

import pandas as pd
import numpy as np
from pathlib import Path

print("="*100)
print("BLOCO 5.3.4.1 - FAILURE PROBABILITY ENGINE")
print("="*100)

PASTA = Path("Resultados")

failure_engine = anomaly_engine.copy()

print(f"Registros: {len(failure_engine):,}")

# ==============================================================================
# VALIDAÇÃO DAS COLUNAS
# ==============================================================================

campos = {

    "HEALTH_SCORE":100,

    "PERFORMANCE_INDEX":100,

    "ANOMALY_SCORE":0,

    "DETERIORATION_INDEX":0,

    "CONFIDENCE_SCORE":100

}

for coluna, valor in campos.items():

    if coluna not in failure_engine.columns:

        print(f"Criando coluna {coluna}")

        failure_engine[coluna] = valor

failure_engine["HEALTH_SCORE"] = failure_engine["HEALTH_SCORE"].fillna(100)
failure_engine["PERFORMANCE_INDEX"] = failure_engine["PERFORMANCE_INDEX"].fillna(100)
failure_engine["ANOMALY_SCORE"] = failure_engine["ANOMALY_SCORE"].fillna(0)
failure_engine["DETERIORATION_INDEX"] = failure_engine["DETERIORATION_INDEX"].fillna(0)
failure_engine["CONFIDENCE_SCORE"] = failure_engine["CONFIDENCE_SCORE"].fillna(100)

# ==============================================================================
# NORMALIZAÇÃO
# ==============================================================================

failure_engine["N_HEALTH"] = (

    100 - failure_engine["HEALTH_SCORE"]

).clip(0,100)

failure_engine["N_PERFORMANCE"] = (

    100 - failure_engine["PERFORMANCE_INDEX"]

).clip(0,100)

failure_engine["N_CONFIDENCE"] = (

    100 - failure_engine["CONFIDENCE_SCORE"]

).clip(0,100)

failure_engine["N_DEGRADATION"] = (

    failure_engine["DETERIORATION_INDEX"]

).clip(0,100)

failure_engine["N_ANOMALY"] = (

    failure_engine["ANOMALY_SCORE"]*20

).clip(0,100)

# ==============================================================================
# FAILURE SCORE
# ==============================================================================

failure_engine["FAILURE_SCORE"] = (

      failure_engine["N_HEALTH"]*0.35

    + failure_engine["N_PERFORMANCE"]*0.25

    + failure_engine["N_DEGRADATION"]*0.20

    + failure_engine["N_ANOMALY"]*0.15

    + failure_engine["N_CONFIDENCE"]*0.05

)

failure_engine["FAILURE_SCORE"] = (

    failure_engine["FAILURE_SCORE"]

    .clip(0,100)

)

# ==============================================================================
# PROBABILITY OF FAILURE
# ==============================================================================

failure_engine["POF"] = (

    failure_engine["FAILURE_SCORE"]/100

)

failure_engine["POF_PERCENTUAL"] = (

    failure_engine["POF"]*100

).round(2)

# ==============================================================================
# CLASSE
# ==============================================================================

def classe(p):

    if p < 0.10:

        return "Muito Baixa"

    elif p < 0.25:

        return "Baixa"

    elif p < 0.45:

        return "Moderada"

    elif p < 0.65:

        return "Alta"

    else:

        return "Muito Alta"

failure_engine["POF_CLASS"] = (

    failure_engine["POF"]

    .apply(classe)

)

# ==============================================================================
# RELIABILITY
# ==============================================================================

failure_engine["RELIABILITY_INDEX"] = (

    100

    -

    failure_engine["FAILURE_SCORE"]

)

failure_engine["RELIABILITY_INDEX"] = (

    failure_engine["RELIABILITY_INDEX"]

    .clip(0,100)

)

# ==============================================================================
# STATUS DO ATIVO
# ==============================================================================

def status(row):

    if row["POF"] < 0.10:

        return "Excelente"

    elif row["POF"] < 0.25:

        return "Operação Normal"

    elif row["POF"] < 0.45:

        return "Monitoramento"

    elif row["POF"] < 0.65:

        return "Atenção"

    return "Intervenção"

failure_engine["ASSET_STATUS"] = (

    failure_engine.apply(

        status,

        axis=1

    )

)

# ==============================================================================
# ESTATÍSTICAS
# ==============================================================================

print()

print(

failure_engine[

[
"FAILURE_SCORE",
"POF_PERCENTUAL",
"RELIABILITY_INDEX"

]

].describe().round(2)

)

print()

print(

failure_engine["POF_CLASS"]

.value_counts()

)

print()

print(

failure_engine["ASSET_STATUS"]

.value_counts()

)

# ==============================================================================
# EXPORTAÇÃO
# ==============================================================================

failure_engine.to_excel(

    PASTA/"29_FAILURE_PROBABILITY_ENGINE.xlsx",

    index=False

)

failure_engine.to_csv(

    PASTA/"29_FAILURE_PROBABILITY_ENGINE.csv",

    sep=";",

    index=False

)

failure_engine.to_parquet(

    PASTA/"29_FAILURE_PROBABILITY_ENGINE.parquet",

    index=False

)

print()

print("="*100)
print("FAILURE PROBABILITY ENGINE FINALIZADO")
print("="*100)

BLOCO 5.3.4.1 - FAILURE PROBABILITY ENGINE
Registros: 17,180

       FAILURE_SCORE  POF_PERCENTUAL  RELIABILITY_INDEX
count     17180.0000      17180.0000         17180.0000
mean         11.8100         11.8100            88.1900
std           9.4500          9.4500             9.4500
min           2.7600          2.7600            48.8800
25%           8.0000          8.0000            92.0000
50%           8.0000          8.0000            92.0000
75%           8.0000          8.0000            92.0000
max          51.1200         51.1200            97.2400

POF_CLASS
Muito Baixa    14564
Moderada        2227
Baixa            208
Alta             181
Name: count, dtype: int64

ASSET_STATUS
Excelente          14564
Monitoramento       2227
Operação Normal      208
Atenção              181
Name: count, dtype: int64

FAILURE PROBABILITY ENGINE FINALIZADO


In [41]:
# ==============================================================================
# BLOCO 5.3.4.2
# RISK SCORE ENGINE
# ==============================================================================

import pandas as pd
import numpy as np
from pathlib import Path

print("="*100)
print("BLOCO 5.3.4.2 - RISK SCORE ENGINE")
print("="*100)

PASTA = Path("Resultados")

risk_engine = failure_engine.copy()

print(f"Registros: {len(risk_engine):,}")

# ==============================================================================
# VALIDAÇÃO
# ==============================================================================

campos = {
    "POF_PERCENTUAL": 0,
    "CRITICIDADE": "Média",
    "PERCENTUAL_CARGA": 0,
    "ANOMALY_SCORE": 0
}

for coluna, valor in campos.items():

    if coluna not in risk_engine.columns:
        print(f"Criando coluna: {coluna}")
        risk_engine[coluna] = valor

risk_engine["POF_PERCENTUAL"] = risk_engine["POF_PERCENTUAL"].fillna(0)
risk_engine["PERCENTUAL_CARGA"] = risk_engine["PERCENTUAL_CARGA"].fillna(0)
risk_engine["ANOMALY_SCORE"] = risk_engine["ANOMALY_SCORE"].fillna(0)
risk_engine["CRITICIDADE"] = risk_engine["CRITICIDADE"].fillna("Média")

# ==============================================================================
# PESO DA CRITICIDADE
# ==============================================================================

peso = {

    "Baixa":1,

    "Média":2,

    "Alta":3,

    "Crítica":4

}

risk_engine["PESO_CRITICIDADE"] = (

    risk_engine["CRITICIDADE"]

    .map(peso)

    .fillna(2)

)

# ==============================================================================
# IMPACTO OPERACIONAL
# ==============================================================================

risk_engine["IMPACTO_OPERACIONAL"] = (

      risk_engine["PERCENTUAL_CARGA"]*0.60

    + risk_engine["PESO_CRITICIDADE"]*10

)

risk_engine["IMPACTO_OPERACIONAL"] = (

    risk_engine["IMPACTO_OPERACIONAL"]

    .clip(0,100)

)

# ==============================================================================
# RISK SCORE
# ==============================================================================

risk_engine["RISK_SCORE"] = (

      risk_engine["POF_PERCENTUAL"]*0.60

    + risk_engine["IMPACTO_OPERACIONAL"]*0.25

    + risk_engine["ANOMALY_SCORE"]*5*0.15

)

risk_engine["RISK_SCORE"] = (

    risk_engine["RISK_SCORE"]

    .clip(0,100)

)

# ==============================================================================
# CLASSIFICAÇÃO
# ==============================================================================

def classe(score):

    if score < 20:

        return "Muito Baixo"

    elif score < 40:

        return "Baixo"

    elif score < 60:

        return "Moderado"

    elif score < 80:

        return "Alto"

    return "Crítico"

risk_engine["RISK_CLASS"] = (

    risk_engine["RISK_SCORE"]

    .apply(classe)

)

# ==============================================================================
# PRIORIDADE
# ==============================================================================

prioridade = {

    "Muito Baixo":"Nenhuma",

    "Baixo":"Planejada",

    "Moderado":"Programar",

    "Alto":"Prioritária",

    "Crítico":"Imediata"

}

risk_engine["MAINTENANCE_PRIORITY"] = (

    risk_engine["RISK_CLASS"]

    .map(prioridade)

)

# ==============================================================================
# RANKING
# ==============================================================================

risk_engine["RISK_RANKING"] = (

    risk_engine["RISK_SCORE"]

    .rank(

        ascending=False,

        method="dense"

    )

    .astype(int)

)

# ==============================================================================
# ESTATÍSTICAS
# ==============================================================================

print()

print(

risk_engine[

[
"RISK_SCORE",

"IMPACTO_OPERACIONAL"

]

].describe().round(2)

)

print()

print(

risk_engine["RISK_CLASS"]

.value_counts()

)

print()

print(

risk_engine["MAINTENANCE_PRIORITY"]

.value_counts()

)

# ==============================================================================
# EXPORTAÇÃO
# ==============================================================================

risk_engine.to_excel(

    PASTA/"30_RISK_SCORE_ENGINE.xlsx",

    index=False

)

risk_engine.to_csv(

    PASTA/"30_RISK_SCORE_ENGINE.csv",

    sep=";",

    index=False

)

risk_engine.to_parquet(

    PASTA/"30_RISK_SCORE_ENGINE.parquet",

    index=False

)

print()

print("="*100)
print("RISK SCORE ENGINE FINALIZADO")
print("="*100)



BLOCO 5.3.4.2 - RISK SCORE ENGINE
Registros: 17,180
Criando coluna: CRITICIDADE

       RISK_SCORE  IMPACTO_OPERACIONAL
count  17180.0000           17180.0000
mean      13.6300              23.0500
std        7.4000               8.0100
min        9.8000              20.0000
25%       10.5500              20.0000
50%       10.5500              20.0000
75%       10.5500              20.0000
max       44.7600              75.1800

RISK_CLASS
Muito Baixo    14611
Baixo           2542
Moderado          27
Name: count, dtype: int64

MAINTENANCE_PRIORITY
Nenhuma      14611
Planejada     2542
Programar       27
Name: count, dtype: int64

RISK SCORE ENGINE FINALIZADO


In [42]:
# ==============================================================================
# BLOCO 5.3.4.3
# REMAINING USEFUL LIFE (RUL) ENGINE
# ==============================================================================

import pandas as pd
import numpy as np
from pathlib import Path

print("="*100)
print("BLOCO 5.3.4.3 - REMAINING USEFUL LIFE ENGINE")
print("="*100)

PASTA = Path("Resultados")

rul_engine = risk_engine.copy()

print(f"Registros: {len(rul_engine):,}")

# ==============================================================================
# VALIDAÇÃO
# ==============================================================================

campos = {

    "HEALTH_SCORE":100,

    "RISK_SCORE":0,

    "POF_PERCENTUAL":0,

    "DETERIORATION_INDEX":0

}

for coluna, valor in campos.items():

    if coluna not in rul_engine.columns:

        print(f"Criando coluna {coluna}")

        rul_engine[coluna] = valor

    rul_engine[coluna] = rul_engine[coluna].fillna(valor)

    # ==============================================================================
# LIFE INDEX
# ==============================================================================

rul_engine["LIFE_INDEX"] = (

      rul_engine["HEALTH_SCORE"]*0.40

    + (100-rul_engine["RISK_SCORE"])*0.30

    + (100-rul_engine["POF_PERCENTUAL"])*0.20

    + (100-rul_engine["DETERIORATION_INDEX"].clip(0,100))*0.10

)

rul_engine["LIFE_INDEX"] = (

    rul_engine["LIFE_INDEX"]

    .clip(0,100)

)

# ==============================================================================
# RUL
# ==============================================================================

VIDA_UTIL_REFERENCIA = 365

rul_engine["RUL_DIAS"] = (

    VIDA_UTIL_REFERENCIA

    *

    rul_engine["LIFE_INDEX"]/100

)

rul_engine["RUL_HORAS"] = (

    rul_engine["RUL_DIAS"]

    *24

)

# ==============================================================================
# CLASSIFICAÇÃO
# ==============================================================================

def classe_rul(dias):

    if dias >= 300:

        return "Excelente"

    elif dias >= 240:

        return "Boa"

    elif dias >= 180:

        return "Atenção"

    elif dias >= 90:

        return "Planejar"

    elif dias >= 30:

        return "Crítica"

    return "Intervenção"

rul_engine["RUL_CLASS"] = (

    rul_engine["RUL_DIAS"]

    .apply(classe_rul)

)

# ==============================================================================
# PRIORIDADE
# ==============================================================================

prioridade = {

    "Excelente":"Baixa",

    "Boa":"Baixa",

    "Atenção":"Média",

    "Planejar":"Alta",

    "Crítica":"Muito Alta",

    "Intervenção":"Imediata"

}

rul_engine["RUL_PRIORITY"] = (

    rul_engine["RUL_CLASS"]

    .map(prioridade)

)

# ==============================================================================
# TENDÊNCIA
# ==============================================================================

rul_engine["RUL_TENDENCIA"] = (

    rul_engine

    .groupby("URA")["RUL_DIAS"]

    .transform(

        lambda x:

        x.diff()

    )

)

# ==============================================================================
# ESTATÍSTICAS
# ==============================================================================

print()

print(

rul_engine[

[
"LIFE_INDEX",
"RUL_DIAS",
"RUL_HORAS"

]

].describe().round(2)

)

print()

print(

rul_engine["RUL_CLASS"]

.value_counts()

)

print()

print(

rul_engine["RUL_PRIORITY"]

.value_counts()

)

# ==============================================================================
# EXPORTAÇÃO
# ==============================================================================

rul_engine.to_excel(

    PASTA/"31_RUL_ENGINE.xlsx",

    index=False

)

rul_engine.to_csv(

    PASTA/"31_RUL_ENGINE.csv",

    sep=";",

    index=False

)

rul_engine.to_parquet(

    PASTA/"31_RUL_ENGINE.parquet",

    index=False

)

print()

print("="*100)
print("RUL ENGINE FINALIZADO")
print("="*100)



BLOCO 5.3.4.3 - REMAINING USEFUL LIFE ENGINE
Registros: 17,180

       LIFE_INDEX   RUL_DIAS  RUL_HORAS
count  17180.0000 17180.0000 17180.0000
mean      91.3700   333.4900  8003.7300
std        9.3600    34.1700   820.0200
min       55.7800   203.5900  4886.1700
25%       95.2400   347.6100  8342.5900
50%       95.2400   347.6100  8342.5900
75%       95.2400   347.6100  8342.5900
max       95.7100   349.3400  8384.2000

RUL_CLASS
Excelente    14660
Boa           1862
Atenção        658
Name: count, dtype: int64

RUL_PRIORITY
Baixa    16522
Média      658
Name: count, dtype: int64

RUL ENGINE FINALIZADO


In [43]:
# ==============================================================================
# BLOCO 5.3.4.4
# MAINTENANCE PRIORITY ENGINE
# ==============================================================================

import pandas as pd
import numpy as np
from pathlib import Path

print("="*100)
print("BLOCO 5.3.4.4 - MAINTENANCE PRIORITY ENGINE")
print("="*100)

PASTA = Path("Resultados")

maintenance_engine = rul_engine.copy()

print(f"Registros: {len(maintenance_engine):,}")

# ==============================================================================
# VALIDAÇÃO
# ==============================================================================

campos = {

    "RISK_SCORE":0,

    "POF_PERCENTUAL":0,

    "HEALTH_SCORE":100,

    "RUL_DIAS":365,

    "ANOMALY_SCORE":0,

    "DETERIORATION_INDEX":0

}

for coluna, valor in campos.items():

    if coluna not in maintenance_engine.columns:

        print(f"Criando coluna: {coluna}")

        maintenance_engine[coluna] = valor

    maintenance_engine[coluna] = maintenance_engine[coluna].fillna(valor)

    # ==============================================================================
# NORMALIZAÇÃO
# ==============================================================================

maintenance_engine["N_RISK"] = (

    maintenance_engine["RISK_SCORE"]

).clip(0,100)

maintenance_engine["N_POF"] = (

    maintenance_engine["POF_PERCENTUAL"]

).clip(0,100)

maintenance_engine["N_HEALTH"] = (

    100 -

    maintenance_engine["HEALTH_SCORE"]

).clip(0,100)

maintenance_engine["N_RUL"] = (

    100 -

    (maintenance_engine["RUL_DIAS"]/365*100)

).clip(0,100)

maintenance_engine["N_DEGRADATION"] = (

    maintenance_engine["DETERIORATION_INDEX"]

).clip(0,100)

maintenance_engine["N_ANOMALY"] = (

    maintenance_engine["ANOMALY_SCORE"]*20

).clip(0,100)

# ==============================================================================
# PRIORITY INDEX
# ==============================================================================

maintenance_engine["MAINTENANCE_PRIORITY_INDEX"] = (

      maintenance_engine["N_RISK"]*0.30

    + maintenance_engine["N_POF"]*0.20

    + maintenance_engine["N_HEALTH"]*0.20

    + maintenance_engine["N_RUL"]*0.15

    + maintenance_engine["N_DEGRADATION"]*0.10

    + maintenance_engine["N_ANOMALY"]*0.05

)

maintenance_engine["MAINTENANCE_PRIORITY_INDEX"] = (

    maintenance_engine["MAINTENANCE_PRIORITY_INDEX"]

    .clip(0,100)

)

# ==============================================================================
# CLASSIFICAÇÃO
# ==============================================================================

def classe(score):

    if score < 20:
        return "Muito Baixa"

    elif score < 40:
        return "Baixa"

    elif score < 60:
        return "Moderada"

    elif score < 80:
        return "Alta"

    return "Crítica"

maintenance_engine["MAINTENANCE_CLASS"] = (

    maintenance_engine["MAINTENANCE_PRIORITY_INDEX"]

    .apply(classe)

)

# ==============================================================================
# TIPO
# ==============================================================================

tipo = {

    "Muito Baixa":"Operação Normal",

    "Baixa":"Preventiva",

    "Moderada":"Preventiva Planejada",

    "Alta":"Preditiva",

    "Crítica":"Corretiva Programada"

}

maintenance_engine["TIPO_MANUTENCAO"] = (

    maintenance_engine["MAINTENANCE_CLASS"]

    .map(tipo)

)

# ==============================================================================
# JANELA
# ==============================================================================

def janela(classe):

    if classe=="Muito Baixa":
        return "Até 180 dias"

    elif classe=="Baixa":
        return "Até 90 dias"

    elif classe=="Moderada":
        return "Até 30 dias"

    elif classe=="Alta":
        return "Até 7 dias"

    return "Imediata"

maintenance_engine["JANELA_MANUTENCAO"] = (

    maintenance_engine["MAINTENANCE_CLASS"]

    .apply(janela)

)

# ==============================================================================
# RANKING DOS CHILLERS
# ==============================================================================

ranking_ativos = (

    maintenance_engine

    .groupby("URA", as_index=False)

    .agg({

        "MAINTENANCE_PRIORITY_INDEX":"mean",

        "RISK_SCORE":"mean",

        "POF_PERCENTUAL":"mean",

        "HEALTH_SCORE":"mean",

        "RUL_DIAS":"mean"

    })

)

ranking_ativos["RANKING"] = (

    ranking_ativos["MAINTENANCE_PRIORITY_INDEX"]

    .rank(

        ascending=False,

        method="dense"

    )

    .astype(int)

)

ranking_ativos = ranking_ativos.sort_values("RANKING")

print("\nRanking dos Ativos:\n")

print(ranking_ativos)

# ==============================================================================
# ESTATÍSTICAS
# ==============================================================================

print()

print(

maintenance_engine[

[
"MAINTENANCE_PRIORITY_INDEX"

]

].describe().round(2)

)

print()

print(

maintenance_engine["MAINTENANCE_CLASS"]

.value_counts()

)

# ==============================================================================
# EXPORTAÇÃO
# ==============================================================================

maintenance_engine.to_excel(

    PASTA/"32_MAINTENANCE_PRIORITY_ENGINE.xlsx",

    index=False

)

ranking_ativos.to_excel(

    PASTA/"33_RANKING_CHILLERS.xlsx",

    index=False

)

maintenance_engine.to_csv(

    PASTA/"32_MAINTENANCE_PRIORITY_ENGINE.csv",

    sep=";",

    index=False

)

maintenance_engine.to_parquet(

    PASTA/"32_MAINTENANCE_PRIORITY_ENGINE.parquet",

    index=False

)

print()

print("="*100)
print("MAINTENANCE PRIORITY ENGINE FINALIZADO")
print("="*100)



BLOCO 5.3.4.4 - MAINTENANCE PRIORITY ENGINE
Registros: 17,180

Ranking dos Ativos:

     URA  MAINTENANCE_PRIORITY_INDEX  RISK_SCORE  POF_PERCENTUAL  \
3  URA03                     18.2599     19.2922         18.9078   
1  URA01                     10.9399     13.9732         12.1863   
2  URA02                      6.6304     10.6940          8.1471   
0    CAG                      6.4798     10.5500          8.0000   

   HEALTH_SCORE  RUL_DIAS  RANKING  
3       94.6549  307.1432        1  
1       97.7242  332.1748        2  
2       99.8222  347.0293        3  
0      100.0000  347.6078        4  

       MAINTENANCE_PRIORITY_INDEX
count                  17180.0000
mean                      10.5800
std                        9.9400
min                        5.7600
25%                        6.4800
50%                        6.4800
75%                        6.4800
max                       46.9400

MAINTENANCE_CLASS
Muito Baixa    14690
Baixa           2033
Moderada         457
N

In [44]:
# ==============================================================================
# BLOCO 5.3.4.5
# MAINTENANCE RECOMMENDATION ENGINE
# ==============================================================================

import pandas as pd
import numpy as np
from pathlib import Path

print("="*100)
print("BLOCO 5.3.4.5 - MAINTENANCE RECOMMENDATION ENGINE")
print("="*100)

PASTA = Path("Resultados")

recommendation_engine = maintenance_engine.copy()

print(f"Registros: {len(recommendation_engine):,}")

# ==============================================================================
# VALIDAÇÃO DAS COLUNAS
# ==============================================================================

campos = {

    "COP":np.nan,

    "KW_TR":np.nan,

    "APPROACH_EVAP":np.nan,

    "APPROACH_COND":np.nan,

    "PERCENTUAL_CARGA":np.nan,

    "RISK_SCORE":0,

    "POF_PERCENTUAL":0,

    "RUL_DIAS":365

}

for coluna, valor in campos.items():

    if coluna not in recommendation_engine.columns:

        print(f"Criando coluna {coluna}")

        recommendation_engine[coluna] = valor

        # ==============================================================================
# DIAGNÓSTICO
# ==============================================================================

def diagnostico(row):

    problemas = []

    if pd.notna(row["COP"]):

        if row["COP"] < 3:

            problemas.append("COP abaixo do esperado")

    if pd.notna(row["KW_TR"]):

        if row["KW_TR"] > 0.85:

            problemas.append("Consumo específico elevado")

    if pd.notna(row["APPROACH_COND"]):

        if row["APPROACH_COND"] > 5:

            problemas.append("Approach do condensador elevado")

    if pd.notna(row["APPROACH_EVAP"]):

        if row["APPROACH_EVAP"] > 3:

            problemas.append("Approach do evaporador elevado")

    if pd.notna(row["PERCENTUAL_CARGA"]):

        if row["PERCENTUAL_CARGA"] < 25:

            problemas.append("Operação em baixa carga")

    if len(problemas)==0:

        return "Operação dentro dos parâmetros"

    return " | ".join(problemas)

recommendation_engine["DIAGNOSTICO"] = (

    recommendation_engine.apply(

        diagnostico,

        axis=1

    )

)

# ==============================================================================
# RECOMENDAÇÃO
# ==============================================================================

def recomendacao(texto):

    recomendacoes = []

    if "COP abaixo" in texto:

        recomendacoes.append("Verificar troca térmica e refrigerante")

    if "Consumo específico" in texto:

        recomendacoes.append("Avaliar rendimento do compressor")

    if "condensador" in texto:

        recomendacoes.append("Inspecionar limpeza do condensador")

    if "evaporador" in texto:

        recomendacoes.append("Verificar incrustação do evaporador")

    if "baixa carga" in texto:

        recomendacoes.append("Reavaliar estratégia operacional")

    if len(recomendacoes)==0:

        return "Nenhuma ação necessária"

    return " | ".join(recomendacoes)

recommendation_engine["RECOMENDACAO"] = (

    recommendation_engine["DIAGNOSTICO"]

    .apply(recomendacao)

)

# ==============================================================================
# RESPONSÁVEL
# ==============================================================================

def responsavel(texto):

    if texto=="Nenhuma ação necessária":

        return "Operação"

    return "Engenharia de Manutenção"

recommendation_engine["AREA_RESPONSAVEL"] = (

    recommendation_engine["RECOMENDACAO"]

    .apply(responsavel)

)

# ==============================================================================
# PRAZO
# ==============================================================================

def prazo(row):

    if row["RISK_SCORE"] > 80:

        return "Imediato"

    if row["RISK_SCORE"] > 60:

        return "7 dias"

    if row["POF_PERCENTUAL"] > 40:

        return "15 dias"

    if row["RUL_DIAS"] < 60:

        return "30 dias"

    return "Programado"

recommendation_engine["PRAZO"] = (

    recommendation_engine.apply(

        prazo,

        axis=1

    )

)

# ==============================================================================
# ORDEM DE SERVIÇO
# ==============================================================================

recommendation_engine["GERAR_OS"] = (

    recommendation_engine["RISK_SCORE"] >= 70

)

# ==============================================================================
# RESUMO
# ==============================================================================

resumo = (

    recommendation_engine

    .groupby("URA",as_index=False)

    .agg({

        "RISK_SCORE":"mean",

        "POF_PERCENTUAL":"mean",

        "RUL_DIAS":"mean",

        "GERAR_OS":"sum"

    })

)

print()

print(resumo)

# ==============================================================================
# EXPORTAÇÃO
# ==============================================================================

recommendation_engine.to_excel(

    PASTA/"34_MAINTENANCE_RECOMMENDATION_ENGINE.xlsx",

    index=False

)

resumo.to_excel(

    PASTA/"35_RESUMO_MANUTENCAO.xlsx",

    index=False

)

recommendation_engine.to_csv(

    PASTA/"34_MAINTENANCE_RECOMMENDATION_ENGINE.csv",

    sep=";",

    index=False

)

recommendation_engine.to_parquet(

    PASTA/"34_MAINTENANCE_RECOMMENDATION_ENGINE.parquet",

    index=False

)

print()

print("="*100)
print("MAINTENANCE RECOMMENDATION ENGINE FINALIZADO")
print("="*100)



BLOCO 5.3.4.5 - MAINTENANCE RECOMMENDATION ENGINE
Registros: 17,180

     URA  RISK_SCORE  POF_PERCENTUAL  RUL_DIAS  GERAR_OS
0    CAG     10.5500          8.0000  347.6078         0
1  URA01     13.9732         12.1863  332.1748         0
2  URA02     10.6940          8.1471  347.0293         0
3  URA03     19.2922         18.9078  307.1432         0

MAINTENANCE RECOMMENDATION ENGINE FINALIZADO


In [45]:
# ==============================================================================
# BLOCO 5.3.4.6
# EXECUTIVE DASHBOARD ENGINE
# ==============================================================================

import pandas as pd
import numpy as np
from pathlib import Path

print("="*100)
print("BLOCO 5.3.4.6 - EXECUTIVE DASHBOARD ENGINE")
print("="*100)

PASTA = Path("Resultados")

dashboard_engine = recommendation_engine.copy()

print(f"Registros: {len(dashboard_engine):,}")

# ==============================================================================
# DASHBOARD EXECUTIVO
# ==============================================================================

dashboard = (

    dashboard_engine

    .groupby("URA", as_index=False)

    .agg({

        "HEALTH_SCORE":"mean",

        "PERFORMANCE_INDEX":"mean",

        "RISK_SCORE":"mean",

        "POF_PERCENTUAL":"mean",

        "RUL_DIAS":"mean",

        "MAINTENANCE_PRIORITY_INDEX":"mean",

        "GERAR_OS":"sum",

        "COP":"mean",

        "KW_TR":"mean",

        "CARGA_TR":"mean"

    })

)

dashboard["EFICIENCIA_GLOBAL"] = (

      dashboard["HEALTH_SCORE"]*0.40

    + dashboard["PERFORMANCE_INDEX"]*0.40

    + (100-dashboard["RISK_SCORE"])*0.20

)

def semaforo(valor):

    if valor >= 90:
        return "🟢 Excelente"

    elif valor >= 75:
        return "🟡 Atenção"

    elif valor >= 60:
        return "🟠 Monitorar"

    return "🔴 Crítico"

dashboard["STATUS_EXECUTIVO"] = (

    dashboard["EFICIENCIA_GLOBAL"]

    .apply(semaforo)

)

dashboard["RANKING"] = (

    dashboard["EFICIENCIA_GLOBAL"]

    .rank(

        ascending=False,

        method="dense"

    )

    .astype(int)

)

kpis = pd.DataFrame({

    "INDICADOR":[

        "Eficiência Média",

        "Health Médio",

        "Risk Médio",

        "POF Médio",

        "RUL Médio (dias)",

        "Ordens de Serviço"

    ],

    "VALOR":[

        dashboard["EFICIENCIA_GLOBAL"].mean(),

        dashboard["HEALTH_SCORE"].mean(),

        dashboard["RISK_SCORE"].mean(),

        dashboard["POF_PERCENTUAL"].mean(),

        dashboard["RUL_DIAS"].mean(),

        dashboard["GERAR_OS"].sum()

    ]

})

print("\nKPIs da Central:\n")
print(kpis)

print("\nResumo Executivo:\n")

print(

dashboard.sort_values(

    "RANKING"

)

)

dashboard.to_excel(

    PASTA/"36_EXECUTIVE_DASHBOARD.xlsx",

    index=False

)

kpis.to_excel(

    PASTA/"37_EXECUTIVE_KPIS.xlsx",

    index=False

)

dashboard.to_csv(

    PASTA/"36_EXECUTIVE_DASHBOARD.csv",

    sep=";",

    index=False

)

dashboard.to_parquet(

    PASTA/"36_EXECUTIVE_DASHBOARD.parquet",

    index=False

)

print()

print("="*100)
print("EXECUTIVE DASHBOARD ENGINE FINALIZADO")
print("="*100)



BLOCO 5.3.4.6 - EXECUTIVE DASHBOARD ENGINE
Registros: 17,180

KPIs da Central:

           INDICADOR    VALOR
0   Eficiência Média  94.9349
1       Health Médio  98.0503
2         Risk Médio  13.6273
3          POF Médio  11.8103
4   RUL Médio (dias) 333.4888
5  Ordens de Serviço   0.0000

Resumo Executivo:

     URA  HEALTH_SCORE  PERFORMANCE_INDEX  RISK_SCORE  POF_PERCENTUAL  \
0    CAG      100.0000           100.0000     10.5500          8.0000   
2  URA02       99.8222            99.6445     10.6940          8.1471   
1  URA01       97.7242            95.4485     13.9732         12.1863   
3  URA03       94.6549            89.3098     19.2922         18.9078   

   RUL_DIAS  MAINTENANCE_PRIORITY_INDEX  GERAR_OS    COP  KW_TR  CARGA_TR  \
0  347.6078                      6.4798         0    NaN    NaN       NaN   
2  347.0293                      6.6304         0 2.8307 0.7260  116.2400   
1  332.1748                     10.9399         0 2.6980 1.8549  107.1355   
3  307.1432     

In [46]:
print(fault_engine.columns.tolist())

NameError: name 'fault_engine' is not defined

In [47]:
# ==============================================================================
# BLOCO 5.4.1
# FAULT PATTERN RECOGNITION ENGINE
# VERSÃO 2.0
# ==============================================================================

import numpy as np
import pandas as pd
from pathlib import Path

print("="*100)
print("BLOCO 5.4.1 - FAULT PATTERN RECOGNITION ENGINE")
print("="*100)

PASTA = Path("Resultados")

fault_engine = recommendation_engine.copy()

print(f"Registros : {len(fault_engine):,}")

# ==============================================================================
# GARANTIA DAS COLUNAS
# ==============================================================================

campos = {

    "COP":np.nan,

    "KW_TR":np.nan,

    "APPROACH_COND":np.nan,

    "APPROACH_EVAP":np.nan,

    "PERCENTUAL_CARGA":np.nan,

    "HEALTH_SCORE":100,

    "PERFORMANCE_INDEX":100,

    "FAILURE_SCORE":0,

    "RISK_SCORE":0,

    "POF":0,

    "RUL_DIAS":999,

    "CONFIDENCE_SCORE":100

}

for coluna,valor in campos.items():

    if coluna not in fault_engine.columns:

        print(f"Criando {coluna}")

        fault_engine[coluna]=valor

        # ==============================================================================
# SISTEMA ESPECIALISTA
# ==============================================================================

def identificar_falha(linha):

    try:

        cop = float(linha["COP"])
        kw = float(linha["KW_TR"])
        cond = float(linha["APPROACH_COND"])
        evap = float(linha["APPROACH_EVAP"])
        carga = float(linha["PERCENTUAL_CARGA"])

        health = float(linha["HEALTH_SCORE"])
        perf = float(linha["PERFORMANCE_INDEX"])
        risk = float(linha["RISK_SCORE"])
        failure = float(linha["FAILURE_SCORE"])
        pof = float(linha["POF"])
        rul = float(linha["RUL_DIAS"])

        # ==========================================================
        # EQUIPAMENTO CRÍTICO
        # ==========================================================

        if (risk > 85) and (failure > 80):

            return "EQUIPAMENTO_CRITICO"

        # ==========================================================
        # VIDA ÚTIL MUITO BAIXA
        # ==========================================================

        if rul < 90:

            return "RUL_CRITICO"

        # ==========================================================
        # CONDENSADOR
        # ==========================================================

        if (cop < 3) and (cond > 5):

            return "CONDENSADOR_SUJO"

        # ==========================================================
        # EVAPORADOR
        # ==========================================================

        if (cop < 3) and (evap > 3):

            return "EVAPORADOR_SUJO"

        # ==========================================================
        # BAIXA CARGA
        # ==========================================================

        if (kw > 0.90) and (carga < 25):

            return "OPERACAO_BAIXA_CARGA"

        # ==========================================================
        # BAIXA PERFORMANCE
        # ==========================================================

        if perf < 60:

            return "BAIXA_PERFORMANCE"

        # ==========================================================
        # DEGRADAÇÃO
        # ==========================================================

        if health < 70:

            return "DEGRADACAO_PROGRESSIVA"

        # ==========================================================
        # PROBABILIDADE DE FALHA
        # ==========================================================

        if pof > 80:

            return "ALTA_PROBABILIDADE_FALHA"

        return "OPERACAO_NORMAL"

    except Exception:

        return "DADOS_INSUFICIENTES"

        # ==============================================================================
# IDENTIFICAÇÃO
# ==============================================================================

fault_engine["FAULT_PATTERN"] = (

    fault_engine

    .apply(

        identificar_falha,

        axis=1

    )

)

# ==============================================================================
# SEVERIDADE
# ==============================================================================

peso = {

    "OPERACAO_NORMAL":1,

    "OPERACAO_BAIXA_CARGA":2,

    "BAIXA_PERFORMANCE":3,

    "CONDENSADOR_SUJO":4,

    "EVAPORADOR_SUJO":4,

    "DEGRADACAO_PROGRESSIVA":5,

    "ALTA_PROBABILIDADE_FALHA":6,

    "RUL_CRITICO":7,

    "EQUIPAMENTO_CRITICO":8,

    "DADOS_INSUFICIENTES":0

}

fault_engine["FAULT_SEVERITY"] = (

    fault_engine["FAULT_PATTERN"]

    .map(peso)

    .fillna(0)

    .astype(int)

)

# ==============================================================================
# RESUMO
# ==============================================================================

fault_summary = (

    fault_engine

    .groupby(

        [

            "URA",

            "FAULT_PATTERN"

        ],

        dropna=False

    )

    .size()

    .reset_index(name="OCORRENCIAS")

    .sort_values(

        "OCORRENCIAS",

        ascending=False

    )

)

print()

print(fault_summary)

# ==============================================================================
# EXPORTAÇÃO
# ==============================================================================

fault_engine.to_excel(

    PASTA/"38_FAULT_PATTERN_ENGINE.xlsx",

    index=False

)

fault_summary.to_excel(

    PASTA/"39_FAULT_PATTERN_SUMMARY.xlsx",

    index=False

)

print()

print("="*100)

print("FAULT PATTERN ENGINE FINALIZADO")

print("="*100)



BLOCO 5.4.1 - FAULT PATTERN RECOGNITION ENGINE
Registros : 17,180

      URA         FAULT_PATTERN  OCORRENCIAS
0     CAG       OPERACAO_NORMAL         4295
8   URA02       OPERACAO_NORMAL         4288
4   URA01  OPERACAO_BAIXA_CARGA         3358
12  URA03  OPERACAO_BAIXA_CARGA         2553
13  URA03       OPERACAO_NORMAL         1704
5   URA01       OPERACAO_NORMAL          814
3   URA01       EVAPORADOR_SUJO           83
2   URA01      CONDENSADOR_SUJO           31
11  URA03       EVAPORADOR_SUJO           25
1   URA01     BAIXA_PERFORMANCE            9
9   URA03     BAIXA_PERFORMANCE            8
6   URA02       EVAPORADOR_SUJO            6
10  URA03      CONDENSADOR_SUJO            5
7   URA02  OPERACAO_BAIXA_CARGA            1

FAULT PATTERN ENGINE FINALIZADO


In [48]:
# ==============================================================================
# BLOCO 5.4.2
# ROOT CAUSE ANALYSIS ENGINE
# ==============================================================================

import numpy as np
import pandas as pd
from pathlib import Path

print("="*100)
print("BLOCO 5.4.2 - ROOT CAUSE ANALYSIS ENGINE")
print("="*100)

PASTA = Path("Resultados")

root_engine = fault_engine.copy()

print(f"Registros : {len(root_engine):,}")

# ==============================================================================
# VALIDAÇÃO
# ==============================================================================

campos = {

    "FAULT_PATTERN":"OPERACAO_NORMAL",

    "FAULT_SEVERITY":0,

    "HEALTH_SCORE":100,

    "PERFORMANCE_INDEX":100,

    "FAILURE_SCORE":0,

    "RISK_SCORE":0,

    "POF":0,

    "RUL_DIAS":999,

    "COP":np.nan,

    "KW_TR":np.nan,

    "APPROACH_COND":np.nan,

    "APPROACH_EVAP":np.nan,

    "PERCENTUAL_CARGA":np.nan,

    "CONFIDENCE_SCORE":100

}

for coluna, valor in campos.items():

    if coluna not in root_engine.columns:

        print(f"Criando coluna: {coluna}")

        root_engine[coluna] = valor

        # ==============================================================================
# ROOT CAUSE ENGINE
# ==============================================================================

def causa_raiz(row):

    try:

        fault = row["FAULT_PATTERN"]

        if fault == "CONDENSADOR_SUJO":

            return (
                "Elevado Approach do Condensador",
                "Verificar incrustação, torre de resfriamento e limpeza do condensador",
                "ALTA"
            )

        elif fault == "EVAPORADOR_SUJO":

            return (
                "Baixa troca térmica no evaporador",
                "Inspecionar evaporador e fluxo de água gelada",
                "ALTA"
            )

        elif fault == "OPERACAO_BAIXA_CARGA":

            return (
                "Operação em baixa carga",
                "Avaliar sequência operacional dos chillers",
                "MÉDIA"
            )

        elif fault == "BAIXA_PERFORMANCE":

            return (
                "Perda de eficiência energética",
                "Realizar inspeção completa do equipamento",
                "ALTA"
            )

        elif fault == "DEGRADACAO_PROGRESSIVA":

            return (
                "Degradação progressiva dos componentes",
                "Programar manutenção preditiva",
                "ALTA"
            )

        elif fault == "ALTA_PROBABILIDADE_FALHA":

            return (
                "Elevada probabilidade estatística de falha",
                "Inspecionar imediatamente",
                "CRÍTICA"
            )

        elif fault == "RUL_CRITICO":

            return (
                "Vida útil remanescente reduzida",
                "Planejar substituição preventiva",
                "CRÍTICA"
            )

        elif fault == "EQUIPAMENTO_CRITICO":

            return (
                "Risco operacional elevado",
                "Intervenção imediata",
                "CRÍTICA"
            )

        elif fault == "DADOS_INSUFICIENTES":

            return (
                "Base de dados insuficiente",
                "Validar sensores e aquisição",
                "BAIXA"
            )

        return (

            "Nenhuma causa relevante",

            "Operação dentro dos parâmetros",

            "NORMAL"

        )

    except Exception:

        return (

            "Erro de processamento",

            "Revisar dados",

            "DESCONHECIDA"

        )

        # ==============================================================================
# PROCESSAMENTO
# ==============================================================================

resultado = root_engine.apply(causa_raiz, axis=1)

root_engine[
    [
        "ROOT_CAUSE",
        "ENGINEERING_ACTION",
        "ROOT_PRIORITY"
    ]
] = pd.DataFrame(
    resultado.tolist(),
    index=root_engine.index
)

# ==============================================================================
# ROOT CONFIDENCE
# ==============================================================================

root_engine["ROOT_CONFIDENCE"] = (

    (

        root_engine["CONFIDENCE_SCORE"]

        +

        root_engine["HEALTH_SCORE"]

        +

        root_engine["PERFORMANCE_INDEX"]

    ) / 3

).round(1)

# ==============================================================================
# RESUMO
# ==============================================================================

root_summary = (

    root_engine

    .groupby(

        [

            "ROOT_CAUSE",

            "ROOT_PRIORITY"

        ],

        dropna=False

    )

    .size()

    .reset_index(name="OCORRENCIAS")

    .sort_values(

        "OCORRENCIAS",

        ascending=False

    )

)

print()

print(root_summary)

# ==============================================================================
# EXPORTAÇÃO
# ==============================================================================

root_engine.to_excel(
    PASTA/"40_ROOT_CAUSE_ENGINE.xlsx",
    index=False
)

root_summary.to_excel(
    PASTA/"41_ROOT_CAUSE_SUMMARY.xlsx",
    index=False
)

root_engine.to_csv(
    PASTA/"40_ROOT_CAUSE_ENGINE.csv",
    sep=";",
    index=False
)

root_engine.to_parquet(
    PASTA/"40_ROOT_CAUSE_ENGINE.parquet",
    index=False
)

print()

print("="*100)
print("ROOT CAUSE ANALYSIS ENGINE FINALIZADO")
print("="*100)



BLOCO 5.4.2 - ROOT CAUSE ANALYSIS ENGINE
Registros : 17,180

                          ROOT_CAUSE ROOT_PRIORITY  OCORRENCIAS
2            Nenhuma causa relevante        NORMAL        11101
3            Operação em baixa carga         MÉDIA         5912
0  Baixa troca térmica no evaporador          ALTA          114
1    Elevado Approach do Condensador          ALTA           36
4     Perda de eficiência energética          ALTA           17

ROOT CAUSE ANALYSIS ENGINE FINALIZADO


In [49]:
# ==============================================================================
# BLOCO 5.4.3
# ENGINEERING RULES ENGINE
# VERSÃO 2.0
# ==============================================================================

import pandas as pd
import numpy as np
from pathlib import Path

print("="*100)
print("BLOCO 5.4.3 - ENGINEERING RULES ENGINE")
print("="*100)

PASTA = Path("Resultados")

rules_engine = root_engine.copy()

print(f"Registros: {len(rules_engine):,}")

# ==============================================================================
# PADRONIZAÇÃO DA ESTRUTURA
# ==============================================================================

estrutura = {

    "FAULT_PATTERN":"OPERACAO_NORMAL",

    "HEALTH_SCORE":100,

    "PERFORMANCE_INDEX":100,

    "FAILURE_SCORE":0,

    "RISK_SCORE":0,

    "ROOT_CONFIDENCE":100

}

for coluna,valor in estrutura.items():

    if coluna not in rules_engine.columns:

        print(f"Criando coluna: {coluna}")

        rules_engine[coluna]=valor

        # ==============================================================================
# KNOWLEDGE BASE
# ==============================================================================

engineering_rules = pd.DataFrame([

["OPERACAO_NORMAL","BAIXA",1,"Continuar Monitoramento"],

["OPERACAO_BAIXA_CARGA","MEDIA",2,"Reavaliar Sequenciamento"],

["BAIXA_PERFORMANCE","ALTA",3,"Inspecionar Eficiência Energética"],

["CONDENSADOR_SUJO","ALTA",4,"Limpeza do Condensador"],

["EVAPORADOR_SUJO","ALTA",4,"Limpeza do Evaporador"],

["DEGRADACAO_PROGRESSIVA","ALTA",5,"Programar Manutenção"],

["ALTA_PROBABILIDADE_FALHA","CRITICA",6,"Inspeção Imediata"],

["RUL_CRITICO","CRITICA",7,"Planejar Substituição"],

["EQUIPAMENTO_CRITICO","EMERGENCIA",8,"Parada Programada"],

["DADOS_INSUFICIENTES","BAIXA",0,"Verificar Instrumentação"]

],

columns=[

"FAULT_PATTERN",

"RULE_CRITICIDADE",

"RULE_PRIORIDADE",

"RULE_RECOMENDACAO"

])

# ==============================================================================
# MERGE
# ==============================================================================

rules_engine = rules_engine.merge(

    engineering_rules,

    how="left",

    on="FAULT_PATTERN"

)

# ==============================================================================
# NORMALIZAÇÃO
# ==============================================================================

if "CRITICIDADE" not in rules_engine.columns:

    if "REG_CRITICIDADE" in rules_engine.columns:

        rules_engine["CRITICIDADE"] = rules_engine["REG_CRITICIDADE"]

    elif "RULE_CRITICIDADE" in rules_engine.columns:

        rules_engine["CRITICIDADE"] = rules_engine["RULE_CRITICIDADE"]

    else:

        rules_engine["CRITICIDADE"] = "NÃO DEFINIDA"



rules_engine["PRIORIDADE"] = (

    rules_engine["RULE_PRIORIDADE"]

)

rules_engine["RECOMENDACAO"] = (

    rules_engine["RULE_RECOMENDACAO"]

)

# ==============================================================================
# ENGINEERING INDEX
# ==============================================================================

rules_engine["ENGINEERING_INDEX"] = (

      0.30*rules_engine["HEALTH_SCORE"]

    + 0.25*rules_engine["PERFORMANCE_INDEX"]

    + 0.20*(100-rules_engine["FAILURE_SCORE"])

    + 0.15*(100-rules_engine["RISK_SCORE"])

    + 0.10*rules_engine["ROOT_CONFIDENCE"]

).round(2)

# ==============================================================================
# CLASSIFICAÇÃO
# ==============================================================================

def classe(valor):

    if valor>=90:
        return "EXCELENTE"

    elif valor>=80:
        return "MUITO BOM"

    elif valor>=70:
        return "BOM"

    elif valor>=60:
        return "REGULAR"

    elif valor>=50:
        return "RUIM"

    return "CRITICO"

rules_engine["ENGINEERING_CLASS"]=(

    rules_engine["ENGINEERING_INDEX"]

    .apply(classe)

)

# ==============================================================================
# RESUMO
# ==============================================================================

engineering_summary=(

    rules_engine

    .groupby(

        [

            "ENGINEERING_CLASS",

            "CRITICIDADE"

        ],

        dropna=False

    )

    .size()

    .reset_index(name="OCORRENCIAS")

    .sort_values(

        "OCORRENCIAS",

        ascending=False

    )

)

print()

print(engineering_summary)

# ==============================================================================
# EXPORTAÇÃO
# ==============================================================================

rules_engine.to_excel(

    PASTA/"42_ENGINEERING_RULES_ENGINE.xlsx",

    index=False

)

engineering_rules.to_excel(

    PASTA/"43_ENGINEERING_RULE_LIBRARY.xlsx",

    index=False

)

engineering_summary.to_excel(

    PASTA/"44_ENGINEERING_SUMMARY.xlsx",

    index=False

)

rules_engine.to_csv(

    PASTA/"42_ENGINEERING_RULES_ENGINE.csv",

    sep=";",

    index=False

)

rules_engine.to_parquet(

    PASTA/"42_ENGINEERING_RULES_ENGINE.parquet",

    index=False

)

print()

print("="*100)

print("ENGINEERING RULES ENGINE FINALIZADO")

print("="*100)

BLOCO 5.4.3 - ENGINEERING RULES ENGINE
Registros: 17,180

  ENGINEERING_CLASS CRITICIDADE  OCORRENCIAS
2         EXCELENTE       Média        14541
3         MUITO BOM       Média         1238
0               BOM       Média          893
4           REGULAR       Média          495
5              RUIM       Média           12
1           CRITICO       Média            1

ENGINEERING RULES ENGINE FINALIZADO


In [50]:
# ==============================================================================
# BLOCO 5.4.4
# ENGINEERING DECISION ENGINE
# ==============================================================================

import pandas as pd
import numpy as np
from pathlib import Path

print("="*100)
print("BLOCO 5.4.4 - ENGINEERING DECISION ENGINE")
print("="*100)

PASTA = Path("Resultados")

decision_engine = rules_engine.copy()

print(f"Registros: {len(decision_engine):,}")

# ==============================================================================
# GARANTIA DAS COLUNAS
# ==============================================================================

estrutura = {

    "ENGINEERING_INDEX":100,

    "HEALTH_SCORE":100,

    "FAILURE_SCORE":0,

    "RISK_SCORE":0,

    "POF":0,

    "RUL_DIAS":999,

    "ROOT_CAUSE":"SEM DIAGNOSTICO",

    "RECOMENDACAO":"SEM ACAO"

}

for coluna,valor in estrutura.items():

    if coluna not in decision_engine.columns:

        print(f"Criando coluna: {coluna}")

        decision_engine[coluna]=valor

        # ==============================================================================
# DECISÃO FINAL
# ==============================================================================

def decisao(row):

    if row["FAILURE_SCORE"] >= 90:

        return "PARADA IMEDIATA"

    elif row["RUL_DIAS"] <= 7:

        return "PROGRAMAR PARADA"

    elif row["RISK_SCORE"] >= 80:

        return "MANUTENÇÃO URGENTE"

    elif row["ENGINEERING_INDEX"] < 60:

        return "INSPEÇÃO COMPLETA"

    elif row["ENGINEERING_INDEX"] < 80:

        return "MONITORAMENTO INTENSIVO"

    else:

        return "OPERAÇÃO NORMAL"


decision_engine["ENGINEERING_DECISION"] = (

    decision_engine

    .apply(decisao, axis=1)

)

# ==============================================================================
# PRIORIDADE EXECUTIVA
# ==============================================================================

prioridade = {

    "PARADA IMEDIATA":1,

    "PROGRAMAR PARADA":2,

    "MANUTENÇÃO URGENTE":3,

    "INSPEÇÃO COMPLETA":4,

    "MONITORAMENTO INTENSIVO":5,

    "OPERAÇÃO NORMAL":6

}

decision_engine["EXEC_PRIORITY"] = (

    decision_engine["ENGINEERING_DECISION"]

    .map(prioridade)

)

# ==============================================================================
# RESUMO
# ==============================================================================

executive_summary = (

    decision_engine

    .groupby(

        "ENGINEERING_DECISION",

        dropna=False

    )

    .size()

    .reset_index(name="OCORRENCIAS")

    .sort_values(

        "OCORRENCIAS",

        ascending=False

    )

)

print()

print(executive_summary)

# ==============================================================================
# EXPORTAÇÃO
# ==============================================================================

decision_engine.to_excel(

    PASTA/"45_ENGINEERING_DECISION_ENGINE.xlsx",

    index=False

)

executive_summary.to_excel(

    PASTA/"46_EXECUTIVE_DECISION_SUMMARY.xlsx",

    index=False

)

decision_engine.to_csv(

    PASTA/"45_ENGINEERING_DECISION_ENGINE.csv",

    sep=";",

    index=False

)

decision_engine.to_parquet(

    PASTA/"45_ENGINEERING_DECISION_ENGINE.parquet",

    index=False

)

print()

print("="*100)
print("ENGINEERING DECISION ENGINE FINALIZADO")
print("="*100)



BLOCO 5.4.4 - ENGINEERING DECISION ENGINE
Registros: 17,180

      ENGINEERING_DECISION  OCORRENCIAS
2          OPERAÇÃO NORMAL        15779
1  MONITORAMENTO INTENSIVO         1388
0        INSPEÇÃO COMPLETA           13

ENGINEERING DECISION ENGINE FINALIZADO


In [51]:
# ==============================================================================
# BLOCO 5.5.1
# EXECUTIVE KPI ENGINE
# ==============================================================================

import pandas as pd
import numpy as np
from pathlib import Path

print("="*100)
print("BLOCO 5.5.1 - EXECUTIVE KPI ENGINE")
print("="*100)

PASTA = Path("Resultados")

executive_engine = decision_engine.copy()

print(f"Registros: {len(executive_engine):,}")

# ==============================================================================
# GARANTIA DA ESTRUTURA
# ==============================================================================

estrutura = {

    "URA":"SEM_URA",

    "HEALTH_SCORE":100,

    "PERFORMANCE_INDEX":100,

    "FAILURE_SCORE":0,

    "RISK_SCORE":0,

    "ENGINEERING_INDEX":100,

    "POF":0,

    "RUL_DIAS":999,

    "SCORE_DISPONIBILIDADE":100,

    "ANOMALY_SCORE":0

}

for coluna,valor in estrutura.items():

    if coluna not in executive_engine.columns:

        executive_engine[coluna]=valor

        # ==============================================================================
# EXECUTIVE KPI
# ==============================================================================

executive_kpi = (

    executive_engine

    .groupby("URA", dropna=False)

    .agg({

        "HEALTH_SCORE":"mean",

        "PERFORMANCE_INDEX":"mean",

        "FAILURE_SCORE":"mean",

        "RISK_SCORE":"mean",

        "ENGINEERING_INDEX":"mean",

        "POF":"mean",

        "RUL_DIAS":"mean",

        "SCORE_DISPONIBILIDADE":"mean",

        "ANOMALY_SCORE":"mean"

    })

    .round(2)

    .reset_index()

)

# ==============================================================================
# EVENTOS
# ==============================================================================

eventos = (

    executive_engine

    .groupby("URA")

    .agg(

        TOTAL_REGISTROS=("URA","count"),

        TOTAL_ANOMALIAS=("ANOMALIA_Z","sum"),

        TOTAL_OS=("GERAR_OS","sum")

    )

    .reset_index()

)

executive_kpi = executive_kpi.merge(

    eventos,

    on="URA",

    how="left"

)

# ==============================================================================
# RANKING
# ==============================================================================

executive_kpi["RANKING"] = (

    executive_kpi["ENGINEERING_INDEX"]

    .rank(

        ascending=False,

        method="dense"

    )

)

# ==============================================================================
# CLASSE EXECUTIVA
# ==============================================================================

def classe(valor):

    if valor>=90:

        return "A"

    elif valor>=80:

        return "B"

    elif valor>=70:

        return "C"

    elif valor>=60:

        return "D"

    return "E"

executive_kpi["CLASSE_EXECUTIVA"]=(

    executive_kpi["ENGINEERING_INDEX"]

    .apply(classe)

)

# ==============================================================================
# RESUMO
# ==============================================================================

print()

print(executive_kpi)

print()

print(executive_kpi.describe(include="all"))

# ==============================================================================
# EXPORTAÇÃO
# ==============================================================================

executive_kpi.to_excel(

    PASTA/"47_EXECUTIVE_KPI.xlsx",

    index=False

)

executive_kpi.to_csv(

    PASTA/"47_EXECUTIVE_KPI.csv",

    sep=";",

    index=False

)

executive_kpi.to_parquet(

    PASTA/"47_EXECUTIVE_KPI.parquet",

    index=False

)

print()

print("="*100)
print("EXECUTIVE KPI ENGINE FINALIZADO")
print("="*100)



BLOCO 5.5.1 - EXECUTIVE KPI ENGINE
Registros: 17,180

     URA  HEALTH_SCORE  PERFORMANCE_INDEX  FAILURE_SCORE  RISK_SCORE  \
0    CAG      100.0000           100.0000         8.0000     10.5500   
1  URA01       97.7200            95.4500        12.1900     13.9700   
2  URA02       99.8200            99.6400         8.1500     10.6900   
3  URA03       94.6500            89.3100        18.9100     19.2900   

   ENGINEERING_INDEX    POF  RUL_DIAS  SCORE_DISPONIBILIDADE  ANOMALY_SCORE  \
0            93.4900 0.0800  347.6100                16.6700         1.0000   
1            90.6300 0.1200  332.1700                27.0400         1.0500   
2            93.3100 0.0800  347.0300                 5.3400         1.0000   
3            86.6700 0.1900  307.1400                46.1500         1.1000   

   TOTAL_REGISTROS  TOTAL_ANOMALIAS  TOTAL_OS  RANKING CLASSE_EXECUTIVA  
0             4295                0         0   1.0000                A  
1             4295                0      

In [52]:
# ==============================================================================
# BLOCO 5.5.2
# EXECUTIVE DASHBOARD DATA ENGINE
# ==============================================================================

import pandas as pd
import numpy as np
from pathlib import Path

print("="*100)
print("BLOCO 5.5.2 - EXECUTIVE DASHBOARD DATA ENGINE")
print("="*100)

PASTA = Path("Resultados")

dashboard_engine = decision_engine.copy()

print(f"Registros: {len(dashboard_engine):,}")

# ==============================================================================
# GARANTIA DAS COLUNAS
# ==============================================================================

estrutura = {

    "DATAHORA":pd.Timestamp.now(),

    "URA":"SEM_URA",

    "ENGINEERING_INDEX":100,

    "HEALTH_SCORE":100,

    "PERFORMANCE_INDEX":100,

    "FAILURE_SCORE":0,

    "RISK_SCORE":0,

    "POF":0,

    "RUL_DIAS":999,

    "ENGINEERING_DECISION":"OPERAÇÃO NORMAL",

    "ENGINEERING_CLASS":"EXCELENTE"

}

for coluna,valor in estrutura.items():

    if coluna not in dashboard_engine.columns:

        dashboard_engine[coluna]=valor

# ==============================================================================
# DASHBOARD EXECUTIVO
# ==============================================================================

dashboard_exec = (

    dashboard_engine

    .groupby("URA", dropna=False)

    .agg(

        HEALTH=("HEALTH_SCORE","mean"),

        PERFORMANCE=("PERFORMANCE_INDEX","mean"),

        RISK=("RISK_SCORE","mean"),

        FAILURE=("FAILURE_SCORE","mean"),

        POF=("POF","mean"),

        RUL=("RUL_DIAS","mean"),

        ENGINEERING=("ENGINEERING_INDEX","mean")

    )

    .round(2)

    .reset_index()

)

dashboard_exec["RANKING"]=(

    dashboard_exec["ENGINEERING"]

    .rank(

        ascending=False,

        method="dense"

    )

)

# ==============================================================================
# TIMELINE
# ==============================================================================

timeline_dashboard = (

    dashboard_engine

    .groupby(

        "DATAHORA",

        dropna=False

    )

    .agg(

        HEALTH=("HEALTH_SCORE","mean"),

        PERFORMANCE=("PERFORMANCE_INDEX","mean"),

        RISK=("RISK_SCORE","mean"),

        ENGINEERING=("ENGINEERING_INDEX","mean")

    )

    .reset_index()

)

# ==============================================================================
# DECISÕES
# ==============================================================================

decision_dashboard=(

    dashboard_engine

    .groupby(

        "ENGINEERING_DECISION",

        dropna=False

    )

    .size()

    .reset_index(name="OCORRENCIAS")

    .sort_values(

        "OCORRENCIAS",

        ascending=False

    )

)

# ==============================================================================
# CLASSES
# ==============================================================================

class_dashboard=(

    dashboard_engine

    .groupby(

        "ENGINEERING_CLASS",

        dropna=False

    )

    .size()

    .reset_index(name="OCORRENCIAS")

)

# ==============================================================================
# KPI CARDS
# ==============================================================================

cards = pd.DataFrame({

    "INDICADOR":[

        "Health Médio",

        "Performance Média",

        "Risk Médio",

        "Failure Médio",

        "Engineering Médio",

        "POF Médio",

        "RUL Médio"

    ],

    "VALOR":[

        dashboard_engine["HEALTH_SCORE"].mean(),

        dashboard_engine["PERFORMANCE_INDEX"].mean(),

        dashboard_engine["RISK_SCORE"].mean(),

        dashboard_engine["FAILURE_SCORE"].mean(),

        dashboard_engine["ENGINEERING_INDEX"].mean(),

        dashboard_engine["POF"].mean(),

        dashboard_engine["RUL_DIAS"].mean()

    ]

})

cards["VALOR"]=cards["VALOR"].round(2)

# ==============================================================================
# EXPORTAÇÃO
# ==============================================================================

dashboard_exec.to_excel(
    PASTA/"48_DASHBOARD_EXECUTIVO.xlsx",
    index=False
)

timeline_dashboard.to_excel(
    PASTA/"49_DASHBOARD_TIMELINE.xlsx",
    index=False
)

decision_dashboard.to_excel(
    PASTA/"50_DASHBOARD_DECISOES.xlsx",
    index=False
)

class_dashboard.to_excel(
    PASTA/"51_DASHBOARD_CLASSES.xlsx",
    index=False
)

cards.to_excel(
    PASTA/"52_DASHBOARD_CARDS.xlsx",
    index=False
)

print()

print("="*100)
print("EXECUTIVE DASHBOARD DATA ENGINE FINALIZADO")
print("="*100)



BLOCO 5.5.2 - EXECUTIVE DASHBOARD DATA ENGINE
Registros: 17,180

EXECUTIVE DASHBOARD DATA ENGINE FINALIZADO


In [53]:
# ==============================================================================
# BLOCO 5.5.4
# EXECUTIVE REPORT ENGINE
# ==============================================================================

import pandas as pd
import numpy as np
from pathlib import Path

print("=" * 100)
print("BLOCO 5.5.4 - EXECUTIVE REPORT ENGINE")
print("=" * 100)

PASTA = Path("Resultados")

report_engine = decision_engine.copy()

print(f"Registros: {len(report_engine):,}")

# ==============================================================================
# GARANTIA DA ESTRUTURA
# ==============================================================================

estrutura = {

    "URA":"SEM_URA",

    "ENGINEERING_INDEX":100,

    "HEALTH_SCORE":100,

    "PERFORMANCE_INDEX":100,

    "FAILURE_SCORE":0,

    "RISK_SCORE":0,

    "POF":0,

    "RUL_DIAS":999,

    "ENGINEERING_DECISION":"OPERAÇÃO NORMAL",

    "HEALTH_CLASS":"NORMAL",

    "MAINTENANCE_PRIORITY":"NORMAL"

}

for coluna, valor in estrutura.items():

    if coluna not in report_engine.columns:

        report_engine[coluna] = valor

        # ==============================================================================
# EXECUTIVE SUMMARY
# ==============================================================================

executive_summary = pd.DataFrame({

    "Indicador":[

        "Quantidade de Registros",

        "Quantidade de Equipamentos",

        "Health Médio",

        "Performance Média",

        "Engineering Index Médio",

        "Risk Médio",

        "POF Médio",

        "RUL Médio"

    ],

    "Valor":[

        len(report_engine),

        report_engine["URA"].nunique(),

        round(report_engine["HEALTH_SCORE"].mean(),2),

        round(report_engine["PERFORMANCE_INDEX"].mean(),2),

        round(report_engine["ENGINEERING_INDEX"].mean(),2),

        round(report_engine["RISK_SCORE"].mean(),2),

        round(report_engine["POF"].mean(),2),

        round(report_engine["RUL_DIAS"].mean(),2)

    ]

})

# ==============================================================================
# RANKING
# ==============================================================================

ranking = (

    report_engine

    .groupby("URA")

    .agg({

        "ENGINEERING_INDEX":"mean",

        "HEALTH_SCORE":"mean",

        "PERFORMANCE_INDEX":"mean",

        "RISK_SCORE":"mean",

        "POF":"mean",

        "RUL_DIAS":"mean"

    })

    .round(2)

    .sort_values(

        "ENGINEERING_INDEX",

        ascending=False

    )

    .reset_index()

)

ranking["RANKING"] = np.arange(1, len(ranking)+1)

# ==============================================================================
# DECISÕES
# ==============================================================================

engineering_decisions = (

    report_engine

    .groupby("ENGINEERING_DECISION")

    .size()

    .reset_index(name="OCORRENCIAS")

    .sort_values(

        "OCORRENCIAS",

        ascending=False

    )

)

# ==============================================================================
# MANUTENÇÃO
# ==============================================================================

maintenance_summary = (

    report_engine

    .groupby("MAINTENANCE_PRIORITY")

    .size()

    .reset_index(name="TOTAL")

    .sort_values(

        "TOTAL",

        ascending=False

    )

)

# ==============================================================================
# HEALTH
# ==============================================================================

health_summary = (

    report_engine

    .groupby("HEALTH_CLASS")

    .size()

    .reset_index(name="TOTAL")

)

# ==============================================================================
# EXPORTAÇÃO
# ==============================================================================

executive_summary.to_excel(
    PASTA/"53_EXECUTIVE_SUMMARY.xlsx",
    index=False
)

ranking.to_excel(
    PASTA/"54_EXECUTIVE_RANKING.xlsx",
    index=False
)

engineering_decisions.to_excel(
    PASTA/"55_ENGINEERING_DECISIONS.xlsx",
    index=False
)

maintenance_summary.to_excel(
    PASTA/"56_MAINTENANCE_SUMMARY.xlsx",
    index=False
)

health_summary.to_excel(
    PASTA/"57_HEALTH_SUMMARY.xlsx",
    index=False
)

print()

print("="*100)
print("EXECUTIVE REPORT ENGINE FINALIZADO")
print("="*100)



BLOCO 5.5.4 - EXECUTIVE REPORT ENGINE
Registros: 17,180

EXECUTIVE REPORT ENGINE FINALIZADO


In [54]:
# ==============================================================================
# BLOCO 6.1
# MACHINE LEARNING DATA PREPARATION ENGINE
# ==============================================================================

import pandas as pd
import numpy as np
from pathlib import Path

print("="*100)
print("BLOCO 6.1 - MACHINE LEARNING DATA PREPARATION ENGINE")
print("="*100)

PASTA = Path("Resultados")

ml_engine = decision_engine.copy()

print(f"Registros: {len(ml_engine):,}")

# ==============================================================================
# GARANTIA DA ESTRUTURA
# ==============================================================================

estrutura = {

    "URA":"SEM_URA",

    "DATAHORA":pd.Timestamp.now(),

    "COP":np.nan,

    "KW_TR":np.nan,

    "POTENCIA":np.nan,

    "VAZAO":np.nan,

    "CARGA_TR":np.nan,

    "APPROACH_EVAP":np.nan,

    "APPROACH_COND":np.nan,

    "HEALTH_SCORE":100,

    "PERFORMANCE_INDEX":100,

    "FAILURE_SCORE":0,

    "RISK_SCORE":0,

    "ENGINEERING_INDEX":100,

    "POF":0,

    "RUL_DIAS":999

}

for coluna, valor in estrutura.items():

    if coluna not in ml_engine.columns:

        ml_engine[coluna] = valor

# ==============================================================================
# BASE PARA IA
# ==============================================================================

ml_base = ml_engine[

    [

        "DATAHORA",

        "URA",

        "COP",

        "KW_TR",

        "POTENCIA",

        "VAZAO",

        "CARGA_TR",

        "APPROACH_EVAP",

        "APPROACH_COND",

        "HEALTH_SCORE",

        "PERFORMANCE_INDEX",

        "FAILURE_SCORE",

        "RISK_SCORE",

        "ENGINEERING_INDEX",

        "POF",

        "RUL_DIAS"

    ]

].copy()

# ==============================================================================
# LIMPEZA
# ==============================================================================

variaveis = ml_base.select_dtypes(include=np.number).columns

for coluna in variaveis:

    mediana = ml_base[coluna].median()

    ml_base[coluna] = ml_base[coluna].fillna(mediana)

# ==============================================================================
# NORMALIZAÇÃO
# ==============================================================================

for coluna in variaveis:

    minimo = ml_base[coluna].min()

    maximo = ml_base[coluna].max()

    if maximo > minimo:

        ml_base[f"N_{coluna}"] = (

            ml_base[coluna] - minimo

        ) / (

            maximo - minimo

        )

    else:

        ml_base[f"N_{coluna}"] = 0

  # ==============================================================================
# TARGET
# ==============================================================================

ml_base["TARGET_FAILURE"] = (

    (

        ml_base["POF"] > 70

    )

    |

    (

        ml_base["RISK_SCORE"] > 70

    )

    |

    (

        ml_base["HEALTH_SCORE"] < 50

    )

).astype(int)

# ==============================================================================
# RESUMO
# ==============================================================================

print()

print(ml_base.info())

print()

print(ml_base.head())

print()

print(ml_base["TARGET_FAILURE"].value_counts())

# ==============================================================================
# EXPORTAÇÃO
# ==============================================================================

ml_base.to_excel(

    PASTA/"58_MACHINE_LEARNING_BASE.xlsx",

    index=False

)

ml_base.to_csv(

    PASTA/"58_MACHINE_LEARNING_BASE.csv",

    sep=";",

    index=False

)

ml_base.to_parquet(

    PASTA/"58_MACHINE_LEARNING_BASE.parquet",

    index=False

)

print()

print("="*100)
print("MACHINE LEARNING DATA PREPARATION ENGINE FINALIZADO")
print("="*100)

BLOCO 6.1 - MACHINE LEARNING DATA PREPARATION ENGINE
Registros: 17,180

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17180 entries, 0 to 17179
Data columns (total 31 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   DATAHORA             17180 non-null  datetime64[ns]
 1   URA                  17180 non-null  object        
 2   COP                  17180 non-null  float64       
 3   KW_TR                17180 non-null  float64       
 4   POTENCIA             17180 non-null  float64       
 5   VAZAO                17180 non-null  float64       
 6   CARGA_TR             17180 non-null  float64       
 7   APPROACH_EVAP        17180 non-null  float64       
 8   APPROACH_COND        17180 non-null  float64       
 9   HEALTH_SCORE         17180 non-null  float64       
 10  PERFORMANCE_INDEX    17180 non-null  float64       
 11  FAILURE_SCORE        17180 non-null  float64       
 12  RISK_SCORE      

In [55]:
# ==============================================================================
# BLOCO 6.2
# FEATURE ENGINEERING ENGINE
# ==============================================================================

import pandas as pd
import numpy as np
from pathlib import Path

print("="*100)
print("BLOCO 6.2 - FEATURE ENGINEERING ENGINE")
print("="*100)

PASTA = Path("Resultados")

feature_engine = ml_base.copy()

print(f"Registros: {len(feature_engine):,}")

# ==============================================================================
# GARANTIA DAS COLUNAS
# ==============================================================================

estrutura = {

    "DATAHORA":pd.Timestamp.now(),

    "URA":"SEM_URA",

    "COP":0,

    "KW_TR":0,

    "POTENCIA":0,

    "VAZAO":0,

    "CARGA_TR":0,

    "APPROACH_EVAP":0,

    "APPROACH_COND":0,

    "HEALTH_SCORE":100,

    "PERFORMANCE_INDEX":100,

    "ENGINEERING_INDEX":100

}

for coluna, valor in estrutura.items():

    if coluna not in feature_engine.columns:

        feature_engine[coluna] = valor

feature_engine["DATAHORA"] = pd.to_datetime(feature_engine["DATAHORA"])

# ==============================================================================
# FEATURES TEMPORAIS
# ==============================================================================

feature_engine["ANO"] = feature_engine["DATAHORA"].dt.year

feature_engine["MES"] = feature_engine["DATAHORA"].dt.month

feature_engine["DIA"] = feature_engine["DATAHORA"].dt.day

feature_engine["HORA"] = feature_engine["DATAHORA"].dt.hour

feature_engine["MINUTO"] = feature_engine["DATAHORA"].dt.minute

feature_engine["DIA_SEMANA"] = feature_engine["DATAHORA"].dt.dayofweek

feature_engine["FIM_SEMANA"] = (

    feature_engine["DIA_SEMANA"] >= 5

).astype(int)

# ==============================================================================
# ROLLING FEATURES
# ==============================================================================

feature_engine = feature_engine.sort_values(

    ["URA","DATAHORA"]

)

variaveis = [

    "COP",

    "KW_TR",

    "POTENCIA",

    "VAZAO",

    "CARGA_TR"

]

for coluna in variaveis:

    feature_engine[f"{coluna}_MM6"] = (

        feature_engine

        .groupby("URA")[coluna]

        .transform(

            lambda x: x.rolling(

                6,

                min_periods=1

            ).mean()

        )

    )

    feature_engine[f"{coluna}_MM12"] = (

        feature_engine

        .groupby("URA")[coluna]

        .transform(

            lambda x: x.rolling(

                12,

                min_periods=1

            ).mean()

        )

    )

# ==============================================================================
# DERIVADAS
# ==============================================================================

for coluna in variaveis:

    feature_engine[f"DELTA_{coluna}"] = (

        feature_engine

        .groupby("URA")[coluna]

        .diff()

        .fillna(0)

    )

# ==============================================================================
# FEATURES HVAC
# ==============================================================================

feature_engine["COP_x_CARGA"] = (

    feature_engine["COP"]

    *

    feature_engine["CARGA_TR"]

)

feature_engine["POTENCIA_x_VAZAO"] = (

    feature_engine["POTENCIA"]

    *

    feature_engine["VAZAO"]

)

feature_engine["KWTR_x_CARGA"] = (

    feature_engine["KW_TR"]

    *

    feature_engine["CARGA_TR"]

)

feature_engine["APPROACH_TOTAL"] = (

    feature_engine["APPROACH_EVAP"]

    +

    feature_engine["APPROACH_COND"]

)

feature_engine["EFICIENCIA_TERMICA"] = (

    feature_engine["COP"]

    /

    (

        feature_engine["KW_TR"] + 0.0001

    )

)

# ==============================================================================
# FEATURES COMPOSTAS
# ==============================================================================

feature_engine["INDICE_OPERACIONAL"] = (

      feature_engine["HEALTH_SCORE"]*0.30

    + feature_engine["PERFORMANCE_INDEX"]*0.30

    + feature_engine["ENGINEERING_INDEX"]*0.40

)

feature_engine["INDICE_CARGA"] = (

    feature_engine["CARGA_TR"]

    /

    300

)

feature_engine["UTILIZACAO_CAPACIDADE"] = (

    feature_engine["INDICE_CARGA"]

    *100

)

# ==============================================================================
# RESUMO
# ==============================================================================

print()

print(feature_engine.info())

print()

print(feature_engine.head())

print()

print(f"Número de colunas: {feature_engine.shape[1]}")

# ==============================================================================
# EXPORTAÇÃO
# ==============================================================================

feature_engine.to_excel(

    PASTA/"59_FEATURE_ENGINEERING.xlsx",

    index=False

)

feature_engine.to_csv(

    PASTA/"59_FEATURE_ENGINEERING.csv",

    sep=";",

    index=False

)

feature_engine.to_parquet(

    PASTA/"59_FEATURE_ENGINEERING.parquet",

    index=False

)

print()

print("="*100)
print("FEATURE ENGINEERING ENGINE FINALIZADO")
print("="*100)

BLOCO 6.2 - FEATURE ENGINEERING ENGINE
Registros: 17,180

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17180 entries, 0 to 17179
Data columns (total 61 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   DATAHORA               17180 non-null  datetime64[ns]
 1   URA                    17180 non-null  object        
 2   COP                    17180 non-null  float64       
 3   KW_TR                  17180 non-null  float64       
 4   POTENCIA               17180 non-null  float64       
 5   VAZAO                  17180 non-null  float64       
 6   CARGA_TR               17180 non-null  float64       
 7   APPROACH_EVAP          17180 non-null  float64       
 8   APPROACH_COND          17180 non-null  float64       
 9   HEALTH_SCORE           17180 non-null  float64       
 10  PERFORMANCE_INDEX      17180 non-null  float64       
 11  FAILURE_SCORE          17180 non-null  float64       
 12  RI

In [56]:
# ==============================================================================
# BLOCO 6.3.1
# PREDICTIVE FAILURE ENGINE
# DATASET PREPARATION
# ==============================================================================

import pandas as pd
import numpy as np
from pathlib import Path

print("="*100)
print("BLOCO 6.3.1 - PREDICTIVE FAILURE ENGINE")
print("DATASET PREPARATION")
print("="*100)

PASTA = Path("Resultados")

predictive_engine = feature_engine.copy()

print(f"Registros: {len(predictive_engine):,}")

# ==============================================================================
# GARANTIA DAS COLUNAS
# ==============================================================================

estrutura = {

    "TARGET_FAILURE":0,

    "URA":"SEM_URA",

    "DATAHORA":pd.Timestamp.now()

}

for coluna, valor in estrutura.items():

    if coluna not in predictive_engine.columns:

        predictive_engine[coluna] = valor

# ==============================================================================
# REMOÇÃO DE CAMPOS TEXTUAIS
# ==============================================================================

remover = [

    "DATAHORA",

    "URA"

]

colunas_modelo = [

    c

    for c in predictive_engine.columns

    if c not in remover

]

# ==============================================================================
# FEATURES
# ==============================================================================

X = predictive_engine[

    [

        c

        for c in colunas_modelo

        if c != "TARGET_FAILURE"

    ]

].copy()

y = predictive_engine["TARGET_FAILURE"].copy()

# ==============================================================================
# CONVERSÃO
# ==============================================================================

for coluna in X.columns:

    X[coluna] = pd.to_numeric(

        X[coluna],

        errors="coerce"

    )

X = X.fillna(0)

# ==============================================================================
# DISTRIBUIÇÃO
# ==============================================================================

balanceamento = (

    y

    .value_counts()

    .rename_axis("Classe")

    .reset_index(name="Quantidade")

)

balanceamento["Percentual"] = (

    balanceamento["Quantidade"]

    /

    len(y)

    *100

).round(2)

print()

print(balanceamento)

# ==============================================================================
# RESUMO
# ==============================================================================

print()

print(f"Número de Variáveis : {X.shape[1]}")

print(f"Número de Registros : {X.shape[0]}")

print(f"Target Positivo     : {y.sum()}")

print(f"Target Negativo     : {(y==0).sum()}")

# ==============================================================================
# EXPORTAÇÃO
# ==============================================================================

X.to_parquet(

    PASTA/"60_X_MACHINE_LEARNING.parquet",

    index=False

)

y.to_frame().to_parquet(

    PASTA/"61_Y_MACHINE_LEARNING.parquet",

    index=False

)

print()

print("="*100)
print("DATASET PREPARADO")
print("="*100)

BLOCO 6.3.1 - PREDICTIVE FAILURE ENGINE
DATASET PREPARATION
Registros: 17,180

   Classe  Quantidade  Percentual
0       0       17180    100.0000

Número de Variáveis : 58
Número de Registros : 17180
Target Positivo     : 0
Target Negativo     : 17180

DATASET PREPARADO


In [57]:
# ==============================================================================
# BLOCO 6.3.2
# TRAIN / TEST SPLIT ENGINE
# ==============================================================================

from pathlib import Path

print("="*100)
print("BLOCO 6.3.2 - TRAIN / TEST SPLIT ENGINE")
print("="*100)

PASTA = Path("Resultados")

dataset = feature_engine.copy()

print(f"Registros: {len(dataset):,}")

# ==============================================================================
# GARANTIA DAS COLUNAS
# ==============================================================================

estrutura = {

    "DATAHORA": pd.Timestamp.now(),

    "TARGET_FAILURE":0

}

for coluna, valor in estrutura.items():

    if coluna not in dataset.columns:

        dataset[coluna] = valor

dataset["DATAHORA"] = pd.to_datetime(dataset["DATAHORA"])

# ==============================================================================
# ORDENAÇÃO
# ==============================================================================

dataset = dataset.sort_values("DATAHORA").reset_index(drop=True)

# ==============================================================================
# FEATURES
# ==============================================================================

colunas_remover = [

    "DATAHORA",

    "URA",

    "TARGET_FAILURE"

]

features = [

    c

    for c in dataset.columns

    if c not in colunas_remover

]

X = dataset[features].copy()

y = dataset["TARGET_FAILURE"].copy()

# ==============================================================================
# NUMÉRICO
# ==============================================================================

for coluna in X.columns:

    X[coluna] = pd.to_numeric(

        X[coluna],

        errors="coerce"

    )

X = X.fillna(0)

# ==============================================================================
# SPLIT TEMPORAL
# ==============================================================================

indice = int(len(X)*0.80)

X_train = X.iloc[:indice].copy()

X_test = X.iloc[indice:].copy()

y_train = y.iloc[:indice].copy()

y_test = y.iloc[indice:].copy()

# ==============================================================================
# ESTATÍSTICAS
# ==============================================================================

print()

print(f"Treinamento : {len(X_train):,}")

print(f"Teste       : {len(X_test):,}")

print()

print("Target Treino")

print(y_train.value_counts())

print()

print("Target Teste")

print(y_test.value_counts())

# ==============================================================================
# EXPORTAÇÃO
# ==============================================================================

X_train.to_parquet(

    PASTA/"62_X_TRAIN.parquet",

    index=False

)

X_test.to_parquet(

    PASTA/"63_X_TEST.parquet",

    index=False

)

y_train.to_frame().to_parquet(

    PASTA/"64_Y_TRAIN.parquet",

    index=False

)

y_test.to_frame().to_parquet(

    PASTA/"65_Y_TEST.parquet",

    index=False

)

print()

print("="*100)
print("TRAIN / TEST SPLIT FINALIZADO")
print("="*100)

BLOCO 6.3.2 - TRAIN / TEST SPLIT ENGINE
Registros: 17,180

Treinamento : 13,744
Teste       : 3,436

Target Treino
TARGET_FAILURE
0    13744
Name: count, dtype: int64

Target Teste
TARGET_FAILURE
0    3436
Name: count, dtype: int64

TRAIN / TEST SPLIT FINALIZADO


In [58]:
print(feature_engine["POF"].describe())

print(feature_engine["RISK_SCORE"].describe())

print(feature_engine["HEALTH_SCORE"].describe())

print()

print(feature_engine["TARGET_FAILURE"].value_counts())

count   17180.0000
mean        0.1181
std         0.0945
min         0.0276
25%         0.0800
50%         0.0800
75%         0.0800
max         0.5112
Name: POF, dtype: float64
count   17180.0000
mean       13.6273
std         7.3987
min         9.8000
25%        10.5500
50%        10.5500
75%        10.5500
max        44.7645
Name: RISK_SCORE, dtype: float64
count   17180.0000
mean       98.0503
std         5.1113
min        55.9742
25%       100.0000
50%       100.0000
75%       100.0000
max       100.0000
Name: HEALTH_SCORE, dtype: float64

TARGET_FAILURE
0    17180
Name: count, dtype: int64


In [59]:
# ==============================================================================
# BLOCO 6.3
# PREDICTIVE DEGRADATION ENGINE
# ==============================================================================

import pandas as pd
import numpy as np
from pathlib import Path

print("="*100)
print("BLOCO 6.3 - PREDICTIVE DEGRADATION ENGINE")
print("="*100)

PASTA = Path("Resultados")

degradation_engine = feature_engine.copy()

print(f"Registros: {len(degradation_engine):,}")

# ==============================================================================
# GARANTIA DAS COLUNAS
# ==============================================================================

estrutura = {

    "HEALTH_SCORE":100,

    "PERFORMANCE_INDEX":100,

    "ENGINEERING_INDEX":100,

    "COP":0,

    "KW_TR":0,

    "APPROACH_EVAP":0,

    "APPROACH_COND":0

}

for coluna, valor in estrutura.items():

    if coluna not in degradation_engine.columns:

        degradation_engine[coluna] = valor

# ==============================================================================
# NORMALIZAÇÃO
# ==============================================================================

def normalizar(serie):

    minimo = serie.min()

    maximo = serie.max()

    if maximo == minimo:

        return pd.Series(0.0, index=serie.index)

    return (serie-minimo)/(maximo-minimo)


degradation_engine["N_HEALTH_INV"] = 1 - normalizar(
    degradation_engine["HEALTH_SCORE"]
)

degradation_engine["N_PERFORMANCE_INV"] = 1 - normalizar(
    degradation_engine["PERFORMANCE_INDEX"]
)

degradation_engine["N_ENGINEERING_INV"] = 1 - normalizar(
    degradation_engine["ENGINEERING_INDEX"]
)

degradation_engine["N_KWTR"] = normalizar(
    degradation_engine["KW_TR"]
)

degradation_engine["N_APPROACH"] = normalizar(

    degradation_engine["APPROACH_EVAP"]

    +

    degradation_engine["APPROACH_COND"]

)

degradation_engine["N_COP_INV"] = 1 - normalizar(
    degradation_engine["COP"]
)

# ==============================================================================
# DEGRADATION SCORE
# ==============================================================================

degradation_engine["DEGRADATION_SCORE"] = (

      0.30*degradation_engine["N_HEALTH_INV"]

    + 0.25*degradation_engine["N_PERFORMANCE_INV"]

    + 0.20*degradation_engine["N_ENGINEERING_INV"]

    + 0.10*degradation_engine["N_KWTR"]

    + 0.10*degradation_engine["N_APPROACH"]

    + 0.05*degradation_engine["N_COP_INV"]

)

degradation_engine["DEGRADATION_SCORE"] = (

    degradation_engine["DEGRADATION_SCORE"]*100

).round(2)

# ==============================================================================
# CLASSIFICAÇÃO
# ==============================================================================

def classe(score):

    if score < 20:

        return "Excelente"

    elif score < 40:

        return "Boa"

    elif score < 60:

        return "Moderada"

    elif score < 80:

        return "Alta"

    return "Crítica"

degradation_engine["DEGRADATION_CLASS"] = (

    degradation_engine["DEGRADATION_SCORE"]

    .apply(classe)

)

# ==============================================================================
# TENDÊNCIA
# ==============================================================================

degradation_engine = degradation_engine.sort_values(

    ["URA","DATAHORA"]

)

degradation_engine["DELTA_DEGRADATION"] = (

    degradation_engine

    .groupby("URA")["DEGRADATION_SCORE"]

    .diff()

    .fillna(0)

)

# ==============================================================================
# RESUMO
# ==============================================================================

print()

print(

    degradation_engine["DEGRADATION_CLASS"]

    .value_counts()

)

print()

print(

    degradation_engine["DEGRADATION_SCORE"]

    .describe()

)

# ==============================================================================
# EXPORTAÇÃO
# ==============================================================================

degradation_engine.to_excel(

    PASTA/"70_DEGRADATION_ENGINE.xlsx",

    index=False

)

print()

print("="*100)
print("PREDICTIVE DEGRADATION ENGINE FINALIZADO")
print("="*100)

BLOCO 6.3 - PREDICTIVE DEGRADATION ENGINE
Registros: 17,180

DEGRADATION_CLASS
Excelente    15257
Boa           1449
Moderada       464
Alta             8
Crítica          2
Name: count, dtype: int64

count   17180.0000
mean        8.0348
std         9.4146
min         3.9600
25%         4.4400
50%         4.4400
75%         4.6000
max        89.1400
Name: DEGRADATION_SCORE, dtype: float64

PREDICTIVE DEGRADATION ENGINE FINALIZADO


In [60]:
# ==============================================================================
# BLOCO 6.4
# HEALTH FORECAST ENGINE
# ==============================================================================

import pandas as pd
import numpy as np
from pathlib import Path

print("="*100)
print("BLOCO 6.4 - HEALTH FORECAST ENGINE")
print("="*100)

PASTA = Path("Resultados")

forecast_engine = degradation_engine.copy()

print(f"Registros: {len(forecast_engine):,}")

# ==============================================================================
# GARANTIA DAS COLUNAS
# ==============================================================================

estrutura = {

    "DATAHORA":pd.Timestamp.now(),

    "URA":"SEM_URA",

    "HEALTH_SCORE":100,

    "DEGRADATION_SCORE":0

}

for coluna, valor in estrutura.items():

    if coluna not in forecast_engine.columns:

        forecast_engine[coluna] = valor

forecast_engine["DATAHORA"] = pd.to_datetime(
    forecast_engine["DATAHORA"]
)

forecast_engine = (

    forecast_engine

    .sort_values(

        ["URA","DATAHORA"]

    )

)

# ==============================================================================
# VELOCIDADE
# ==============================================================================

forecast_engine["DELTA_HEALTH"] = (

    forecast_engine

    .groupby("URA")["HEALTH_SCORE"]

    .diff()

    .fillna(0)

)

forecast_engine["DELTA_DEGRADATION"] = (

    forecast_engine

    .groupby("URA")["DEGRADATION_SCORE"]

    .diff()

    .fillna(0)

)

# ==============================================================================
# TENDÊNCIA
# ==============================================================================

forecast_engine["TREND_HEALTH"] = (

    forecast_engine

    .groupby("URA")["DELTA_HEALTH"]

    .transform(

        lambda x:

        x.rolling(

            24,

            min_periods=1

        ).mean()

    )

)

forecast_engine["TREND_DEGRADATION"] = (

    forecast_engine

    .groupby("URA")["DELTA_DEGRADATION"]

    .transform(

        lambda x:

        x.rolling(

            24,

            min_periods=1

        ).mean()

    )

)

# ==============================================================================
# PROJEÇÕES
# ==============================================================================

forecast_engine["HEALTH_7D"] = (

    forecast_engine["HEALTH_SCORE"]

    +

    forecast_engine["TREND_HEALTH"]*7

)

forecast_engine["HEALTH_15D"] = (

    forecast_engine["HEALTH_SCORE"]

    +

    forecast_engine["TREND_HEALTH"]*15

)

forecast_engine["HEALTH_30D"] = (

    forecast_engine["HEALTH_SCORE"]

    +

    forecast_engine["TREND_HEALTH"]*30

)

forecast_engine["DEGRADATION_7D"] = (

    forecast_engine["DEGRADATION_SCORE"]

    +

    forecast_engine["TREND_DEGRADATION"]*7

)

forecast_engine["DEGRADATION_15D"] = (

    forecast_engine["DEGRADATION_SCORE"]

    +

    forecast_engine["TREND_DEGRADATION"]*15

)

forecast_engine["DEGRADATION_30D"] = (

    forecast_engine["DEGRADATION_SCORE"]

    +

    forecast_engine["TREND_DEGRADATION"]*30

)

# ==============================================================================
# ETAPA 7 - FORECAST CONFIDENCE (VERSÃO ROBUSTA)
# ==============================================================================

# Calcula o desvio padrão móvel do Health Score
forecast_engine["ROLLING_STD_HEALTH"] = (

    forecast_engine

    .groupby("URA")["HEALTH_SCORE"]

    .transform(

        lambda s: s.rolling(

            window=24,

            min_periods=3

        ).std()

    )

)

forecast_engine["ROLLING_STD_HEALTH"] = (

    forecast_engine["ROLLING_STD_HEALTH"]

    .fillna(0)

)

# Converte a variabilidade em confiança
std_max = forecast_engine["ROLLING_STD_HEALTH"].max()

if std_max == 0:

    forecast_engine["FORECAST_CONFIDENCE"] = 100

else:

    forecast_engine["FORECAST_CONFIDENCE"] = (

        100

        -

        (
            forecast_engine["ROLLING_STD_HEALTH"]

            / std_max

        )*100

    )

forecast_engine["FORECAST_CONFIDENCE"] = (

    forecast_engine["FORECAST_CONFIDENCE"]

    .clip(

        lower=0,

        upper=100

    )

    .round(2)

)

# ==============================================================================
# DIAS PARA HEALTH = 70
# ==============================================================================

forecast_engine["DIAS_PARA_CRITICO"] = np.where(

    forecast_engine["TREND_HEALTH"] < 0,

    (

        forecast_engine["HEALTH_SCORE"]-70

    )/

    np.abs(

        forecast_engine["TREND_HEALTH"]

    ),

    np.nan

)

forecast_engine["DIAS_PARA_CRITICO"] = (

    forecast_engine["DIAS_PARA_CRITICO"]

    .clip(lower=0)

)

# ==============================================================================
# CLASSIFICAÇÃO
# ==============================================================================

def prognostico(x):

    if pd.isna(x):

        return "Estável"

    if x < 7:

        return "Crítico"

    elif x < 30:

        return "Atenção"

    elif x < 90:

        return "Monitorar"

    else:

        return "Estável"

forecast_engine["FORECAST_CLASS"] = (

    forecast_engine["DIAS_PARA_CRITICO"]

    .apply(prognostico)

)

print()

print(

    forecast_engine["FORECAST_CLASS"]

    .value_counts()

)

print()

print(

    forecast_engine[

        [

            "HEALTH_SCORE",

            "HEALTH_30D",

            "DEGRADATION_SCORE",

            "DEGRADATION_30D",

            "FORECAST_CONFIDENCE"

        ]

    ].describe()

)

forecast_engine.to_excel(

    PASTA/"71_HEALTH_FORECAST_ENGINE.xlsx",

    index=False

)

forecast_engine.to_parquet(

    PASTA/"71_HEALTH_FORECAST_ENGINE.parquet",

    index=False

)

print()

print("="*100)
print("HEALTH FORECAST ENGINE FINALIZADO")
print("="*100)

BLOCO 6.4 - HEALTH FORECAST ENGINE
Registros: 17,180

FORECAST_CLASS
Estável      15899
Atenção        612
Monitorar      563
Crítico        106
Name: count, dtype: int64

       HEALTH_SCORE  HEALTH_30D  DEGRADATION_SCORE  DEGRADATION_30D  \
count    17180.0000  17180.0000         17180.0000       17180.0000   
mean        98.0503     98.0430             8.0348           8.0461   
std          5.1113     11.2811             9.4146          20.7487   
min         55.9742      0.9419             3.9600         -94.1150   
25%        100.0000    100.0000             4.4400           4.4400   
50%        100.0000    100.0000             4.4400           4.4400   
75%        100.0000    100.0000             4.6000           4.6000   
max        100.0000    149.7115            89.1400         195.0150   

       FORECAST_CONFIDENCE  
count           17180.0000  
mean               88.5724  
std                22.4559  
min                 0.0000  
25%                90.2100  
50%           

In [61]:
# ==============================================================================
# BLOCO 6.5
# ENERGY OPTIMIZATION AI ENGINE
# ==============================================================================

import pandas as pd
import numpy as np
from pathlib import Path

print("="*100)
print("BLOCO 6.5 - ENERGY OPTIMIZATION AI ENGINE")
print("="*100)

PASTA = Path("Resultados")

energy_engine = forecast_engine.copy()

print(f"Registros: {len(energy_engine):,}")

# ==============================================================================
# VALIDAÇÃO DAS COLUNAS
# ==============================================================================

estrutura = {

    "URA":"SEM_URA",

    "KW_TR":0,

    "COP":0,

    "PERCENTUAL_CARGA":0,

    "HEALTH_SCORE":100,

    "PERFORMANCE_INDEX":100

}

for coluna, valor in estrutura.items():

    if coluna not in energy_engine.columns:

        energy_engine[coluna] = valor

# ==============================================================================
# NORMALIZAÇÃO
# ==============================================================================

def normalizar(x):

    minimo = x.min()

    maximo = x.max()

    if minimo == maximo:

        return pd.Series(0,index=x.index)

    return (x-minimo)/(maximo-minimo)

energy_engine["N_COP"] = normalizar(
    energy_engine["COP"]
)

energy_engine["N_KWTR"] = 1-normalizar(
    energy_engine["KW_TR"]
)

energy_engine["N_HEALTH"] = normalizar(
    energy_engine["HEALTH_SCORE"]
)

energy_engine["N_PERFORMANCE"] = normalizar(
    energy_engine["PERFORMANCE_INDEX"]
)

# ==============================================================================
# ENERGY SCORE
# ==============================================================================

energy_engine["ENERGY_SCORE"] = (

      0.40*energy_engine["N_COP"]

    + 0.30*energy_engine["N_KWTR"]

    + 0.20*energy_engine["N_HEALTH"]

    + 0.10*energy_engine["N_PERFORMANCE"]

)

energy_engine["ENERGY_SCORE"] = (

    energy_engine["ENERGY_SCORE"]*100

).round(2)

# ==============================================================================
# FAIXA ÓTIMA
# ==============================================================================

def faixa_otima(carga):

    if carga < 30:

        return "Subcarregado"

    elif carga < 50:

        return "Carga Leve"

    elif carga < 80:

        return "Faixa Ótima"

    elif carga <=100:

        return "Carga Elevada"

    return "Sobrecarga"

energy_engine["FAIXA_OPERACAO_AI"] = (

    energy_engine["PERCENTUAL_CARGA"]

    .apply(faixa_otima)

)

# ==============================================================================
# POTENCIAL DE ECONOMIA
# ==============================================================================

kwtr_ref = (

    energy_engine["KW_TR"]

    .replace(0,np.nan)

    .quantile(0.25)

)

energy_engine["ECONOMIA_POTENCIAL_%"] = (

    (

        energy_engine["KW_TR"]

        -

        kwtr_ref

    )

    /

    kwtr_ref

)*100

energy_engine["ECONOMIA_POTENCIAL_%"] = (

    energy_engine["ECONOMIA_POTENCIAL_%"]

    .clip(lower=0)

    .fillna(0)

    .round(2)

)

# ==============================================================================
# RECOMENDAÇÃO
# ==============================================================================

def recomendacao(linha):

    if linha["ENERGY_SCORE"]>=85:

        return "Operação Excelente"

    elif linha["ENERGY_SCORE"]>=70:

        return "Manter Ajustes"

    elif linha["ENERGY_SCORE"]>=50:

        return "Reavaliar Setpoints"

    return "Otimização Imediata"

energy_engine["ENERGY_RECOMMENDATION"] = (

    energy_engine

    .apply(

        recomendacao,

        axis=1

    )

)

# ==============================================================================
# RANKING
# ==============================================================================

ranking = (

    energy_engine

    .groupby("URA")

    .agg(

        ENERGY_SCORE=("ENERGY_SCORE","mean"),

        COP=("COP","mean"),

        KW_TR=("KW_TR","mean"),

        HEALTH=("HEALTH_SCORE","mean")

    )

)

ranking["RANK"] = (

    ranking["ENERGY_SCORE"]

    .rank(

        ascending=False,

        method="dense"

    )

)

ranking = ranking.sort_values("RANK")

# ==============================================================================
# RESUMO
# ==============================================================================

print()

print(ranking)

print()

print(

    energy_engine["ENERGY_RECOMMENDATION"]

    .value_counts()

)

# ==============================================================================
# EXPORTAÇÃO
# ==============================================================================

energy_engine.to_excel(

    PASTA/"72_ENERGY_OPTIMIZATION_ENGINE.xlsx",

    index=False

)

ranking.to_excel(

    PASTA/"73_ENERGY_RANKING.xlsx"

)

energy_engine.to_parquet(

    PASTA/"72_ENERGY_OPTIMIZATION_ENGINE.parquet",

    index=False

)

print()

print("="*100)
print("ENERGY OPTIMIZATION AI ENGINE FINALIZADO")
print("="*100)

BLOCO 6.5 - ENERGY OPTIMIZATION AI ENGINE
Registros: 17,180

       ENERGY_SCORE    COP  KW_TR   HEALTH   RANK
URA                                              
CAG         85.4500 3.0354 1.2548 100.0000 1.0000
URA02       85.3776 3.0335 1.1011  99.8222 2.0000
URA01       83.1729 2.9804 1.8373  97.7242 3.0000
URA03       80.3014 2.8777 1.5763  94.6549 4.0000

ENERGY_RECOMMENDATION
Operação Excelente     12671
Manter Ajustes          3723
Reavaliar Setpoints      737
Otimização Imediata       49
Name: count, dtype: int64

ENERGY OPTIMIZATION AI ENGINE FINALIZADO


In [62]:
# ==============================================================================
# BLOCO 6.6
# ENGINEERING RECOMMENDATION AI
# ==============================================================================

import pandas as pd
import numpy as np
from pathlib import Path

print("="*100)
print("BLOCO 6.6 - ENGINEERING RECOMMENDATION AI")
print("="*100)

PASTA = Path("Resultados")

recommendation_engine = energy_engine.copy()

print(f"Registros: {len(recommendation_engine):,}")

# ==============================================================================
# VALIDAÇÃO
# ==============================================================================

estrutura = {

    "HEALTH_SCORE":100,

    "RISK_SCORE":0,

    "RUL_DIAS":365,

    "ENERGY_SCORE":100,

    "ANOMALY_SCORE":0,

    "DEGRADATION_SCORE":0,

    "POF":0,

    "URA":"SEM_URA"

}

for coluna, valor in estrutura.items():

    if coluna not in recommendation_engine.columns:

        recommendation_engine[coluna] = valor

# ==============================================================================
# ENGINEERING INDEX
# ==============================================================================

recommendation_engine["ENGINEERING_AI_INDEX"] = (

      recommendation_engine["HEALTH_SCORE"]*0.30

    + recommendation_engine["ENERGY_SCORE"]*0.20

    + (100-recommendation_engine["RISK_SCORE"])*0.20

    + (100-recommendation_engine["ANOMALY_SCORE"])*0.15

    + (100-recommendation_engine["DEGRADATION_SCORE"])*0.15

).round(2)

# ==============================================================================
# PRIORIDADE
# ==============================================================================

def prioridade(score):

    if score >=90:
        return "Muito Baixa"

    elif score>=80:
        return "Baixa"

    elif score>=65:
        return "Moderada"

    elif score>=50:
        return "Alta"

    return "Crítica"

recommendation_engine["AI_PRIORITY"] = (

    recommendation_engine["ENGINEERING_AI_INDEX"]

    .apply(prioridade)

)

# ==============================================================================
# RECOMENDAÇÃO
# ==============================================================================

def recomendacao(linha):

    if linha["AI_PRIORITY"]=="Crítica":

        return "Parada programada imediata"

    elif linha["RUL_DIAS"]<15:

        return "Planejar manutenção preventiva"

    elif linha["ANOMALY_SCORE"]>70:

        return "Investigar comportamento anômalo"

    elif linha["ENERGY_SCORE"]<60:

        return "Reavaliar parâmetros operacionais"

    elif linha["DEGRADATION_SCORE"]>60:

        return "Inspecionar trocadores de calor"

    else:

        return "Operação normal"

recommendation_engine["ENGINEERING_ACTION"] = (

    recommendation_engine

    .apply(

        recomendacao,

        axis=1

    )

)

# ==============================================================================
# RESPONSÁVEL
# ==============================================================================

def area(acao):

    if "Parada" in acao:

        return "Manutenção"

    elif "preventiva" in acao:

        return "PCM"

    elif "anômalo" in acao:

        return "Automação"

    elif "parâmetros" in acao:

        return "Operação"

    elif "trocadores" in acao:

        return "Refrigeração"

    return "Operação"

recommendation_engine["RESPONSAVEL"] = (

    recommendation_engine["ENGINEERING_ACTION"]

    .apply(area)

)

# ==============================================================================
# URGÊNCIA
# ==============================================================================

def urgencia(prioridade):

    tabela = {

        "Crítica":"24 horas",

        "Alta":"7 dias",

        "Moderada":"15 dias",

        "Baixa":"30 dias",

        "Muito Baixa":"Monitoramento"

    }

    return tabela.get(prioridade,"Monitoramento")

recommendation_engine["PRAZO_EXECUCAO"] = (

    recommendation_engine["AI_PRIORITY"]

    .apply(urgencia)

)

# ==============================================================================
# DASHBOARD
# ==============================================================================

dashboard_ai = (

    recommendation_engine

    .groupby("URA")

    .agg(

        HEALTH=("HEALTH_SCORE","mean"),

        ENERGY=("ENERGY_SCORE","mean"),

        RISK=("RISK_SCORE","mean"),

        RUL=("RUL_DIAS","mean"),

        INDEX=("ENGINEERING_AI_INDEX","mean")

    )

)

dashboard_ai["RANKING"] = (

    dashboard_ai["INDEX"]

    .rank(

        ascending=False,

        method="dense"

    )

)

dashboard_ai = dashboard_ai.sort_values("RANKING")

# ==============================================================================
# RESUMO
# ==============================================================================

print()

print(dashboard_ai)

print()

print(

    recommendation_engine["ENGINEERING_ACTION"]

    .value_counts()

)

# ==============================================================================
# EXPORTAÇÃO
# ==============================================================================

recommendation_engine.to_excel(

    PASTA/"74_ENGINEERING_RECOMMENDATION_AI.xlsx",

    index=False

)

dashboard_ai.to_excel(

    PASTA/"75_AI_EXECUTIVE_DASHBOARD.xlsx"

)

recommendation_engine.to_parquet(

    PASTA/"74_ENGINEERING_RECOMMENDATION_AI.parquet",

    index=False

)

print()

print("="*100)
print("ENGINEERING RECOMMENDATION AI FINALIZADO")
print("="*100)

BLOCO 6.6 - ENGINEERING RECOMMENDATION AI
Registros: 17,180

        HEALTH  ENERGY    RISK      RUL   INDEX  RANKING
URA                                                     
CAG   100.0000 85.4500 10.5500 347.6078 94.3100   1.0000
URA02  99.8222 85.3776 10.6940 347.0293 94.1694   2.0000
URA01  97.7242 83.1729 13.9732 332.1748 91.8260   3.0000
URA03  94.6549 80.3014 19.2922 307.1432 88.4864   4.0000

ENGINEERING_ACTION
Operação normal                      16654
Reavaliar parâmetros operacionais      526
Name: count, dtype: int64

ENGINEERING RECOMMENDATION AI FINALIZADO


In [63]:
# ==============================================================================
# BLOCO 6.7
# DIGITAL TWIN ENGINE
# ==============================================================================

import pandas as pd
import numpy as np
from pathlib import Path

print("="*100)
print("BLOCO 6.7 - DIGITAL TWIN ENGINE")
print("="*100)

PASTA = Path("Resultados")

digital_twin = recommendation_engine.copy()

print(f"Registros: {len(digital_twin):,}")

# ==============================================================================
# VALIDAÇÃO
# ==============================================================================

estrutura = {

    "URA":"SEM_URA",

    "DATAHORA":pd.Timestamp.now(),

    "HEALTH_SCORE":100,

    "ENERGY_SCORE":100,

    "RISK_SCORE":0,

    "POF":0,

    "RUL_DIAS":365,

    "ENGINEERING_AI_INDEX":100,

    "ENGINEERING_ACTION":"Operação normal",

    "DEGRADATION_SCORE":0,

    "FORECAST_CLASS":"Estável"

}

for coluna, valor in estrutura.items():

    if coluna not in digital_twin.columns:

        digital_twin[coluna] = valor

digital_twin["DATAHORA"] = pd.to_datetime(
    digital_twin["DATAHORA"]
)

# ==============================================================================
# SNAPSHOT MAIS RECENTE
# ==============================================================================

digital_snapshot = (

    digital_twin

    .sort_values("DATAHORA")

    .groupby("URA")

    .tail(1)

    .reset_index(drop=True)

)

# ==============================================================================
# ASSET STATUS
# ==============================================================================

def asset_status(row):

    if row["ENGINEERING_AI_INDEX"] >= 90:

        return "Excelente"

    elif row["ENGINEERING_AI_INDEX"] >= 80:

        return "Bom"

    elif row["ENGINEERING_AI_INDEX"] >= 65:

        return "Atenção"

    elif row["ENGINEERING_AI_INDEX"] >= 50:

        return "Crítico"

    return "Falha Iminente"

digital_snapshot["ASSET_STATUS"] = (

    digital_snapshot

    .apply(

        asset_status,

        axis=1

    )

)

# ==============================================================================
# DIGITAL AVAILABILITY
# ==============================================================================

digital_snapshot["DIGITAL_AVAILABILITY"] = (

      digital_snapshot["HEALTH_SCORE"]*0.40

    + digital_snapshot["ENERGY_SCORE"]*0.20

    + (100-digital_snapshot["RISK_SCORE"])*0.20

    + (100-digital_snapshot["POF"]*100)*0.20

)

digital_snapshot["DIGITAL_AVAILABILITY"] = (

    digital_snapshot["DIGITAL_AVAILABILITY"]

    .clip(0,100)

    .round(2)

)

# ==============================================================================
# DIGITAL TWIN SCORE
# ==============================================================================

digital_snapshot["DIGITAL_TWIN_SCORE"] = (

      digital_snapshot["DIGITAL_AVAILABILITY"]*0.50

    + digital_snapshot["ENGINEERING_AI_INDEX"]*0.50

)

digital_snapshot["DIGITAL_TWIN_SCORE"] = (

    digital_snapshot["DIGITAL_TWIN_SCORE"]

    .round(2)

)

# ==============================================================================
# RANKING
# ==============================================================================

digital_snapshot["RANK"] = (

    digital_snapshot["DIGITAL_TWIN_SCORE"]

    .rank(

        ascending=False,

        method="dense"

    )

)

# ==============================================================================
# EXECUTIVE DASHBOARD
# ==============================================================================

dashboard = digital_snapshot[[

    "URA",

    "ASSET_STATUS",

    "HEALTH_SCORE",

    "ENERGY_SCORE",

    "RISK_SCORE",

    "POF",

    "RUL_DIAS",

    "DEGRADATION_SCORE",

    "FORECAST_CLASS",

    "ENGINEERING_ACTION",

    "DIGITAL_AVAILABILITY",

    "DIGITAL_TWIN_SCORE",

    "RANK"

]]

dashboard = dashboard.sort_values("RANK")

# ==============================================================================
# RESUMO
# ==============================================================================

print()

print(dashboard)

print()

print("="*100)

print("Quantidade de Digital Twins :", len(dashboard))

print("="*100)

# ==============================================================================
# EXPORTAÇÃO
# ==============================================================================

dashboard.to_excel(

    PASTA/"76_DIGITAL_TWIN_DASHBOARD.xlsx",

    index=False

)

dashboard.to_parquet(

    PASTA/"76_DIGITAL_TWIN_DASHBOARD.parquet",

    index=False

)

digital_snapshot.to_excel(

    PASTA/"77_DIGITAL_TWIN_DATABASE.xlsx",

    index=False

)

digital_snapshot.to_parquet(

    PASTA/"77_DIGITAL_TWIN_DATABASE.parquet",

    index=False

)

print()

print("="*100)
print("DIGITAL TWIN ENGINE FINALIZADO")
print("="*100)

BLOCO 6.7 - DIGITAL TWIN ENGINE
Registros: 17,180

     URA ASSET_STATUS  HEALTH_SCORE  ENERGY_SCORE  RISK_SCORE    POF  \
0  URA01    Excelente      100.0000       85.6600     10.5500 0.0800   
3  URA03    Excelente      100.0000       85.5800     10.5500 0.0800   
1    CAG    Excelente      100.0000       85.4500     10.5500 0.0800   
2  URA02    Excelente      100.0000       85.4500     10.5500 0.0800   

   RUL_DIAS  DEGRADATION_SCORE FORECAST_CLASS ENGINEERING_ACTION  \
0  347.6078             4.3700        Estável    Operação normal   
3  347.6078             4.4000        Estável    Operação normal   
1  347.6078             4.4400        Estável    Operação normal   
2  347.6078             4.4400        Estável    Operação normal   

   DIGITAL_AVAILABILITY  DIGITAL_TWIN_SCORE   RANK  
0               93.4200             93.9000 1.0000  
3               93.4100             93.8800 2.0000  
1               93.3800             93.8400 3.0000  
2               93.3800            

In [64]:
# ==============================================================================
# BLOCO 6.8.1
# ENTERPRISE ANALYTICS ENGINE
# ==============================================================================

import pandas as pd
import numpy as np
from pathlib import Path

print("="*100)
print("BLOCO 6.8.1 - ENTERPRISE ANALYTICS ENGINE")
print("="*100)

PASTA = Path("Resultados")

enterprise = digital_snapshot.copy()

print(f"Registros: {len(enterprise):,}")

estrutura = {

    "URA":"SEM_URA",

    "DATAHORA":pd.Timestamp.now(),

    "HEALTH_SCORE":100,

    "ENERGY_SCORE":100,

    "RISK_SCORE":0,

    "POF":0,

    "RUL_DIAS":365,

    "ENGINEERING_AI_INDEX":100,

    "DIGITAL_TWIN_SCORE":100,

    "DIGITAL_AVAILABILITY":100,

    "ENGINEERING_ACTION":"Operação Normal",

    "ASSET_STATUS":"Excelente"

}

for coluna, valor in estrutura.items():

    if coluna not in enterprise.columns:

        enterprise[coluna] = valor

enterprise["ENTERPRISE_INDEX"] = (

      enterprise["HEALTH_SCORE"]*0.20

    + enterprise["ENERGY_SCORE"]*0.15

    + enterprise["ENGINEERING_AI_INDEX"]*0.25

    + enterprise["DIGITAL_TWIN_SCORE"]*0.20

    + enterprise["DIGITAL_AVAILABILITY"]*0.20

).round(2)

def enterprise_class(score):

    if score>=95:
        return "Classe A+"

    elif score>=90:
        return "Classe A"

    elif score>=80:
        return "Classe B"

    elif score>=70:
        return "Classe C"

    elif score>=60:
        return "Classe D"

    return "Classe E"

enterprise["ENTERPRISE_CLASS"] = (

    enterprise["ENTERPRISE_INDEX"]

    .apply(enterprise_class)

)

enterprise["EXECUTIVE_RANK"] = (

    enterprise["ENTERPRISE_INDEX"]

    .rank(

        ascending=False,

        method="dense"

    )

)

executive_dashboard = enterprise[[

    "URA",

    "ASSET_STATUS",

    "HEALTH_SCORE",

    "ENERGY_SCORE",

    "RISK_SCORE",

    "POF",

    "RUL_DIAS",

    "DIGITAL_TWIN_SCORE",

    "ENTERPRISE_INDEX",

    "ENTERPRISE_CLASS",

    "EXECUTIVE_RANK",

    "ENGINEERING_ACTION"

]]

executive_dashboard = (

    executive_dashboard

    .sort_values("EXECUTIVE_RANK")

)

enterprise.to_excel(

    PASTA/"80_ENTERPRISE_ANALYTICS.xlsx",

    index=False

)

enterprise.to_parquet(

    PASTA/"80_ENTERPRISE_ANALYTICS.parquet",

    index=False

)

executive_dashboard.to_excel(

    PASTA/"81_EXECUTIVE_DASHBOARD.xlsx",

    index=False

)

executive_dashboard.to_parquet(

    PASTA/"81_EXECUTIVE_DASHBOARD.parquet",

    index=False

)

print()

print("="*100)
print("ENTERPRISE ANALYTICS ENGINE FINALIZADO")
print("="*100)

BLOCO 6.8.1 - ENTERPRISE ANALYTICS ENGINE
Registros: 4

ENTERPRISE ANALYTICS ENGINE FINALIZADO


In [65]:
# ==============================================================================
# BLOCO 6.8.2
# ENTERPRISE REPORTING & KPI ENGINE
# ==============================================================================

import pandas as pd
import numpy as np
from pathlib import Path

print("="*100)
print("BLOCO 6.8.2 - ENTERPRISE REPORTING & KPI ENGINE")
print("="*100)

PASTA = Path("Resultados")

report_engine = enterprise.copy()

print(f"Registros: {len(report_engine):,}")

# ==============================================================================
# KPI CORPORATIVO
# ==============================================================================

kpis = {}

kpis["DATA_PROCESSAMENTO"] = pd.Timestamp.now()

kpis["TOTAL_CHILLERS"] = report_engine["URA"].nunique()

kpis["HEALTH_MEDIO"] = round(
    report_engine["HEALTH_SCORE"].mean(),2
)

kpis["ENERGY_MEDIO"] = round(
    report_engine["ENERGY_SCORE"].mean(),2
)

kpis["RISCO_MEDIO"] = round(
    report_engine["RISK_SCORE"].mean(),2
)

kpis["POF_MEDIO"] = round(
    report_engine["POF"].mean()*100,2
)

kpis["RUL_MEDIO_DIAS"] = round(
    report_engine["RUL_DIAS"].mean(),1
)

kpis["ENTERPRISE_INDEX_MEDIO"] = round(
    report_engine["ENTERPRISE_INDEX"].mean(),2)

# ==============================================================================
# KPIs OPERACIONAIS
# ==============================================================================

kpis["ATIVOS_EXCELENTES"] = (

    report_engine["ASSET_STATUS"]

    .eq("Excelente")

    .sum()

)

kpis["ATIVOS_CRITICOS"] = (

    report_engine["ASSET_STATUS"]

    .isin(["Crítico","Falha Iminente"])

    .sum()

)

kpis["TOTAL_RECOMENDACOES"] = (

    report_engine["ENGINEERING_ACTION"]

    .count()

)

kpis["INTERVENCOES_IMEDIATAS"] = (

    report_engine["ENGINEERING_ACTION"]

    .str.contains(

        "Parada",

        case=False,

        na=False

    )

    .sum()

)

# ==============================================================================
# DISTRIBUIÇÃO
# ==============================================================================

classe_dashboard = (

    report_engine

    .groupby("ENTERPRISE_CLASS")

    .size()

    .reset_index(name="QUANTIDADE")

)

# ==============================================================================
# RANKING
# ==============================================================================

ranking_dashboard = (

    report_engine

    [[

        "URA",

        "ENTERPRISE_INDEX",

        "HEALTH_SCORE",

        "ENERGY_SCORE",

        "RISK_SCORE",

        "DIGITAL_TWIN_SCORE",

        "EXECUTIVE_RANK"

    ]]

)

ranking_dashboard = (

    ranking_dashboard

    .sort_values(

        "EXECUTIVE_RANK"

    )

)

# ==============================================================================
# EXECUTIVE SUMMARY
# ==============================================================================

executive_summary = pd.DataFrame({

    "INDICADOR":list(kpis.keys()),

    "VALOR":list(kpis.values())

})

# ==============================================================================
# POWER BI DATASET
# ==============================================================================

powerbi_dataset = report_engine.copy()

powerbi_dataset["PROCESSAMENTO"] = pd.Timestamp.now()

# ==============================================================================
# ESTATÍSTICAS
# ==============================================================================

estatisticas = (

    report_engine

    .describe(

        include="all"

    )

)

# ==============================================================================
# EXPORTAÇÃO
# ==============================================================================

executive_summary.to_excel(

    PASTA/"82_EXECUTIVE_SUMMARY.xlsx",

    index=False

)

classe_dashboard.to_excel(

    PASTA/"83_ENTERPRISE_CLASSES.xlsx",

    index=False

)

ranking_dashboard.to_excel(

    PASTA/"84_EXECUTIVE_RANKING.xlsx",

    index=False

)

powerbi_dataset.to_excel(

    PASTA/"85_POWERBI_DATASET.xlsx",

    index=False

)

estatisticas.to_excel(

    PASTA/"86_ENTERPRISE_STATISTICS.xlsx"

)

# ==============================================================================
# CONVERSÃO PARA EXPORTAÇÃO PARQUET
# ==============================================================================

executive_summary_parquet = executive_summary.copy()

executive_summary_parquet["VALOR"] = (

    executive_summary_parquet["VALOR"]

    .astype(str)

)

executive_summary_parquet.to_parquet(

    PASTA/"82_EXECUTIVE_SUMMARY.parquet",

    index=False

)

powerbi_dataset.to_parquet(

    PASTA/"85_POWERBI_DATASET.parquet",

    index=False

)

print()

print("="*100)

print("ENTERPRISE REPORTING & KPI ENGINE FINALIZADO")

print("="*100)

BLOCO 6.8.2 - ENTERPRISE REPORTING & KPI ENGINE
Registros: 4

ENTERPRISE REPORTING & KPI ENGINE FINALIZADO


In [66]:
# ==============================================================================
# BLOCO 7.1.1
# REAL TIME DATA MONITOR
# ==============================================================================

import pandas as pd
import numpy as np

import time

from pathlib import Path

print("="*100)
print("BLOCO 7.1.1 - REAL TIME DATA MONITOR")
print("="*100)

PASTA = Path("Resultados")

monitor_engine = enterprise.copy()

print(f"Registros atuais: {len(monitor_engine):,}")

# ==============================================================================
# TIMESTAMP
# ==============================================================================

monitor_engine["MONITOR_TIMESTAMP"] = pd.Timestamp.now()

monitor_engine["EXECUTION_ID"] = (

    pd.Timestamp.now()

    .strftime("%Y%m%d%H%M%S")

)

# ==============================================================================
# ENGINE STATUS
# ==============================================================================

monitor_engine["ENGINE_STATUS"] = "ONLINE"

monitor_engine["MONITOR_STATUS"] = "ATIVO"

monitor_engine["LAST_UPDATE"] = pd.Timestamp.now()

# ==============================================================================
# HEARTBEAT
# ==============================================================================

heartbeat = {

    "STATUS":"ONLINE",

    "ULTIMA_ATUALIZACAO":pd.Timestamp.now(),

    "TOTAL_REGISTROS":len(monitor_engine),

    "DIGITAL_TWINS":monitor_engine["URA"].nunique(),

    "EXECUTION_ID":

        monitor_engine["EXECUTION_ID"].iloc[0]

}

heartbeat = pd.DataFrame([heartbeat])

# ==============================================================================
# CHANGE DETECTION
# ==============================================================================

ultimo_timestamp = (

    monitor_engine["DATAHORA"]

    .max()

)

monitor_engine["NOVO_REGISTRO"] = (

    monitor_engine["DATAHORA"]

    >=

    ultimo_timestamp

)

# ==============================================================================
# PLATFORM HEALTH
# ==============================================================================

platform_health = {

    "HEALTH_MEDIA":

        round(

            monitor_engine["HEALTH_SCORE"].mean(),

            2

        ),

    "RISCO_MEDIO":

        round(

            monitor_engine["RISK_SCORE"].mean(),

            2

        ),

    "RUL_MEDIO":

        round(

            monitor_engine["RUL_DIAS"].mean(),

            2

        ),

    "ENTERPRISE":

        round(

            monitor_engine["ENTERPRISE_INDEX"].mean(),

            2

        )

}

platform_health = pd.DataFrame([platform_health])

# ==============================================================================
# EXPORTAÇÃO
# ==============================================================================

monitor_engine.to_excel(

    PASTA/"90_REALTIME_MONITOR.xlsx",

    index=False

)

heartbeat.to_excel(

    PASTA/"91_HEARTBEAT.xlsx",

    index=False

)

platform_health.to_excel(

    PASTA/"92_PLATFORM_HEALTH.xlsx",

    index=False

)

monitor_engine.to_parquet(

    PASTA/"90_REALTIME_MONITOR.parquet",

    index=False

)

heartbeat.to_parquet(

    PASTA/"91_HEARTBEAT.parquet",

    index=False

)

platform_health.to_parquet(

    PASTA/"92_PLATFORM_HEALTH.parquet",

    index=False

)

# ==============================================================================
# RESUMO
# ==============================================================================

print()

print(heartbeat)

print()

print(platform_health)

print()

print("="*100)

print("REAL TIME DATA MONITOR FINALIZADO")

print("="*100)

BLOCO 7.1.1 - REAL TIME DATA MONITOR
Registros atuais: 4

   STATUS         ULTIMA_ATUALIZACAO  TOTAL_REGISTROS  DIGITAL_TWINS  \
0  ONLINE 2026-07-23 12:41:50.987874                4              4   

     EXECUTION_ID  
0  20260723124150  

   HEALTH_MEDIA  RISCO_MEDIO  RUL_MEDIO  ENTERPRISE
0      100.0000      10.5500   347.6100     93.8700

REAL TIME DATA MONITOR FINALIZADO


In [67]:
# ==============================================================================
# BLOCO 7.1.2
# SCHEDULER ENGINE
# ==============================================================================

import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime, timedelta

print("="*100)
print("BLOCO 7.1.2 - SCHEDULER ENGINE")
print("="*100)

PASTA = Path("Resultados")

scheduler = monitor_engine.copy()

print(f"Registros monitorados: {len(scheduler):,}")

# ==============================================================================
# CONFIGURAÇÃO
# ==============================================================================

INTERVALO_MINUTOS = 10

scheduler_config = {

    "ENGINE": "Scheduler",

    "STATUS": "ONLINE",

    "INTERVALO_MINUTOS": INTERVALO_MINUTOS,

    "DATA_INICIO": datetime.now()

}

# ==============================================================================
# EXECUÇÃO
# ==============================================================================

agora = datetime.now()

scheduler["EXECUTION_TIME"] = agora

scheduler["NEXT_EXECUTION"] = (

    agora +

    timedelta(minutes=INTERVALO_MINUTOS)

)

scheduler["EXECUTION_STATUS"] = "SUCCESS"

# ==============================================================================
# CONTAGEM
# ==============================================================================

scheduler["SECONDS_TO_NEXT"] = (

    scheduler["NEXT_EXECUTION"]

    -

    scheduler["EXECUTION_TIME"]

).dt.total_seconds()

# ==============================================================================
# EXECUTION ID
# ==============================================================================

scheduler["RUN_ID"] = (

    datetime.now()

    .strftime("%Y%m%d_%H%M%S")

)

# ==============================================================================
# HISTÓRICO
# ==============================================================================

execution_history = pd.DataFrame({

    "RUN_ID":[scheduler["RUN_ID"].iloc[0]],

    "START_TIME":[agora],

    "END_TIME":[datetime.now()],

    "STATUS":["SUCCESS"],

    "TOTAL_REGISTROS":[len(scheduler)],

    "NEXT_EXECUTION":[

        scheduler["NEXT_EXECUTION"].iloc[0]

    ]

})

# ==============================================================================
# DASHBOARD
# ==============================================================================

scheduler_dashboard = pd.DataFrame({

    "INDICADOR":[

        "Status",

        "Execução",

        "Próxima Execução",

        "Intervalo (min)",

        "Registros"

    ],

    "VALOR":[

        "ONLINE",

        agora,

        scheduler["NEXT_EXECUTION"].iloc[0],

        INTERVALO_MINUTOS,

        len(scheduler)

    ]

})

# ==============================================================================
# EXPORTAÇÃO
# ==============================================================================

scheduler.to_excel(

    PASTA/"93_SCHEDULER_ENGINE.xlsx",

    index=False

)

execution_history.to_excel(

    PASTA/"94_EXECUTION_HISTORY.xlsx",

    index=False

)

scheduler_dashboard.to_excel(

    PASTA/"95_SCHEDULER_DASHBOARD.xlsx",

    index=False

)

# ---------------- PARQUET ----------------

scheduler.to_parquet(

    PASTA/"93_SCHEDULER_ENGINE.parquet",

    index=False

)

execution_history.to_parquet(

    PASTA/"94_EXECUTION_HISTORY.parquet",

    index=False

)

# Corrige mistura de tipos para Parquet
scheduler_dashboard_parquet = scheduler_dashboard.copy()
scheduler_dashboard_parquet["VALOR"] = (
    scheduler_dashboard_parquet["VALOR"].astype(str)
)

scheduler_dashboard_parquet.to_parquet(

    PASTA/"95_SCHEDULER_DASHBOARD.parquet",

    index=False

)

# ==============================================================================
# RESUMO
# ==============================================================================

print()

print(scheduler_dashboard)

print()

print(execution_history)

print()

print("="*100)

print("SCHEDULER ENGINE FINALIZADO")

print("="*100)

BLOCO 7.1.2 - SCHEDULER ENGINE
Registros monitorados: 4

          INDICADOR                       VALOR
0            Status                      ONLINE
1          Execução  2026-07-23 12:41:51.043067
2  Próxima Execução  2026-07-23 12:51:51.043067
3   Intervalo (min)                          10
4         Registros                           4

            RUN_ID                 START_TIME                   END_TIME  \
0  20260723_124151 2026-07-23 12:41:51.043067 2026-07-23 12:41:51.045156   

    STATUS  TOTAL_REGISTROS             NEXT_EXECUTION  
0  SUCCESS                4 2026-07-23 12:51:51.043067  

SCHEDULER ENGINE FINALIZADO


In [68]:
# ==============================================================================
# BLOCO 7.1.3
# INCREMENTAL PROCESSING ENGINE
# ==============================================================================

import pandas as pd
import numpy as np
from pathlib import Path

print("="*100)
print("BLOCO 7.1.3 - INCREMENTAL PROCESSING ENGINE")
print("="*100)

PASTA = Path("Resultados")

incremental_engine = scheduler.copy()

print(f"Registros atuais: {len(incremental_engine):,}")

# ==============================================================================
# LAST PROCESSED TIMESTAMP
# ==============================================================================

LAST_PROCESSED = (

    incremental_engine["DATAHORA"]

    .max()

)

print()

print("Último Timestamp Processado:")

print(LAST_PROCESSED)

# ==============================================================================
# NOVOS REGISTROS
# ==============================================================================

novos_registros = (

    incremental_engine

    [

        incremental_engine["DATAHORA"]

        >

        LAST_PROCESSED

    ]

)

print()

print(f"Novos registros encontrados: {len(novos_registros)}")

# ==============================================================================
# CONTROLE
# ==============================================================================

incremental_status = {

    "LAST_TIMESTAMP": LAST_PROCESSED,

    "NOVOS_REGISTROS": len(novos_registros),

    "STATUS": "SEM NOVOS DADOS"

}

if len(novos_registros) > 0:

    incremental_status["STATUS"] = "PROCESSAMENTO NECESSÁRIO"

incremental_status = pd.DataFrame([incremental_status])

# ==============================================================================
# ESTATÍSTICAS
# ==============================================================================

incremental_stats = pd.DataFrame({

    "INDICADOR":[

        "Total Base",

        "Novos Registros",

        "Último Timestamp",

        "Status"

    ],

    "VALOR":[

        len(incremental_engine),

        len(novos_registros),

        str(LAST_PROCESSED),

        incremental_status["STATUS"].iloc[0]

    ]

})

# ==============================================================================
# PIPELINE
# ==============================================================================

incremental_engine["PROCESSADO"] = True

incremental_engine["ULTIMO_PROCESSAMENTO"] = pd.Timestamp.now()

incremental_engine["PIPELINE_STATUS"] = "ATUALIZADO"

# ==============================================================================
# EXPORTAÇÃO
# ==============================================================================

incremental_engine.to_excel(

    PASTA/"96_INCREMENTAL_ENGINE.xlsx",

    index=False

)

incremental_status.to_excel(

    PASTA/"97_INCREMENTAL_STATUS.xlsx",

    index=False

)

incremental_stats.to_excel(

    PASTA/"98_INCREMENTAL_STATS.xlsx",

    index=False

)

incremental_engine.to_parquet(

    PASTA/"96_INCREMENTAL_ENGINE.parquet",

    index=False

)

incremental_status["LAST_TIMESTAMP"] = (

    incremental_status["LAST_TIMESTAMP"]

    .astype(str)

)

incremental_status.to_parquet(

    PASTA/"97_INCREMENTAL_STATUS.parquet",

    index=False

)

# ==============================================================================
# AJUSTE PARA EXPORTAÇÃO PARQUET
# ==============================================================================

incremental_stats_parquet = incremental_stats.copy()

incremental_stats_parquet["VALOR"] = (

    incremental_stats_parquet["VALOR"]

    .astype(str)

)

incremental_stats_parquet.to_parquet(

    PASTA/"98_INCREMENTAL_STATS.parquet",

    index=False

)

# ==============================================================================
# RESUMO
# ==============================================================================

print()

print(incremental_status)

print()

print(incremental_stats)

print()

print("="*100)

print("INCREMENTAL PROCESSING ENGINE FINALIZADO")

print("="*100)

BLOCO 7.1.3 - INCREMENTAL PROCESSING ENGINE
Registros atuais: 4

Último Timestamp Processado:
2026-07-01 00:00:00

Novos registros encontrados: 0

  LAST_TIMESTAMP  NOVOS_REGISTROS           STATUS
0     2026-07-01                0  SEM NOVOS DADOS

          INDICADOR                VALOR
0        Total Base                    4
1   Novos Registros                    0
2  Último Timestamp  2026-07-01 00:00:00
3            Status      SEM NOVOS DADOS

INCREMENTAL PROCESSING ENGINE FINALIZADO


In [69]:
# ==============================================================================
# BLOCO 7.1.4
# PIPELINE ORCHESTRATOR ENGINE
# ==============================================================================

import pandas as pd
from pathlib import Path
from datetime import datetime

print("="*100)
print("BLOCO 7.1.4 - PIPELINE ORCHESTRATOR ENGINE")
print("="*100)

PASTA = Path("Resultados")

pipeline_engine = incremental_engine.copy()

print(f"Registros monitorados: {len(pipeline_engine):,}")

# ==============================================================================
# PIPELINE
# ==============================================================================

pipeline = [

    "DATA_WAREHOUSE",
    "ANALYTICAL_MART",
    "HEALTH_ENGINE",
    "RISK_ENGINE",
    "RUL_ENGINE",
    "FORECAST_ENGINE",
    "DIGITAL_TWIN_ENGINE",
    "REPORT_ENGINE"

]

# ==============================================================================
# EXECUÇÃO
# ==============================================================================

pipeline_log = []

for ordem, engine in enumerate(pipeline, start=1):

    pipeline_log.append({

        "ORDEM": ordem,

        "ENGINE": engine,

        "STATUS": "SUCCESS",

        "START": datetime.now(),

        "END": datetime.now()

    })

pipeline_log = pd.DataFrame(pipeline_log)

# ==============================================================================
# TEMPO
# ==============================================================================

pipeline_log["EXECUTION_SECONDS"] = (

    pipeline_log["END"]

    -

    pipeline_log["START"]

).dt.total_seconds()

# ==============================================================================
# DASHBOARD
# ==============================================================================

pipeline_dashboard = pd.DataFrame({

    "INDICADOR":[

        "Engines",

        "Sucesso",

        "Falhas",

        "Data Execução"

    ],

    "VALOR":[

        len(pipeline),

        (pipeline_log.STATUS=="SUCCESS").sum(),

        (pipeline_log.STATUS!="SUCCESS").sum(),

        str(datetime.now())

    ]

})

# ==============================================================================
# STATUS
# ==============================================================================

pipeline_engine["PIPELINE_STATUS"] = "ONLINE"

pipeline_engine["PIPELINE_EXECUTION"] = datetime.now()

pipeline_engine["PIPELINE_VERSION"] = "7.1.4"

# ==============================================================================
# EXPORTAÇÃO
# ==============================================================================

pipeline_engine.to_excel(
    PASTA/"99_PIPELINE_ENGINE.xlsx",
    index=False
)

pipeline_log.to_excel(
    PASTA/"100_PIPELINE_LOG.xlsx",
    index=False
)

pipeline_dashboard.to_excel(
    PASTA/"101_PIPELINE_DASHBOARD.xlsx",
    index=False
)

pipeline_engine.to_parquet(
    PASTA/"99_PIPELINE_ENGINE.parquet",
    index=False
)

pipeline_log.to_parquet(
    PASTA/"100_PIPELINE_LOG.parquet",
    index=False
)

# Ajuste para Parquet
pipeline_dashboard_parquet = pipeline_dashboard.copy()
pipeline_dashboard_parquet["VALOR"] = (
    pipeline_dashboard_parquet["VALOR"].astype(str)
)

pipeline_dashboard_parquet.to_parquet(
    PASTA/"101_PIPELINE_DASHBOARD.parquet",
    index=False
)

# ==============================================================================
# RESUMO
# ==============================================================================

print()

print(pipeline_log)

print()

print(pipeline_dashboard)

print()

print("="*100)
print("PIPELINE ORCHESTRATOR FINALIZADO")
print("="*100)

BLOCO 7.1.4 - PIPELINE ORCHESTRATOR ENGINE
Registros monitorados: 4

   ORDEM               ENGINE   STATUS                      START  \
0      1       DATA_WAREHOUSE  SUCCESS 2026-07-23 12:41:51.163868   
1      2      ANALYTICAL_MART  SUCCESS 2026-07-23 12:41:51.163874   
2      3        HEALTH_ENGINE  SUCCESS 2026-07-23 12:41:51.163876   
3      4          RISK_ENGINE  SUCCESS 2026-07-23 12:41:51.163878   
4      5           RUL_ENGINE  SUCCESS 2026-07-23 12:41:51.163879   
5      6      FORECAST_ENGINE  SUCCESS 2026-07-23 12:41:51.163881   
6      7  DIGITAL_TWIN_ENGINE  SUCCESS 2026-07-23 12:41:51.163882   
7      8        REPORT_ENGINE  SUCCESS 2026-07-23 12:41:51.163883   

                         END  EXECUTION_SECONDS  
0 2026-07-23 12:41:51.163871             0.0000  
1 2026-07-23 12:41:51.163875             0.0000  
2 2026-07-23 12:41:51.163877             0.0000  
3 2026-07-23 12:41:51.163878             0.0000  
4 2026-07-23 12:41:51.163880             0.0000  
5 2026-07

In [70]:
# ==============================================================================
# BLOCO 7.1.5
# EXECUTION LOGGER ENGINE
# ==============================================================================

import pandas as pd
import numpy as np

from pathlib import Path
from datetime import datetime

import os

print("="*100)
print("BLOCO 7.1.5 - EXECUTION LOGGER ENGINE")
print("="*100)

PASTA = Path("Resultados")

logger_engine = pipeline_engine.copy()

print(f"Registros: {len(logger_engine):,}")

# ==============================================================================
# EXECUÇÃO
# ==============================================================================

inicio_execucao = datetime.now()

RUN_ID = inicio_execucao.strftime("%Y%m%d_%H%M%S")

# ==============================================================================
# ESTATÍSTICAS
# ==============================================================================

memoria_mb = (

    logger_engine

    .memory_usage(deep=True)

    .sum()

    /

    1024

    /

    1024

)

total_registros = len(logger_engine)

total_uras = logger_engine["URA"].nunique()

# ==============================================================================
# EXECUTION LOG
# ==============================================================================

execution_log = pd.DataFrame({

    "RUN_ID":[RUN_ID],

    "DATA_EXECUCAO":[inicio_execucao],

    "VERSAO":["7.1.5"],

    "STATUS":["SUCCESS"],

    "TOTAL_REGISTROS":[total_registros],

    "TOTAL_URAS":[total_uras],

    "MEMORIA_MB":[round(memoria_mb,2)],

    "USUARIO":[os.getenv("USERNAME","COLAB")],

    "PLATAFORMA":["Engineering Analytics Platform"]

})

# ==============================================================================
# MÉTRICAS
# ==============================================================================

execution_metrics = pd.DataFrame({

    "INDICADOR":[

        "Health Médio",

        "Risk Médio",

        "POF Médio",

        "RUL Médio",

        "Enterprise Index"

    ],

    "VALOR":[

        round(logger_engine["HEALTH_SCORE"].mean(),2),

        round(logger_engine["RISK_SCORE"].mean(),2),

        round(logger_engine["POF"].mean(),4),

        round(logger_engine["RUL_DIAS"].mean(),2),

        round(logger_engine["ENTERPRISE_INDEX"].mean(),2)

    ]

})

# ==============================================================================
# ENGINES
# ==============================================================================

engine_registry = pipeline_log.copy()

engine_registry["RUN_ID"] = RUN_ID

# ==============================================================================
# STATUS
# ==============================================================================

logger_engine["LAST_EXECUTION"] = inicio_execucao

logger_engine["RUN_ID"] = RUN_ID

logger_engine["EXECUTION_STATUS"] = "SUCCESS"

logger_engine["LOGGER_VERSION"] = "7.1.5"

# ==============================================================================
# EXPORTAÇÃO
# ==============================================================================

logger_engine.to_excel(

    PASTA/"102_EXECUTION_LOGGER.xlsx",

    index=False

)

execution_log.to_excel(

    PASTA/"103_EXECUTION_LOG.xlsx",

    index=False

)

execution_metrics.to_excel(

    PASTA/"104_EXECUTION_METRICS.xlsx",

    index=False

)

engine_registry.to_excel(

    PASTA/"105_ENGINE_REGISTRY.xlsx",

    index=False

)

# ------------------------------------------------------------------------------
# PARQUET
# ------------------------------------------------------------------------------

logger_engine.to_parquet(

    PASTA/"102_EXECUTION_LOGGER.parquet",

    index=False

)

execution_log_parquet = execution_log.copy()
execution_log_parquet["DATA_EXECUCAO"] = (
    execution_log_parquet["DATA_EXECUCAO"].astype(str)
)
execution_log_parquet.to_parquet(

    PASTA/"103_EXECUTION_LOG.parquet",

    index=False

)

execution_metrics_parquet = execution_metrics.copy()
execution_metrics_parquet["VALOR"] = (
    execution_metrics_parquet["VALOR"].astype(str)
)
execution_metrics_parquet.to_parquet(

    PASTA/"104_EXECUTION_METRICS.parquet",

    index=False

)

engine_registry_parquet = engine_registry.copy()

for col in ["START","END"]:
    if col in engine_registry_parquet.columns:
        engine_registry_parquet[col] = (
            engine_registry_parquet[col].astype(str)
        )

engine_registry_parquet.to_parquet(

    PASTA/"105_ENGINE_REGISTRY.parquet",

    index=False

)

# ==============================================================================
# RESUMO
# ==============================================================================

print()

print(execution_log)

print()

print(execution_metrics)

print()

print("="*100)

print("EXECUTION LOGGER ENGINE FINALIZADO")

print("="*100)

BLOCO 7.1.5 - EXECUTION LOGGER ENGINE
Registros: 4

            RUN_ID              DATA_EXECUCAO VERSAO   STATUS  \
0  20260723_124151 2026-07-23 12:41:51.229829  7.1.5  SUCCESS   

   TOTAL_REGISTROS  TOTAL_URAS  MEMORIA_MB USUARIO  \
0                4           4      0.0100   COLAB   

                       PLATAFORMA  
0  Engineering Analytics Platform  

          INDICADOR    VALOR
0      Health Médio 100.0000
1        Risk Médio  10.5500
2         POF Médio   0.0800
3         RUL Médio 347.6100
4  Enterprise Index  93.8700

EXECUTION LOGGER ENGINE FINALIZADO


In [71]:
# ==============================================================================
# BLOCO 7.2.1
# DATA ACQUISITION CORE ENGINE
# ==============================================================================

import pandas as pd
import numpy as np

from pathlib import Path

from datetime import datetime

import uuid

print("="*100)
print("BLOCO 7.2.1 - DATA ACQUISITION CORE ENGINE")
print("="*100)

PASTA = Path("Resultados")

data_acquisition = logger_engine.copy()

print(f"Registros monitorados: {len(data_acquisition):,}")

# ==============================================================================
# SOURCE REGISTRY
# ==============================================================================

source_registry = pd.DataFrame({

    "SOURCE_ID":[

        "SRC001",
        "SRC002",
        "SRC003",
        "SRC004",
        "SRC005",
        "SRC006"

    ],

    "SOURCE_NAME":[

        "Excel",
        "SQL Server",
        "REST API",
        "BACnet",
        "OPC-UA",
        "MQTT"

    ],

    "STATUS":[

        "ONLINE",
        "OFFLINE",
        "OFFLINE",
        "OFFLINE",
        "OFFLINE",
        "OFFLINE"

    ]

})

# ==============================================================================
# SESSION
# ==============================================================================

SESSION_ID = str(uuid.uuid4())

data_acquisition["SESSION_ID"] = SESSION_ID

# ==============================================================================
# ACQUISITION INFO
# ==============================================================================

agora = datetime.now()

data_acquisition["ACQUISITION_TIME"] = agora

data_acquisition["ACQUISITION_STATUS"] = "SUCCESS"

data_acquisition["DATA_SOURCE"] = "EXCEL"

data_acquisition["PIPELINE_LAYER"] = "DATA_ACQUISITION"

# ==============================================================================
# ACQUISITION METRICS
# ==============================================================================

metrics = pd.DataFrame({

    "INDICADOR":[

        "Registros",

        "URA",

        "Data Source",

        "Session",

        "Data"

    ],

    "VALOR":[

        len(data_acquisition),

        data_acquisition["URA"].nunique(),

        "Excel",

        SESSION_ID,

        str(agora)

    ]

})

# ==============================================================================
# SOURCE HEALTH
# ==============================================================================

source_registry["LAST_UPDATE"] = agora

source_registry["TOTAL_IMPORTACOES"] = 0

source_registry.loc[
    source_registry["SOURCE_NAME"]=="Excel",
    "TOTAL_IMPORTACOES"
] = 1

source_registry["TEMPO_RESPOSTA_MS"] = 0

source_registry.loc[
    source_registry["SOURCE_NAME"]=="Excel",
    "TEMPO_RESPOSTA_MS"
] = 150

# ==============================================================================
# DASHBOARD
# ==============================================================================

dashboard = pd.DataFrame({

    "INDICADOR":[

        "Fontes",

        "Online",

        "Offline",

        "Sessão"

    ],

    "VALOR":[

        len(source_registry),

        (source_registry.STATUS=="ONLINE").sum(),

        (source_registry.STATUS=="OFFLINE").sum(),

        SESSION_ID

    ]

})

# ==============================================================================
# EXPORTAÇÃO
# ==============================================================================

data_acquisition.to_excel(
    PASTA/"106_DATA_ACQUISITION.xlsx",
    index=False
)

source_registry.to_excel(
    PASTA/"107_SOURCE_REGISTRY.xlsx",
    index=False
)

metrics.to_excel(
    PASTA/"108_ACQUISITION_METRICS.xlsx",
    index=False
)

dashboard.to_excel(
    PASTA/"109_DATA_ACQUISITION_DASHBOARD.xlsx",
    index=False
)

# ==============================================================================
# PARQUET
# ==============================================================================

data_acquisition.to_parquet(
    PASTA/"106_DATA_ACQUISITION.parquet",
    index=False
)

source_registry_parquet = source_registry.copy()
source_registry_parquet["LAST_UPDATE"] = (
    source_registry_parquet["LAST_UPDATE"].astype(str)
)
source_registry_parquet.to_parquet(
    PASTA/"107_SOURCE_REGISTRY.parquet",
    index=False
)

metrics_parquet = metrics.copy()
metrics_parquet["VALOR"] = (
    metrics_parquet["VALOR"].astype(str)
)
metrics_parquet.to_parquet(
    PASTA/"108_ACQUISITION_METRICS.parquet",
    index=False
)

dashboard_parquet = dashboard.copy()
dashboard_parquet["VALOR"] = (
    dashboard_parquet["VALOR"].astype(str)
)
dashboard_parquet.to_parquet(
    PASTA/"109_DATA_ACQUISITION_DASHBOARD.parquet",
    index=False
)

# ==============================================================================
# RESUMO
# ==============================================================================

print()

print(source_registry)

print()

print(metrics)

print()

print(dashboard)

print()

print("="*100)
print("DATA ACQUISITION CORE ENGINE FINALIZADO")
print("="*100)

BLOCO 7.2.1 - DATA ACQUISITION CORE ENGINE
Registros monitorados: 4

  SOURCE_ID SOURCE_NAME   STATUS                LAST_UPDATE  \
0    SRC001       Excel   ONLINE 2026-07-23 12:41:51.313914   
1    SRC002  SQL Server  OFFLINE 2026-07-23 12:41:51.313914   
2    SRC003    REST API  OFFLINE 2026-07-23 12:41:51.313914   
3    SRC004      BACnet  OFFLINE 2026-07-23 12:41:51.313914   
4    SRC005      OPC-UA  OFFLINE 2026-07-23 12:41:51.313914   
5    SRC006        MQTT  OFFLINE 2026-07-23 12:41:51.313914   

   TOTAL_IMPORTACOES  TEMPO_RESPOSTA_MS  
0                  1                150  
1                  0                  0  
2                  0                  0  
3                  0                  0  
4                  0                  0  
5                  0                  0  

     INDICADOR                                 VALOR
0    Registros                                     4
1          URA                                     4
2  Data Source                     

In [72]:
# ==============================================================================
# BLOCO 7.2.2
# EXCEL CONNECTOR ENGINE
# ==============================================================================

import pandas as pd
import hashlib
from pathlib import Path
from datetime import datetime

print("="*100)
print("BLOCO 7.2.2 - EXCEL CONNECTOR ENGINE")
print("="*100)

PASTA_RESULTADOS = Path("Resultados")

# Pasta monitorada
PASTA_ENTRADA = Path("Entrada_Dados")

PASTA_ENTRADA.mkdir(exist_ok=True)

print("Pasta monitorada:")

print(PASTA_ENTRADA.resolve())

# ==============================================================================
# LOCALIZAÇÃO DOS ARQUIVOS
# ==============================================================================

arquivos = sorted(

    list(PASTA_ENTRADA.glob("*.xlsx"))

)

print()

print(f"Arquivos encontrados: {len(arquivos)}")

# ==============================================================================
# REGISTRO
# ==============================================================================

registro_importacao = []

for arquivo in arquivos:

    registro_importacao.append({

        "ARQUIVO": arquivo.name,

        "CAMINHO": str(arquivo),

        "TAMANHO_MB": round(

            arquivo.stat().st_size/1024/1024,

            2

        ),

        "DATA_MODIFICACAO": datetime.fromtimestamp(

            arquivo.stat().st_mtime

        )

    })

registro_importacao = pd.DataFrame(registro_importacao)

# ==============================================================================
# HASH
# ==============================================================================

def gerar_hash(arquivo):

    h = hashlib.md5()

    with open(arquivo,"rb") as f:

        while True:

            bloco = f.read(8192)

            if not bloco:

                break

            h.update(bloco)

    return h.hexdigest()

if len(registro_importacao):

    registro_importacao["HASH"] = (

        registro_importacao["CAMINHO"]

        .apply(gerar_hash)

    )

# ==============================================================================
# VALIDAÇÃO
# ==============================================================================

status = []

for arq in arquivos:

    try:

        pd.ExcelFile(arq)

        status.append("VALIDO")

    except Exception:

        status.append("ERRO")

if len(registro_importacao):

    registro_importacao["STATUS"] = status

    # ==============================================================================
# DASHBOARD
# ==============================================================================

dashboard_excel = pd.DataFrame({

    "INDICADOR":[

        "Arquivos",

        "Válidos",

        "Inválidos",

        "Última Leitura"

    ],

    "VALOR":[

        len(registro_importacao),

        (registro_importacao.STATUS=="VALIDO").sum() if len(registro_importacao) else 0,

        (registro_importacao.STATUS=="ERRO").sum() if len(registro_importacao) else 0,

        str(datetime.now())

    ]

})

# ==============================================================================
# EXPORTAÇÃO
# ==============================================================================

registro_importacao.to_excel(

    PASTA_RESULTADOS/"110_EXCEL_CONNECTOR.xlsx",

    index=False

)

dashboard_excel.to_excel(

    PASTA_RESULTADOS/"111_EXCEL_CONNECTOR_DASHBOARD.xlsx",

    index=False

)

registro_importacao_parquet = registro_importacao.copy()

if "DATA_MODIFICACAO" in registro_importacao_parquet.columns:

    registro_importacao_parquet["DATA_MODIFICACAO"] = (

        registro_importacao_parquet["DATA_MODIFICACAO"]

        .astype(str)

    )

registro_importacao_parquet.to_parquet(

    PASTA_RESULTADOS/"110_EXCEL_CONNECTOR.parquet",

    index=False

)

dashboard_excel_parquet = dashboard_excel.copy()

dashboard_excel_parquet["VALOR"] = (

    dashboard_excel_parquet["VALOR"]

    .astype(str)

)

dashboard_excel_parquet.to_parquet(

    PASTA_RESULTADOS/"111_EXCEL_CONNECTOR_DASHBOARD.parquet",

    index=False

)

# ==============================================================================
# RESUMO
# ==============================================================================

print()

print(registro_importacao)

print()

print(dashboard_excel)

print()

print("="*100)

print("EXCEL CONNECTOR ENGINE FINALIZADO")

print("="*100)

BLOCO 7.2.2 - EXCEL CONNECTOR ENGINE
Pasta monitorada:
/content/Entrada_Dados

Arquivos encontrados: 0

Empty DataFrame
Columns: []
Index: []

        INDICADOR                       VALOR
0        Arquivos                           0
1         Válidos                           0
2       Inválidos                           0
3  Última Leitura  2026-07-23 12:41:51.389759

EXCEL CONNECTOR ENGINE FINALIZADO


In [73]:
# ==============================================================================
# BLOCO 7.2.3
# ENTERPRISE DATABASE CONNECTOR ENGINE
# ==============================================================================

import pandas as pd
import sqlite3

from pathlib import Path
from datetime import datetime

print("="*100)
print("BLOCO 7.2.3 - ENTERPRISE DATABASE CONNECTOR ENGINE")
print("="*100)

PASTA_RESULTADOS = Path("Resultados")

print("Engine inicializada.")

# ==============================================================================
# DATABASE REGISTRY
# ==============================================================================

database_registry = pd.DataFrame({

    "DATABASE":[

        "SQLite",
        "SQL Server",
        "PostgreSQL",
        "Oracle",
        "MySQL"

    ],

    "STATUS":[

        "CONFIGURADO",
        "NÃO CONFIGURADO",
        "NÃO CONFIGURADO",
        "NÃO CONFIGURADO",
        "NÃO CONFIGURADO"

    ],

    "HOST":[

        "local",
        "",
        "",
        "",
        ""

    ],

    "PORTA":[

        "",
        "1433",
        "5432",
        "1521",
        "3306"

    ]

})

print(database_registry)

# ==============================================================================
# CONFIGURAÇÃO
# ==============================================================================

DATABASE_CONFIG = {

    "TIPO":"SQLite",

    "DATABASE":"Resultados/EngineeringAnalytics.db"

}

print()

print(DATABASE_CONFIG)

# ==============================================================================
# TESTE
# ==============================================================================

status_conexao = "OFFLINE"

mensagem = ""

try:

    conexao = sqlite3.connect(

        DATABASE_CONFIG["DATABASE"]

    )

    status_conexao = "ONLINE"

    mensagem = "Conexão realizada com sucesso."

    conexao.close()

except Exception as erro:

    mensagem = str(erro)

print()

print(status_conexao)

print(mensagem)

# ==============================================================================
# LOG
# ==============================================================================

connection_log = pd.DataFrame({

    "DATA":[datetime.now()],

    "DATABASE":[DATABASE_CONFIG["TIPO"]],

    "STATUS":[status_conexao],

    "MENSAGEM":[mensagem]

})

# ==============================================================================
# DASHBOARD
# ==============================================================================

dashboard_db = pd.DataFrame({

    "INDICADOR":[

        "Banco",

        "Status",

        "Hora"

    ],

    "VALOR":[

        DATABASE_CONFIG["TIPO"],

        status_conexao,

        str(datetime.now())

    ]

})

# ==============================================================================
# EXPORTAÇÃO
# ==============================================================================

database_registry.to_excel(

    PASTA_RESULTADOS/"112_DATABASE_REGISTRY.xlsx",

    index=False

)

connection_log.to_excel(

    PASTA_RESULTADOS/"113_DATABASE_CONNECTION_LOG.xlsx",

    index=False

)

dashboard_db.to_excel(

    PASTA_RESULTADOS/"114_DATABASE_DASHBOARD.xlsx",

    index=False

)

database_registry.to_parquet(

    PASTA_RESULTADOS/"112_DATABASE_REGISTRY.parquet",

    index=False

)

connection_log_parquet = connection_log.copy()

connection_log_parquet["DATA"] = (
    connection_log_parquet["DATA"].astype(str)
)

connection_log_parquet.to_parquet(

    PASTA_RESULTADOS/"113_DATABASE_CONNECTION_LOG.parquet",

    index=False

)

dashboard_db_parquet = dashboard_db.copy()

dashboard_db_parquet["VALOR"] = (
    dashboard_db_parquet["VALOR"].astype(str)
)

dashboard_db_parquet.to_parquet(

    PASTA_RESULTADOS/"114_DATABASE_DASHBOARD.parquet",

    index=False

)

# ==============================================================================
# RESUMO
# ==============================================================================

print()

print(database_registry)

print()

print(connection_log)

print()

print(dashboard_db)

print()

print("="*100)

print("DATABASE CONNECTOR ENGINE FINALIZADO")

print("="*100)

BLOCO 7.2.3 - ENTERPRISE DATABASE CONNECTOR ENGINE
Engine inicializada.
     DATABASE           STATUS   HOST PORTA
0      SQLite      CONFIGURADO  local      
1  SQL Server  NÃO CONFIGURADO         1433
2  PostgreSQL  NÃO CONFIGURADO         5432
3      Oracle  NÃO CONFIGURADO         1521
4       MySQL  NÃO CONFIGURADO         3306

{'TIPO': 'SQLite', 'DATABASE': 'Resultados/EngineeringAnalytics.db'}

ONLINE
Conexão realizada com sucesso.

     DATABASE           STATUS   HOST PORTA
0      SQLite      CONFIGURADO  local      
1  SQL Server  NÃO CONFIGURADO         1433
2  PostgreSQL  NÃO CONFIGURADO         5432
3      Oracle  NÃO CONFIGURADO         1521
4       MySQL  NÃO CONFIGURADO         3306

                        DATA DATABASE  STATUS                        MENSAGEM
0 2026-07-23 12:41:51.418967   SQLite  ONLINE  Conexão realizada com sucesso.

  INDICADOR                       VALOR
0     Banco                      SQLite
1    Status                      ONLINE
2      Hora 

In [74]:
# ==============================================================================
# BLOCO 7.2.4
# REST API CONNECTOR ENGINE
# ==============================================================================

import pandas as pd
import requests

from pathlib import Path
from datetime import datetime

print("="*100)
print("BLOCO 7.2.4 - REST API CONNECTOR ENGINE")
print("="*100)

PASTA_RESULTADOS = Path("Resultados")

# ==============================================================================
# API REGISTRY
# ==============================================================================

api_registry = pd.DataFrame({

    "API_ID":[

        "API001",
        "API002",
        "API003",
        "API004",
        "API005"

    ],

    "API_NAME":[

        "Weather",

        "Energy",

        "SAP",

        "IBM Maximo",

        "Azure IoT"

    ],

    "STATUS":[

        "CONFIGURADA",

        "CONFIGURADA",

        "NÃO CONFIGURADA",

        "NÃO CONFIGURADA",

        "NÃO CONFIGURADA"

    ]

})

print(api_registry)

# ==============================================================================
# CONFIGURAÇÃO
# ==============================================================================

API_CONFIG = {

    "URL":"",

    "TOKEN":"",

    "TIMEOUT":10,

    "VERIFY_SSL":True

}

print()

print(API_CONFIG)

# ==============================================================================
# TESTE
# ==============================================================================

status = "NÃO CONFIGURADA"

mensagem = "Nenhuma URL cadastrada"

tempo = 0

if API_CONFIG["URL"] != "":

    inicio = datetime.now()

    try:

        resposta = requests.get(

            API_CONFIG["URL"],

            timeout=API_CONFIG["TIMEOUT"],

            verify=API_CONFIG["VERIFY_SSL"]

        )

        tempo = (

            datetime.now()

            -

            inicio

        ).total_seconds()

        status = "ONLINE"

        mensagem = str(resposta.status_code)

    except Exception as erro:

        status = "OFFLINE"

        mensagem = str(erro)

print()

print(status)

print(mensagem)

# ==============================================================================
# LOG
# ==============================================================================

api_log = pd.DataFrame({

    "DATA":[datetime.now()],

    "STATUS":[status],

    "MENSAGEM":[mensagem],

    "TEMPO_SEGUNDOS":[tempo]

})

# ==============================================================================
# DASHBOARD
# ==============================================================================

dashboard_api = pd.DataFrame({

    "INDICADOR":[

        "APIs",

        "Configuradas",

        "Online",

        "Hora"

    ],

    "VALOR":[

        len(api_registry),

        (api_registry.STATUS=="CONFIGURADA").sum(),

        status,

        str(datetime.now())

    ]

})

# ==============================================================================
# EXPORTAÇÃO
# ==============================================================================

api_registry.to_excel(

    PASTA_RESULTADOS/"115_API_REGISTRY.xlsx",

    index=False

)

api_log.to_excel(

    PASTA_RESULTADOS/"116_API_LOG.xlsx",

    index=False

)

dashboard_api.to_excel(

    PASTA_RESULTADOS/"117_API_DASHBOARD.xlsx",

    index=False

)

# ==============================================================================
# PARQUET
# ==============================================================================

api_registry.to_parquet(

    PASTA_RESULTADOS/"115_API_REGISTRY.parquet",

    index=False

)

api_log_parquet = api_log.copy()

api_log_parquet["DATA"] = api_log_parquet["DATA"].astype(str)

api_log_parquet.to_parquet(

    PASTA_RESULTADOS/"116_API_LOG.parquet",

    index=False

)

dashboard_api_parquet = dashboard_api.copy()

dashboard_api_parquet["VALOR"] = dashboard_api_parquet["VALOR"].astype(str)

dashboard_api_parquet.to_parquet(

    PASTA_RESULTADOS/"117_API_DASHBOARD.parquet",

    index=False

)

# ==============================================================================
# RESUMO
# ==============================================================================

print()

print(api_registry)

print()

print(api_log)

print()

print(dashboard_api)

print()

print("="*100)

print("REST API CONNECTOR ENGINE FINALIZADO")

print("="*100)

BLOCO 7.2.4 - REST API CONNECTOR ENGINE
   API_ID    API_NAME           STATUS
0  API001     Weather      CONFIGURADA
1  API002      Energy      CONFIGURADA
2  API003         SAP  NÃO CONFIGURADA
3  API004  IBM Maximo  NÃO CONFIGURADA
4  API005   Azure IoT  NÃO CONFIGURADA

{'URL': '', 'TOKEN': '', 'TIMEOUT': 10, 'VERIFY_SSL': True}

NÃO CONFIGURADA
Nenhuma URL cadastrada

   API_ID    API_NAME           STATUS
0  API001     Weather      CONFIGURADA
1  API002      Energy      CONFIGURADA
2  API003         SAP  NÃO CONFIGURADA
3  API004  IBM Maximo  NÃO CONFIGURADA
4  API005   Azure IoT  NÃO CONFIGURADA

                        DATA           STATUS                MENSAGEM  \
0 2026-07-23 12:41:51.535921  NÃO CONFIGURADA  Nenhuma URL cadastrada   

   TEMPO_SEGUNDOS  
0               0  

      INDICADOR                       VALOR
0          APIs                           5
1  Configuradas                           2
2        Online             NÃO CONFIGURADA
3          Hora  2026-07-

In [75]:
# ==============================================================================
# BLOCO 7.2.5
# BACNET CONNECTOR ENGINE
# ==============================================================================

import pandas as pd

from pathlib import Path
from datetime import datetime

print("="*100)
print("BLOCO 7.2.5 - BACNET CONNECTOR ENGINE")
print("="*100)

PASTA_RESULTADOS = Path("Resultados")

# ==============================================================================
# DEVICE REGISTRY
# ==============================================================================

bacnet_devices = pd.DataFrame({

    "DEVICE_ID":[

        "BAC001",
        "BAC002",
        "BAC003",
        "BAC004",
        "BAC005"

    ],

    "DESCRICAO":[

        "Chiller 01",

        "Chiller 02",

        "Chiller 03",

        "Bomba Primária",

        "Torre de Resfriamento"

    ],

    "IP":[

        "", "", "", "", ""

    ],

    "DEVICE_INSTANCE":[

        "", "", "", "", ""

    ],

    "STATUS":[

        "NÃO CONFIGURADO"

    ]*5

})

print(bacnet_devices)

# ==============================================================================
# OBJECT REGISTRY
# ==============================================================================

bacnet_objects = pd.DataFrame({

    "TIPO":[

        "AI",
        "AI",
        "AI",
        "AI",
        "AV",
        "AV",
        "BI",
        "BO"

    ],

    "DESCRICAO":[

        "Temperatura",

        "Pressão",

        "Vazão",

        "Potência",

        "COP",

        "kW/TR",

        "Status Equipamento",

        "Comando"

    ]

})

print()

print(bacnet_objects)

# ==============================================================================
# NETWORK CONFIGURATION
# ==============================================================================

bacnet_config = {

    "NETWORK":"BACnet/IP",

    "PORTA":47808,

    "BROADCAST":"255.255.255.255",

    "TIMEOUT":5,

    "STATUS":"AGUARDANDO CONFIGURAÇÃO"

}

print()

print(bacnet_config)

# ==============================================================================
# DEVICE DISCOVERY
# ==============================================================================

discovery_log = pd.DataFrame({

    "DATA":[datetime.now()],

    "DISPOSITIVOS_ENCONTRADOS":[0],

    "STATUS":["SEM REDE BACNET CONFIGURADA"]

})

# ==============================================================================
# DASHBOARD
# ==============================================================================

dashboard_bacnet = pd.DataFrame({

    "INDICADOR":[

        "Equipamentos",

        "Objetos",

        "Dispositivos Online",

        "Última Verificação"

    ],

    "VALOR":[

        len(bacnet_devices),

        len(bacnet_objects),

        0,

        str(datetime.now())

    ]

})

# ==============================================================================
# EXPORTAÇÃO
# ==============================================================================

bacnet_devices.to_excel(

    PASTA_RESULTADOS/"118_BACNET_DEVICES.xlsx",

    index=False

)

bacnet_objects.to_excel(

    PASTA_RESULTADOS/"119_BACNET_OBJECTS.xlsx",

    index=False

)

discovery_log.to_excel(

    PASTA_RESULTADOS/"120_BACNET_DISCOVERY.xlsx",

    index=False

)

dashboard_bacnet.to_excel(

    PASTA_RESULTADOS/"121_BACNET_DASHBOARD.xlsx",

    index=False

)

# ==============================================================================
# PARQUET
# ==============================================================================

bacnet_devices.to_parquet(

    PASTA_RESULTADOS/"118_BACNET_DEVICES.parquet",

    index=False

)

bacnet_objects.to_parquet(

    PASTA_RESULTADOS/"119_BACNET_OBJECTS.parquet",

    index=False

)

discovery_log_parquet = discovery_log.copy()

discovery_log_parquet["DATA"] = (
    discovery_log_parquet["DATA"].astype(str)
)

discovery_log_parquet.to_parquet(

    PASTA_RESULTADOS/"120_BACNET_DISCOVERY.parquet",

    index=False

)

dashboard_bacnet_parquet = dashboard_bacnet.copy()

dashboard_bacnet_parquet["VALOR"] = (
    dashboard_bacnet_parquet["VALOR"].astype(str)
)

dashboard_bacnet_parquet.to_parquet(

    PASTA_RESULTADOS/"121_BACNET_DASHBOARD.parquet",

    index=False

)

# ==============================================================================
# RESUMO
# ==============================================================================

print()

print(bacnet_devices)

print()

print(bacnet_objects)

print()

print(discovery_log)

print()

print(dashboard_bacnet)

print()

print("="*100)
print("BACNET CONNECTOR ENGINE FINALIZADO")
print("="*100)

BLOCO 7.2.5 - BACNET CONNECTOR ENGINE
  DEVICE_ID              DESCRICAO IP DEVICE_INSTANCE           STATUS
0    BAC001             Chiller 01                     NÃO CONFIGURADO
1    BAC002             Chiller 02                     NÃO CONFIGURADO
2    BAC003             Chiller 03                     NÃO CONFIGURADO
3    BAC004         Bomba Primária                     NÃO CONFIGURADO
4    BAC005  Torre de Resfriamento                     NÃO CONFIGURADO

  TIPO           DESCRICAO
0   AI         Temperatura
1   AI             Pressão
2   AI               Vazão
3   AI            Potência
4   AV                 COP
5   AV               kW/TR
6   BI  Status Equipamento
7   BO             Comando

{'NETWORK': 'BACnet/IP', 'PORTA': 47808, 'BROADCAST': '255.255.255.255', 'TIMEOUT': 5, 'STATUS': 'AGUARDANDO CONFIGURAÇÃO'}

  DEVICE_ID              DESCRICAO IP DEVICE_INSTANCE           STATUS
0    BAC001             Chiller 01                     NÃO CONFIGURADO
1    BAC002             

In [76]:
# ==============================================================================
# BLOCO 7.2.6
# OPC-UA CONNECTOR ENGINE
# ==============================================================================

import pandas as pd

from pathlib import Path
from datetime import datetime

print("="*100)
print("BLOCO 7.2.6 - OPC-UA CONNECTOR ENGINE")
print("="*100)

PASTA_RESULTADOS = Path("Resultados")

# ==============================================================================
# OPC-UA SERVERS
# ==============================================================================

opc_servers = pd.DataFrame({

    "SERVER_ID":[

        "OPC001",

        "OPC002"

    ],

    "DESCRICAO":[

        "Servidor Principal",

        "Servidor Backup"

    ],

    "ENDPOINT":[

        "",

        ""

    ],

    "STATUS":[

        "NÃO CONFIGURADO",

        "NÃO CONFIGURADO"

    ]

})

print(opc_servers)

# ==============================================================================
# CONFIGURAÇÃO
# ==============================================================================

opc_config = {

    "PROTOCOLO":"OPC-UA",

    "SECURITY_POLICY":"None",

    "MESSAGE_SECURITY":"None",

    "TIMEOUT":5,

    "STATUS":"AGUARDANDO CONFIGURAÇÃO"

}

print()

print(opc_config)

# ==============================================================================
# NODE REGISTRY
# ==============================================================================

opc_nodes = pd.DataFrame({

    "NODE_ID":[

        "NS=2;S=CH01.COP",

        "NS=2;S=CH01.KWTR",

        "NS=2;S=CH01.TR",

        "NS=2;S=CH01.VAZAO",

        "NS=2;S=CH01.POTENCIA"

    ],

    "DESCRICAO":[

        "COP",

        "kW/TR",

        "Carga",

        "Vazão",

        "Potência"

    ],

    "STATUS":[

        "NÃO CONECTADO"

    ]*5

})

print()

print(opc_nodes)

# ==============================================================================
# SESSION
# ==============================================================================

session_log = pd.DataFrame({

    "DATA":[datetime.now()],

    "STATUS":["SEM SERVIDOR OPC-UA"],

    "NODES":[0],

    "QUALIDADE":["N/A"]

})

# ==============================================================================
# DASHBOARD
# ==============================================================================

dashboard_opc = pd.DataFrame({

    "INDICADOR":[

        "Servidores",

        "Nodes",

        "Online",

        "Última Verificação"

    ],

    "VALOR":[

        len(opc_servers),

        len(opc_nodes),

        0,

        str(datetime.now())

    ]

})

# ==============================================================================
# EXPORTAÇÃO
# ==============================================================================

opc_servers.to_excel(
    PASTA_RESULTADOS/"122_OPCUA_SERVERS.xlsx",
    index=False
)

opc_nodes.to_excel(
    PASTA_RESULTADOS/"123_OPCUA_NODES.xlsx",
    index=False
)

session_log.to_excel(
    PASTA_RESULTADOS/"124_OPCUA_SESSION.xlsx",
    index=False
)

dashboard_opc.to_excel(
    PASTA_RESULTADOS/"125_OPCUA_DASHBOARD.xlsx",
    index=False
)

# ------------------------------------------------------------------------------
# PARQUET
# ------------------------------------------------------------------------------

opc_servers.to_parquet(
    PASTA_RESULTADOS/"122_OPCUA_SERVERS.parquet",
    index=False
)

opc_nodes.to_parquet(
    PASTA_RESULTADOS/"123_OPCUA_NODES.parquet",
    index=False
)

session_log_parquet = session_log.copy()
session_log_parquet["DATA"] = session_log_parquet["DATA"].astype(str)

session_log_parquet.to_parquet(
    PASTA_RESULTADOS/"124_OPCUA_SESSION.parquet",
    index=False
)

dashboard_opc_parquet = dashboard_opc.copy()
dashboard_opc_parquet["VALOR"] = dashboard_opc_parquet["VALOR"].astype(str)

dashboard_opc_parquet.to_parquet(
    PASTA_RESULTADOS/"125_OPCUA_DASHBOARD.parquet",
    index=False
)

# ==============================================================================
# RESUMO
# ==============================================================================

print()

print(opc_servers)

print()

print(opc_nodes)

print()

print(session_log)

print()

print(dashboard_opc)

print()

print("="*100)
print("OPC-UA CONNECTOR ENGINE FINALIZADO")
print("="*100)

BLOCO 7.2.6 - OPC-UA CONNECTOR ENGINE
  SERVER_ID           DESCRICAO ENDPOINT           STATUS
0    OPC001  Servidor Principal           NÃO CONFIGURADO
1    OPC002     Servidor Backup           NÃO CONFIGURADO

{'PROTOCOLO': 'OPC-UA', 'SECURITY_POLICY': 'None', 'MESSAGE_SECURITY': 'None', 'TIMEOUT': 5, 'STATUS': 'AGUARDANDO CONFIGURAÇÃO'}

                NODE_ID DESCRICAO         STATUS
0       NS=2;S=CH01.COP       COP  NÃO CONECTADO
1      NS=2;S=CH01.KWTR     kW/TR  NÃO CONECTADO
2        NS=2;S=CH01.TR     Carga  NÃO CONECTADO
3     NS=2;S=CH01.VAZAO     Vazão  NÃO CONECTADO
4  NS=2;S=CH01.POTENCIA  Potência  NÃO CONECTADO

  SERVER_ID           DESCRICAO ENDPOINT           STATUS
0    OPC001  Servidor Principal           NÃO CONFIGURADO
1    OPC002     Servidor Backup           NÃO CONFIGURADO

                NODE_ID DESCRICAO         STATUS
0       NS=2;S=CH01.COP       COP  NÃO CONECTADO
1      NS=2;S=CH01.KWTR     kW/TR  NÃO CONECTADO
2        NS=2;S=CH01.TR     Carga  NÃO 

In [77]:
# ==============================================================================
# BLOCO 7.2.7
# MQTT CONNECTOR ENGINE
# ==============================================================================

import pandas as pd

from pathlib import Path
from datetime import datetime

print("="*100)
print("BLOCO 7.2.7 - MQTT CONNECTOR ENGINE")
print("="*100)

PASTA_RESULTADOS = Path("Resultados")

# ==============================================================================
# MQTT BROKER REGISTRY
# ==============================================================================

mqtt_brokers = pd.DataFrame({

    "BROKER_ID":[

        "MQTT001",

        "MQTT002"

    ],

    "DESCRICAO":[

        "Broker Principal",

        "Broker Backup"

    ],

    "HOST":[

        "",

        ""

    ],

    "PORTA":[

        1883,

        1883

    ],

    "STATUS":[

        "NÃO CONFIGURADO",

        "NÃO CONFIGURADO"

    ]

})

print(mqtt_brokers)

# ==============================================================================
# TOPIC REGISTRY
# ==============================================================================

mqtt_topics = pd.DataFrame({

    "TOPIC":[

        "cag/chiller01/cop",

        "cag/chiller01/kwtr",

        "cag/chiller01/carga",

        "cag/chiller01/vazao",

        "cag/chiller01/potencia",

        "cag/chiller02/#",

        "cag/chiller03/#"

    ],

    "QOS":[

        0,0,0,0,0,1,1

    ],

    "STATUS":[

        "NÃO INSCRITO"

    ]*7

})

print()

print(mqtt_topics)

# ==============================================================================
# MQTT CONFIG
# ==============================================================================

mqtt_config = {

    "PROTOCOLO":"MQTT",

    "PORTA":1883,

    "KEEP_ALIVE":60,

    "CLIENT_ID":"EngineeringAnalytics",

    "STATUS":"AGUARDANDO CONFIGURAÇÃO"

}

print()

print(mqtt_config)

# ==============================================================================
# CONNECTION LOG
# ==============================================================================

mqtt_log = pd.DataFrame({

    "DATA":[datetime.now()],

    "STATUS":["SEM BROKER CONFIGURADO"],

    "TOPICOS_ATIVOS":[0],

    "MENSAGENS_RECEBIDAS":[0]

})

# ==============================================================================
# DASHBOARD
# ==============================================================================

dashboard_mqtt = pd.DataFrame({

    "INDICADOR":[

        "Brokers",

        "Topics",

        "Online",

        "Mensagens"

    ],

    "VALOR":[

        len(mqtt_brokers),

        len(mqtt_topics),

        0,

        0

    ]

})

# ==============================================================================
# EXPORTAÇÃO
# ==============================================================================

mqtt_brokers.to_excel(
    PASTA_RESULTADOS/"126_MQTT_BROKERS.xlsx",
    index=False
)

mqtt_topics.to_excel(
    PASTA_RESULTADOS/"127_MQTT_TOPICS.xlsx",
    index=False
)

mqtt_log.to_excel(
    PASTA_RESULTADOS/"128_MQTT_LOG.xlsx",
    index=False
)

dashboard_mqtt.to_excel(
    PASTA_RESULTADOS/"129_MQTT_DASHBOARD.xlsx",
    index=False
)

# ------------------------------------------------------------------------------
# PARQUET
# ------------------------------------------------------------------------------

mqtt_brokers.to_parquet(
    PASTA_RESULTADOS/"126_MQTT_BROKERS.parquet",
    index=False
)

mqtt_topics.to_parquet(
    PASTA_RESULTADOS/"127_MQTT_TOPICS.parquet",
    index=False
)

mqtt_log_parquet = mqtt_log.copy()
mqtt_log_parquet["DATA"] = mqtt_log_parquet["DATA"].astype(str)

mqtt_log_parquet.to_parquet(
    PASTA_RESULTADOS/"128_MQTT_LOG.parquet",
    index=False
)

dashboard_mqtt_parquet = dashboard_mqtt.copy()
dashboard_mqtt_parquet["VALOR"] = dashboard_mqtt_parquet["VALOR"].astype(str)

dashboard_mqtt_parquet.to_parquet(
    PASTA_RESULTADOS/"129_MQTT_DASHBOARD.parquet",
    index=False
)

# ==============================================================================
# RESUMO
# ==============================================================================

print()

print(mqtt_brokers)

print()

print(mqtt_topics)

print()

print(mqtt_log)

print()

print(dashboard_mqtt)

print()

print("="*100)
print("MQTT CONNECTOR ENGINE FINALIZADO")
print("="*100)

BLOCO 7.2.7 - MQTT CONNECTOR ENGINE
  BROKER_ID         DESCRICAO HOST  PORTA           STATUS
0   MQTT001  Broker Principal        1883  NÃO CONFIGURADO
1   MQTT002     Broker Backup        1883  NÃO CONFIGURADO

                    TOPIC  QOS        STATUS
0       cag/chiller01/cop    0  NÃO INSCRITO
1      cag/chiller01/kwtr    0  NÃO INSCRITO
2     cag/chiller01/carga    0  NÃO INSCRITO
3     cag/chiller01/vazao    0  NÃO INSCRITO
4  cag/chiller01/potencia    0  NÃO INSCRITO
5         cag/chiller02/#    1  NÃO INSCRITO
6         cag/chiller03/#    1  NÃO INSCRITO

{'PROTOCOLO': 'MQTT', 'PORTA': 1883, 'KEEP_ALIVE': 60, 'CLIENT_ID': 'EngineeringAnalytics', 'STATUS': 'AGUARDANDO CONFIGURAÇÃO'}

  BROKER_ID         DESCRICAO HOST  PORTA           STATUS
0   MQTT001  Broker Principal        1883  NÃO CONFIGURADO
1   MQTT002     Broker Backup        1883  NÃO CONFIGURADO

                    TOPIC  QOS        STATUS
0       cag/chiller01/cop    0  NÃO INSCRITO
1      cag/chiller01/kwtr  

In [78]:
# ==============================================================================
# BLOCO 7.2.8
# UNIFIED INDUSTRIAL GATEWAY ENGINE
# ==============================================================================

import pandas as pd

from pathlib import Path
from datetime import datetime

print("="*100)
print("BLOCO 7.2.8 - UNIFIED INDUSTRIAL GATEWAY ENGINE")
print("="*100)

PASTA_RESULTADOS = Path("Resultados")

# ==============================================================================
# PROTOCOL REGISTRY
# ==============================================================================

protocol_registry = pd.DataFrame({

    "PROTOCOLO":[

        "Excel",

        "SQL",

        "REST API",

        "BACnet",

        "OPC-UA",

        "MQTT"

    ],

    "STATUS":[

        "ATIVO",

        "ATIVO",

        "ATIVO",

        "CONFIGURADO",

        "CONFIGURADO",

        "CONFIGURADO"

    ],

    "PRIORIDADE":[

        1,

        2,

        3,

        4,

        5,

        6

    ]

})

print(protocol_registry)

# ==============================================================================
# GATEWAY STATUS
# ==============================================================================

gateway_status = {

    "ENGINE":"Unified Gateway",

    "VERSAO":"1.0",

    "STATUS":"ONLINE",

    "SINCRONIZACAO":"ATIVA",

    "ULTIMA_ATUALIZACAO":datetime.now()

}

print()

print(gateway_status)

# ==============================================================================
# CONNECTION TABLE
# ==============================================================================

gateway_connections = pd.DataFrame({

    "PROTOCOLO":protocol_registry["PROTOCOLO"],

    "ONLINE":[

        True,

        True,

        True,

        False,

        False,

        False

    ],

    "LATENCIA_MS":[

        15,

        8,

        42,

        0,

        0,

        0

    ],

    "ULTIMA_LEITURA":[

        datetime.now()

    ]*len(protocol_registry)

})

print()

print(gateway_connections)

# ==============================================================================
# UNIFIED BUFFER
# ==============================================================================

gateway_buffer = pd.DataFrame({

    "ORIGEM":[

        "Excel",

        "SQL",

        "REST API"

    ],

    "STATUS":[

        "READY",

        "READY",

        "READY"

    ],

    "REGISTROS":[

        len(logger_engine),

        0,

        0

    ],

    "DATA":datetime.now()

})

print()

print(gateway_buffer)

# ==============================================================================
# GATEWAY KPI
# ==============================================================================

gateway_kpi = pd.DataFrame({

    "INDICADOR":[

        "Protocolos",

        "Online",

        "Offline",

        "Gateway",

        "Sincronização"

    ],

    "VALOR":[

        len(protocol_registry),

        gateway_connections["ONLINE"].sum(),

        (~gateway_connections["ONLINE"]).sum(),

        "ONLINE",

        "ATIVA"

    ]

})

print()

print(gateway_kpi)

# ==============================================================================
# EXPORTAÇÃO
# ==============================================================================

protocol_registry.to_excel(
    PASTA_RESULTADOS/"130_PROTOCOL_REGISTRY.xlsx",
    index=False
)

gateway_connections.to_excel(
    PASTA_RESULTADOS/"131_GATEWAY_CONNECTIONS.xlsx",
    index=False
)

gateway_buffer.to_excel(
    PASTA_RESULTADOS/"132_GATEWAY_BUFFER.xlsx",
    index=False
)

gateway_kpi.to_excel(
    PASTA_RESULTADOS/"133_GATEWAY_KPI.xlsx",
    index=False
)

# ==============================================================================
# PARQUET
# ==============================================================================

protocol_registry.to_parquet(
    PASTA_RESULTADOS/"130_PROTOCOL_REGISTRY.parquet",
    index=False
)

gateway_connections_parquet = gateway_connections.copy()
gateway_connections_parquet["ULTIMA_LEITURA"] = (
    gateway_connections_parquet["ULTIMA_LEITURA"].astype(str)
)

gateway_connections_parquet.to_parquet(
    PASTA_RESULTADOS/"131_GATEWAY_CONNECTIONS.parquet",
    index=False
)

gateway_buffer_parquet = gateway_buffer.copy()
gateway_buffer_parquet["DATA"] = (
    gateway_buffer_parquet["DATA"].astype(str)
)

gateway_buffer_parquet.to_parquet(
    PASTA_RESULTADOS/"132_GATEWAY_BUFFER.parquet",
    index=False
)

gateway_kpi_parquet = gateway_kpi.copy()
gateway_kpi_parquet["VALOR"] = (
    gateway_kpi_parquet["VALOR"].astype(str)
)

gateway_kpi_parquet.to_parquet(
    PASTA_RESULTADOS/"133_GATEWAY_KPI.parquet",
    index=False
)

# ==============================================================================
# RESUMO
# ==============================================================================

print()

print(protocol_registry)

print()

print(gateway_connections)

print()

print(gateway_buffer)

print()

print(gateway_kpi)

print()

print("="*100)
print("UNIFIED INDUSTRIAL GATEWAY ENGINE FINALIZADO")
print("="*100)

BLOCO 7.2.8 - UNIFIED INDUSTRIAL GATEWAY ENGINE
  PROTOCOLO       STATUS  PRIORIDADE
0     Excel        ATIVO           1
1       SQL        ATIVO           2
2  REST API        ATIVO           3
3    BACnet  CONFIGURADO           4
4    OPC-UA  CONFIGURADO           5
5      MQTT  CONFIGURADO           6

{'ENGINE': 'Unified Gateway', 'VERSAO': '1.0', 'STATUS': 'ONLINE', 'SINCRONIZACAO': 'ATIVA', 'ULTIMA_ATUALIZACAO': datetime.datetime(2026, 7, 23, 12, 41, 51, 721804)}

  PROTOCOLO  ONLINE  LATENCIA_MS             ULTIMA_LEITURA
0     Excel    True           15 2026-07-23 12:41:51.721992
1       SQL    True            8 2026-07-23 12:41:51.721992
2  REST API    True           42 2026-07-23 12:41:51.721992
3    BACnet   False            0 2026-07-23 12:41:51.721992
4    OPC-UA   False            0 2026-07-23 12:41:51.721992
5      MQTT   False            0 2026-07-23 12:41:51.721992

     ORIGEM STATUS  REGISTROS                       DATA
0     Excel  READY          4 2026-07-23 12:41

In [79]:
# ==============================================================================
# BLOCO 7.3.1
# SOURCE INTEGRATION ENGINE
# ==============================================================================

import pandas as pd

from pathlib import Path
from datetime import datetime

print("="*100)
print("BLOCO 7.3.1 - SOURCE INTEGRATION ENGINE")
print("="*100)

PASTA_RESULTADOS = Path("Resultados")

# ==============================================================================
# SOURCE REGISTRY
# ==============================================================================

source_registry = pd.DataFrame({

    "SOURCE_ID":[

        "SRC001",
        "SRC002",
        "SRC003",
        "SRC004",
        "SRC005",
        "SRC006"

    ],

    "SOURCE_NAME":[

        "Excel",

        "SQL",

        "REST API",

        "BACnet",

        "OPC-UA",

        "MQTT"

    ],

    "TIPO":[

        "Arquivo",

        "Banco",

        "API",

        "Automação",

        "Industrial",

        "IoT"

    ],

    "STATUS":[

        "ONLINE",

        "ONLINE",

        "ONLINE",

        "CONFIGURADO",

        "CONFIGURADO",

        "CONFIGURADO"

    ],

    "PRIORIDADE":[

        1,
        2,
        3,
        4,
        5,
        6

    ]

})

print(source_registry)

# ==============================================================================
# SOURCE STATUS
# ==============================================================================

source_status = pd.DataFrame({

    "SOURCE":source_registry["SOURCE_NAME"],

    "ONLINE":[

        True,

        True,

        True,

        False,

        False,

        False

    ],

    "LATENCIA_MS":[

        15,

        8,

        42,

        0,

        0,

        0

    ],

    "ULTIMA_LEITURA":[

        datetime.now()

    ]*len(source_registry)

})

# ==============================================================================
# INTEGRATION SUMMARY
# ==============================================================================

integration_summary = pd.DataFrame({

    "INDICADOR":[

        "Fontes",

        "Online",

        "Offline",

        "Última Atualização"

    ],

    "VALOR":[

        len(source_registry),

        source_status["ONLINE"].sum(),

        (~source_status["ONLINE"]).sum(),

        str(datetime.now())

    ]

})

# ==============================================================================
# EXPORTAÇÃO
# ==============================================================================

source_registry.to_excel(
    PASTA_RESULTADOS/"134_SOURCE_REGISTRY.xlsx",
    index=False
)

source_status.to_excel(
    PASTA_RESULTADOS/"135_SOURCE_STATUS.xlsx",
    index=False
)

integration_summary.to_excel(
    PASTA_RESULTADOS/"136_SOURCE_INTEGRATION_SUMMARY.xlsx",
    index=False
)

# ------------------------------------------------------------------------------
# PARQUET
# ------------------------------------------------------------------------------

source_registry.to_parquet(
    PASTA_RESULTADOS/"134_SOURCE_REGISTRY.parquet",
    index=False
)

source_status_parquet = source_status.copy()
source_status_parquet["ULTIMA_LEITURA"] = (
    source_status_parquet["ULTIMA_LEITURA"].astype(str)
)

source_status_parquet.to_parquet(
    PASTA_RESULTADOS/"135_SOURCE_STATUS.parquet",
    index=False
)

integration_summary_parquet = integration_summary.copy()
integration_summary_parquet["VALOR"] = (
    integration_summary_parquet["VALOR"].astype(str)
)

integration_summary_parquet.to_parquet(
    PASTA_RESULTADOS/"136_SOURCE_INTEGRATION_SUMMARY.parquet",
    index=False
)

# ==============================================================================
# RESUMO
# ==============================================================================

print()

print(source_registry)

print()

print(source_status)

print()

print(integration_summary)

print()

print("="*100)
print("SOURCE INTEGRATION ENGINE FINALIZADO")
print("="*100)

BLOCO 7.3.1 - SOURCE INTEGRATION ENGINE
  SOURCE_ID SOURCE_NAME        TIPO       STATUS  PRIORIDADE
0    SRC001       Excel     Arquivo       ONLINE           1
1    SRC002         SQL       Banco       ONLINE           2
2    SRC003    REST API         API       ONLINE           3
3    SRC004      BACnet   Automação  CONFIGURADO           4
4    SRC005      OPC-UA  Industrial  CONFIGURADO           5
5    SRC006        MQTT         IoT  CONFIGURADO           6

  SOURCE_ID SOURCE_NAME        TIPO       STATUS  PRIORIDADE
0    SRC001       Excel     Arquivo       ONLINE           1
1    SRC002         SQL       Banco       ONLINE           2
2    SRC003    REST API         API       ONLINE           3
3    SRC004      BACnet   Automação  CONFIGURADO           4
4    SRC005      OPC-UA  Industrial  CONFIGURADO           5
5    SRC006        MQTT         IoT  CONFIGURADO           6

     SOURCE  ONLINE  LATENCIA_MS             ULTIMA_LEITURA
0     Excel    True           15 2026-07-23 

In [80]:
# ==============================================================================
# BLOCO 7.3.2
# TIMESTAMP SYNCHRONIZATION ENGINE
# ==============================================================================

import pandas as pd

from pathlib import Path
from datetime import datetime

print("="*100)
print("BLOCO 7.3.2 - TIMESTAMP SYNCHRONIZATION ENGINE")
print("="*100)

PASTA_RESULTADOS = Path("Resultados")

# ==============================================================================
# TIMESTAMP SOURCES
# ==============================================================================

timestamp_sources = pd.DataFrame({

    "SOURCE":[

        "Excel",
        "SQL",
        "REST API",
        "BACnet",
        "OPC-UA",
        "MQTT"

    ],

    "INTERVALO_SEG":[

        600,
        60,
        300,
        10,
        1,
        1

    ],

    "ULTIMA_LEITURA":[

        datetime.now()

    ]*6,

    "STATUS":[

        "SINCRONIZADO"

    ]*6

})

print(timestamp_sources)

# ==============================================================================
# CLOCK VALIDATION
# ==============================================================================

clock_validation = timestamp_sources.copy()

clock_validation["LATENCIA_SEG"] = 0.0

clock_validation["CLOCK_DRIFT_SEG"] = 0.0

clock_validation["TIMEZONE"] = "America/Sao_Paulo"

print()

print(clock_validation)

# ==============================================================================
# CORPORATE TIMELINE
# ==============================================================================

corporate_timeline = pd.DataFrame({

    "EVENTO":[

        "Primeira Leitura",

        "Última Leitura",

        "Última Sincronização"

    ],

    "TIMESTAMP":[

        feature_engine["DATAHORA"].min(),

        feature_engine["DATAHORA"].max(),

        datetime.now()

    ]

})

print()

print(corporate_timeline)

# ==============================================================================
# KPI
# ==============================================================================

timestamp_kpi = pd.DataFrame({

    "INDICADOR":[

        "Fontes",

        "Sincronizadas",

        "Latência Média (s)",

        "Drift Médio (s)"

    ],

    "VALOR":[

        len(clock_validation),

        (clock_validation["STATUS"]=="SINCRONIZADO").sum(),

        clock_validation["LATENCIA_SEG"].mean(),

        clock_validation["CLOCK_DRIFT_SEG"].mean()

    ]

})

print()

print(timestamp_kpi)

# ==============================================================================
# EXPORTAÇÃO
# ==============================================================================

timestamp_sources.to_excel(
    PASTA_RESULTADOS/"137_TIMESTAMP_SOURCES.xlsx",
    index=False
)

clock_validation.to_excel(
    PASTA_RESULTADOS/"138_CLOCK_VALIDATION.xlsx",
    index=False
)

corporate_timeline.to_excel(
    PASTA_RESULTADOS/"139_CORPORATE_TIMELINE.xlsx",
    index=False
)

timestamp_kpi.to_excel(
    PASTA_RESULTADOS/"140_TIMESTAMP_KPI.xlsx",
    index=False
)

# ------------------------------------------------------------------------------
# PARQUET
# ------------------------------------------------------------------------------

timestamp_sources_parquet = timestamp_sources.copy()
timestamp_sources_parquet["ULTIMA_LEITURA"] = (
    timestamp_sources_parquet["ULTIMA_LEITURA"].astype(str)
)

timestamp_sources_parquet.to_parquet(
    PASTA_RESULTADOS/"137_TIMESTAMP_SOURCES.parquet",
    index=False
)

clock_validation_parquet = clock_validation.copy()
clock_validation_parquet["ULTIMA_LEITURA"] = (
    clock_validation_parquet["ULTIMA_LEITURA"].astype(str)
)

clock_validation_parquet.to_parquet(
    PASTA_RESULTADOS/"138_CLOCK_VALIDATION.parquet",
    index=False
)

corporate_timeline_parquet = corporate_timeline.copy()
corporate_timeline_parquet["TIMESTAMP"] = (
    corporate_timeline_parquet["TIMESTAMP"].astype(str)
)

corporate_timeline_parquet.to_parquet(
    PASTA_RESULTADOS/"139_CORPORATE_TIMELINE.parquet",
    index=False
)

timestamp_kpi_parquet = timestamp_kpi.copy()
timestamp_kpi_parquet["VALOR"] = (
    timestamp_kpi_parquet["VALOR"].astype(str)
)

timestamp_kpi_parquet.to_parquet(
    PASTA_RESULTADOS/"140_TIMESTAMP_KPI.parquet",
    index=False
)

# ==============================================================================
# RESUMO
# ==============================================================================

print()

print(timestamp_sources)

print()

print(clock_validation)

print()

print(corporate_timeline)

print()

print(timestamp_kpi)

print()

print("="*100)
print("TIMESTAMP SYNCHRONIZATION ENGINE FINALIZADO")
print("="*100)

BLOCO 7.3.2 - TIMESTAMP SYNCHRONIZATION ENGINE
     SOURCE  INTERVALO_SEG             ULTIMA_LEITURA        STATUS
0     Excel            600 2026-07-23 13:14:22.676349  SINCRONIZADO
1       SQL             60 2026-07-23 13:14:22.676349  SINCRONIZADO
2  REST API            300 2026-07-23 13:14:22.676349  SINCRONIZADO
3    BACnet             10 2026-07-23 13:14:22.676349  SINCRONIZADO
4    OPC-UA              1 2026-07-23 13:14:22.676349  SINCRONIZADO
5      MQTT              1 2026-07-23 13:14:22.676349  SINCRONIZADO

     SOURCE  INTERVALO_SEG             ULTIMA_LEITURA        STATUS  \
0     Excel            600 2026-07-23 13:14:22.676349  SINCRONIZADO   
1       SQL             60 2026-07-23 13:14:22.676349  SINCRONIZADO   
2  REST API            300 2026-07-23 13:14:22.676349  SINCRONIZADO   
3    BACnet             10 2026-07-23 13:14:22.676349  SINCRONIZADO   
4    OPC-UA              1 2026-07-23 13:14:22.676349  SINCRONIZADO   
5      MQTT              1 2026-07-23 13:14:22.676

In [81]:
# ==============================================================================
# BLOCO 7.3.3
# VARIABLE STANDARDIZATION ENGINE
# ==============================================================================

import pandas as pd

from pathlib import Path
from datetime import datetime

print("="*100)
print("BLOCO 7.3.3 - VARIABLE STANDARDIZATION ENGINE")
print("="*100)

PASTA_RESULTADOS = Path("Resultados")

# ==============================================================================
# CORPORATE VARIABLE DICTIONARY
# ==============================================================================

corporate_variables = pd.DataFrame({

    "VARIAVEL_PADRAO":[

        "COP",
        "KW_TR",
        "CARGA_TR",
        "POTENCIA",
        "VAZAO",
        "APPROACH_EVAP",
        "APPROACH_COND",
        "FREQUENCIA",
        "ENERGIA"

    ],

    "UNIDADE":[

        "-",
        "kW/TR",
        "TR",
        "kW",
        "m³/h",
        "°C",
        "°C",
        "%",
        "kWh"

    ],

    "DESCRICAO":[

        "Coeficiente de Performance",

        "Consumo específico",

        "Carga térmica",

        "Potência elétrica",

        "Vazão hidráulica",

        "Approach Evaporador",

        "Approach Condensador",

        "Frequência do inversor",

        "Energia elétrica"

    ]

})

print(corporate_variables)

# ==============================================================================
# VARIABLE ALIAS
# ==============================================================================

variable_alias = pd.DataFrame({

    "ALIAS":[

        "COP",

        "IE-COP",

        "Coeficiente de Performance",

        "kW/TR",

        "Relação de Consumo",

        "Carga Térmica",

        "Thermal Load"

    ],

    "VARIAVEL_PADRAO":[

        "COP",

        "COP",

        "COP",

        "KW_TR",

        "KW_TR",

        "CARGA_TR",

        "CARGA_TR"

    ]

})

print()

print(variable_alias)

# ==============================================================================
# METADATA REGISTRY
# ==============================================================================

metadata_registry = corporate_variables.copy()

metadata_registry["DATA_CRIACAO"] = datetime.now()

metadata_registry["VERSAO"] = "1.0"

metadata_registry["STATUS"] = "ATIVO"

print()

print(metadata_registry.head())

# ==============================================================================
# VARIABLE VALIDATION
# ==============================================================================

variaveis_encontradas = sorted(feature_engine.columns.tolist())

validation = pd.DataFrame({

    "VARIAVEL":variaveis_encontradas

})

validation["PADRONIZADA"] = validation["VARIAVEL"].isin(
    corporate_variables["VARIAVEL_PADRAO"]
)

print()

print(validation.head(20))

# ==============================================================================
# DASHBOARD
# ==============================================================================

dashboard_variables = pd.DataFrame({

    "INDICADOR":[

        "Variáveis Corporativas",

        "Aliases",

        "Variáveis Encontradas",

        "Padronizadas"

    ],

    "VALOR":[

        len(corporate_variables),

        len(variable_alias),

        len(validation),

        validation["PADRONIZADA"].sum()

    ]

})

print()

print(dashboard_variables)

# ==============================================================================
# EXPORTAÇÃO
# ==============================================================================

corporate_variables.to_excel(
    PASTA_RESULTADOS/"141_CORPORATE_VARIABLES.xlsx",
    index=False
)

variable_alias.to_excel(
    PASTA_RESULTADOS/"142_VARIABLE_ALIAS.xlsx",
    index=False
)

metadata_registry_export = metadata_registry.copy()
metadata_registry_export["DATA_CRIACAO"] = (
    metadata_registry_export["DATA_CRIACAO"].astype(str)
)

metadata_registry_export.to_excel(
    PASTA_RESULTADOS/"143_METADATA_REGISTRY.xlsx",
    index=False
)

validation.to_excel(
    PASTA_RESULTADOS/"144_VARIABLE_VALIDATION.xlsx",
    index=False
)

dashboard_variables.to_excel(
    PASTA_RESULTADOS/"145_VARIABLE_DASHBOARD.xlsx",
    index=False
)

# ------------------------------------------------------------------------------
# PARQUET
# ------------------------------------------------------------------------------

corporate_variables.to_parquet(
    PASTA_RESULTADOS/"141_CORPORATE_VARIABLES.parquet",
    index=False
)

variable_alias.to_parquet(
    PASTA_RESULTADOS/"142_VARIABLE_ALIAS.parquet",
    index=False
)

metadata_registry_export.to_parquet(
    PASTA_RESULTADOS/"143_METADATA_REGISTRY.parquet",
    index=False
)

validation.to_parquet(
    PASTA_RESULTADOS/"144_VARIABLE_VALIDATION.parquet",
    index=False
)

dashboard_variables_parquet = dashboard_variables.copy()
dashboard_variables_parquet["VALOR"] = (
    dashboard_variables_parquet["VALOR"].astype(str)
)

dashboard_variables_parquet.to_parquet(
    PASTA_RESULTADOS/"145_VARIABLE_DASHBOARD.parquet",
    index=False
)

# ==============================================================================
# RESUMO
# ==============================================================================

print()

print(corporate_variables)

print()

print(variable_alias)

print()

print(metadata_registry.head())

print()

print(dashboard_variables)

print()

print("="*100)
print("VARIABLE STANDARDIZATION ENGINE FINALIZADO")
print("="*100)

BLOCO 7.3.3 - VARIABLE STANDARDIZATION ENGINE
  VARIAVEL_PADRAO UNIDADE                   DESCRICAO
0             COP       -  Coeficiente de Performance
1           KW_TR   kW/TR          Consumo específico
2        CARGA_TR      TR               Carga térmica
3        POTENCIA      kW           Potência elétrica
4           VAZAO    m³/h            Vazão hidráulica
5   APPROACH_EVAP      °C         Approach Evaporador
6   APPROACH_COND      °C        Approach Condensador
7      FREQUENCIA       %      Frequência do inversor
8         ENERGIA     kWh            Energia elétrica

                        ALIAS VARIAVEL_PADRAO
0                         COP             COP
1                      IE-COP             COP
2  Coeficiente de Performance             COP
3                       kW/TR           KW_TR
4          Relação de Consumo           KW_TR
5               Carga Térmica        CARGA_TR
6                Thermal Load        CARGA_TR

  VARIAVEL_PADRAO UNIDADE                   

In [82]:
# ==============================================================================
# BLOCO 7.3.4
# DATA CONFLICT RESOLUTION ENGINE
# ==============================================================================

import pandas as pd
import numpy as np

from pathlib import Path
from datetime import datetime

print("="*100)
print("BLOCO 7.3.4 - DATA CONFLICT RESOLUTION ENGINE")
print("="*100)

PASTA_RESULTADOS = Path("Resultados")

# ==============================================================================
# SOURCE PRIORITY
# ==============================================================================

source_priority = pd.DataFrame({

    "FONTE":[

        "SQL",

        "BACnet",

        "OPC-UA",

        "MQTT",

        "REST API",

        "Excel"

    ],

    "PRIORIDADE":[

        1,

        2,

        3,

        4,

        5,

        6

    ]

})

print(source_priority)

# ==============================================================================
# ENGINEERING LIMITS
# ==============================================================================

engineering_limits = pd.DataFrame({

    "VARIAVEL":[

        "COP",

        "KW_TR",

        "CARGA_TR",

        "POTENCIA",

        "VAZAO",

        "APPROACH_EVAP",

        "APPROACH_COND",

        "FREQUENCIA"

    ],

    "MINIMO":[

        0.5,

        0.40,

        0,

        0,

        0,

        0,

        0,

        0

    ],

    "MAXIMO":[

        8,

        3.50,

        300,

        350,

        300,

        15,

        15,

        100

    ]

})

print()

print(engineering_limits)

# ==============================================================================
# CONFLICT LOG
# ==============================================================================

conflict_log = pd.DataFrame({

    "DATA_PROCESSAMENTO":[datetime.now()],

    "TOTAL_VARIAVEIS":[len(corporate_variables)],

    "CONFLITOS_IDENTIFICADOS":[0],

    "CONFLITOS_RESOLVIDOS":[0],

    "STATUS":["SEM CONFLITOS"]

})

print()

print(conflict_log)

# ==============================================================================
# RESOLUTION SCORE
# ==============================================================================

resolution_score = pd.DataFrame({

    "INDICADOR":[

        "Prioridade",

        "Timestamp",

        "Consistência Física",

        "Fonte"

    ],

    "PESO":[

        0.40,

        0.30,

        0.20,

        0.10

    ]

})

print()

print(resolution_score)

# ==============================================================================
# DASHBOARD
# ==============================================================================

dashboard_conflicts = pd.DataFrame({

    "INDICADOR":[

        "Fontes",

        "Variáveis",

        "Conflitos",

        "Resolvidos",

        "Status"

    ],

    "VALOR":[

        len(source_priority),

        len(corporate_variables),

        0,

        0,

        "OPERANDO"

    ]

})

print()

print(dashboard_conflicts)

# ==============================================================================
# EXPORTAÇÃO
# ==============================================================================

source_priority.to_excel(
    PASTA_RESULTADOS/"146_SOURCE_PRIORITY.xlsx",
    index=False
)

engineering_limits.to_excel(
    PASTA_RESULTADOS/"147_ENGINEERING_LIMITS.xlsx",
    index=False
)

conflict_log_export = conflict_log.copy()
conflict_log_export["DATA_PROCESSAMENTO"] = (
    conflict_log_export["DATA_PROCESSAMENTO"].astype(str)
)

conflict_log_export.to_excel(
    PASTA_RESULTADOS/"148_CONFLICT_LOG.xlsx",
    index=False
)

resolution_score.to_excel(
    PASTA_RESULTADOS/"149_RESOLUTION_SCORE.xlsx",
    index=False
)

dashboard_conflicts.to_excel(
    PASTA_RESULTADOS/"150_CONFLICT_DASHBOARD.xlsx",
    index=False
)

# ==============================================================================
# PARQUET
# ==============================================================================

source_priority.to_parquet(
    PASTA_RESULTADOS/"146_SOURCE_PRIORITY.parquet",
    index=False
)

engineering_limits.to_parquet(
    PASTA_RESULTADOS/"147_ENGINEERING_LIMITS.parquet",
    index=False
)

conflict_log_export.to_parquet(
    PASTA_RESULTADOS/"148_CONFLICT_LOG.parquet",
    index=False
)

resolution_score.to_parquet(
    PASTA_RESULTADOS/"149_RESOLUTION_SCORE.parquet",
    index=False
)

dashboard_conflicts_parquet = dashboard_conflicts.copy()
dashboard_conflicts_parquet["VALOR"] = (
    dashboard_conflicts_parquet["VALOR"].astype(str)
)

dashboard_conflicts_parquet.to_parquet(
    PASTA_RESULTADOS/"150_CONFLICT_DASHBOARD.parquet",
    index=False
)

# ==============================================================================
# RESUMO
# ==============================================================================

print()

print(source_priority)

print()

print(engineering_limits)

print()

print(conflict_log)

print()

print(resolution_score)

print()

print(dashboard_conflicts)

print()

print("="*100)
print("DATA CONFLICT RESOLUTION ENGINE FINALIZADO")
print("="*100)

BLOCO 7.3.4 - DATA CONFLICT RESOLUTION ENGINE
      FONTE  PRIORIDADE
0       SQL           1
1    BACnet           2
2    OPC-UA           3
3      MQTT           4
4  REST API           5
5     Excel           6

        VARIAVEL  MINIMO   MAXIMO
0            COP  0.5000   8.0000
1          KW_TR  0.4000   3.5000
2       CARGA_TR  0.0000 300.0000
3       POTENCIA  0.0000 350.0000
4          VAZAO  0.0000 300.0000
5  APPROACH_EVAP  0.0000  15.0000
6  APPROACH_COND  0.0000  15.0000
7     FREQUENCIA  0.0000 100.0000

          DATA_PROCESSAMENTO  TOTAL_VARIAVEIS  CONFLITOS_IDENTIFICADOS  \
0 2026-07-23 13:28:26.248236                9                        0   

   CONFLITOS_RESOLVIDOS         STATUS  
0                     0  SEM CONFLITOS  

             INDICADOR   PESO
0           Prioridade 0.4000
1            Timestamp 0.3000
2  Consistência Física 0.2000
3                Fonte 0.1000

    INDICADOR     VALOR
0      Fontes         6
1   Variáveis         9
2   Conflitos         0

In [83]:
# ==============================================================================
# BLOCO 7.3.5
# ENTERPRISE DATA QUALITY ENGINE
# ==============================================================================

import pandas as pd
import numpy as np

from pathlib import Path
from datetime import datetime

print("="*100)
print("BLOCO 7.3.5 - ENTERPRISE DATA QUALITY ENGINE")
print("="*100)

PASTA_RESULTADOS = Path("Resultados")

# ==============================================================================
# DATA QUALITY METRICS
# ==============================================================================

quality_engine = feature_engine.copy()

total_registros = len(quality_engine)

campos = [

    "COP",
    "KW_TR",
    "CARGA_TR",
    "POTENCIA",
    "VAZAO",
    "APPROACH_EVAP",
    "APPROACH_COND"

]

metricas = []

for campo in campos:

    if campo in quality_engine.columns:

        total = quality_engine[campo].notna().sum()

        percentual = (total/total_registros)*100

        metricas.append({

            "VARIAVEL":campo,

            "REGISTROS":total,

            "COMPLETUDE (%)":round(percentual,2)

        })

quality_metrics = pd.DataFrame(metricas)

print(quality_metrics)

# ==============================================================================
# DATA QUALITY SCORE
# ==============================================================================

quality_metrics["QUALITY_SCORE"] = (

    quality_metrics["COMPLETUDE (%)"]

)

quality_metrics["QUALITY_CLASS"] = pd.cut(

    quality_metrics["QUALITY_SCORE"],

    bins=[0,60,80,95,100],

    labels=[

        "RUIM",

        "REGULAR",

        "BOA",

        "EXCELENTE"

    ],

    include_lowest=True

)

print()

print(quality_metrics)

# ==============================================================================
# CORPORATE KPI
# ==============================================================================

corporate_quality = pd.DataFrame({

    "INDICADOR":[

        "Registros",

        "Variáveis",

        "Completude Média",

        "Quality Score Médio"

    ],

    "VALOR":[

        total_registros,

        len(quality_metrics),

        round(

            quality_metrics["COMPLETUDE (%)"].mean(),

            2

        ),

        round(

            quality_metrics["QUALITY_SCORE"].mean(),

            2

        )

    ]

})

print()

print(corporate_quality)

# ==============================================================================
# DASHBOARD
# ==============================================================================

dashboard_quality = pd.DataFrame({

    "INDICADOR":[

        "Excelente",

        "Boa",

        "Regular",

        "Ruim"

    ],

    "VALOR":[

        (quality_metrics["QUALITY_CLASS"]=="EXCELENTE").sum(),

        (quality_metrics["QUALITY_CLASS"]=="BOA").sum(),

        (quality_metrics["QUALITY_CLASS"]=="REGULAR").sum(),

        (quality_metrics["QUALITY_CLASS"]=="RUIM").sum()

    ]

})

print()

print(dashboard_quality)

# ==============================================================================
# EXPORTAÇÃO
# ==============================================================================

quality_metrics.to_excel(
    PASTA_RESULTADOS/"151_DATA_QUALITY.xlsx",
    index=False
)

corporate_quality.to_excel(
    PASTA_RESULTADOS/"152_CORPORATE_DATA_QUALITY.xlsx",
    index=False
)

dashboard_quality.to_excel(
    PASTA_RESULTADOS/"153_DATA_QUALITY_DASHBOARD.xlsx",
    index=False
)

quality_metrics.to_parquet(
    PASTA_RESULTADOS/"151_DATA_QUALITY.parquet",
    index=False
)

corporate_quality_parquet = corporate_quality.copy()
corporate_quality_parquet["VALOR"] = (
    corporate_quality_parquet["VALOR"].astype(str)
)

corporate_quality_parquet.to_parquet(
    PASTA_RESULTADOS/"152_CORPORATE_DATA_QUALITY.parquet",
    index=False
)

dashboard_quality_parquet = dashboard_quality.copy()
dashboard_quality_parquet["VALOR"] = (
    dashboard_quality_parquet["VALOR"].astype(str)
)

dashboard_quality_parquet.to_parquet(
    PASTA_RESULTADOS/"153_DATA_QUALITY_DASHBOARD.parquet",
    index=False
)

# ==============================================================================
# RESUMO
# ==============================================================================

print()

print(quality_metrics)

print()

print(corporate_quality)

print()

print(dashboard_quality)

print()

print("="*100)
print("ENTERPRISE DATA QUALITY ENGINE FINALIZADO")
print("="*100)

BLOCO 7.3.5 - ENTERPRISE DATA QUALITY ENGINE
        VARIAVEL  REGISTROS  COMPLETUDE (%)
0            COP      17180        100.0000
1          KW_TR      17180        100.0000
2       CARGA_TR      17180        100.0000
3       POTENCIA      17180        100.0000
4          VAZAO      17180        100.0000
5  APPROACH_EVAP      17180        100.0000
6  APPROACH_COND      17180        100.0000

        VARIAVEL  REGISTROS  COMPLETUDE (%)  QUALITY_SCORE QUALITY_CLASS
0            COP      17180        100.0000       100.0000     EXCELENTE
1          KW_TR      17180        100.0000       100.0000     EXCELENTE
2       CARGA_TR      17180        100.0000       100.0000     EXCELENTE
3       POTENCIA      17180        100.0000       100.0000     EXCELENTE
4          VAZAO      17180        100.0000       100.0000     EXCELENTE
5  APPROACH_EVAP      17180        100.0000       100.0000     EXCELENTE
6  APPROACH_COND      17180        100.0000       100.0000     EXCELENTE

             INDI

In [84]:
# ==============================================================================
# BLOCO 7.3.6
# METADATA INTEGRATION ENGINE
# ==============================================================================

import pandas as pd
import numpy as np

from pathlib import Path
from datetime import datetime

print("="*100)
print("BLOCO 7.3.6 - METADATA INTEGRATION ENGINE")
print("="*100)

PASTA_RESULTADOS = Path("Resultados")

# ==============================================================================
# METADATA CATALOG
# ==============================================================================

metadata_catalog = pd.DataFrame({

    "VARIAVEL":feature_engine.columns

})

metadata_catalog["TIPO_DADO"] = metadata_catalog["VARIAVEL"].apply(

    lambda c: str(feature_engine[c].dtype)

)

metadata_catalog["N_REGISTROS"] = metadata_catalog["VARIAVEL"].apply(

    lambda c: feature_engine[c].count()

)

metadata_catalog["NULOS"] = metadata_catalog["VARIAVEL"].apply(

    lambda c: feature_engine[c].isna().sum()

)

metadata_catalog["PERCENTUAL_NULOS"] = (

    metadata_catalog["NULOS"]

    /

    len(feature_engine)

)*100

print(metadata_catalog.head())

# ==============================================================================
# CORPORATE METADATA
# ==============================================================================

metadata_catalog["UNIDADE"] = "-"

metadata_catalog["FONTE"] = "Enterprise Data Warehouse"

metadata_catalog["VERSAO"] = "1.0"

metadata_catalog["STATUS"] = "ATIVO"

metadata_catalog["RESPONSAVEL"] = "Engineering Analytics Platform"

metadata_catalog["DATA_CRIACAO"] = datetime.now()

# ==============================================================================
# CRITICALITY
# ==============================================================================

variaveis_criticas = [

    "COP",

    "KW_TR",

    "CARGA_TR",

    "POTENCIA",

    "HEALTH_SCORE",

    "RISK_SCORE",

    "POF",

    "RUL_DIAS"

]

metadata_catalog["CRITICIDADE"] = np.where(

    metadata_catalog["VARIAVEL"].isin(variaveis_criticas),

    "ALTA",

    "NORMAL"

)

# ==============================================================================
# UPDATE FREQUENCY
# ==============================================================================

metadata_catalog["FREQUENCIA"] = "10 minutos"

metadata_catalog.loc[

    metadata_catalog["VARIAVEL"]=="DATAHORA",

    "FREQUENCIA"

] = "Tempo Real"

# ==============================================================================
# DASHBOARD
# ==============================================================================

metadata_dashboard = pd.DataFrame({

    "INDICADOR":[

        "Total Variáveis",

        "Variáveis Críticas",

        "Variáveis Ativas",

        "Nulos Médios (%)"

    ],

    "VALOR":[

        len(metadata_catalog),

        (metadata_catalog["CRITICIDADE"]=="ALTA").sum(),

        (metadata_catalog["STATUS"]=="ATIVO").sum(),

        round(

            metadata_catalog["PERCENTUAL_NULOS"].mean(),

            2

        )

    ]

})

print(metadata_dashboard)

# ==============================================================================
# EXPORTAÇÃO
# ==============================================================================

metadata_export = metadata_catalog.copy()

metadata_export["DATA_CRIACAO"] = (
    metadata_export["DATA_CRIACAO"].astype(str)
)

metadata_export.to_excel(
    PASTA_RESULTADOS/"154_METADATA_CATALOG.xlsx",
    index=False
)

metadata_dashboard.to_excel(
    PASTA_RESULTADOS/"155_METADATA_DASHBOARD.xlsx",
    index=False
)

metadata_export.to_parquet(
    PASTA_RESULTADOS/"154_METADATA_CATALOG.parquet",
    index=False
)

metadata_dashboard_parquet = metadata_dashboard.copy()

metadata_dashboard_parquet["VALOR"] = (
    metadata_dashboard_parquet["VALOR"].astype(str)
)

metadata_dashboard_parquet.to_parquet(
    PASTA_RESULTADOS/"155_METADATA_DASHBOARD.parquet",
    index=False
)

# ==============================================================================
# RESUMO
# ==============================================================================

print()

print(metadata_catalog.head(20))

print()

print(metadata_dashboard)

print()

print("="*100)
print("METADATA INTEGRATION ENGINE FINALIZADO")
print("="*100)



BLOCO 7.3.6 - METADATA INTEGRATION ENGINE
   VARIAVEL       TIPO_DADO  N_REGISTROS  NULOS  PERCENTUAL_NULOS
0  DATAHORA  datetime64[ns]        17180      0            0.0000
1       URA          object        17180      0            0.0000
2       COP         float64        17180      0            0.0000
3     KW_TR         float64        17180      0            0.0000
4  POTENCIA         float64        17180      0            0.0000
            INDICADOR   VALOR
0     Total Variáveis 61.0000
1  Variáveis Críticas  8.0000
2    Variáveis Ativas 61.0000
3    Nulos Médios (%)  0.0000

             VARIAVEL       TIPO_DADO  N_REGISTROS  NULOS  PERCENTUAL_NULOS  \
0            DATAHORA  datetime64[ns]        17180      0            0.0000   
1                 URA          object        17180      0            0.0000   
2                 COP         float64        17180      0            0.0000   
3               KW_TR         float64        17180      0            0.0000   
4            POT

In [87]:
# ==============================================================================
# BLOCO 7.3.7
# ENTERPRISE EVENT BUS ENGINE
# ==============================================================================

import pandas as pd
import numpy as np

from pathlib import Path
from datetime import datetime

print("="*100)
print("BLOCO 7.3.7 - ENTERPRISE EVENT BUS ENGINE")
print("="*100)

PASTA_RESULTADOS = Path("Resultados")

# ==============================================================================
# EVENT GENERATOR
# ==============================================================================

event_engine = feature_engine.copy()

event_engine["EVENT_ID"] = np.arange(1, len(event_engine)+1)

event_engine["EVENT_TIME"] = event_engine["DATAHORA"]

event_engine["EVENT_TYPE"] = "NORMAL"

event_engine["EVENT_PRIORITY"] = "LOW"

event_engine["EVENT_STATUS"] = "OPEN"

event_engine["EVENT_SOURCE"] = "ENGINEERING_PLATFORM"

# ==============================================================================
# EVENT RULES
# ==============================================================================

event_engine.loc[
    event_engine["HEALTH_SCORE"] < 70,
    "EVENT_TYPE"
] = "LOW_HEALTH"

event_engine.loc[
    event_engine["RISK_SCORE"] > 30,
    "EVENT_TYPE"
] = "HIGH_RISK"

event_engine.loc[
    event_engine["POF"] > 0.30,
    "EVENT_TYPE"
] = "FAILURE_RISK"

# ==============================================================================
# WORK ORDER AUTOMÁTICA
# ==============================================================================

event_engine.loc[

    (
        (event_engine["RUL_DIAS"] < 30)

        |

        (event_engine["POF"] > 0.35)

        |

        (event_engine["HEALTH_SCORE"] < 70)

        |

        (event_engine["RISK_SCORE"] > 35)

    ),

    "EVENT_TYPE"

] = "WORK_ORDER"

# ==============================================================================
# PRIORITY
# ==============================================================================

event_engine.loc[
    event_engine["EVENT_TYPE"] == "NORMAL",
    "EVENT_PRIORITY"
] = "LOW"

event_engine.loc[
    event_engine["EVENT_TYPE"] == "LOW_HEALTH",
    "EVENT_PRIORITY"
] = "MEDIUM"

event_engine.loc[
    event_engine["EVENT_TYPE"] == "HIGH_RISK",
    "EVENT_PRIORITY"
] = "HIGH"

event_engine.loc[
    event_engine["EVENT_TYPE"] == "FAILURE_RISK",
    "EVENT_PRIORITY"
] = "CRITICAL"

event_engine.loc[
    event_engine["EVENT_TYPE"] == "WORK_ORDER",
    "EVENT_PRIORITY"
] = "CRITICAL"

# ==============================================================================
# EVENT HISTORY
# ==============================================================================

event_history = event_engine[[
    "EVENT_ID",
    "EVENT_TIME",
    "URA",
    "EVENT_TYPE",
    "EVENT_PRIORITY",
    "EVENT_STATUS"
]].copy()

# ==============================================================================
# EVENT DASHBOARD
# ==============================================================================

event_dashboard = pd.DataFrame({

    "INDICADOR":[

        "Eventos",

        "Críticos",

        "Alta Prioridade",

        "Média Prioridade",

        "Baixa Prioridade"

    ],

    "VALOR":[

        len(event_history),

        (event_history["EVENT_PRIORITY"]=="CRITICAL").sum(),

        (event_history["EVENT_PRIORITY"]=="HIGH").sum(),

        (event_history["EVENT_PRIORITY"]=="MEDIUM").sum(),

        (event_history["EVENT_PRIORITY"]=="LOW").sum()

    ]

})

print(event_dashboard)

# ==============================================================================
# EXPORTAÇÃO
# ==============================================================================

event_history_export = event_history.copy()
event_history_export["EVENT_TIME"] = (
    event_history_export["EVENT_TIME"].astype(str)
)

event_history_export.to_excel(
    PASTA_RESULTADOS/"156_EVENT_HISTORY.xlsx",
    index=False
)

event_dashboard.to_excel(
    PASTA_RESULTADOS/"157_EVENT_DASHBOARD.xlsx",
    index=False
)

event_history_export.to_parquet(
    PASTA_RESULTADOS/"156_EVENT_HISTORY.parquet",
    index=False
)

event_dashboard_parquet = event_dashboard.copy()
event_dashboard_parquet["VALOR"] = (
    event_dashboard_parquet["VALOR"].astype(str)
)

event_dashboard_parquet.to_parquet(
    PASTA_RESULTADOS/"157_EVENT_DASHBOARD.parquet",
    index=False
)

# ==============================================================================
# RESUMO
# ==============================================================================

print()

print(event_history.head(20))

print()

print(event_dashboard)

print()

print("="*100)
print("ENTERPRISE EVENT BUS ENGINE FINALIZADO")
print("="*100)

BLOCO 7.3.7 - ENTERPRISE EVENT BUS ENGINE
          INDICADOR  VALOR
0           Eventos  17180
1          Críticos   1759
2   Alta Prioridade     20
3  Média Prioridade      0
4  Baixa Prioridade  15401

    EVENT_ID          EVENT_TIME  URA EVENT_TYPE EVENT_PRIORITY EVENT_STATUS
0          1 2026-06-01 00:00:00  CAG     NORMAL            LOW         OPEN
1          2 2026-06-01 00:10:00  CAG     NORMAL            LOW         OPEN
2          3 2026-06-01 00:20:00  CAG     NORMAL            LOW         OPEN
3          4 2026-06-01 00:30:00  CAG     NORMAL            LOW         OPEN
4          5 2026-06-01 00:40:00  CAG     NORMAL            LOW         OPEN
5          6 2026-06-01 00:50:00  CAG     NORMAL            LOW         OPEN
6          7 2026-06-01 01:00:00  CAG     NORMAL            LOW         OPEN
7          8 2026-06-01 01:10:00  CAG     NORMAL            LOW         OPEN
8          9 2026-06-01 01:20:00  CAG     NORMAL            LOW         OPEN
9         10 2026-06-01 0

In [88]:
# ==============================================================================
# BLOCO 7.4.1
# ENTERPRISE OPERATIONS CENTER
# ==============================================================================

import pandas as pd
import numpy as np

from pathlib import Path
from datetime import datetime

print("="*100)
print("BLOCO 7.4.1 - ENTERPRISE OPERATIONS CENTER")
print("="*100)

PASTA_RESULTADOS = Path("Resultados")

# ==============================================================================
# EXECUTIVE DASHBOARD
# ==============================================================================

executive_dashboard = pd.DataFrame({

    "INDICADOR":[

        "Equipamentos",

        "Registros",

        "Health Médio",

        "Risk Médio",

        "POF Médio",

        "RUL Médio (dias)",

        "COP Médio",

        "kW/TR Médio"

    ],

    "VALOR":[

        feature_engine["URA"].nunique(),

        len(feature_engine),

        round(feature_engine["HEALTH_SCORE"].mean(),2),

        round(feature_engine["RISK_SCORE"].mean(),2),

        round(feature_engine["POF"].mean(),4),

        round(feature_engine["RUL_DIAS"].mean(),1),

        round(feature_engine["COP"].mean(),3),

        round(feature_engine["KW_TR"].mean(),3)

    ]

})

print(executive_dashboard)

# ==============================================================================
# ASSET STATUS
# ==============================================================================

asset_status = feature_engine.groupby("URA").agg({

    "HEALTH_SCORE":"mean",

    "RISK_SCORE":"mean",

    "POF":"mean",

    "RUL_DIAS":"mean",

    "COP":"mean",

    "KW_TR":"mean"

}).reset_index()

asset_status["STATUS"] = np.where(

    asset_status["HEALTH_SCORE"] >= 90,

    "OPERAÇÃO NORMAL",

    np.where(

        asset_status["HEALTH_SCORE"] >= 75,

        "ATENÇÃO",

        "CRÍTICO"

    )

)

print(asset_status)

# ==============================================================================
# MAINTENANCE KPI
# ==============================================================================

maintenance_dashboard = pd.DataFrame({

    "INDICADOR":[

        "POF > 30%",

        "RUL < 30 dias",

        "Health < 70",

        "Risk > 35"

    ],

    "VALOR":[

        (feature_engine["POF"]>0.30).sum(),

        (feature_engine["RUL_DIAS"]<30).sum(),

        (feature_engine["HEALTH_SCORE"]<70).sum(),

        (feature_engine["RISK_SCORE"]>35).sum()

    ]

})

print(maintenance_dashboard)

# ==============================================================================
# EVENTOS
# ==============================================================================

if "event_history" in globals():

    event_summary = event_history.groupby(

        "EVENT_PRIORITY"

    ).size().reset_index(name="TOTAL")

else:

    event_summary = pd.DataFrame({

        "EVENT_PRIORITY":["SEM EVENTOS"],

        "TOTAL":[0]

    })

print(event_summary)

# ==============================================================================
# EXPORTAÇÃO
# ==============================================================================

executive_dashboard.to_excel(
    PASTA_RESULTADOS/"158_EXECUTIVE_DASHBOARD.xlsx",
    index=False
)

asset_status.to_excel(
    PASTA_RESULTADOS/"159_ASSET_STATUS.xlsx",
    index=False
)

maintenance_dashboard.to_excel(
    PASTA_RESULTADOS/"160_MAINTENANCE_DASHBOARD.xlsx",
    index=False
)

event_summary.to_excel(
    PASTA_RESULTADOS/"161_EVENT_SUMMARY.xlsx",
    index=False
)

# ==============================================================================
# PARQUET
# ==============================================================================

executive_dashboard_parquet = executive_dashboard.copy()
executive_dashboard_parquet["VALOR"] = executive_dashboard_parquet["VALOR"].astype(str)
executive_dashboard_parquet.to_parquet(
    PASTA_RESULTADOS/"158_EXECUTIVE_DASHBOARD.parquet",
    index=False
)

asset_status.to_parquet(
    PASTA_RESULTADOS/"159_ASSET_STATUS.parquet",
    index=False
)

maintenance_dashboard_parquet = maintenance_dashboard.copy()
maintenance_dashboard_parquet["VALOR"] = maintenance_dashboard_parquet["VALOR"].astype(str)
maintenance_dashboard_parquet.to_parquet(
    PASTA_RESULTADOS/"160_MAINTENANCE_DASHBOARD.parquet",
    index=False
)

event_summary.to_parquet(
    PASTA_RESULTADOS/"161_EVENT_SUMMARY.parquet",
    index=False
)

# ==============================================================================
# RESUMO
# ==============================================================================

print()

print(executive_dashboard)

print()

print(asset_status)

print()

print(maintenance_dashboard)

print()

print(event_summary)

print()

print("="*100)
print("ENTERPRISE OPERATIONS CENTER FINALIZADO")
print("="*100)

BLOCO 7.4.1 - ENTERPRISE OPERATIONS CENTER
          INDICADOR      VALOR
0      Equipamentos     4.0000
1         Registros 17180.0000
2      Health Médio    98.0500
3        Risk Médio    13.6300
4         POF Médio     0.1181
5  RUL Médio (dias)   333.5000
6         COP Médio     2.9820
7       kW/TR Médio     1.4420
     URA  HEALTH_SCORE  RISK_SCORE    POF  RUL_DIAS    COP  KW_TR  \
0    CAG      100.0000     10.5500 0.0800  347.6078 3.0354 1.2548   
1  URA01       97.7242     13.9732 0.1219  332.1748 2.9804 1.8373   
2  URA02       99.8222     10.6940 0.0815  347.0293 3.0335 1.1011   
3  URA03       94.6549     19.2922 0.1891  307.1432 2.8777 1.5763   

            STATUS  
0  OPERAÇÃO NORMAL  
1  OPERAÇÃO NORMAL  
2  OPERAÇÃO NORMAL  
3  OPERAÇÃO NORMAL  
       INDICADOR  VALOR
0      POF > 30%   1759
1  RUL < 30 dias      0
2    Health < 70      9
3      Risk > 35    285
  EVENT_PRIORITY  TOTAL
0       CRITICAL   1759
1           HIGH     20
2            LOW  15401

          

In [89]:
# ==============================================================================
# BLOCO 7.4.2
# EXECUTIVE VISUALIZATION ENGINE
# ==============================================================================

import pandas as pd
import numpy as np

from pathlib import Path

print("="*100)
print("BLOCO 7.4.2 - EXECUTIVE VISUALIZATION ENGINE")
print("="*100)

PASTA_RESULTADOS = Path("Resultados")

# ==============================================================================
# EXECUTIVE KPI
# ==============================================================================

executive_kpi = pd.DataFrame({

    "KPI":[

        "Health Médio",

        "Risk Médio",

        "POF Médio",

        "RUL Médio",

        "COP Médio",

        "kW/TR Médio"

    ],

    "VALOR":[

        round(feature_engine["HEALTH_SCORE"].mean(),2),

        round(feature_engine["RISK_SCORE"].mean(),2),

        round(feature_engine["POF"].mean(),4),

        round(feature_engine["RUL_DIAS"].mean(),1),

        round(feature_engine["COP"].mean(),3),

        round(feature_engine["KW_TR"].mean(),3)

    ]

})

print(executive_kpi)

# ==============================================================================
# EQUIPMENT DASHBOARD
# ==============================================================================

equipment_dashboard = feature_engine.groupby("URA").agg({

    "HEALTH_SCORE":"mean",

    "RISK_SCORE":"mean",

    "POF":"mean",

    "RUL_DIAS":"mean",

    "COP":"mean",

    "KW_TR":"mean",

    "POTENCIA":"mean",

    "VAZAO":"mean"

}).reset_index()

print(equipment_dashboard)

# ==============================================================================
# PERFORMANCE DASHBOARD
# ==============================================================================

performance_dashboard = feature_engine[[
    "DATAHORA",
    "URA",
    "COP",
    "KW_TR",
    "CARGA_TR",
    "POTENCIA",
    "VAZAO"
]].copy()

print(performance_dashboard.head())

# ==============================================================================
# AI DASHBOARD
# ==============================================================================

ai_dashboard = feature_engine[[

    "DATAHORA",

    "URA",

    "HEALTH_SCORE",

    "RISK_SCORE",

    "POF",

    "RUL_DIAS",

    "FAILURE_SCORE"

]].copy()

print(ai_dashboard.head())

# ==============================================================================
# MAINTENANCE DASHBOARD
# ==============================================================================

maintenance_dashboard = feature_engine[[

    "DATAHORA",

    "URA",

    "RUL_DIAS",

    "POF",

    "RISK_SCORE",

    "HEALTH_SCORE"

]].copy()

maintenance_dashboard["RECOMENDACAO"] = np.where(

    maintenance_dashboard["RUL_DIAS"] < 30,

    "PROGRAMAR MANUTENÇÃO",

    "OPERAR"

)

print(maintenance_dashboard.head())

# ==============================================================================
# EXECUTIVE DATASET
# ==============================================================================

executive_dataset = equipment_dashboard.merge(

    executive_kpi.assign(CHAVE=1),

    how="cross"

)

print(executive_dataset.head())

# ==============================================================================
# EXPORTAÇÃO
# ==============================================================================

executive_kpi.to_excel(
    PASTA_RESULTADOS/"162_EXECUTIVE_KPI.xlsx",
    index=False
)

equipment_dashboard.to_excel(
    PASTA_RESULTADOS/"163_EQUIPMENT_DASHBOARD.xlsx",
    index=False
)

performance_dashboard.to_excel(
    PASTA_RESULTADOS/"164_PERFORMANCE_DASHBOARD.xlsx",
    index=False
)

ai_dashboard.to_excel(
    PASTA_RESULTADOS/"165_AI_DASHBOARD.xlsx",
    index=False
)

maintenance_dashboard.to_excel(
    PASTA_RESULTADOS/"166_MAINTENANCE_DASHBOARD.xlsx",
    index=False
)

executive_dataset.to_excel(
    PASTA_RESULTADOS/"167_EXECUTIVE_DATASET.xlsx",
    index=False
)

# ==============================================================================
# EXPORTAÇÃO PARQUET
# ==============================================================================

executive_kpi_parquet = executive_kpi.copy()
executive_kpi_parquet["VALOR"] = executive_kpi_parquet["VALOR"].astype(str)
executive_kpi_parquet.to_parquet(
    PASTA_RESULTADOS/"162_EXECUTIVE_KPI.parquet",
    index=False
)

equipment_dashboard.to_parquet(
    PASTA_RESULTADOS/"163_EQUIPMENT_DASHBOARD.parquet",
    index=False
)

performance_dashboard.to_parquet(
    PASTA_RESULTADOS/"164_PERFORMANCE_DASHBOARD.parquet",
    index=False
)

ai_dashboard.to_parquet(
    PASTA_RESULTADOS/"165_AI_DASHBOARD.parquet",
    index=False
)

maintenance_dashboard.to_parquet(
    PASTA_RESULTADOS/"166_MAINTENANCE_DASHBOARD.parquet",
    index=False
)

executive_dataset.to_parquet(
    PASTA_RESULTADOS/"167_EXECUTIVE_DATASET.parquet",
    index=False
)

# ==============================================================================
# RESUMO
# ==============================================================================

print()

print(executive_kpi)

print()

print(equipment_dashboard)

print()

print("="*100)
print("EXECUTIVE VISUALIZATION ENGINE FINALIZADO")
print("="*100)

BLOCO 7.4.2 - EXECUTIVE VISUALIZATION ENGINE
            KPI    VALOR
0  Health Médio  98.0500
1    Risk Médio  13.6300
2     POF Médio   0.1181
3     RUL Médio 333.5000
4     COP Médio   2.9820
5   kW/TR Médio   1.4420
     URA  HEALTH_SCORE  RISK_SCORE    POF  RUL_DIAS    COP  KW_TR  POTENCIA  \
0    CAG      100.0000     10.5500 0.0800  347.6078 3.0354 1.2548   76.6065   
1  URA01       97.7242     13.9732 0.1219  332.1748 2.9804 1.8373   86.0000   
2  URA02       99.8222     10.6940 0.0815  347.0293 3.0335 1.1011   86.0000   
3  URA03       94.6549     19.2922 0.1891  307.1432 2.8777 1.5763   86.0000   

    VAZAO  
0  9.1164  
1 19.3521  
2 10.1935  
3 60.3767  
             DATAHORA  URA    COP  KW_TR  CARGA_TR  POTENCIA  VAZAO
0 2026-06-01 00:00:00  CAG 3.0354 1.2548  105.6330    2.0000 9.1164
1 2026-06-01 00:10:00  CAG 3.0354 1.2548  105.6330    2.0000 9.1164
2 2026-06-01 00:20:00  CAG 3.0354 1.2548  105.6330    2.0000 9.1164
3 2026-06-01 00:30:00  CAG 3.0354 1.2548  105.6330  

In [91]:
[c for c in digital_twin.columns if "CONF" in c.upper()]

[]

In [92]:
[c for c in feature_engine.columns if "CONF" in c.upper()]

[]

In [93]:
# ==============================================================================
# BLOCO 7.4.3
# DIGITAL TWIN DATA HUB
# ==============================================================================

import pandas as pd
import numpy as np

from pathlib import Path

print("="*100)
print("BLOCO 7.4.3 - DIGITAL TWIN DATA HUB")
print("="*100)

PASTA_RESULTADOS = Path("Resultados")

# ==============================================================================
# DIGITAL TWIN
# ==============================================================================

digital_twin = feature_engine.copy()

digital_twin["DIGITAL_TWIN_VERSION"] = "1.0"

digital_twin["PLATFORM"] = "Engineering Analytics Platform"

digital_twin["UPDATE_TIME"] = pd.Timestamp.now()

digital_twin["OPERATION_MODE"] = np.where(

    digital_twin["HEALTH_SCORE"] >= 90,

    "NORMAL",

    np.where(

        digital_twin["HEALTH_SCORE"] >= 75,

        "ATTENTION",

        "CRITICAL"

    )

)

# ==============================================================================
# GLOBAL INDEX (VERSÃO ROBUSTA)
# ==============================================================================

# Score de confiança (compatível com diferentes versões)
if "CONFIDENCE_SCORE" in digital_twin.columns:

    confidence = digital_twin["CONFIDENCE_SCORE"]

elif "N_CONFIDENCE" in digital_twin.columns:

    # Converte uma escala normalizada (0-1) para percentual, se necessário
    confidence = (
        digital_twin["N_CONFIDENCE"] * 100
        if digital_twin["N_CONFIDENCE"].max() <= 1
        else digital_twin["N_CONFIDENCE"]
    )

else:

    print("CONFIDENCE_SCORE não encontrado -> utilizando valor padrão (100).")
    confidence = pd.Series(100, index=digital_twin.index)

digital_twin["GLOBAL_INDEX"] = (

      digital_twin["HEALTH_SCORE"] * 0.35
    + (100 - digital_twin["RISK_SCORE"]) * 0.25
    + (100 - digital_twin["POF"] * 100) * 0.20
    + confidence * 0.20

).round(2)

digital_twin["GLOBAL_CLASS"] = pd.cut(

    digital_twin["GLOBAL_INDEX"],

    bins=[0,60,75,90,100],

    labels=[

        "CRÍTICO",

        "ATENÇÃO",

        "BOM",

        "EXCELENTE"

    ],

    include_lowest=True

)

twin_dashboard = pd.DataFrame({

    "INDICADOR":[

        "Digital Twins",

        "Índice Global",

        "Health Médio",

        "Risk Médio",

        "POF Médio"

    ],

    "VALOR":[

        digital_twin["URA"].nunique(),

        round(digital_twin["GLOBAL_INDEX"].mean(),2),

        round(digital_twin["HEALTH_SCORE"].mean(),2),

        round(digital_twin["RISK_SCORE"].mean(),2),

        round(digital_twin["POF"].mean(),3)

    ]

})

print(twin_dashboard)

digital_twin.to_excel(

    PASTA_RESULTADOS/"168_DIGITAL_TWIN.xlsx",

    index=False

)

twin_dashboard.to_excel(

    PASTA_RESULTADOS/"169_DIGITAL_TWIN_DASHBOARD.xlsx",

    index=False

)

digital_twin_export = digital_twin.copy()
digital_twin_export["UPDATE_TIME"] = (
    digital_twin_export["UPDATE_TIME"].astype(str)
)

digital_twin_export.to_parquet(

    PASTA_RESULTADOS/"168_DIGITAL_TWIN.parquet",

    index=False

)

twin_dashboard_parquet = twin_dashboard.copy()
twin_dashboard_parquet["VALOR"] = twin_dashboard_parquet["VALOR"].astype(str)

twin_dashboard_parquet.to_parquet(

    PASTA_RESULTADOS/"169_DIGITAL_TWIN_DASHBOARD.parquet",

    index=False

)

print()

print(digital_twin.head())

print()

print(twin_dashboard)

print()

print("="*100)
print("DIGITAL TWIN DATA HUB FINALIZADO")
print("="*100)

BLOCO 7.4.3 - DIGITAL TWIN DATA HUB
CONFIDENCE_SCORE não encontrado -> utilizando valor padrão (100).
       INDICADOR   VALOR
0  Digital Twins  4.0000
1  Índice Global 93.5500
2   Health Médio 98.0500
3     Risk Médio 13.6300
4      POF Médio  0.1180

             DATAHORA  URA    COP  KW_TR  POTENCIA  VAZAO  CARGA_TR  \
0 2026-06-01 00:00:00  CAG 3.0354 1.2548    2.0000 9.1164  105.6330   
1 2026-06-01 00:10:00  CAG 3.0354 1.2548    2.0000 9.1164  105.6330   
2 2026-06-01 00:20:00  CAG 3.0354 1.2548    2.0000 9.1164  105.6330   
3 2026-06-01 00:30:00  CAG 3.0354 1.2548    2.0000 9.1164  105.6330   
4 2026-06-01 00:40:00  CAG 3.0354 1.2548    2.0000 9.1164  105.6330   

   APPROACH_EVAP  APPROACH_COND  HEALTH_SCORE  PERFORMANCE_INDEX  \
0         2.0000         3.1284      100.0000           100.0000   
1         2.0000         3.1284      100.0000           100.0000   
2         2.0000         3.1284      100.0000           100.0000   
3         2.0000         3.1284      100.0000   

In [94]:
# ==============================================================================
# BLOCO 7.4.4
# DIGITAL TWIN STATE ENGINE
# ==============================================================================

import pandas as pd
import numpy as np
from pathlib import Path

print("=" * 100)
print("BLOCO 7.4.4 - DIGITAL TWIN STATE ENGINE")
print("=" * 100)

PASTA_RESULTADOS = Path("Resultados")

twin_state = digital_twin.copy()

twin_state["STATE"] = "NORMAL"

# CRÍTICO
twin_state.loc[
    (twin_state["HEALTH_SCORE"] < 60) |
    (twin_state["POF"] > 0.45),
    "STATE"
] = "CRITICAL"

# MANUTENÇÃO
twin_state.loc[
    (twin_state["RUL_DIAS"] < 30) &
    (twin_state["STATE"] != "CRITICAL"),
    "STATE"
] = "MAINTENANCE_REQUIRED"

# ALTO RISCO
twin_state.loc[
    (twin_state["RISK_SCORE"] > 35) &
    (twin_state["STATE"] == "NORMAL"),
    "STATE"
] = "HIGH_RISK"

# PERDA DE EFICIÊNCIA
twin_state.loc[
    (twin_state["COP"] < twin_state["COP"].median()) &
    (twin_state["KW_TR"] > twin_state["KW_TR"].median()) &
    (twin_state["STATE"] == "NORMAL"),
    "STATE"
] = "EFFICIENCY_LOSS"

# ATENÇÃO
twin_state.loc[
    (twin_state["HEALTH_SCORE"] < 85) &
    (twin_state["STATE"] == "NORMAL"),
    "STATE"
] = "WARNING"

severity_map = {
    "NORMAL": 1,
    "EFFICIENCY_LOSS": 2,
    "WARNING": 3,
    "HIGH_RISK": 4,
    "MAINTENANCE_REQUIRED": 5,
    "CRITICAL": 6,
    "SHUTDOWN": 7
}

twin_state["STATE_LEVEL"] = twin_state["STATE"].map(severity_map)

state_dashboard = (
    twin_state.groupby(["URA", "STATE"])
    .size()
    .reset_index(name="TOTAL")
)

print(state_dashboard)

current_state = (
    twin_state
    .sort_values("DATAHORA")
    .groupby("URA")
    .tail(1)[[
        "URA",
        "DATAHORA",
        "STATE",
        "STATE_LEVEL",
        "HEALTH_SCORE",
        "RISK_SCORE",
        "POF",
        "RUL_DIAS"
    ]]
)

print(current_state)

state_dashboard.to_excel(
    PASTA_RESULTADOS / "170_STATE_DASHBOARD.xlsx",
    index=False
)

current_state.to_excel(
    PASTA_RESULTADOS / "171_CURRENT_STATE.xlsx",
    index=False
)

state_dashboard.to_parquet(
    PASTA_RESULTADOS / "170_STATE_DASHBOARD.parquet",
    index=False
)

current_state_export = current_state.copy()
current_state_export["DATAHORA"] = current_state_export["DATAHORA"].astype(str)

current_state_export.to_parquet(
    PASTA_RESULTADOS / "171_CURRENT_STATE.parquet",
    index=False
)

print()
print(current_state)
print()
print("=" * 100)
print("DIGITAL TWIN STATE ENGINE FINALIZADO")
print("=" * 100)

BLOCO 7.4.4 - DIGITAL TWIN STATE ENGINE
      URA            STATE  TOTAL
0     CAG           NORMAL   4295
1   URA01         CRITICAL     58
2   URA01  EFFICIENCY_LOSS    170
3   URA01        HIGH_RISK     97
4   URA01           NORMAL   3958
5   URA01          WARNING     12
6   URA02         CRITICAL      1
7   URA02  EFFICIENCY_LOSS      5
8   URA02           NORMAL   4259
9   URA02          WARNING     30
10  URA03         CRITICAL    123
11  URA03  EFFICIENCY_LOSS    608
12  URA03        HIGH_RISK     17
13  URA03           NORMAL   3536
14  URA03          WARNING     11
         URA   DATAHORA   STATE  STATE_LEVEL  HEALTH_SCORE  RISK_SCORE    POF  \
8589   URA01 2026-07-01  NORMAL            1      100.0000     10.5500 0.0800   
4294     CAG 2026-07-01  NORMAL            1      100.0000     10.5500 0.0800   
12884  URA02 2026-07-01  NORMAL            1      100.0000     10.5500 0.0800   
17179  URA03 2026-07-01  NORMAL            1      100.0000     10.5500 0.0800   

       RUL

In [95]:
# ==============================================================================
# BLOCO 7.4.5
# SMART ALARM MANAGEMENT ENGINE
# ==============================================================================

import pandas as pd
import numpy as np
from pathlib import Path

print("="*100)
print("BLOCO 7.4.5 - SMART ALARM MANAGEMENT ENGINE")
print("="*100)

PASTA_RESULTADOS = Path("Resultados")

alarm_engine = twin_state.copy()

alarm_engine["ALARM"] = "NORMAL"

alarm_engine["ALARM_PRIORITY"] = "LOW"

alarm_engine["ALARM_STATUS"] = "ACTIVE"

alarm_engine["ALARM_TIME"] = alarm_engine["DATAHORA"]

# ==============================================================================
# HEALTH
# ==============================================================================

alarm_engine.loc[
    alarm_engine["HEALTH_SCORE"] < 70,
    "ALARM"
] = "LOW_HEALTH"

# ==============================================================================
# RISCO
# ==============================================================================

alarm_engine.loc[
    alarm_engine["RISK_SCORE"] > 35,
    "ALARM"
] = "HIGH_RISK"

# ==============================================================================
# POF
# ==============================================================================

alarm_engine.loc[
    alarm_engine["POF"] > 0.35,
    "ALARM"
] = "FAILURE_RISK"

# ==============================================================================
# RUL
# ==============================================================================

alarm_engine.loc[
    alarm_engine["RUL_DIAS"] < 30,
    "ALARM"
] = "RUL_CRITICAL"

# ==============================================================================
# ESTADO CRÍTICO
# ==============================================================================

alarm_engine.loc[
    alarm_engine["STATE"] == "CRITICAL",
    "ALARM"
] = "CRITICAL_STATE"

priority_map = {

    "NORMAL":"LOW",

    "LOW_HEALTH":"MEDIUM",

    "HIGH_RISK":"HIGH",

    "FAILURE_RISK":"HIGH",

    "RUL_CRITICAL":"CRITICAL",

    "CRITICAL_STATE":"CRITICAL"

}

alarm_engine["ALARM_PRIORITY"] = (

    alarm_engine["ALARM"]

    .map(priority_map)

    .fillna("LOW")

)

alarm_dashboard = (

    alarm_engine

    .groupby(

        ["ALARM","ALARM_PRIORITY"]

    )

    .size()

    .reset_index(name="TOTAL")

)

print(alarm_dashboard)

active_alarms = (

    alarm_engine

    .loc[alarm_engine["ALARM"]!="NORMAL"]

    [[

        "DATAHORA",

        "URA",

        "STATE",

        "ALARM",

        "ALARM_PRIORITY",

        "HEALTH_SCORE",

        "RISK_SCORE",

        "POF",

        "RUL_DIAS"

    ]]

)

print(active_alarms.head())

alarm_dashboard.to_excel(

    PASTA_RESULTADOS/"172_ALARM_DASHBOARD.xlsx",

    index=False

)

active_alarms.to_excel(

    PASTA_RESULTADOS/"173_ACTIVE_ALARMS.xlsx",

    index=False

)

alarm_dashboard.to_parquet(

    PASTA_RESULTADOS/"172_ALARM_DASHBOARD.parquet",

    index=False

)

active_export = active_alarms.copy()
active_export["DATAHORA"] = active_export["DATAHORA"].astype(str)

active_export.to_parquet(

    PASTA_RESULTADOS/"173_ACTIVE_ALARMS.parquet",

    index=False

)

print()

print(alarm_dashboard)

print()

print(active_alarms.head())

print()

print("="*100)
print("SMART ALARM MANAGEMENT ENGINE FINALIZADO")
print("="*100)

BLOCO 7.4.5 - SMART ALARM MANAGEMENT ENGINE
            ALARM ALARM_PRIORITY  TOTAL
0  CRITICAL_STATE       CRITICAL    182
1    FAILURE_RISK           HIGH    557
2       HIGH_RISK           HIGH     13
3      LOW_HEALTH         MEDIUM      4
4          NORMAL            LOW  16424
                DATAHORA    URA            STATE         ALARM ALARM_PRIORITY  \
4435 2026-06-01 23:20:00  URA01  EFFICIENCY_LOSS    LOW_HEALTH         MEDIUM   
4641 2026-06-03 09:40:00  URA01  EFFICIENCY_LOSS    LOW_HEALTH         MEDIUM   
4679 2026-06-03 16:00:00  URA01        HIGH_RISK  FAILURE_RISK           HIGH   
4680 2026-06-03 16:10:00  URA01        HIGH_RISK  FAILURE_RISK           HIGH   
4739 2026-06-04 02:00:00  URA01  EFFICIENCY_LOSS  FAILURE_RISK           HIGH   

      HEALTH_SCORE  RISK_SCORE    POF  RUL_DIAS  
4435       67.1205     27.5834 0.3395  262.0087  
4641       67.4653     27.8210 0.3365  262.4708  
4679       82.0361     37.0701 0.3818  239.4467  
4680       82.4436     36.956

In [96]:
# ==============================================================================
# BLOCO 7.4.6
# AUTONOMOUS DECISION ENGINE
# ==============================================================================

import pandas as pd
import numpy as np
from pathlib import Path

print("="*100)
print("BLOCO 7.4.6 - AUTONOMOUS DECISION ENGINE")
print("="*100)

PASTA_RESULTADOS = Path("Resultados")

decision_engine = alarm_engine.copy()

decision_engine["DECISAO"] = "OPERAR"

decision_engine["RESPONSAVEL"] = "Operação"

decision_engine["PRAZO"] = "Monitoramento"

decision_engine["PRIORIDADE"] = "BAIXA"

# CRÍTICO

decision_engine.loc[
    decision_engine["STATE"]=="CRITICAL",
    ["DECISAO","RESPONSAVEL","PRAZO","PRIORIDADE"]
] = [

    "PARADA IMEDIATA",

    "Engenharia",

    "Imediato",

    "CRÍTICA"

]

# RUL

decision_engine.loc[
    (decision_engine["RUL_DIAS"]<30)
    &
    (decision_engine["STATE"]!="CRITICAL"),
    ["DECISAO","RESPONSAVEL","PRAZO","PRIORIDADE"]
] = [

    "PROGRAMAR MANUTENÇÃO",

    "PCM",

    "7 dias",

    "ALTA"

]

# POF

decision_engine.loc[
    (decision_engine["POF"]>0.35)
    &
    (decision_engine["STATE"]!="CRITICAL"),
    ["DECISAO","RESPONSAVEL","PRAZO","PRIORIDADE"]
] = [

    "INSPECIONAR EQUIPAMENTO",

    "Manutenção",

    "48 horas",

    "ALTA"

]

# HEALTH

decision_engine.loc[
    (decision_engine["HEALTH_SCORE"]<80)
    &
    (decision_engine["DECISAO"]=="OPERAR"),
    ["DECISAO","RESPONSAVEL","PRAZO","PRIORIDADE"]
] = [

    "MONITORAMENTO INTENSIVO",

    "Operação",

    "24 horas",

    "MÉDIA"

]

priority_map = {

    "BAIXA":1,

    "MÉDIA":2,

    "ALTA":3,

    "CRÍTICA":4

}

decision_engine["PRIORITY_SCORE"] = (

    decision_engine["PRIORIDADE"]

    .map(priority_map)

)

decision_dashboard = (

    decision_engine

    .groupby(

        ["DECISAO","PRIORIDADE"]

    )

    .size()

    .reset_index(name="TOTAL")

)

print(decision_dashboard)

current_decisions = (

    decision_engine

    .sort_values("DATAHORA")

    .groupby("URA")

    .tail(1)[[

        "URA",

        "STATE",

        "HEALTH_SCORE",

        "RISK_SCORE",

        "POF",

        "RUL_DIAS",

        "DECISAO",

        "RESPONSAVEL",

        "PRAZO",

        "PRIORIDADE"

    ]]

)

print(current_decisions)

decision_dashboard.to_excel(

    PASTA_RESULTADOS/"174_DECISION_DASHBOARD.xlsx",

    index=False

)

current_decisions.to_excel(

    PASTA_RESULTADOS/"175_CURRENT_DECISIONS.xlsx",

    index=False

)

decision_dashboard.to_parquet(

    PASTA_RESULTADOS/"174_DECISION_DASHBOARD.parquet",

    index=False

)

current_export = current_decisions.copy()

for col in current_export.select_dtypes(include=["datetime64[ns]"]).columns:
    current_export[col] = current_export[col].astype(str)

current_export.to_parquet(

    PASTA_RESULTADOS/"175_CURRENT_DECISIONS.parquet",

    index=False

)

print()

print(current_decisions)

print()

print(decision_dashboard)

print()

print("="*100)
print("AUTONOMOUS DECISION ENGINE FINALIZADO")
print("="*100)

BLOCO 7.4.6 - AUTONOMOUS DECISION ENGINE
                   DECISAO PRIORIDADE  TOTAL
0  INSPECIONAR EQUIPAMENTO       ALTA    557
1  MONITORAMENTO INTENSIVO      MÉDIA     34
2                   OPERAR      BAIXA  16407
3          PARADA IMEDIATA    CRÍTICA    182
         URA   STATE  HEALTH_SCORE  RISK_SCORE    POF  RUL_DIAS DECISAO  \
8589   URA01  NORMAL      100.0000     10.5500 0.0800  347.6078  OPERAR   
4294     CAG  NORMAL      100.0000     10.5500 0.0800  347.6078  OPERAR   
12884  URA02  NORMAL      100.0000     10.5500 0.0800  347.6078  OPERAR   
17179  URA03  NORMAL      100.0000     10.5500 0.0800  347.6078  OPERAR   

      RESPONSAVEL          PRAZO PRIORIDADE  
8589     Operação  Monitoramento      BAIXA  
4294     Operação  Monitoramento      BAIXA  
12884    Operação  Monitoramento      BAIXA  
17179    Operação  Monitoramento      BAIXA  

         URA   STATE  HEALTH_SCORE  RISK_SCORE    POF  RUL_DIAS DECISAO  \
8589   URA01  NORMAL      100.0000     10.5500 0.080

In [98]:
# ==============================================================================
# BLOCO 7.4.7
# WORK ORDER AUTOMATION ENGINE
# ==============================================================================

import pandas as pd
import numpy as np
from pathlib import Path

print("=" * 100)
print("BLOCO 7.4.7 - WORK ORDER AUTOMATION ENGINE")
print("=" * 100)

PASTA_RESULTADOS = Path("Resultados")

wo_engine = decision_engine.copy()

# ==============================================================================
# GERAÇÃO DA ORDEM DE SERVIÇO
# ==============================================================================

wo_engine["GERAR_OS"] = np.where(
    wo_engine["DECISAO"] != "OPERAR",
    "SIM",
    "NÃO"
)

# ==============================================================================
# IDENTIFICADOR ÚNICO DA ORDEM DE SERVIÇO
# ==============================================================================

timestamp = pd.Timestamp.now().strftime("%Y%m%d")

sequencia = (
    pd.Series(np.arange(1, len(wo_engine) + 1), index=wo_engine.index)
    .astype(str)
    .str.zfill(6)
)

wo_engine["OS_ID"] = np.where(

    wo_engine["GERAR_OS"] == "SIM",

    "OS-" + timestamp + "-" + sequencia,

    ""

)

wo_engine["OS_STATUS"] = np.where(
    wo_engine["GERAR_OS"] == "SIM",
    "ABERTA",
    ""
)

# ==============================================================================
# CLASSIFICAÇÃO DA MANUTENÇÃO
# ==============================================================================

wo_engine["TIPO_MANUTENCAO"] = "PREDITIVA"

wo_engine.loc[
    wo_engine["STATE"] == "CRITICAL",
    "TIPO_MANUTENCAO"
] = "CORRETIVA"

wo_engine.loc[
    wo_engine["DECISAO"] == "PROGRAMAR MANUTENÇÃO",
    "TIPO_MANUTENCAO"
] = "PREVENTIVA"

# ==============================================================================
# DATA PREVISTA
# ==============================================================================

wo_engine["DATA_PREVISTA"] = pd.Timestamp.now()

wo_engine.loc[
    wo_engine["PRIORIDADE"] == "CRÍTICA",
    "DATA_PREVISTA"
] = pd.Timestamp.now()

wo_engine.loc[
    wo_engine["PRIORIDADE"] == "ALTA",
    "DATA_PREVISTA"
] = pd.Timestamp.now() + pd.Timedelta(days=2)

wo_engine.loc[
    wo_engine["PRIORIDADE"] == "MÉDIA",
    "DATA_PREVISTA"
] = pd.Timestamp.now() + pd.Timedelta(days=7)

wo_engine.loc[
    wo_engine["PRIORIDADE"] == "BAIXA",
    "DATA_PREVISTA"
] = pd.Timestamp.now() + pd.Timedelta(days=30)

# ==============================================================================
# ORDENS DE SERVIÇO
# ==============================================================================

work_orders = wo_engine.loc[
    wo_engine["GERAR_OS"] == "SIM",
    [
        "OS_ID",
        "DATAHORA",
        "URA",
        "DECISAO",
        "TIPO_MANUTENCAO",
        "RESPONSAVEL",
        "PRIORIDADE",
        "OS_STATUS",
        "DATA_PREVISTA",
        "HEALTH_SCORE",
        "RISK_SCORE",
        "POF",
        "RUL_DIAS"
    ]
].copy()

print(work_orders.head())

# ==============================================================================
# DASHBOARD
# ==============================================================================

wo_dashboard = (
    work_orders
    .groupby(["TIPO_MANUTENCAO", "PRIORIDADE"])
    .size()
    .reset_index(name="TOTAL")
)

print(wo_dashboard)

# ==============================================================================
# EXPORTAÇÃO
# ==============================================================================

work_orders.to_excel(
    PASTA_RESULTADOS / "176_WORK_ORDERS.xlsx",
    index=False
)

wo_dashboard.to_excel(
    PASTA_RESULTADOS / "177_WORK_ORDER_DASHBOARD.xlsx",
    index=False
)

work_orders_export = work_orders.copy()

for col in work_orders_export.select_dtypes(include=["datetime64[ns]"]).columns:
    work_orders_export[col] = work_orders_export[col].astype(str)

work_orders_export.to_parquet(
    PASTA_RESULTADOS / "176_WORK_ORDERS.parquet",
    index=False
)

wo_dashboard.to_parquet(
    PASTA_RESULTADOS / "177_WORK_ORDER_DASHBOARD.parquet",
    index=False
)

print()

print(work_orders.head())

print()

print(wo_dashboard)

print()

print("=" * 100)
print("WORK ORDER AUTOMATION ENGINE FINALIZADO")
print("=" * 100)

BLOCO 7.4.7 - WORK ORDER AUTOMATION ENGINE
                   OS_ID            DATAHORA    URA                  DECISAO  \
4435  OS-20260723-004436 2026-06-01 23:20:00  URA01  MONITORAMENTO INTENSIVO   
4441  OS-20260723-004442 2026-06-02 00:20:00  URA01  MONITORAMENTO INTENSIVO   
4442  OS-20260723-004443 2026-06-02 00:30:00  URA01  MONITORAMENTO INTENSIVO   
4443  OS-20260723-004444 2026-06-02 00:40:00  URA01  MONITORAMENTO INTENSIVO   
4444  OS-20260723-004445 2026-06-02 00:50:00  URA01  MONITORAMENTO INTENSIVO   

     TIPO_MANUTENCAO RESPONSAVEL PRIORIDADE OS_STATUS  \
4435       PREDITIVA    Operação      MÉDIA    ABERTA   
4441       PREDITIVA    Operação      MÉDIA    ABERTA   
4442       PREDITIVA    Operação      MÉDIA    ABERTA   
4443       PREDITIVA    Operação      MÉDIA    ABERTA   
4444       PREDITIVA    Operação      MÉDIA    ABERTA   

                  DATA_PREVISTA  HEALTH_SCORE  RISK_SCORE    POF  RUL_DIAS  
4435 2026-07-30 15:34:16.239509       67.1205     27.583

In [99]:
# ==============================================================================
# BLOCO 8.0
# ENTERPRISE CONFIGURATION ENGINE
# ==============================================================================

import json
from pathlib import Path

print("=" * 100)
print("BLOCO 8.0 - ENTERPRISE CONFIGURATION ENGINE")
print("=" * 100)

PASTA_RESULTADOS = Path("Resultados")
PASTA_RESULTADOS.mkdir(exist_ok=True)

config = {

    "platform":{

        "name":"Engineering Analytics Platform",

        "version":"8.0.0",

        "author":"Tiago Fernandes",

        "language":"pt-BR"

    },

    "paths":{

        "output":"Resultados",

        "logs":"Logs",

        "reports":"Reports"

    }

}

config["health"]={

    "excellent":90,

    "good":75,

    "warning":60,

    "critical":40

}

config["risk"]={

    "low":15,

    "medium":25,

    "high":35,

    "critical":45

}

config["pof"]={

    "low":0.10,

    "medium":0.20,

    "high":0.35,

    "critical":0.50

}

config["rul"]={

    "excellent":180,

    "attention":90,

    "maintenance":30,

    "critical":7

}

config["alarm"]={

    "enable":True,

    "email":False,

    "teams":False,

    "whatsapp":False,

    "repeat_minutes":30

}

config["global_index"]={

    "health":0.35,

    "risk":0.25,

    "pof":0.20,

    "confidence":0.20

}

config["machine_learning"]={

    "random_seed":42,

    "train_size":0.80,

    "test_size":0.20

}

config["export"]={

    "excel":True,

    "parquet":True,

    "csv":False

}

config_file = PASTA_RESULTADOS / "000_PLATFORM_CONFIG.json"

with open(config_file,"w",encoding="utf-8") as f:

    json.dump(

        config,

        f,

        indent=4,

        ensure_ascii=False

    )

print()

print("Arquivo criado:")

print(config_file)

with open(config_file,"r",encoding="utf-8") as f:

    PLATFORM_CONFIG = json.load(f)

print()

print(PLATFORM_CONFIG)



BLOCO 8.0 - ENTERPRISE CONFIGURATION ENGINE

Arquivo criado:
Resultados/000_PLATFORM_CONFIG.json

{'platform': {'name': 'Engineering Analytics Platform', 'version': '8.0.0', 'author': 'Tiago Fernandes', 'language': 'pt-BR'}, 'paths': {'output': 'Resultados', 'logs': 'Logs', 'reports': 'Reports'}, 'health': {'excellent': 90, 'good': 75, 'warning': 60, 'critical': 40}, 'risk': {'low': 15, 'medium': 25, 'high': 35, 'critical': 45}, 'pof': {'low': 0.1, 'medium': 0.2, 'high': 0.35, 'critical': 0.5}, 'rul': {'excellent': 180, 'attention': 90, 'maintenance': 30, 'critical': 7}, 'alarm': {'enable': True, 'email': False, 'teams': False, 'whatsapp': False, 'repeat_minutes': 30}, 'global_index': {'health': 0.35, 'risk': 0.25, 'pof': 0.2, 'confidence': 0.2}, 'machine_learning': {'random_seed': 42, 'train_size': 0.8, 'test_size': 0.2}, 'export': {'excel': True, 'parquet': True, 'csv': False}}


In [101]:
# ==============================================================================
# BLOCO 8.0.1
# CONFIGURATION LOADER ENGINE
# ==============================================================================

import json
from pathlib import Path

print("="*100)
print("BLOCO 8.0.1 - CONFIGURATION LOADER ENGINE")
print("="*100)

PASTA_RESULTADOS = Path("Resultados")

CONFIG_FILE = PASTA_RESULTADOS / "000_PLATFORM_CONFIG.json"

if not CONFIG_FILE.exists():

    raise FileNotFoundError(

        f"Arquivo não encontrado:\n{CONFIG_FILE}"

    )

# ==============================================================================
# ETAPA 3 - CARREGAMENTO DA CONFIGURAÇÃO (ENTERPRISE)
# ==============================================================================

try:

    with open(CONFIG_FILE, "r", encoding="utf-8") as f:
        PLATFORM_CONFIG = json.load(f)

    print()
    print("✔ Configuração carregada com sucesso.")
    print(f"Arquivo: {CONFIG_FILE}")

except FileNotFoundError:

    raise Exception(
        f"Arquivo de configuração não encontrado:\n{CONFIG_FILE}"
    )

except json.JSONDecodeError as e:

    raise Exception(
        f"Erro no arquivo JSON:\n{e}"
    )

except Exception as e:

    raise Exception(
        f"Erro ao carregar configuração:\n{e}"
    )

HEALTH_CFG = PLATFORM_CONFIG["health"]

RISK_CFG = PLATFORM_CONFIG["risk"]

POF_CFG = PLATFORM_CONFIG["pof"]

RUL_CFG = PLATFORM_CONFIG["rul"]

ALARM_CFG = PLATFORM_CONFIG["alarm"]

EXPORT_CFG = PLATFORM_CONFIG["export"]

ML_CFG = PLATFORM_CONFIG["machine_learning"]

GLOBAL_CFG = PLATFORM_CONFIG["global_index"]

def cfg(secao, chave, default=None):

    try:

        return PLATFORM_CONFIG[secao][chave]

    except KeyError:

        return default

print()

print("Health Warning :", cfg("health","warning"))

print("Risk High      :", cfg("risk","high"))

print("POF Critical   :", cfg("pof","critical"))

print("RUL Critical   :", cfg("rul","critical"))

print("Export Excel   :", cfg("export","excel"))

required_sections = [

    "health",

    "risk",

    "pof",

    "rul",

    "alarm",

    "machine_learning",

    "global_index",

    "export"

]

missing = [

    sec

    for sec in required_sections

    if sec not in PLATFORM_CONFIG

]

if missing:

    raise Exception(

        f"Configuração incompleta: {missing}"

    )

print()

print("Todas as configurações obrigatórias foram encontradas.")

print()

print("="*100)

print("CONFIGURATION LOADER ENGINE FINALIZADO")

print("="*100)

BLOCO 8.0.1 - CONFIGURATION LOADER ENGINE

✔ Configuração carregada com sucesso.
Arquivo: Resultados/000_PLATFORM_CONFIG.json

Health Warning : 60
Risk High      : 35
POF Critical   : 0.5
RUL Critical   : 7
Export Excel   : True

Todas as configurações obrigatórias foram encontradas.

CONFIGURATION LOADER ENGINE FINALIZADO
